# СПАРК‑Интерфакс: автономный конструктор датасета

**Версия ядра:** 4.1.0  
**Сборка ноутбука:** 4.1.0-r3


**Target-runtime:** `spark_target_runtime_v1`

Этот Colab превращает ваш список ИНН/ОГРН в понятный набор
Excel, CSV, Parquet и ZIP‑файлов с выбранными методами,
полями и расчётными признаками.

**Каждый аналитик работает независимо.** Скопируйте этот
`.ipynb` в свой Google Drive, подключите свой Диск и свои
Colab Secrets. Весь исполняемый код уже находится внутри
ноутбука; общий сервер, общий компьютер и отдельный архив
проекта не нужны.

| Шаг | Результат |
|---:|---|
| 1–3 | личный Диск, автономное ядро и подтверждённая сессия СПАРК |
| 4–5.3 | методы API, target, параметры, поля и ML‑признаки |
| 6 | проверка той же конфигурации на одной компании |
| 7–7.1 | входной файл, checkpoint и накопительный датасет |
| 8 | массовый сбор с продолжением из совместимого checkpoint |
| 9 | готовые файлы и кнопки скачивания |

При первом запуске выполняйте ячейки сверху вниз. Полную ранее
сохранённую конфигурацию можно восстановить в пункте 4.

Для продолжения ранее сохранённого массового запуска после
перезапуска Colab достаточно выполнить пункты
**1 → 2 → 2.1 → 3 → 8**. Если вы изменили методы, target,
параметры, поля или признаки, повторите зависимые нижние этапы,
которые укажет программа.


## 1. Подключение личного Google Диска

Ячейка запрашивает стандартное разрешение Google Colab и
подключает только ваш Диск. Все устойчивые файлы проекта
будут находиться в `MyDrive/spark_api_research/`.


In [ ]:
# @title 1. Подключить личный Google Диск

from pathlib import Path
from html import escape

from google.colab import drive
from IPython.display import HTML, display


def _показать_статус_диска(
    *,
    успешно: bool,
    заголовок: str,
    сообщение: str,
) -> None:
    """Показывает компактный цветной статус подключения Google Диска."""

    if успешно:
        цвет = "#137333"
        фон = "#e6f4ea"
        граница = "#34a853"
        значок = "●"
    else:
        цвет = "#b3261e"
        фон = "#fce8e6"
        граница = "#d93025"
        значок = "●"

    display(
        HTML(
            f"""
            <div style="
                max-width: 760px;
                padding: 12px 16px;
                margin: 6px 0;
                border: 1px solid {граница};
                border-left: 5px solid {граница};
                border-radius: 8px;
                background: {фон};
                color: {цвет};
                font-family: Arial, sans-serif;
                line-height: 1.45;
            ">
                <div style="font-size: 16px; font-weight: 700;">
                    <span style="margin-right: 7px;">{значок}</span>
                    {escape(заголовок)}
                </div>
                <div style="
                    margin-top: 3px;
                    color: #3c4043;
                    font-size: 14px;
                ">
                    {escape(сообщение)}
                </div>
            </div>
            """
        )
    )


try:
    drive.mount(
        "/content/drive",
        force_remount=False,
    )

    MY_DRIVE = Path(
        "/content/drive/MyDrive"
    )

    if not MY_DRIVE.is_dir():
        raise RuntimeError(
            "Папка MyDrive не найдена после подключения."
        )

    _показать_статус_диска(
        успешно=True,
        заголовок="Google Диск подключён",
        сообщение=(
            f"Личная папка доступна: {MY_DRIVE}"
        ),
    )

except Exception as error:
    _показать_статус_диска(
        успешно=False,
        заголовок="Google Диск не подключён",
        сообщение=(
            f"{error} Повторно запустите ячейку "
            "и разрешите доступ к Google Диску."
        ),
    )
    MY_DRIVE = None


## 2. Установка автономного проекта

Запустите ячейку один раз в новой среде Colab. Она:

- проверит SHA‑256 встроенного пакета;
- установит код в вашу личную папку на Диске;
- создаст нужные каталоги;
- установит только отсутствующие Python‑библиотеки;
- сохранит резервную копию прежнего `src/` при обновлении;
- не затронет входные данные, кэш, checkpoints и результаты.

Повторный запуск безопасен. Пользовательская функция
`src/custom_features.py` сохраняется при обновлении.


In [ ]:
# @title 2. Установить и проверить автономный проект
from pathlib import Path
import base64
import compileall
import hashlib
import importlib
import importlib.util
import io
import json
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone

PROJECT_VERSION = "4.1.0"
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/spark_api_research"
)
EMBEDDED_ARCHIVE_SHA256 = (
    "c451c5c642dab2f8797fbe3b6a51df8c9caf4306c24414a3d109462da91c022f"
)
EMBEDDED_ARCHIVE_BASE64 = """
    ...
"""

try:
    if not Path(
        "/content/drive/MyDrive"
    ).is_dir():
        raise RuntimeError(
            "Google Диск не подключён. "
            "Сначала выполните ячейку 1."
        )

    archive_bytes = base64.b64decode(
        "".join(
            EMBEDDED_ARCHIVE_BASE64.split()
        )
    )
    actual_archive_sha256 = (
        hashlib.sha256(
            archive_bytes
        ).hexdigest()
    )

    if (
        actual_archive_sha256
        != EMBEDDED_ARCHIVE_SHA256
    ):
        raise RuntimeError(
            "Встроенный пакет повреждён: "
            "контрольная сумма не совпала."
        )

    with zipfile.ZipFile(
        io.BytesIO(archive_bytes)
    ) as embedded_zip:
        archive_names = set(
            embedded_zip.namelist()
        )

        if (
            "install_manifest.json"
            not in archive_names
        ):
            raise RuntimeError(
                "Во встроенном пакете нет manifest."
            )

        for archive_name in archive_names:
            archive_path = Path(
                archive_name
            )

            if (
                archive_path.is_absolute()
                or ".." in archive_path.parts
            ):
                raise RuntimeError(
                    "Обнаружен небезопасный путь "
                    f"в архиве: {archive_name}"
                )

        install_manifest = json.loads(
            embedded_zip.read(
                "install_manifest.json"
            ).decode("utf-8")
        )
        expected_names = (
            set(
                install_manifest[
                    "files"
                ]
            )
            | {
                "install_manifest.json"
            }
        )

        if archive_names != expected_names:
            raise RuntimeError(
                "Состав встроенного пакета "
                "не совпал с manifest."
            )

        for (
            relative_name,
            file_info,
        ) in install_manifest[
            "files"
        ].items():
            file_bytes = embedded_zip.read(
                relative_name
            )
            file_sha256 = hashlib.sha256(
                file_bytes
            ).hexdigest()

            if (
                file_sha256
                != file_info["sha256"]
            ):
                raise RuntimeError(
                    "Повреждён встроенный файл: "
                    f"{relative_name}"
                )

        PROJECT_ROOT.mkdir(
            parents=True,
            exist_ok=True,
        )
        releases_root = (
            PROJECT_ROOT
            / "releases"
            / PROJECT_VERSION
        )
        releases_root.mkdir(
            parents=True,
            exist_ok=True,
        )
        release_archive_path = (
            releases_root
            / (
                "spark_api_research_"
                f"{PROJECT_VERSION}.zip"
            )
        )

        if (
            not release_archive_path.is_file()
            or hashlib.sha256(
                release_archive_path.read_bytes()
            ).hexdigest()
            != EMBEDDED_ARCHIVE_SHA256
        ):
            release_archive_temporary = (
                release_archive_path
                .with_suffix(".zip.tmp")
            )
            release_archive_temporary.write_bytes(
                archive_bytes
            )
            release_archive_temporary.replace(
                release_archive_path
            )

        installed_marker_path = (
            PROJECT_ROOT
            / "installed_release.json"
        )
        installed_marker = {}

        if installed_marker_path.is_file():
            try:
                installed_marker = json.loads(
                    installed_marker_path.read_text(
                        encoding="utf-8"
                    )
                )
            except Exception:
                installed_marker = {}

        previous_manifest_path = (
            PROJECT_ROOT / "install_manifest.json"
        )
        previous_manifest = {}

        if previous_manifest_path.is_file():
            try:
                previous_manifest = json.loads(
                    previous_manifest_path.read_text(encoding="utf-8")
                )
            except Exception:
                previous_manifest = {}

        installed_files_valid = True

        for relative_name, file_info in install_manifest["files"].items():
            installed_path = PROJECT_ROOT / relative_name
            if (
                not installed_path.is_file()
                or hashlib.sha256(installed_path.read_bytes()).hexdigest()
                != file_info["sha256"]
            ):
                installed_files_valid = False
                break

        needs_install = (
            installed_marker.get("archive_sha256") != EMBEDDED_ARCHIVE_SHA256
            or not installed_files_valid
        )

        if needs_install:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            existing_src = PROJECT_ROOT / "src"
            legacy_custom_path = existing_src / "custom_features.py"
            legacy_custom_bytes = (
                legacy_custom_path.read_bytes()
                if legacy_custom_path.is_file()
                else None
            )
            previous_custom_sha = str(
                (
                    (previous_manifest.get("files") or {}).get(
                        "src/custom_features.py"
                    )
                    or {}
                ).get("sha256")
                or ""
            )
            legacy_custom_sha = (
                hashlib.sha256(legacy_custom_bytes).hexdigest()
                if legacy_custom_bytes is not None
                else ""
            )
            legacy_custom_modified = bool(
                legacy_custom_bytes is not None
                and (
                    not previous_custom_sha
                    or legacy_custom_sha != previous_custom_sha
                )
            )
            backup_root = PROJECT_ROOT / "code_backups" / timestamp

            if existing_src.is_dir():
                backup_root.mkdir(parents=True, exist_ok=False)
                shutil.copytree(existing_src, backup_root / "src")

            if legacy_custom_modified:
                user_features_root = PROJECT_ROOT / "user_features"
                user_features_root.mkdir(parents=True, exist_ok=True)
                user_custom_path = user_features_root / "custom_features.py"
                if not user_custom_path.is_file():
                    user_custom_path.write_bytes(legacy_custom_bytes)
                    print(
                        "Пользовательские формулы перенесены из "
                        "src/custom_features.py в user_features/custom_features.py."
                    )
                elif user_custom_path.read_bytes() != legacy_custom_bytes:
                    backup_root.mkdir(parents=True, exist_ok=True)
                    (backup_root / "legacy_custom_features.py").write_bytes(
                        legacy_custom_bytes
                    )
                    print(
                        "Существующий user_features/custom_features.py сохранён. "
                        "Старая изменённая копия помещена в code_backups."
                    )

            for relative_name in sorted(install_manifest["files"]):
                destination = PROJECT_ROOT / relative_name
                destination.parent.mkdir(parents=True, exist_ok=True)
                temporary = destination.with_suffix(
                    destination.suffix + ".installing"
                )
                temporary.write_bytes(embedded_zip.read(relative_name))
                temporary.replace(destination)

            for relative_name in (
                install_manifest.get("deprecated_managed_files") or []
            ):
                deprecated_path = PROJECT_ROOT / relative_name
                if deprecated_path.is_file():
                    deprecated_path.unlink()

            manifest_path = PROJECT_ROOT / "install_manifest.json"
            manifest_temporary = manifest_path.with_suffix(".json.tmp")
            manifest_temporary.write_text(
                json.dumps(
                    install_manifest,
                    ensure_ascii=False,
                    indent=2,
                    sort_keys=True,
                ),
                encoding="utf-8",
            )
            manifest_temporary.replace(manifest_path)

            installed_marker = {
                "project": "spark_api_research",
                "version": PROJECT_VERSION,
                "archive_sha256": EMBEDDED_ARCHIVE_SHA256,
                "installed_at": datetime.now(timezone.utc).isoformat(
                    timespec="seconds"
                ),
                "project_root": str(PROJECT_ROOT),
                "shared_server_required": False,
            }
            marker_temporary = installed_marker_path.with_suffix(".json.tmp")
            marker_temporary.write_text(
                json.dumps(
                    installed_marker,
                    ensure_ascii=False,
                    indent=2,
                    sort_keys=True,
                ),
                encoding="utf-8",
            )
            marker_temporary.replace(installed_marker_path)

        user_override_path = (
            PROJECT_ROOT / "user_features" / "custom_features.py"
        )
        override_manifest_path = (
            PROJECT_ROOT / "user_features" / "override_manifest.json"
        )
        if user_override_path.is_file():
            override_manifest_path.parent.mkdir(parents=True, exist_ok=True)
            override_manifest = {
                "format": "spark_user_feature_override_v1",
                "relative_path": "user_features/custom_features.py",
                "size_bytes": user_override_path.stat().st_size,
                "sha256": hashlib.sha256(
                    user_override_path.read_bytes()
                ).hexdigest(),
                "checked_at": datetime.now(timezone.utc).isoformat(
                    timespec="seconds"
                ),
            }
            override_temporary = override_manifest_path.with_suffix(".json.tmp")
            override_temporary.write_text(
                json.dumps(
                    override_manifest,
                    ensure_ascii=False,
                    indent=2,
                    sort_keys=True,
                ),
                encoding="utf-8",
            )
            override_temporary.replace(override_manifest_path)
        elif override_manifest_path.is_file():
            override_manifest_path.unlink()

    dependency_specs = {
        "pandas": "pandas>=2.0,<3.0",
        "requests": "requests>=2.31,<3.0",
        "pyarrow": "pyarrow>=14,<25",
        "openpyxl": "openpyxl>=3.1,<4.0",
        "xlrd": "xlrd>=2.0,<3.0",
        "pyxlsb": "pyxlsb>=1.0.10,<2.0",
        "ipywidgets": "ipywidgets>=8.0,<9.0",
    }
    missing_dependencies = [
        package_spec
        for import_name, package_spec
        in dependency_specs.items()
        if (
            importlib.util.find_spec(
                import_name
            )
            is None
        )
    ]

    if missing_dependencies:
        print(
            "Устанавливаю недостающие "
            "библиотеки: "
            + ", ".join(
                missing_dependencies
            )
        )
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                *missing_dependencies,
            ]
        )

    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(
            0,
            str(PROJECT_ROOT),
        )

    for module_name in list(
        sys.modules
    ):
        if (
            module_name == "src"
            or module_name.startswith(
                "src."
            )
        ):
            del sys.modules[
                module_name
            ]

    importlib.invalidate_caches()

    if not compileall.compile_dir(
        PROJECT_ROOT / "src",
        quiet=1,
        force=True,
    ):
        raise RuntimeError(
            "Встроенный Python-код "
            "не прошёл компиляцию."
        )

    from src.notebook_steps import (
        bootstrap_project,
    )
    from src.target_runtime import (
        TARGET_RUNTIME_VERSION,
        target_runtime_state,
    )

    TARGET_RUNTIME_STATE = target_runtime_state()
    if (
        not TARGET_RUNTIME_STATE.get("ready")
        or TARGET_RUNTIME_STATE.get("version")
        != TARGET_RUNTIME_VERSION
    ):
        raise RuntimeError(
            "Штатный обработчик target не прошёл проверку загрузки."
        )

    BOOTSTRAP_RESULT = (
        bootstrap_project(
            project_root=PROJECT_ROOT
        )
    )
    BOOTSTRAP_RESULT["target_runtime"] = dict(
        TARGET_RUNTIME_STATE
    )
except Exception as error:
    BOOTSTRAP_RESULT = {
        "success": False,
        "error": str(error),
        "error_type": type(error).__name__,
    }
    try:
        from html import escape
        from IPython.display import HTML, display
        display(HTML(
            "<div style='border-left:5px solid #c62828;padding:12px;background:#ffebee'>"
            "<b>Автономный проект не установлен</b><br>"
            + escape(str(error))
            + "<br><b>Что сделать:</b> проверьте пункт 1 и повторите пункт 2."
            + "</div>"
        ))
    except Exception:
        print(
            "Автономный проект не установлен. "
            f"Причина: {type(error).__name__}: {error}"
        )


### 2.1. Подготовка методов детализации

Некоторые методы СПАРК требуют последовательного выполнения нескольких запросов. Сначала программа получает список объектов компании, а затем автоматически запрашивает подробную информацию по каждому найденному объекту.

**Что нужно сделать**

1. Запустите кодовую ячейку 2.1 после технической подготовки проекта в пункте 2.
2. Выполняйте этот пункт заново после каждого перезапуска среды Google Colab.
3. Убедитесь, что появилась строка:

```text
Внутренняя проверка правил: успешно
```

4. После успешной подготовки переходите к пункту 3 — авторизации в СПАРК.

Ничего вводить или настраивать вручную не требуется. Пропускать этот пункт нельзя, если в выбранном наборе используются методы детализации.

<details>
<summary><b>Техническая справка: как работают методы детализации</b></summary>

Некоторые методы REST API СПАРК нельзя выполнить напрямую только по ИНН или ОГРН организации.

Для получения подробных данных необходимо выполнить цепочку запросов:

1. Основной метод получает перечень объектов, связанных с компанией.
2. Из ответа извлекаются идентификаторы найденных объектов.
3. Для каждого идентификатора вызывается соответствующий метод детализации.
4. Полученные карточки добавляются в результат обработки организации.

В качестве идентификаторов могут использоваться:

* VIN автомобиля;
* номер договора;
* кадастровый номер;
* идентификатор сертификата;
* идентификатор проверки;
* идентификатор патента;
* другие технические ключи объектов СПАРК.

Ячейка автоматически подключает правила выполнения для **11 методов детализации**.

**Ограничение количества карточек**

По умолчанию для одного метода и одной организации обрабатывается не более **50 карточек**.

Ограничение защищает массовый конвейер от чрезмерного количества вложенных запросов, если у компании найдено очень много связанных объектов.

Абсолютный технический максимум составляет **100 карточек одного метода на одну организацию**.

**Обработка отсутствующих данных**

Если основной метод не нашёл объектов нужного типа, это считается нормальным результатом:

* дополнительные запросы не выполняются;
* обработка компании продолжается;
* в журнале фиксируется отсутствие связанных объектов;
* ситуация не считается технической ошибкой.

Например, у организации могут отсутствовать:

* транспортные средства;
* договоры лизинга;
* залоги;
* недвижимость;
* сертификаты;
* проверки;
* патенты;
* другие детализируемые объекты.

**Обработка ошибок**

Если основной или зависимый метод завершился ошибкой, программа:

* сохраняет технический статус;
* фиксирует проблемный endpoint;
* не подменяет ошибку сообщением «данных нет»;
* продолжает обработку остальных доступных методов, когда это возможно.

Это позволяет отличить реальное отсутствие объектов у компании от ошибки авторизации, параметров, лицензии или ответа API.

**Что сохраняется в текущем сеансе**

После запуска правила детализации подключаются к центральному производственному конвейеру в оперативной памяти текущей среды Colab.

Они используются:

* при проверке одной компании;
* при массовой обработке;
* при формировании связанных таблиц;
* при ведении журналов методов и HTTP-запросов.

После перезапуска среды Colab подключение из памяти исчезает. Поэтому пункт 2.1 необходимо выполнять заново после пункта 2.

**Что эта ячейка не делает**

В пункте 2.1:

* не выполняются запросы к СПАРК;
* не обрабатываются организации;
* не изменяется список выбранных методов;
* не изменяются признаки и поля;
* не рассчитывается `target_default`;
* не создаётся датасет;
* не подключаются банковские методы;
* не расходуются запросы API.

Ячейка только подготавливает технические правила, которые будут использованы последующими этапами.

**Успешный результат**

Успешным считается результат, при котором:

* все правила детализации загружены;
* внутренняя проверка завершилась без ошибок;
* указано количество подключённых правил;
* появилась строка:

```text
Внутренняя проверка правил: успешно
```

После этого переходите к пункту 3 и выполняйте авторизацию в СПАРК.

Код служебной ячейки 2.1 не предназначен для ручного редактирования.

</details>


In [ ]:
# @title 2.1. Подготовить методы детализации

try:
    from src.notebook_runtime import run_notebook_cell
except Exception as _cell_runtime_error:
    print(
        "Не удалось запустить этап «Подготовка методов детализации». "
        "Сначала выполните пункт 2. "
        f"Причина: {type(_cell_runtime_error).__name__}: {_cell_runtime_error}"
    )
else:
    with run_notebook_cell(
        stage="Подготовка методов детализации",
        next_step="Повторно выполните пункт 2, затем пункт 2.1.",
        project_root=globals().get("PROJECT_ROOT"),
    ):
        from collections.abc import Mapping
        from urllib.parse import urlsplit
        import json

        import pandas as pd

        import src.method_executor as method_executor
        import src.pipeline_v2 as pipeline_v2
        from src.field_registry import extract_path_values


        # Версия служебного обработчика. Последующие этапы смогут проверить,
        # что эта ячейка действительно была выполнена в текущей среде Colab.
        DETAIL_EXECUTOR_VERSION = "spark_detail_executor_v2"

        # Обычный предел — 50 карточек одного метода детализации на компанию.
        # Абсолютный предел согласован с защитой универсального исполнителя,
        # который принимает не более 100 запросов одного метода за один вызов.
        DEFAULT_DETAIL_LIMIT = 50
        MAX_DETAIL_LIMIT = 100


        # Для каждого метода детализации указаны:
        # 1) родительский метод, возвращающий список объектов;
        # 2) путь к этому списку в JSON;
        # 3) соответствие «параметр запроса: поле объекта».
        DETAIL_RULES = {
            "/rest/CheckVIN": [
                (
                    "/rest/GetCompanyVehicles",
                    "data[].vehicles[]",
                    {"vin": "vin"},
                ),
                (
                    "/rest/GetCompanyVehicles",
                    "data[].taxiPermits[].vehicles[]",
                    {"vin": "vin"},
                ),
            ],
            "/rest/GetCertificateReport": [
                (
                    "/rest/GetCompanyCertificates",
                    "data[].certificates[]",
                    {"number": "number"},
                ),
            ],
            "/rest/GetDomainReport": [
                (
                    "/rest/GetCompanyContacts",
                    "data[].domains[]",
                    {"domain": "name"},
                ),
                (
                    "/rest/GetCompanyContacts",
                    "data[].sites[]",
                    {"domain": "$value"},
                ),
            ],
            "/rest/GetInspectionReport": [
                (
                    "/rest/GetCompanyInspections",
                    "data[].inspections[]",
                    {
                        "id": "id",
                        "number": "number",
                    },
                ),
            ],
            "/rest/GetLeasingReport": [
                (
                    "/rest/GetCompanyLeasings",
                    "data[].leasings[]",
                    {
                        "id": "id",
                        "contractNumber": "contractNumber",
                    },
                ),
            ],
            "/rest/GetPatentReport": [
                (
                    "/rest/GetCompanyPatents",
                    "data[].patents[]",
                    {
                        "id": "id",
                        "number": "number",
                        "typeId": "type.id",
                    },
                ),
            ],
            "/rest/GetPledgeReport": [
                (
                    "/rest/GetCompanyPledges",
                    "data[].pledges[]",
                    {
                        "id": "id",
                        "contractNumber": "contractNumber",
                        "notificationNumber": "notificationNumber",
                    },
                ),
            ],
            "/rest/GetRealPropertyReport": [
                (
                    "/rest/GetCompanyRealEstate",
                    "data[].realProperties[]",
                    {"cadastralNumber": "cadastralNumber"},
                ),
            ],
            "/rest/GetStateContractPenalties": [
                (
                    "/rest/GetCompanyStateContracts",
                    "data[].stateContracts[]",
                    {
                        "id": "id",
                        "contractNumber": "contractNumber",
                    },
                ),
                (
                    "/rest/GetSupplierStateContracts",
                    "data[].stateContracts[]",
                    {
                        "id": "id",
                        "contractNumber": "number",
                    },
                ),
            ],
            "/rest/GetStateContractReport": [
                (
                    "/rest/GetCompanyStateContracts",
                    "data[].stateContracts[]",
                    {
                        "id": "id",
                        "contractNumber": "contractNumber",
                        "tenderNumber": "tenderNumber",
                    },
                ),
                (
                    "/rest/GetSupplierStateContracts",
                    "data[].stateContracts[]",
                    {
                        "id": "id",
                        "contractNumber": "number",
                    },
                ),
            ],
            "/rest/GetTenderReport": [
                (
                    "/rest/GetCompanyStateContracts",
                    "data[].stateContracts[]",
                    {"tenderNumber": "tenderNumber"},
                ),
            ],
        }


        # Для запуска карточки достаточно хотя бы одного пригодного
        # идентификатора из соответствующего набора.
        DETAIL_IDENTIFIER_FIELDS = {
            "/rest/CheckVIN": {"vin"},
            "/rest/GetCertificateReport": {"number"},
            "/rest/GetDomainReport": {"domain"},
            "/rest/GetInspectionReport": {"id", "number"},
            "/rest/GetLeasingReport": {"id", "contractNumber"},
            "/rest/GetPatentReport": {"id", "number"},
            "/rest/GetPledgeReport": {
                "id",
                "contractNumber",
                "notificationNumber",
            },
            "/rest/GetRealPropertyReport": {"cadastralNumber"},
            "/rest/GetStateContractPenalties": {
                "id",
                "contractNumber",
            },
            "/rest/GetStateContractReport": {
                "id",
                "contractNumber",
                "tenderNumber",
            },
            "/rest/GetTenderReport": {"tenderNumber"},
        }


        CRITICAL_METHOD_STATUSES = {
            "authentication_error",
            "temporary_error",
            "http_error",
            "technical_error",
            "invalid_parameters",
            "unexpected_response",
        }


        def _nested_value(value, path):
            """Получает вложенное значение одного найденного объекта."""

            if path == "$value":
                return value

            current = value

            for key in str(path).split("."):
                if not isinstance(current, Mapping):
                    return None
                current = current.get(key)

            return current


        def _not_empty(value):
            """Проверяет, можно ли использовать значение как идентификатор."""

            if value is None:
                return False

            if isinstance(value, str):
                return bool(value.strip())

            return True


        def _normalized_domain(value):
            """Приводит сайт или URL к чистому доменному имени."""

            text = str(value or "").strip()

            if not text:
                return ""

            parsed = urlsplit(
                text if "://" in text else f"//{text}",
                scheme="https",
            )
            domain = str(
                parsed.hostname
                or parsed.path.split("/", 1)[0]
            ).lower().rstrip(".")

            return (
                domain[4:]
                if domain.startswith("www.")
                else domain
            )


        def _detail_limit(parameters):
            """Читает пользовательский лимит и ограничивает его безопасно."""

            raw_limit = (
                parameters.get(
                    "__max_dependency_requests__",
                    DEFAULT_DETAIL_LIMIT,
                )
                if isinstance(parameters, Mapping)
                else DEFAULT_DETAIL_LIMIT
            )

            try:
                return max(
                    1,
                    min(int(raw_limit), MAX_DETAIL_LIMIT),
                )
            except (TypeError, ValueError):
                return DEFAULT_DETAIL_LIMIT


        def _response_items(response):
            """Разворачивает один ответ или серию ответов одного метода."""

            if response is None:
                return []

            if (
                isinstance(response, Mapping)
                and response.get("_spark_multi_response")
            ):
                items = response.get("_spark_response_items")
                return items if isinstance(items, list) else []

            return [response]


        def _canonical_payload(payload):
            """Создаёт стабильный ключ для удаления одинаковых запросов."""

            return json.dumps(
                payload,
                ensure_ascii=False,
                sort_keys=True,
                separators=(",", ":"),
                default=str,
            )


        def _build_detail_payloads(endpoint, responses, parameters):
            """Строит уникальные запросы карточек и возвращает статистику."""

            defaults = {
                str(key): value
                for key, value in (
                    parameters.items()
                    if isinstance(parameters, Mapping)
                    else []
                )
                if not str(key).startswith("__")
            }
            payloads = []
            seen = set()
            source_object_count = 0
            source_endpoints_seen = set()

            for parent_endpoint, object_path, field_map in DETAIL_RULES[endpoint]:
                if parent_endpoint in responses:
                    source_endpoints_seen.add(parent_endpoint)

                for response_item in _response_items(
                    responses.get(parent_endpoint)
                ):
                    extracted_objects = extract_path_values(
                        response_item,
                        object_path,
                    )
                    source_object_count += len(extracted_objects)

                    for extracted in extracted_objects:
                        source = extracted.get("value")
                        payload = dict(defaults)

                        for target_field, source_path in field_map.items():
                            value = _nested_value(source, source_path)

                            if target_field == "domain" and _not_empty(value):
                                value = _normalized_domain(value)

                            if _not_empty(value):
                                payload[target_field] = value

                        if not any(
                            _not_empty(payload.get(field_name))
                            for field_name in DETAIL_IDENTIFIER_FIELDS[endpoint]
                        ):
                            continue

                        fingerprint = _canonical_payload(payload)

                        if fingerprint in seen:
                            continue

                        seen.add(fingerprint)
                        payloads.append(payload)

            found_request_count = len(payloads)
            limit = _detail_limit(parameters)
            selected_payloads = payloads[:limit]

            return {
                "payloads": selected_payloads,
                "source_object_count": source_object_count,
                "source_endpoint_count": len(source_endpoints_seen),
                "found_request_count": found_request_count,
                "request_count": len(selected_payloads),
                "truncated_count": max(
                    0,
                    found_request_count - len(selected_payloads),
                ),
                "limit": limit,
            }


        def _method_log_rows(dataframe, endpoints):
            """Возвращает строки журнала заданных родительских методов."""

            if (
                not isinstance(dataframe, pd.DataFrame)
                or dataframe.empty
                or "endpoint" not in dataframe.columns
            ):
                return pd.DataFrame()

            return dataframe[
                dataframe["endpoint"].astype(str).isin(endpoints)
            ].copy()


        def _dependency_diagnostic(endpoint, base_log, responses, statistics):
            """Объясняет, почему карточка не была запрошена."""

            parent_endpoints = sorted(
                {
                    parent_endpoint
                    for parent_endpoint, _, _ in DETAIL_RULES[endpoint]
                }
            )
            parent_rows = _method_log_rows(base_log, parent_endpoints)
            failed_rows = (
                parent_rows[
                    parent_rows["result_status"].isin(
                        CRITICAL_METHOD_STATUSES
                    )
                ]
                if (
                    not parent_rows.empty
                    and "result_status" in parent_rows.columns
                )
                else pd.DataFrame()
            )

            if not failed_rows.empty:
                failed_names = ", ".join(
                    sorted(
                        set(
                            failed_rows["endpoint"]
                            .dropna()
                            .astype(str)
                        )
                    )
                )
                return (
                    "dependency_source_error",
                    "Не получен исходный список объектов",
                    (
                        "Родительский метод завершился с ошибкой: "
                        f"{failed_names}. После исправления причины "
                        "карточки будут запрошены автоматически."
                    ),
                    True,
                )

            if statistics["source_endpoint_count"] == 0:
                return (
                    "dependency_source_missing",
                    "Не выполнен родительский метод",
                    (
                        "Для метода детализации отсутствует результат "
                        "родительского метода: "
                        + ", ".join(parent_endpoints)
                        + ". Повторно сохраните набор методов."
                    ),
                    True,
                )

            if statistics["source_object_count"] == 0:
                return (
                    "available_without_related_objects",
                    "Связанные объекты не найдены",
                    (
                        "Родительский метод выполнен корректно, но компания "
                        "не имеет объектов этого типа. Это нормальный результат."
                    ),
                    False,
                )

            return (
                "available_without_object_identifiers",
                "Не найдены идентификаторы объектов",
                (
                    "Связанные объекты получены, но в ответе нет пригодного "
                    "идентификатора для запроса карточки. HTTP-запрос не выполнялся."
                ),
                False,
            )


        def _log_without_request(
            plan_row,
            company_result,
            company_code,
            result_status,
            status_label,
            message,
            statistics=None,
        ):
            """Создаёт понятную запись метода без выполнения HTTP-запроса."""

            statistics = statistics or {}

            return {
                **plan_row,
                "company_code": str(
                    company_result.get("company_code")
                    or company_code
                ),
                "spark_id": company_result.get("spark_id"),
                "result_status": result_status,
                "status_label": status_label,
                "message": message,
                "http_status_code": None,
                "http_request_count": 0,
                "cache_hit_count": 0,
                "elapsed_seconds": 0.0,
                "reauthenticated": False,
                "data_object_count": 0,
                "runtime_field_count": 0,
                "dependency_object_count": int(
                    statistics.get("source_object_count", 0)
                ),
                "dependency_request_count": 0,
                "dependency_truncated_count": int(
                    statistics.get("truncated_count", 0)
                ),
            }


        def _join_frames(frames, *, sort_by_position=False):
            """Безопасно объединяет только непустые таблицы."""

            available = [
                frame
                for frame in frames
                if isinstance(frame, pd.DataFrame) and not frame.empty
            ]

            if not available:
                return pd.DataFrame()

            result = pd.concat(
                available,
                ignore_index=True,
                sort=False,
            )

            if sort_by_position and "position" in result.columns:
                result = result.sort_values(
                    "position",
                    kind="stable",
                    na_position="last",
                )

            return result.reset_index(drop=True)


        def _validate_detail_configuration():
            """Проверяет все 11 правил до подключения к конвейеру."""

            if len(DETAIL_RULES) != 11:
                raise RuntimeError(
                    "Ожидалось 11 методов детализации, "
                    f"получено: {len(DETAIL_RULES)}."
                )

            if set(DETAIL_RULES) != set(DETAIL_IDENTIFIER_FIELDS):
                raise RuntimeError(
                    "Список методов детализации не совпадает со списком "
                    "их идентификаторов."
                )

            if not (1 <= DEFAULT_DETAIL_LIMIT <= MAX_DETAIL_LIMIT <= 100):
                raise RuntimeError(
                    "Нарушены безопасные пределы количества запросов детализации."
                )

            for endpoint, rules in DETAIL_RULES.items():
                if not endpoint.startswith("/rest/") or not rules:
                    raise RuntimeError(
                        f"Некорректное правило детализации: {endpoint}."
                    )

                mapped_fields = set()

                for parent_endpoint, object_path, field_map in rules:
                    if (
                        not str(parent_endpoint).startswith("/rest/")
                        or not str(object_path).strip()
                        or not isinstance(field_map, Mapping)
                        or not field_map
                    ):
                        raise RuntimeError(
                            f"Повреждено правило источника для {endpoint}."
                        )
                    mapped_fields.update(str(name) for name in field_map)

                if not (
                    DETAIL_IDENTIFIER_FIELDS[endpoint]
                    & mapped_fields
                ):
                    raise RuntimeError(
                        f"Для {endpoint} не настроен ни один идентификатор."
                    )


        def _set_nested_value(container, path, value):
            """Служебно создаёт объект для внутренней самопроверки."""

            current = container
            parts = str(path).split(".")

            for key in parts[:-1]:
                current = current.setdefault(key, {})

            current[parts[-1]] = value


        def _wrap_object_path(path, value):
            """Служебно оборачивает объект по JSON-пути с массивами."""

            current = value

            for part in reversed(str(path).split(".")):
                is_array = part.endswith("[]")
                key = part[:-2] if is_array else part
                current = {key: [current] if is_array else current}

            return current


        def _run_detail_self_check():
            """Без API проверяет извлечение идентификаторов всех 11 методов."""

            sample_values = {
                "vin": "TESTVIN123456789",
                "number": "TEST-001",
                "domain": "https://www.example.org/path",
                "id": 101,
                "contractNumber": "CONTRACT-001",
                "notificationNumber": "NOTICE-001",
                "cadastralNumber": "77:01:0000000:1",
                "tenderNumber": "TENDER-001",
                "typeId": 1,
            }

            for endpoint, rules in DETAIL_RULES.items():
                parent_endpoint, object_path, field_map = rules[0]

                if list(field_map.values()) == ["$value"]:
                    source_object = sample_values[
                        next(iter(field_map))
                    ]
                else:
                    source_object = {}
                    for target_field, source_path in field_map.items():
                        if source_path == "$value":
                            source_object = sample_values[target_field]
                            break
                        _set_nested_value(
                            source_object,
                            source_path,
                            sample_values[target_field],
                        )

                response = _wrap_object_path(
                    object_path,
                    source_object,
                )
                check = _build_detail_payloads(
                    endpoint,
                    {parent_endpoint: response},
                    {},
                )

                if check["request_count"] != 1:
                    raise RuntimeError(
                        "Внутренняя проверка не смогла построить запрос "
                        f"для {endpoint}."
                    )


        # Сохраняем именно исходный универсальный исполнитель. Повторный запуск
        # этой ячейки заменит обработчик новой версией, но не создаст цепочку
        # вложенных обёрток и не умножит HTTP-запросы.
        if not hasattr(
            method_executor,
            "_spark_detail_base_execute_company_methods",
        ):
            method_executor._spark_detail_base_execute_company_methods = getattr(
                method_executor,
                "_spark_base_execute_company_methods",
                method_executor.execute_company_methods,
            )


        def _execute_with_details(**kwargs):
            """Выполняет прямые методы, а затем карточки найденных объектов."""

            base_executor = (
                method_executor
                ._spark_detail_base_execute_company_methods
            )
            registry = kwargs.get("execution_registry_df")

            if not isinstance(registry, pd.DataFrame):
                return base_executor(**kwargs)

            endpoints = registry.get(
                "endpoint",
                pd.Series("", index=registry.index),
            ).astype(str)
            detail_mask = endpoints.isin(DETAIL_RULES)
            detail_registry = registry[detail_mask].copy()

            if detail_registry.empty:
                return base_executor(**kwargs)

            base_kwargs = dict(kwargs)
            base_kwargs["execution_registry_df"] = registry[
                ~detail_mask
            ].copy()
            base_result = base_executor(**base_kwargs)

            responses = dict(base_result.get("responses") or {})
            company_result = dict(
                base_result.get("company_result") or {}
            )
            company_code = str(kwargs.get("company_code") or "")
            parameter_map = (
                kwargs.get("method_parameters")
                if isinstance(
                    kwargs.get("method_parameters"),
                    Mapping,
                )
                else {}
            )
            base_log = base_result.get("method_log_df")
            method_logs = [base_log]
            runtime_tables = [base_result.get("runtime_fields_df")]
            total_http = int(
                base_result.get("http_request_count") or 0
            )
            total_cache = int(
                base_result.get("cache_hit_count") or 0
            )
            authentication_failed = bool(
                base_result.get("authentication_failed")
            )
            overall_success = bool(
                base_result.get("success", True)
            )
            complete_without_errors = bool(
                base_result.get(
                    "complete_without_method_errors",
                    True,
                )
            )
            company_ready = bool(
                company_result.get("success", overall_success)
            )

            for plan_row in detail_registry.to_dict(orient="records"):
                endpoint = str(plan_row.get("endpoint") or "")

                if authentication_failed:
                    method_logs.append(
                        pd.DataFrame(
                            [
                                _log_without_request(
                                    plan_row,
                                    company_result,
                                    company_code,
                                    "skipped_after_authentication_error",
                                    "Не выполнен после потери сессии",
                                    (
                                        "Предыдущий запрос подтвердил ошибку "
                                        "авторизации. Повторно выполните пункт 3."
                                    ),
                                )
                            ]
                        )
                    )
                    continue

                if not company_ready:
                    method_logs.append(
                        pd.DataFrame(
                            [
                                _log_without_request(
                                    plan_row,
                                    company_result,
                                    company_code,
                                    "skipped_company_unavailable",
                                    "Организация не подготовлена",
                                    (
                                        "Карточки не запрашивались, потому что "
                                        "не удалось определить организацию."
                                    ),
                                )
                            ]
                        )
                    )
                    continue

                parameters = parameter_map.get(endpoint, {})
                statistics = _build_detail_payloads(
                    endpoint,
                    responses,
                    parameters,
                )
                payloads = statistics["payloads"]

                if not payloads:
                    (
                        result_status,
                        status_label,
                        message,
                        is_error,
                    ) = _dependency_diagnostic(
                        endpoint,
                        base_log,
                        responses,
                        statistics,
                    )
                    method_logs.append(
                        pd.DataFrame(
                            [
                                _log_without_request(
                                    plan_row,
                                    company_result,
                                    company_code,
                                    result_status,
                                    status_label,
                                    message,
                                    statistics,
                                )
                            ]
                        )
                    )
                    complete_without_errors = bool(
                        complete_without_errors and not is_error
                    )
                    continue

                executable_row = {
                    **plan_row,
                    "execution_status": "configured",
                    "execution_status_label": (
                        "Выполняется по найденным объектам"
                    ),
                    "execution_allowed": True,
                    "reason": (
                        "Параметры автоматически получены "
                        "из родительского метода."
                    ),
                }
                detail_kwargs = {
                    **kwargs,
                    "execution_registry_df": pd.DataFrame(
                        [executable_row]
                    ),
                    "method_parameters": {
                        endpoint: {"__requests__": payloads}
                    },
                    "preloaded_responses": None,
                    "preloaded_response_sources": None,
                    "company_result": company_result,
                }
                detail_result = base_executor(**detail_kwargs)
                detail_log = detail_result.get("method_log_df")

                if isinstance(detail_log, pd.DataFrame):
                    detail_log = detail_log.copy()
                    detail_log["dependency_object_count"] = (
                        statistics["source_object_count"]
                    )
                    detail_log["dependency_request_count"] = (
                        statistics["request_count"]
                    )
                    detail_log["dependency_truncated_count"] = (
                        statistics["truncated_count"]
                    )

                    if statistics["truncated_count"]:
                        suffix = (
                            f" Обработано первых {statistics['limit']} "
                            "объектов; ещё "
                            f"{statistics['truncated_count']} не запрашивались "
                            "из-за защитного лимита."
                        )
                        detail_log["message"] = (
                            detail_log["message"]
                            .fillna("")
                            .astype(str)
                            + suffix
                        )

                method_logs.append(detail_log)
                runtime_tables.append(
                    detail_result.get("runtime_fields_df")
                )
                responses.update(
                    detail_result.get("responses") or {}
                )
                total_http += int(
                    detail_result.get("http_request_count") or 0
                )
                total_cache += int(
                    detail_result.get("cache_hit_count") or 0
                )
                authentication_failed = bool(
                    authentication_failed
                    or detail_result.get("authentication_failed")
                )
                overall_success = bool(
                    overall_success
                    and detail_result.get("success", True)
                )
                complete_without_errors = bool(
                    complete_without_errors
                    and detail_result.get(
                        "complete_without_method_errors",
                        True,
                    )
                )

            method_log_df = _join_frames(
                method_logs,
                sort_by_position=True,
            )
            runtime_fields_df = _join_frames(runtime_tables)
            duplicate_subset = [
                column
                for column in (
                    "company_code",
                    "endpoint",
                    "json_path",
                    "runtime_type",
                )
                if column in runtime_fields_df.columns
            ]

            if duplicate_subset:
                runtime_fields_df = (
                    runtime_fields_df.drop_duplicates(
                        subset=duplicate_subset,
                        keep="first",
                    ).reset_index(drop=True)
                )

            return {
                **base_result,
                "success": bool(
                    overall_success and not authentication_failed
                ),
                "complete_without_method_errors": (
                    complete_without_errors
                ),
                "message": (
                    "План методов выполнен, включая детализацию объектов."
                    if not authentication_failed
                    else (
                        "План остановлен после ошибки авторизации."
                    )
                ),
                "responses": responses,
                "method_log_df": method_log_df,
                "runtime_fields_df": runtime_fields_df,
                "http_request_count": total_http,
                "cache_hit_count": total_cache,
                "authentication_failed": authentication_failed,
                "detail_executor_version": DETAIL_EXECUTOR_VERSION,
            }


        try:
            _validate_detail_configuration()
            _run_detail_self_check()

            method_executor.execute_company_methods = (
                _execute_with_details
            )
            pipeline_v2.execute_company_methods = (
                _execute_with_details
            )

            detail_state = {
                "ready": True,
                "version": DETAIL_EXECUTOR_VERSION,
                "detail_method_count": len(DETAIL_RULES),
                "default_limit": DEFAULT_DETAIL_LIMIT,
                "maximum_limit": MAX_DETAIL_LIMIT,
            }
            method_executor._spark_detail_executor_state = (
                detail_state
            )
            pipeline_v2._spark_detail_executor_state = (
                detail_state
            )

            print("МЕТОДЫ ДЕТАЛИЗАЦИИ ПОДГОТОВЛЕНЫ")
            print(f"Подключено зависимых методов: {len(DETAIL_RULES)}")
            print("Внутренняя проверка правил: успешно")
            print(
                "Защитный лимит: до "
                f"{DEFAULT_DETAIL_LIMIT} карточек одного метода "
                "на организацию."
            )
            print(
                "Допустимый технический максимум: "
                f"{MAX_DETAIL_LIMIT} карточек."
            )
            print(
                "Банковские методы не подключались "
                "и запрашиваться не будут."
            )
            print(
                "Запросы к СПАРК в этой ячейке не выполнялись."
            )
            print(
                "Следующий шаг: авторизуйтесь в СПАРК в пункте 3."
            )

        except Exception as error:
            method_executor._spark_detail_executor_state = {
                "ready": False,
                "version": DETAIL_EXECUTOR_VERSION,
                "error": str(error),
            }
            pipeline_v2._spark_detail_executor_state = (
                method_executor._spark_detail_executor_state
            )
            raise RuntimeError(
                "Не удалось подготовить методы детализации. "
                "Повторно выполните пункт 2, затем пункт 2.1. "
                f"Причина: {type(error).__name__}: {error}"
            )


## 3. Авторизация в СПАРК

До запуска создайте в панели **Secrets** Google Colab два
секрета и разрешите этому ноутбуку доступ к ним:

- `SPARK_LOGIN`
- `SPARK_PASSWORD`

Секреты и cookie не записываются на Google Диск. После
перезапуска среды эту ячейку нужно выполнить снова.


In [ ]:
# @title 3. Авторизоваться в СПАРК

try:
    from src.notebook_runtime import run_notebook_cell
except Exception as _cell_runtime_error:
    print(
        "Не удалось запустить этап «Авторизация в СПАРК». "
        "Сначала выполните пункт 2. "
        f"Причина: {type(_cell_runtime_error).__name__}: {_cell_runtime_error}"
    )
else:
    with run_notebook_cell(
        stage="Авторизация в СПАРК",
        next_step="Проверьте Secrets и повторите пункт 3.",
        project_root=globals().get("PROJECT_ROOT"),
    ):
        from src.notebook_steps import (
            authentication_step,
        )

        AUTH_RESULT = authentication_step(
            project_root=PROJECT_ROOT
        )
        SESSION = AUTH_RESULT.get(
            "session"
        )


## 4. Методы API и сохранённые конфигурации


В этом пункте выберите одно из двух действий:

- **Открыть сохранённую конфигурацию** — восстановить методы API, target, параметры, поля и признаки из одного профиля;
- **Настроить методы вручную** — создать новый набор и продолжить обычную настройку в пунктах 5.1–5.3.


<details>
<summary><b>Техническая справка</b></summary>

В этом пункте определяется, какие данные СПАРК будут собираться для каждой нефинансовой организации. Сохранённый набор будет одинаково использоваться при проверке одной компании и при массовой выгрузке.

Банковские методы, включая формы 101 и 102, в интерфейс не включаются и запрашиваться не будут.

**Режимы выбора**

- **Все прямые данные — 38 методов** — получает основные сведения компании без дополнительных карточек отдельных объектов.
- **Максимальная детализация — 38 + 11 методов** — дополнительно получает карточки автомобилей, лизинга, залогов, недвижимости, контрактов и других найденных объектов.
- **Ручная настройка** — позволяет выбрать только необходимые методы с помощью категорий, поиска и галочек.

Методы детализации выполняются только тогда, когда у организации найден соответствующий объект. Например, карточка автомобиля запрашивается только при наличии VIN. Защитный лимит пункта 2.1 ограничивает обработку первыми 50 карточками одного метода на организацию.

**Почему итоговое количество может быть больше выбранного**

Программа самостоятельно добавляет обязательные служебные методы:

1. поиск организации;
2. получение краткой справки;
3. получение списка отчётных периодов — если используется бухгалтерская отчётность.

Поэтому:

- 38 прямых методов образуют технический план из 41 метода;
- 38 прямых + 11 методов детализации образуют план из 52 методов.

Интерфейс показывает выбранные, зависимые и служебные методы отдельными строками.

**Дополнительные параметры разработчика**

Обычно оставляйте JSON пустым:

```json
{}
```

</details>


In [ ]:
# @title 4. Выбрать методы или открыть сохранённую конфигурацию

from __future__ import annotations

from collections.abc import Mapping
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import html
import json


ОБНОВИТЬ_КАТАЛОГ_ЗАНОВО = False  # @param {type:"boolean"}

BUILD_CONFIGURATION_FORMAT = "spark_build_configuration_profile_v1"
BUILD_CONFIGURATION_DIRECTORY = "configuration_profiles"
METHOD_PARAMETER_PROFILE_FORMAT = "spark_method_parameter_profile_v2"


def _безопасный_текст(value) -> str:
    """Экранирует текст перед выводом в HTML-интерфейс."""

    return html.escape(str(value or ""))


def _json_готово(value):
    """Преобразует значение к стабильному JSON-виду."""

    if isinstance(value, Mapping):
        return {
            str(key): _json_готово(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set, frozenset)):
        return [_json_готово(item) for item in value]

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, datetime):
        return value.isoformat(timespec="seconds")

    if value is None or isinstance(value, (str, int, float, bool)):
        return value

    if hasattr(value, "item"):
        try:
            return _json_готово(value.item())
        except Exception:
            pass

    return str(value)


def _контрольная_сумма(value) -> str:
    """Считает стабильный SHA-256 JSON-объекта."""

    serialized = json.dumps(
        _json_готово(value),
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()


def _следующий_пункт(error) -> str:
    """Возвращает конкретное действие по фактической причине ошибки."""

    text = str(error).lower()

    if "пункт 1" in text or "google диск" in text:
        return (
            "Выполните пункт 1 «Подключить личный Google Диск», "
            "затем повторите пункт 4."
        )

    if (
        "пункт 2" in text
        or "no module named 'src'" in text
        or "ipywidgets" in text
        or "интерфейсные библиотеки" in text
    ):
        return (
            "Выполните пункт 2 «Установить и проверить автономный проект», "
            "затем повторите пункт 4."
        )

    if "пункт 3" in text or "авториз" in text:
        return (
            "Выполните пункт 3 «Авторизоваться в СПАРК», "
            "затем повторите пункт 4."
        )

    if (
        "каталог" in text
        or "openapi" in text
        or "network" in text
        or "connection" in text
    ):
        return (
            "Проверьте подключение к интернету "
            "и повторите только пункт 4."
        )

    return (
        "Повторите только пункт 4. "
        "Предыдущие пункты повторно запускать не нужно."
    )


def _понятная_причина(error) -> str:
    """Переводит известные технические ошибки интерфейса."""

    text = str(error) or type(error).__name__
    lowered = text.lower()

    if "typedtuple" in lowered and "expected a widget" in lowered:
        return (
            "Интерфейс выбора методов получил обычный HTML-вывод "
            "вместо элемента ipywidgets."
        )

    return text


try:
    import ipywidgets as widgets
    import pandas as pd
    from IPython.display import clear_output, display

    import src.notebook_selectors as notebook_selectors
    from src.catalog_service import prepare_and_save_catalog
    from src.feature_registry import build_feature_registry
    from src.method_policy import validate_method_policy
    from src.notebook_runtime import _write_traceback
    from src.notebook_ui import show_error
    from src.user_features_loader import active_feature_source_state
    from src.workflow import (
        invalidate_after,
        load_workflow,
        mark_stage,
        save_workflow,
    )

    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError(
            "Требуется пункт 1: Google Диск не подключён."
        )

    _project_root = globals().get("PROJECT_ROOT")

    if (
        not _project_root
        or not (Path(_project_root) / "src").is_dir()
    ):
        raise RuntimeError(
            "Требуется пункт 2: автономный проект не установлен."
        )

    _project_root = Path(_project_root)
    _workflow_root = _project_root / "workflow"
    _configuration_root = (
        _project_root / BUILD_CONFIGURATION_DIRECTORY
    )

    _workflow = load_workflow(
        workflow_root=_workflow_root,
    )
    _stages = _workflow.get("stages") or {}

    if not bool(
        (_stages.get("bootstrap") or {}).get("ready")
    ):
        raise RuntimeError(
            "Требуется пункт 2: автономный проект не подготовлен."
        )

    if not bool(
        (_stages.get("authentication") or {}).get("ready")
    ):
        raise RuntimeError(
            "Требуется пункт 3: авторизация в СПАРК не завершена."
        )

    _catalog = prepare_and_save_catalog(
        project_root=_project_root,
        workflow=_workflow,
        force_refresh=ОБНОВИТЬ_КАТАЛОГ_ЗАНОВО,
        export_when_refreshed=True,
    )

    if not _catalog.get("success"):
        raise RuntimeError(
            _catalog.get("message")
            or "Официальный каталог API не удалось подготовить."
        )

    METHOD_POLICY_STATE = validate_method_policy(
        _catalog["catalog_result"]
    )

    _feature_registry_df = build_feature_registry()
    _known_feature_names = {
        str(value)
        for value in _feature_registry_df.loc[
            _feature_registry_df["included_in_ml"].fillna(False),
            "feature_name",
        ].astype(str)
    }

    def _загрузить_профиль(profile_path: str | Path) -> dict:
        """Читает и полностью проверяет сохранённую конфигурацию."""

        path = Path(profile_path)

        if not path.is_file():
            raise FileNotFoundError(
                f"Файл конфигурации не найден: {path}"
            )

        try:
            payload = json.loads(
                path.read_text(encoding="utf-8")
            )
        except Exception as error:
            raise RuntimeError(
                "JSON конфигурации повреждён или не читается."
            ) from error

        if not isinstance(payload, Mapping):
            raise RuntimeError(
                "Файл конфигурации имеет неправильную структуру."
            )

        if payload.get("format") != BUILD_CONFIGURATION_FORMAT:
            raise RuntimeError(
                "Файл создан в неподдерживаемом формате конфигурации."
            )

        configuration = payload.get("configuration")

        if not isinstance(configuration, Mapping):
            raise RuntimeError(
                "В файле отсутствует снимок настроек пунктов 4–5.3."
            )

        expected_hash = str(
            payload.get("configuration_hash") or ""
        ).strip()
        actual_hash = _контрольная_сумма(configuration)

        if not expected_hash or expected_hash != actual_hash:
            raise RuntimeError(
                "Контрольная сумма конфигурации не совпала. "
                "Файл мог быть изменён вручную или повреждён."
            )

        methods = configuration.get("methods") or {}
        target = configuration.get("target") or {}
        parameter_profile = (
            configuration.get("method_parameter_profile") or {}
        )
        fields = configuration.get("fields") or {}

        if not isinstance(methods, Mapping) or not (
            methods.get("selected_endpoints") or []
        ):
            raise RuntimeError(
                "В конфигурации отсутствует сохранённый набор методов API."
            )

        if (
            not isinstance(target, Mapping)
            or target.get("format") != "spark_target_profile_v1"
            or not str(
                target.get("configuration_hash") or ""
            ).strip()
        ):
            raise RuntimeError(
                "В конфигурации отсутствует корректное правило target."
            )

        if (
            not isinstance(parameter_profile, Mapping)
            or parameter_profile.get("format")
            != METHOD_PARAMETER_PROFILE_FORMAT
        ):
            raise RuntimeError(
                "В конфигурации отсутствуют параметры методов пункта 5.2."
            )

        if not isinstance(fields, Mapping) or not (
            fields.get("selected_feature_names")
            or fields.get("selected_field_ids")
        ):
            raise RuntimeError(
                "В конфигурации отсутствует сохранённый набор признаков."
            )

        result = dict(payload)
        result["profile_path"] = str(path)
        return result

    def _найти_профили() -> tuple[list[dict], list[str]]:
        """Находит текущие профили без истории старых версий."""

        profiles: list[dict] = []
        errors: list[str] = []

        if not _configuration_root.is_dir():
            return profiles, errors

        for path in sorted(
            _configuration_root.glob("*.json"),
            key=lambda item: item.stat().st_mtime,
            reverse=True,
        ):
            try:
                payload = _загрузить_профиль(path)
                payload["modified_at_timestamp"] = (
                    path.stat().st_mtime
                )
                profiles.append(payload)
            except Exception as error:
                errors.append(f"{path.name}: {error}")

        return profiles, errors

    def _метка_профиля(payload: Mapping) -> str:
        """Формирует короткую строку выпадающего списка."""

        summary = payload.get("summary") or {}
        name = str(
            payload.get("profile_name")
            or Path(
                str(
                    payload.get("profile_path")
                    or "configuration"
                )
            ).stem
        )
        feature_mode = str(
            summary.get("feature_mode")
            or "Ручная настройка"
        )
        method_count = int(
            summary.get("method_count") or 0
        )
        feature_count = int(
            summary.get("feature_count") or 0
        )
        field_count = int(
            summary.get("field_count") or 0
        )
        revision = int(payload.get("revision") or 1)
        return (
            f"{name} — {method_count} методов — "
            f"{feature_mode}: {feature_count} ML, "
            f"{field_count} полей API — v{revision}"
        )

    def _применить_профиль(payload: Mapping) -> dict:
        """Применяет снимок, не затрагивая авторизацию и входной файл."""

        configuration = payload.get("configuration") or {}
        methods = deepcopy(
            dict(configuration.get("methods") or {})
        )
        target = deepcopy(
            dict(configuration.get("target") or {})
        )
        parameter_profile = deepcopy(
            dict(
                configuration.get("method_parameter_profile")
                or {}
            )
        )
        fields = deepcopy(
            dict(configuration.get("fields") or {})
        )

        policy_state = validate_method_policy(
            _catalog["catalog_result"]
        )
        saved_policy_version = str(
            methods.get("runtime_policy_version") or ""
        ).strip()
        current_policy_version = str(
            policy_state.get("version") or ""
        ).strip()

        if (
            saved_policy_version
            and current_policy_version
            and saved_policy_version != current_policy_version
        ):
            raise RuntimeError(
                "Конфигурация создана для другой версии политики "
                "методов API. Выберите более новый профиль либо "
                "настройте методы вручную."
            )

        registry = _catalog["catalog_result"].get(
            "endpoint_registry_df"
        )
        catalog_endpoints = (
            set(registry["endpoint"].astype(str))
            if isinstance(registry, pd.DataFrame)
            and not registry.empty
            and "endpoint" in registry.columns
            else set()
        )
        saved_endpoints = {
            str(value)
            for value in (
                methods.get("selected_endpoints") or []
            )
            if str(value).strip()
        }
        missing_endpoints = sorted(
            saved_endpoints - catalog_endpoints
        )

        if missing_endpoints:
            preview = ", ".join(missing_endpoints[:8])
            suffix = (
                f" и ещё {len(missing_endpoints) - 8}"
                if len(missing_endpoints) > 8
                else ""
            )
            raise RuntimeError(
                "В текущем каталоге СПАРК отсутствуют методы из "
                f"профиля: {preview}{suffix}."
            )

        saved_features = {
            str(value)
            for value in (
                fields.get("selected_feature_names") or []
            )
            if str(value).strip()
        }
        unknown_features = sorted(
            saved_features - _known_feature_names
        )

        if unknown_features:
            preview = ", ".join(unknown_features[:8])
            suffix = (
                f" и ещё {len(unknown_features) - 8}"
                if len(unknown_features) > 8
                else ""
            )
            raise RuntimeError(
                "В текущем реестре отсутствуют признаки из профиля: "
                f"{preview}{suffix}."
            )

        current_feature_state = _json_готово(
            active_feature_source_state(_project_root)
        )
        current_feature_sha = str(
            (
                current_feature_state
                if isinstance(current_feature_state, Mapping)
                else {}
            ).get("sha256")
            or ""
        ).strip()
        saved_feature_sha = str(
            fields.get("custom_features_sha256") or ""
        ).strip()

        if (
            saved_feature_sha
            and current_feature_sha
            and saved_feature_sha != current_feature_sha
        ):
            raise RuntimeError(
                "Файл расчётных формул изменился после сохранения "
                "этой конфигурации. Применение остановлено, чтобы "
                "не смешать разные формулы в одном датасете."
            )

        method_profile_path = str(
            methods.get("profile_path") or ""
        ).strip()
        field_profile_path = str(
            fields.get("profile_path") or ""
        ).strip()

        if not method_profile_path or not Path(
            method_profile_path
        ).is_file():
            raise RuntimeError(
                "Связанный файл параметров методов не найден на "
                "Google Диске. Откройте профиль, сохранённый в этой "
                "же папке проекта, либо повторно подготовьте пункт 5.2."
            )

        if not field_profile_path or not Path(
            field_profile_path
        ).is_file():
            raise RuntimeError(
                "Связанный файл полей и признаков не найден на "
                "Google Диске. Откройте профиль, сохранённый в этой "
                "же папке проекта, либо повторно сохраните пункт 5.3."
            )

        updated = deepcopy(dict(_workflow))
        updated["methods"] = methods
        updated["target"] = target
        updated["method_parameter_profile"] = parameter_profile
        updated["fields"] = fields

        saved_policies = configuration.get("policies")
        if isinstance(saved_policies, Mapping):
            updated["policies"] = deepcopy(
                dict(saved_policies)
            )

        updated["active_build_configuration"] = {
            "format": BUILD_CONFIGURATION_FORMAT,
            "profile_name": str(
                payload.get("profile_name") or ""
            ),
            "profile_path": str(
                payload.get("profile_path") or ""
            ),
            "configuration_hash": str(
                payload.get("configuration_hash") or ""
            ),
            "revision": int(payload.get("revision") or 1),
            "applied_at": datetime.now(
                timezone.utc
            ).isoformat(timespec="seconds"),
        }
        updated["test"] = {
            "required": True,
            "completed": False,
            "company_code": None,
            "configuration_hash": None,
            "summary": {},
            "output_directory": None,
            "runtime_fields_path": None,
            "excel_path": None,
            "zip_path": None,
        }
        updated = mark_stage(
            updated,
            "methods",
            ready=True,
            message=(
                "Методы и параметры восстановлены из полной "
                "конфигурации."
            ),
        )
        updated = mark_stage(
            updated,
            "fields",
            ready=True,
            message=(
                "Поля и признаки восстановлены из полной "
                "конфигурации."
            ),
        )
        updated = invalidate_after(
            updated,
            "fields",
            reason=(
                "Применена сохранённая полная конфигурация. "
                "Повторите тест одной компании."
            ),
        )
        saved = save_workflow(
            updated,
            workflow_root=_workflow_root,
        )
        return saved["workflow"]

    # ------------------------------------------------------------
    # Интерфейс: две независимые ответственности
    # ------------------------------------------------------------

    _button_open_profile = widgets.Button(
        description="Открыть сохранённую конфигурацию",
        icon="folder-open",
        button_style="info",
        layout=widgets.Layout(
            width="340px",
            min_width="340px",
        ),
    )
    _button_manual = widgets.Button(
        description="Настроить методы вручную",
        icon="sliders",
        layout=widgets.Layout(
            width="300px",
            min_width="300px",
        ),
    )
    _mode_hint = widgets.HTML(
        value=(
            "<div style='padding:10px 12px;background:#f5f7fa;"
            "border-left:4px solid #78909c;line-height:1.5'>"
            "Выберите одно действие. Ничего не применяется "
            "автоматически.</div>"
        )
    )

    _profile_refresh_button = widgets.Button(
        description="Обновить список",
        icon="refresh",
        layout=widgets.Layout(width="190px"),
    )
    _profile_dropdown = widgets.Dropdown(
        options=[("Сначала обновите список", "")],
        value="",
        description="Профиль:",
        disabled=True,
        layout=widgets.Layout(width="100%"),
        style={"description_width": "90px"},
    )
    _profile_preview = widgets.HTML(
        value=(
            "<div style='padding:10px 12px;background:#f5f7fa;"
            "border-left:4px solid #78909c'>"
            "Нажмите «Обновить список». Профиль не будет "
            "выбран автоматически.</div>"
        )
    )
    _profile_apply_button = widgets.Button(
        description="Применить выбранную конфигурацию",
        icon="check",
        button_style="success",
        disabled=True,
        layout=widgets.Layout(
            width="330px",
            min_width="330px",
        ),
    )
    _profile_output = widgets.Output(
        layout=widgets.Layout(width="100%")
    )
    _profile_state = {
        "profiles": {},
        "errors": [],
    }

    _profile_panel = widgets.VBox(
        [
            widgets.HTML(
                "<h3>Открыть сохранённую полную конфигурацию</h3>"
                "<p>Профиль восстанавливает методы API, target, "
                "параметры методов, ML-признаки и дополнительные "
                "поля. Выбор в списке служит только для просмотра.</p>"
            ),
            widgets.HBox(
                [_profile_refresh_button],
                layout=widgets.Layout(
                    gap="8px",
                    flex_flow="row wrap",
                ),
            ),
            _profile_dropdown,
            _profile_preview,
            _profile_apply_button,
            _profile_output,
        ],
        layout=widgets.Layout(
            display="none",
            width="100%",
            border="1px solid #d9e2ec",
            padding="12px",
        ),
    )

    _manual_output = widgets.Output(
        layout=widgets.Layout(width="100%")
    )
    _manual_panel = widgets.VBox(
        [
            widgets.HTML(
                "<h3>Настроить только методы API вручную</h3>"
                "<p>Используйте этот раздел, когда создаёте новую "
                "конфигурацию или хотите изменить состав методов. "
                "После сохранения переходите к пункту 5.1.</p>"
            ),
            _manual_output,
        ],
        layout=widgets.Layout(
            display="none",
            width="100%",
            border="1px solid #d9e2ec",
            padding="12px",
        ),
    )
    _manual_state = {
        "built": False,
        "interface": None,
    }

    def _показать_профили(_=None) -> None:
        """Открывает только меню сохранённых конфигураций."""

        _manual_panel.layout.display = "none"
        _profile_panel.layout.display = "flex"
        _mode_hint.value = (
            "<div style='padding:10px 12px;background:#e3f2fd;"
            "border-left:4px solid #1976d2'>"
            "Открыт раздел сохранённых конфигураций. "
            "Текущий workflow пока не изменён.</div>"
        )

    def _создать_ручной_интерфейс() -> None:
        """Лениво строит прежний интерфейс выбора методов."""

        if _manual_state["built"]:
            return

        with _manual_output:
            clear_output(wait=True)
            _original_widgets_factory = (
                notebook_selectors._widgets
            )

            def _widgets_для_пункта_4():
                notebook_selectors.require_widget_environment()
                return (
                    widgets,
                    widgets.HTML,
                    clear_output,
                    display,
                )

            notebook_selectors._widgets = (
                _widgets_для_пункта_4
            )

            try:
                latest_workflow = load_workflow(
                    workflow_root=_workflow_root,
                )
                _manual_state["interface"] = (
                    notebook_selectors.show_method_selector(
                        project_root=_project_root,
                        workflow=latest_workflow,
                        catalog_result=_catalog[
                            "catalog_result"
                        ],
                    )
                )
                _manual_state["built"] = True
            finally:
                notebook_selectors._widgets = (
                    _original_widgets_factory
                )

    def _показать_ручной_выбор(_=None) -> None:
        """Открывает только прежний ручной выбор методов."""

        _profile_panel.layout.display = "none"
        _manual_panel.layout.display = "flex"
        _mode_hint.value = (
            "<div style='padding:10px 12px;background:#fff8e1;"
            "border-left:4px solid #f9a825'>"
            "Открыт ручной выбор методов API. Сохранённая полная "
            "конфигурация не применяется.</div>"
        )

        try:
            _создать_ручной_интерфейс()
        except Exception as error:
            with _manual_output:
                clear_output(wait=True)
                show_error(
                    stage="Ручной выбор методов",
                    error=_понятная_причина(error),
                    action=(
                        "Повторите только пункт 4 и снова откройте "
                        "ручной выбор методов."
                    ),
                )

    def _обновить_профили(_=None) -> None:
        """Обновляет список без автоматического выбора."""

        _profile_dropdown.disabled = True
        _profile_apply_button.disabled = True

        try:
            profiles, errors = _найти_профили()
            _profile_state["profiles"] = {
                str(profile["profile_path"]): profile
                for profile in profiles
            }
            _profile_state["errors"] = errors
            options = [
                ("Выберите сохранённую конфигурацию", "")
            ]
            options.extend(
                (
                    _метка_профиля(profile),
                    str(profile["profile_path"]),
                )
                for profile in profiles
            )
            _profile_dropdown.options = options
            _profile_dropdown.value = ""
            _profile_preview.value = (
                "<div style='padding:10px 12px;background:#f5f7fa;"
                "border-left:4px solid #78909c;line-height:1.5'>"
                f"Найдено конфигураций: <b>{len(profiles)}</b>. "
                "Ничего не выбрано и текущие настройки не изменены."
                + (
                    "<br><span style='color:#b26a00'>"
                    f"Повреждённых или неподдерживаемых файлов: "
                    f"{len(errors)}.</span>"
                    if errors
                    else ""
                )
                + "</div>"
            )
        except Exception as error:
            _profile_dropdown.options = [
                ("Не удалось прочитать список", "")
            ]
            _profile_dropdown.value = ""
            _profile_preview.value = (
                "<div style='padding:10px 12px;background:#ffebee;"
                "border-left:4px solid #c62828'>"
                f"{_безопасный_текст(error)}</div>"
            )
        finally:
            _profile_dropdown.disabled = False

    def _показать_сводку_профиля(change) -> None:
        """Показывает профиль без применения."""

        if change.get("name") != "value":
            return

        selected_path = str(change.get("new") or "")
        payload = _profile_state["profiles"].get(
            selected_path
        )

        if not isinstance(payload, Mapping):
            _profile_apply_button.disabled = True
            return

        summary = payload.get("summary") or {}
        report_year = summary.get("target_report_year")
        horizon = summary.get("target_horizon_months")
        target_text = str(
            summary.get("target_name") or "target_default"
        )

        if report_year is not None:
            target_text += f", отчётный год {report_year}"

        if horizon is not None:
            target_text += f", горизонт {horizon} мес."

        _profile_preview.value = (
            "<div style='padding:12px 14px;background:#eef4fb;"
            "border-left:5px solid #1976d2;line-height:1.65'>"
            "<b>Конфигурация открыта для просмотра</b><br>"
            f"Название: <b>{_безопасный_текст(payload.get('profile_name'))}</b><br>"
            f"Методы API: <b>{int(summary.get('method_count') or 0)}</b><br>"
            "Набор признаков: "
            f"<b>{_безопасный_текст(summary.get('feature_mode') or 'Ручная настройка')}</b><br>"
            f"ML-признаков: <b>{int(summary.get('feature_count') or 0)}</b><br>"
            "Дополнительных полей API: "
            f"<b>{int(summary.get('field_count') or 0)}</b><br>"
            f"Target: <b>{_безопасный_текст(target_text)}</b><br>"
            f"Версия профиля: <b>{int(payload.get('revision') or 1)}</b><br>"
            "Текущие настройки пока не изменены."
            "</div>"
        )
        _profile_apply_button.disabled = False

    def _применить_выбранный_профиль(_=None) -> None:
        """Применяет профиль и показывает точный следующий шаг."""

        selected_path = str(
            _profile_dropdown.value or ""
        )
        _profile_apply_button.disabled = True

        with _profile_output:
            clear_output(wait=True)

            try:
                if not selected_path:
                    raise ValueError(
                        "Сначала выберите конфигурацию в списке."
                    )

                payload = _загрузить_профиль(selected_path)
                restored_workflow = _применить_профиль(payload)
                _workflow.clear()
                _workflow.update(restored_workflow)
                _catalog["workflow"] = restored_workflow
                summary = payload.get("summary") or {}

                try:
                    from src import runtime_readiness

                    target_runtime_ready = (
                        runtime_readiness._target_runtime_ready()
                    )
                    detail_runtime_ready, _ = (
                        runtime_readiness._detail_runtime_ready(
                            list(
                                (
                                    restored_workflow.get("methods")
                                    or {}
                                ).get("selected_endpoints")
                                or []
                            )
                        )
                    )
                except Exception:
                    target_runtime_ready = False
                    detail_runtime_ready = False

                next_actions = []

                if not target_runtime_ready:
                    next_actions.append(
                        "повторно выполните пункты 2, 2.1 и 3; "
                        "сохранённую конфигурацию повторно применять не нужно"
                    )
                elif not detail_runtime_ready:
                    next_actions.append(
                        "выполните пункт 2.1"
                    )

                if next_actions:
                    next_text = (
                        "После применения "
                        + "; затем ".join(next_actions)
                        + ". Пункты 5.2 и 5.3 повторять не нужно. "
                        "После этого переходите к пункту 6."
                    )
                else:
                    next_text = (
                        "Все служебные обработчики уже подключены. "
                        "Переходите к пункту 6."
                    )

                _profile_preview.value = (
                    "<div style='padding:12px 14px;background:#e8f5e9;"
                    "border-left:5px solid #2e7d32;line-height:1.65'>"
                    "<b>Полная конфигурация применена</b><br>"
                    f"Название: <b>{_безопасный_текст(payload.get('profile_name'))}</b><br>"
                    f"Методы API: <b>{int(summary.get('method_count') or 0)}</b><br>"
                    "Набор признаков: "
                    f"<b>{_безопасный_текст(summary.get('feature_mode') or 'Ручная настройка')}</b><br>"
                    f"ML-признаков: <b>{int(summary.get('feature_count') or 0)}</b><br>"
                    "Дополнительных полей API: "
                    f"<b>{int(summary.get('field_count') or 0)}</b><br>"
                    f"{_безопасный_текст(next_text)}"
                    "</div>"
                )
                print("ПОЛНАЯ КОНФИГУРАЦИЯ ПРИМЕНЕНА")
                print(
                    "Название: "
                    f"{payload.get('profile_name')}"
                )
                print(
                    "Методов API: "
                    f"{int(summary.get('method_count') or 0)}"
                )
                print(
                    "Набор признаков: "
                    f"{summary.get('feature_mode') or 'Ручная настройка'}"
                )
                print(
                    "ML-признаков: "
                    f"{int(summary.get('feature_count') or 0)}"
                )
                print(
                    "Дополнительных полей API: "
                    f"{int(summary.get('field_count') or 0)}"
                )
                print()
                print(next_text)
                _manual_panel.layout.display = "none"

            except Exception as error:
                show_error(
                    stage="Открытие полной конфигурации",
                    error=error,
                    action=(
                        "Выберите другой профиль или обновите список. "
                        "Текущие настройки и сохранённые файлы не "
                        "изменялись."
                    ),
                )
            finally:
                _profile_apply_button.disabled = not bool(
                    _profile_dropdown.value
                )

    _button_open_profile.on_click(_показать_профили)
    _button_manual.on_click(_показать_ручной_выбор)
    _profile_refresh_button.on_click(_обновить_профили)
    _profile_dropdown.observe(
        _показать_сводку_профиля,
        names="value",
    )
    _profile_apply_button.on_click(
        _применить_выбранный_профиль
    )

    _interface = widgets.VBox(
        [
            widgets.HTML(
                "<h2>Методы API и сохранённые конфигурации</h2>"
                "<p><b>Открыть сохранённую конфигурацию</b> — "
                "восстановить пункты 4–5.3 целиком. "
                "<b>Настроить методы вручную</b> — создать новый "
                "набор методов и продолжить обычную настройку.</p>"
            ),
            widgets.HBox(
                [_button_open_profile, _button_manual],
                layout=widgets.Layout(
                    gap="10px",
                    flex_flow="row wrap",
                ),
            ),
            _mode_hint,
            _profile_panel,
            _manual_panel,
        ],
        layout=widgets.Layout(
            width="100%",
            max_width="1150px",
        ),
    )

    display(_interface)

    METHOD_STEP = {
        "success": True,
        "workflow": _workflow,
        "catalog_result": _catalog["catalog_result"],
        "interface": _interface,
    }

except Exception as _cell_error:
    METHOD_STEP = {
        "success": False,
        "error": str(_cell_error),
    }

    _action = _следующий_пункт(_cell_error)

    try:
        _log_path = _write_traceback(
            project_root=globals().get("PROJECT_ROOT"),
            stage="Выбор методов",
            error=_cell_error,
        )
    except Exception:
        _log_path = None

    if _log_path:
        _action += (
            " Технический журнал сохранён: "
            f"{_log_path}"
        )

    try:
        show_error(
            stage="Выбор методов",
            error=_понятная_причина(_cell_error),
            action=_action,
        )
    except Exception:
        print("Операция не завершена")
        print("Этап: Выбор методов")
        print(
            "Причина: "
            f"{_понятная_причина(_cell_error)}"
        )
        print(f"Что сделать: {_action}")
        print(
            "Ранее сохранённые файлы не изменялись."
        )


## 5. Настройка датасета

### 5.1. Целевая переменная

В этой ячейке задаётся правило, по которому программа сформирует столбец **`target_default`** — целевую переменную для обучения модели дефолта.

> **Что сделать:** выберите правило дефолта, горизонт прогноза и период истории, затем нажмите **«Сохранить правило цели»**.  
> Текущая базовая настройка: **строгий юридический дефолт, горизонт 12 месяцев, последние 3 завершённых года**. Меняйте её только при наличии другого согласованного определения дефолта.

Эта ячейка **не отправляет запросы в СПАРК**. Сохранённое правило будет автоматически применено при проверке одной компании и при массовой выгрузке.

<details>
<summary><b>Техническая справка: как рассчитывается target_default</b></summary>

**Что является объектом наблюдения**

Одна строка датасета соответствует сочетанию:

- организация;
- отчётный год;
- дата оценки;
- горизонт наблюдения.

Дата оценки зафиксирована как **1 июня года, следующего за отчётным**.

Например, для отчётности за **2024 год**:

- отчётный год — 2024;
- дата оценки — 1 июня 2025 года;
- при горизонте 12 месяцев период наблюдения заканчивается 1 июня 2026 года;
- после завершения этого периода можно определить `target_default`.

Последний полностью наблюдаемый отчётный год программа рассчитывает автоматически с учётом текущей даты и выбранного горизонта.

**Значения целевой переменной**

| Значение | Что означает |
|---|---|
| `1` | Выбранное событие дефолта наступило после даты оценки и до окончания горизонта включительно |
| `0` | Горизонт полностью завершён, необходимые источники проверены, событие дефолта не обнаружено |
| пусто | Горизонт ещё не завершён, данных недостаточно либо компания уже находилась в состоянии дефолта или прекратила деятельность до даты оценки |

Пустой результат **никогда не заменяется нулём**. Это защищает модель от ложного вывода, что отсутствие данных означает отсутствие дефолта.

Строки с пустым `target_default` сохраняются для контроля, но не включаются в готовую ML-матрицу для обучения.

**Доступные правила**

| Правило | Какие события учитываются |
|---|---|
| **Строгий юридический дефолт** | Введена подтверждённая процедура банкротства |
| **Расширенный финансовый дефолт** | Банкротство, ликвидация вследствие несостоятельности, подтверждённая длительная просрочка и новое активное исполнительное производство |
| **Ручная настройка** | Аналитик самостоятельно выбирает события, пороги и условие их объединения |

В строгом правиле простое принятие заявления о банкротстве не считается дефолтом. Учитывается подтверждённая процедура: наблюдение, финансовое оздоровление, внешнее управление либо признание должника банкротом.

Обычная добровольная ликвидация также не считается дефолтом.

**Горизонт прогноза**

Можно выбрать горизонт:

- 6 месяцев;
- 12 месяцев;
- 18 месяцев.

Горизонт отсчитывается от даты оценки. Чем длиннее горизонт, тем более ранний отчётный год потребуется для полностью наблюдаемой целевой переменной.

**Объединение событий**

В ручном режиме доступны два варианта:

- **ИЛИ** — для `target_default = 1` достаточно одного выбранного события;
- **И** — должны наступить все выбранные события.

В готовых правилах состав событий зафиксирован. Для изменения отдельных событий выберите режим **«Ручная настройка»**.

**Пороговые значения**

Пороги используются только для соответствующих событий:

- **Просрочка от 90 или 120 дней** — минимальная подтверждённая продолжительность просрочки;
- **Производств от** — минимальное количество новых активных исполнительных производств;
- **Общая сумма от** — минимальная совокупная сумма таких производств.

Если минимальная сумма равна нулю, решение принимается по количеству производств.

**Откуда берутся данные для target**

Программа автоматически подключает необходимые служебные источники СПАРК:

| Событие | Источник |
|---|---|
| Банкротство и ликвидация вследствие несостоятельности | Краткая справка и история статуса организации |
| Длительная просрочка | Данные о платёжной дисциплине |
| Исполнительные производства | Реестр исполнительных производств компании |

Эти источники подключаются технически и **не изменяют пользовательский набор методов**, сохранённый в пункте 4.

Если источник не содержит достаточной истории или не покрывает весь горизонт, программа оставит `target_default` пустым, а причину запишет в диагностические поля.

**Защита от утечки целевой переменной**

Сведения, по которым определяется дефолт, не должны одновременно использоваться как признаки модели после даты оценки.

Поэтому программа:

- использует для признаков только сведения, относящиеся к дате оценки или более раннему периоду;
- использует последующие события только для расчёта `target_default`;
- исключает поля, раскрывающие будущий дефолт, из листа **«ML-матрица»**;
- сохраняет диагностические поля `target_*` в основной выгрузке для проверки расчёта.

В готовой ML-матрице `target_default` располагается первым, а технические идентификаторы и диагностические поля не используются как признаки модели.

**Ежегодное накопление истории**

При каждом ежегодном массовом запуске программа сохраняет новый неизменяемый снимок данных на подключённом Google Диске.

Настройка истории определяет, какие совместимые снимки войдут в обучающую выборку:

- последние 3 завершённых года;
- последние 5 завершённых лет;
- вся накопленная история.

Выбор трёх или пяти лет **не создаёт исторические признаки задним числом**. При первом запуске в истории может находиться только один реально собранный год. Следующие годы будут добавляться при последующих ежегодных запусках.

Старые признаки не пересчитываются по современным данным. Для ранее сохранённых строк программа может дозаполнить только `target_default`, когда соответствующий горизонт полностью завершится.

Если изменить правило target или состав признаков, программа создаст отдельную совместимую серию истории и не смешает разные определения датасета.

**Что сохраняет эта ячейка**

После нажатия **«Сохранить правило цели»** программа:

1. проверяет выбранные события и пороги;
2. определяет последний полностью наблюдаемый отчётный год;
3. сохраняет JSON-профиль правила на Google Диске;
4. подключает необходимые источники target к техническому плану;
5. передаёт правило в тестовый и массовый конвейер.

Фактический расчёт `target_default` выполняется позже — при проверке одной компании и при массовой обработке списка организаций.

Если вы изменили правило, сохраните его заново и повторно выполните все последующие ячейки, начиная с пункта 5.2.

</details>


In [ ]:
# @title 5.1. Настроить целевую переменную

from __future__ import annotations

try:
    from src.notebook_runtime import run_notebook_cell
except Exception as _cell_runtime_error:
    print(
        "Не удалось запустить этап «Настройка target». "
        "Сначала выполните пункт 2. "
        f"Причина: {type(_cell_runtime_error).__name__}: {_cell_runtime_error}"
    )
else:
    with run_notebook_cell(
        stage="Настройка target",
        next_step="Повторите пункт 5.1.",
        project_root=globals().get("PROJECT_ROOT"),
    ):
        import hashlib
        import json
        import re
        import shutil
        from calendar import monthrange
        from collections.abc import Mapping
        from datetime import date, datetime, timezone
        from pathlib import Path

        import ipywidgets as widgets
        from IPython.display import clear_output, display

        import src.workflow as workflow_module


        # Интерфейс сохраняет правило target. Штатное ядро проекта
        # автоматически использует его в тестовом и массовом конвейере.
        # Эта ячейка запросы к СПАРК и обработку организаций не выполняет.
        TARGET_CONFIG_READY = False
        TARGET_PROFILE_SAVED = False
        TARGET_CONFIG = None
        TARGET_PROFILE_PATH = None


        TARGET_MODE_OPTIONS = (
            (
                "Строгий юридический дефолт",
                "strict_legal_default",
            ),
            (
                "Расширенный финансовый дефолт",
                "broad_financial_default",
            ),
            (
                "Ручная настройка",
                "custom",
            ),
        )

        TARGET_MODE_LABELS = dict(
            (value, label)
            for label, value in TARGET_MODE_OPTIONS
        )

        TARGET_CALCULATOR_VERSION = "spark_target_calculator_v2"

        EVENT_DEFINITIONS = {
            "bankruptcy_procedure": {
                "label": "Введена подтверждённая процедура банкротства",
                "description": (
                    "Наблюдение, финансовое оздоровление, внешнее "
                    "управление либо признание должника банкротом. "
                    "Простое принятие заявления не учитывается."
                ),
                "required_endpoints": [
                    "/rest/GetCompanyShortReport",
                ],
                "coverage": "public_event_history",
            },
            "insolvency_liquidation": {
                "label": "Ликвидация вследствие подтверждённой несостоятельности",
                "description": (
                    "Обычная добровольная ликвидация дефолтом не считается."
                ),
                "required_endpoints": [
                    "/rest/GetCompanyShortReport",
                ],
                "coverage": "status_history_when_reason_is_known",
            },
            "payment_overdue": {
                "label": "Подтверждённая длительная просрочка платежей",
                "description": (
                    "Используется только при наличии достаточных данных "
                    "СПАРК о платёжной дисциплине. Отсутствие данных не "
                    "приравнивается к отсутствию просрочки."
                ),
                "required_endpoints": [
                    "/rest/GetCompanyPaymentDiscipline",
                ],
                "coverage": "limited_payment_monitoring",
            },
            "execution_proceeding": {
                "label": "Новое активное исполнительное производство",
                "description": (
                    "Событие учитывается только после применения заданного "
                    "порога по количеству и сумме."
                ),
                "required_endpoints": [
                    "/rest/GetCompanyExecutionProceedings",
                ],
                "coverage": "dated_public_event_history",
            },
        }

        MODE_EVENTS = {
            "strict_legal_default": (
                "bankruptcy_procedure",
            ),
            "broad_financial_default": (
                "bankruptcy_procedure",
                "insolvency_liquidation",
                "payment_overdue",
                "execution_proceeding",
            ),
        }


        def _add_months(value: date, months: int) -> date:
            """Добавляет к дате целое число календарных месяцев."""

            month_index = value.month - 1 + int(months)
            year = value.year + month_index // 12
            month = month_index % 12 + 1
            day = min(value.day, monthrange(year, month)[1])
            return date(year, month, day)


        def _latest_matured_report_year(
            *,
            as_of_date: date,
            horizon_months: int,
            observation_month: int = 6,
            observation_day: int = 1,
        ) -> int:
            """Возвращает последний год отчётности с завершённым горизонтом."""

            candidate = as_of_date.year - 1

            while candidate >= 1900:
                observation_date = date(
                    candidate + 1,
                    observation_month,
                    observation_day,
                )
                horizon_end = _add_months(
                    observation_date,
                    horizon_months,
                )

                if horizon_end <= as_of_date:
                    return candidate

                candidate -= 1

            raise ValueError(
                "Не удалось определить завершённый отчётный период."
            )


        def _safe_profile_name(value: str) -> str:
            """Возвращает безопасное имя файла профиля."""

            normalized = re.sub(
                r"[^0-9A-Za-zА-Яа-яЁё_-]+",
                "_",
                str(value).strip(),
            ).strip("_-. ")

            if not normalized:
                raise ValueError(
                    "Введите название правила целевой переменной."
                )

            return normalized[:100]


        def _canonical_json(value: dict) -> str:
            """Сериализует конфигурацию в стабильном порядке."""

            return json.dumps(
                value,
                ensure_ascii=False,
                sort_keys=True,
                separators=(",", ":"),
            )


        def _config_hash(value: dict) -> str:
            """Возвращает отпечаток настроек, влияющих на датасет."""

            return hashlib.sha256(
                _canonical_json(value).encode("utf-8")
            ).hexdigest()


        def _target_rule_hash(value: Mapping) -> str:
            """Хеширует только смысл target, без даты запуска и окна экспорта."""

            semantic_rule = {
                "format": value.get("format"),
                "calculator_version": value.get(
                    "calculator_version",
                    TARGET_CALCULATOR_VERSION,
                ),
                "target_column": value.get("target_column"),
                "target_mode": value.get("target_mode"),
                "forecast_horizon_months": value.get(
                    "forecast_horizon_months"
                ),
                "observation_rule": value.get("observation_rule"),
                "combination_operator": value.get(
                    "combination_operator"
                ),
                "selected_events": list(
                    value.get("selected_events") or []
                ),
                "thresholds": value.get("thresholds") or {},
                "label_policy": value.get("label_policy") or {},
            }
            return _config_hash(semantic_rule)


        def _project_root() -> Path:
            """Получает личный каталог проекта из предыдущих ячеек."""

            root = globals().get("PROJECT_ROOT")

            if root is None:
                raise RuntimeError(
                    "Рабочий каталог проекта не найден. "
                    "Сначала выполните подключение Google Диска "
                    "и техническую подготовку."
                )

            root = Path(root)

            if not root.exists():
                raise RuntimeError(
                    "Рабочий каталог проекта недоступен: "
                    f"{root}"
                )

            return root


        def _selected_event_codes() -> list[str]:
            """Возвращает отмеченные события в стабильном порядке."""

            return [
                event_code
                for event_code in EVENT_DEFINITIONS
                if EVENT_CHECKBOXES[event_code].value
            ]


        def _required_endpoints(
            event_codes: list[str],
        ) -> list[str]:
            """Возвращает методы СПАРК, необходимые выбранному правилу."""

            result = []

            for event_code in event_codes:
                for endpoint in EVENT_DEFINITIONS[
                    event_code
                ]["required_endpoints"]:
                    if endpoint not in result:
                        result.append(endpoint)

            return result


        def _apply_mode_template(*_):
            """Применяет готовый профиль либо открывает ручной выбор."""

            mode = TARGET_MODE.value
            template_events = set(
                MODE_EVENTS.get(mode, ())
            )
            is_custom = mode == "custom"

            for event_code, checkbox in (
                EVENT_CHECKBOXES.items()
            ):
                if not is_custom:
                    checkbox.value = (
                        event_code in template_events
                    )
                checkbox.disabled = not is_custom

            COMBINATION_OPERATOR.disabled = not is_custom

            if not is_custom:
                COMBINATION_OPERATOR.value = "OR"

            _refresh_threshold_state()
            _refresh_preview()


        def _refresh_threshold_state(*_):
            """Показывает активность порогов выбранных событий."""

            PAYMENT_OVERDUE_DAYS.disabled = not (
                EVENT_CHECKBOXES[
                    "payment_overdue"
                ].value
            )
            EXECUTION_MIN_COUNT.disabled = not (
                EVENT_CHECKBOXES[
                    "execution_proceeding"
                ].value
            )
            EXECUTION_MIN_AMOUNT_RUB.disabled = not (
                EVENT_CHECKBOXES[
                    "execution_proceeding"
                ].value
            )


        def _build_target_config() -> dict:
            """Формирует и проверяет полную конфигурацию target."""

            event_codes = _selected_event_codes()

            if not event_codes:
                raise ValueError(
                    "Выберите хотя бы одно событие дефолта."
                )

            horizon_months = int(
                FORECAST_HORIZON.value
            )

            if horizon_months not in {6, 12, 18}:
                raise ValueError(
                    "Допустимый горизонт: 6, 12 или 18 месяцев."
                )

            history_window_years = int(
                TRAINING_HISTORY_WINDOW.value
            )

            if history_window_years not in {0, 3, 5}:
                raise ValueError(
                    "Допустимый период обучения: 3 года, 5 лет "
                    "или вся накопленная история."
                )

            today = datetime.now(timezone.utc).date()
            latest_report_year = (
                _latest_matured_report_year(
                    as_of_date=today,
                    horizon_months=horizon_months,
                )
            )

            target_core = {
                "format": "spark_target_profile_v1",
                "calculator_version": TARGET_CALCULATOR_VERSION,
                "profile_name": str(
                    TARGET_PROFILE_NAME.value
                ).strip(),
                "target_column": "target_default",
                "target_mode": TARGET_MODE.value,
                "target_mode_label": (
                    TARGET_MODE_LABELS[
                        TARGET_MODE.value
                    ]
                ),
                "forecast_horizon_months": (
                    horizon_months
                ),
                "observation_rule": {
                    "type": "fixed_after_report_year",
                    "month": 6,
                    "day": 1,
                    "description": (
                        "1 июня года, следующего за отчётным"
                    ),
                },
                "combination_operator": (
                    COMBINATION_OPERATOR.value
                ),
                "selected_events": event_codes,
                "event_rules": {
                    event_code: {
                        "label": EVENT_DEFINITIONS[
                            event_code
                        ]["label"],
                        "coverage": EVENT_DEFINITIONS[
                            event_code
                        ]["coverage"],
                    }
                    for event_code in event_codes
                },
                "thresholds": {
                    "payment_overdue_days": int(
                        PAYMENT_OVERDUE_DAYS.value
                    ),
                    "execution_min_count": int(
                        EXECUTION_MIN_COUNT.value
                    ),
                    "execution_min_amount_rub": float(
                        EXECUTION_MIN_AMOUNT_RUB.value
                    ),
                },
                "label_policy": {
                    "positive_value": 1,
                    "negative_value": 0,
                    "unknown_value": None,
                    "positive_rule": (
                        "Выбранное сочетание событий произошло "
                        "после даты оценки и не позднее конца горизонта."
                    ),
                    "negative_rule": (
                        "Полный горизонт завершён, необходимые источники "
                        "проверены и выбранное сочетание событий не произошло."
                    ),
                    "unknown_rule": (
                        "Незавершённый горизонт или недостаточное покрытие "
                        "источников не подменяется нулём."
                    ),
                },
                "training_history": {
                    "update_frequency": "annual",
                    "window_mode": (
                        "all_history"
                        if history_window_years == 0
                        else "rolling"
                    ),
                    "window_years": (
                        None
                        if history_window_years == 0
                        else history_window_years
                    ),
                    "use_only_matured_observations": True,
                    "save_annual_snapshots": True,
                    "latest_matured_report_year_at_save": (
                        latest_report_year
                    ),
                    "as_of_date": today.isoformat(),
                },
                "required_endpoints": (
                    _required_endpoints(event_codes)
                ),
                "leakage_policy": {
                    "exclude_target_source_fields_from_ml": True,
                    "exclude_events_after_observation_from_features": True,
                    "require_feature_date_not_after_observation": True,
                },
            }

            target_core["target_rule_hash"] = (
                _target_rule_hash(target_core)
            )
            hash_payload = dict(target_core)
            hash_payload["training_history"] = dict(
                target_core["training_history"]
            )
            hash_payload["training_history"].pop(
                "as_of_date",
                None,
            )
            hash_payload["training_history"].pop(
                "latest_matured_report_year_at_save",
                None,
            )
            target_core["configuration_hash"] = _config_hash(
                hash_payload
            )

            return target_core


        def _refresh_preview(*_):
            """Обновляет краткое описание текущих настроек."""

            try:
                event_codes = _selected_event_codes()
                horizon_months = int(
                    FORECAST_HORIZON.value
                )
                today = datetime.now(
                    timezone.utc
                ).date()
                latest_year = (
                    _latest_matured_report_year(
                        as_of_date=today,
                        horizon_months=horizon_months,
                    )
                )
                event_labels = [
                    EVENT_DEFINITIONS[event_code][
                        "label"
                    ]
                    for event_code in event_codes
                ]
                operator_label = (
                    "И — должны выполниться все выбранные события"
                    if COMBINATION_OPERATOR.value == "AND"
                    else "ИЛИ — достаточно одного выбранного события"
                )
                history_years = int(TRAINING_HISTORY_WINDOW.value)
                history_label = (
                    "вся накопленная история"
                    if history_years == 0
                    else (
                        f"последние {history_years} завершённых "
                        + ("года" if history_years == 3 else "лет")
                    )
                )

                event_text = (
                    "<br>".join(
                        f"• {label}"
                        for label in event_labels
                    )
                    if event_labels
                    else "События пока не выбраны"
                )

                PREVIEW.value = (
                    "<div style='padding:12px;border-left:4px solid #4285f4;"
                    "background:#eef5ff'>"
                    "<b>Текущее правило</b><br>"
                    f"Дата оценки: 1 июня после отчётного года<br>"
                    f"Горизонт: {horizon_months} мес.<br>"
                    f"Последний полностью наблюдаемый отчётный год "
                    f"на {today:%d.%m.%Y}: <b>{latest_year}</b><br><br>"
                    f"Объединение: {operator_label}<br>"
                    f"История обучения: {history_label}<br><br>"
                    f"{event_text}"
                    "</div>"
                )

            except Exception as error:
                PREVIEW.value = (
                    "<div style='color:#b3261e'>"
                    "Не удалось сформировать предварительную сводку: "
                    f"{type(error).__name__}: {error}"
                    "</div>"
                )


        def _save_target_profile(_):
            """Проверяет и сохраняет активное правило target."""

            global TARGET_CONFIG_READY
            global TARGET_PROFILE_SAVED
            global TARGET_CONFIG
            global TARGET_PROFILE_PATH

            with SAVE_OUTPUT:
                clear_output()

                try:
                    root = _project_root()
                    config = _build_target_config()
                    workflow = workflow_module.load_workflow(
                        workflow_root=(
                            root / "workflow"
                        )
                    )
                    workflow_module.require_stages(
                        workflow,
                        "bootstrap",
                        "authentication",
                        "methods",
                    )
                    safe_name = _safe_profile_name(
                        config["profile_name"]
                    )

                    profile_root = (
                        root / "target_profiles"
                    )
                    history_root = (
                        profile_root / "history"
                    )
                    profile_root.mkdir(
                        parents=True,
                        exist_ok=True,
                    )
                    history_root.mkdir(
                        parents=True,
                        exist_ok=True,
                    )

                    profile_path = (
                        profile_root
                        / f"{safe_name}.json"
                    )

                    if profile_path.exists():
                        timestamp = datetime.now(
                            timezone.utc
                        ).strftime("%Y%m%d_%H%M%S")
                        shutil.copy2(
                            profile_path,
                            history_root
                            / f"{safe_name}_{timestamp}.json",
                        )

                    config_to_save = dict(config)
                    config_to_save["saved_at_utc"] = (
                        datetime.now(timezone.utc)
                        .isoformat(timespec="seconds")
                    )

                    temporary_path = (
                        profile_path.with_suffix(
                            ".json.tmp"
                        )
                    )
                    temporary_path.write_text(
                        json.dumps(
                            config_to_save,
                            ensure_ascii=False,
                            indent=2,
                        ),
                        encoding="utf-8",
                    )
                    temporary_path.replace(profile_path)

                    active_path = (
                        profile_root
                        / "active_target_profile.json"
                    )
                    active_temporary_path = (
                        active_path.with_suffix(
                            ".json.tmp"
                        )
                    )
                    active_temporary_path.write_text(
                        json.dumps(
                            config_to_save,
                            ensure_ascii=False,
                            indent=2,
                        ),
                        encoding="utf-8",
                    )
                    active_temporary_path.replace(
                        active_path
                    )

                    previous_target_hash = str(
                        (
                            workflow.get("target")
                            or {}
                        ).get(
                            "configuration_hash"
                        )
                        or ""
                    )
                    current_target_hash = str(
                        config_to_save[
                            "configuration_hash"
                        ]
                    )
                    workflow = (
                        workflow_module.update_workflow(
                            workflow,
                            target=config_to_save,
                            test={
                                **(
                                    workflow.get("test")
                                    or {}
                                ),
                                "required": True,
                                "completed": False,
                                "company_code": None,
                                "configuration_hash": None,
                                "summary": {},
                                "output_directory": None,
                            },
                        )
                    )

                    if (
                        previous_target_hash
                        != current_target_hash
                    ):
                        workflow = (
                            workflow_module.invalidate_after(
                                workflow,
                                "methods",
                                reason=(
                                    "Правило целевой переменной "
                                    "изменено. Повторно сохраните "
                                    "поля и признаки."
                                ),
                            )
                        )

                    workflow_module.save_workflow(
                        workflow,
                        workflow_root=(
                            root / "workflow"
                        ),
                    )

                    TARGET_CONFIG = config_to_save
                    TARGET_PROFILE_PATH = str(
                        profile_path
                    )
                    TARGET_CONFIG_READY = True
                    TARGET_PROFILE_SAVED = True

                    selected_labels = [
                        EVENT_DEFINITIONS[event_code][
                            "label"
                        ]
                        for event_code in config[
                            "selected_events"
                        ]
                    ]

                    print(
                        "ПРАВИЛО ЦЕЛЕВОЙ ПЕРЕМЕННОЙ СОХРАНЕНО"
                    )
                    print(
                        f"Столбец результата: "
                        f"{config['target_column']}"
                    )
                    print(
                        "Основное правило: "
                        f"{config['target_mode_label']}"
                    )
                    print(
                        "Дата оценки: 1 июня года, "
                        "следующего за отчётным"
                    )
                    print(
                        "Горизонт прогноза: "
                        f"{config['forecast_horizon_months']} мес."
                    )
                    print(
                        "Последний завершённый отчётный год: "
                        f"{config['training_history']['latest_matured_report_year_at_save']}"
                    )
                    print("События:")

                    for label in selected_labels:
                        print(f"- {label}")

                    print(
                        "Объединение событий: "
                        + (
                            "должны выполниться все выбранные события (И)"
                            if config["combination_operator"] == "AND"
                            else "достаточно одного выбранного события (ИЛИ)"
                        )
                    )
                    history_years = config["training_history"].get(
                        "window_years"
                    )
                    print(
                        "История обучения: "
                        + (
                            "вся накопленная история"
                            if history_years is None
                            else (
                                f"последние {history_years} завершённых "
                                + ("года" if history_years == 3 else "лет")
                            )
                        )
                    )

                    if "payment_overdue" in config["selected_events"]:
                        print(
                            "Порог просрочки: от "
                            f"{config['thresholds']['payment_overdue_days']} дней"
                        )

                    if "execution_proceeding" in config["selected_events"]:
                        print(
                            "Порог исполнительных производств: от "
                            f"{config['thresholds']['execution_min_count']} шт. "
                            "и от "
                            f"{config['thresholds']['execution_min_amount_rub']:,.2f} руб."
                        )

                    print(
                        "Неизвестный результат не будет "
                        "подменяться нулём."
                    )
                    print(
                        "Поля, формирующие target, будут исключены "
                        "из признаков модели."
                    )
                    print(
                        "Источники target будут подключены программой "
                        "как служебные и не изменят ваш выбор в пункте 4."
                    )
                    print(f"Файл: {profile_path}")
                    print(
                        "В этой ячейке запросы к СПАРК "
                        "не выполнялись."
                    )
                    print(
                        "Следующий шаг: подготовьте параметры методов "
                        "в пункте 5.2."
                    )

                except Exception as error:
                    TARGET_CONFIG_READY = False
                    TARGET_PROFILE_SAVED = False
                    TARGET_CONFIG = None
                    TARGET_PROFILE_PATH = None

                    print(
                        "ПРАВИЛО ЦЕЛЕВОЙ ПЕРЕМЕННОЙ НЕ СОХРАНЕНО"
                    )
                    print(
                        f"Причина: {type(error).__name__}: {error}"
                    )
                    print(
                        "Что сделать: исправьте отмеченные настройки "
                        "и снова нажмите «Сохранить правило цели»."
                    )
                    print(
                        "Ранее сохранённые профили не изменялись."
                    )


        # ------------------------------------------------------------

        # Производственная логика target теперь находится в ядре проекта.
        # Ячейка 5.1 отвечает только за интерфейс и сохранение правила.
        from src.target_runtime import (
            TARGET_RUNTIME_VERSION,
            target_runtime_state,
        )

        TARGET_RUNTIME_STATE = target_runtime_state()
        if (
            not TARGET_RUNTIME_STATE.get("ready")
            or TARGET_RUNTIME_STATE.get("version")
            != TARGET_RUNTIME_VERSION
        ):
            raise RuntimeError(
                "Штатный обработчик target не загружен. "
                "Повторите пункт 2 и снова откройте пункт 5.1."
            )


        TARGET_PROFILE_NAME = widgets.Text(
            value="Строгий дефолт 12 месяцев",
            description="Название:",
            layout=widgets.Layout(width="760px"),
            style={"description_width": "190px"},
        )

        TARGET_MODE = widgets.Dropdown(
            options=TARGET_MODE_OPTIONS,
            value="strict_legal_default",
            description="Правило:",
            layout=widgets.Layout(width="760px"),
            style={"description_width": "190px"},
        )

        FORECAST_HORIZON = widgets.Dropdown(
            options=(
                ("6 месяцев", 6),
                ("12 месяцев", 12),
                ("18 месяцев", 18),
            ),
            value=12,
            description="Горизонт:",
            layout=widgets.Layout(width="500px"),
            style={"description_width": "190px"},
        )

        COMBINATION_OPERATOR = widgets.Dropdown(
            options=(
                ("Достаточно одного события — ИЛИ", "OR"),
                ("Необходимы все события — И", "AND"),
            ),
            value="OR",
            description="Объединение:",
            layout=widgets.Layout(width="760px"),
            style={"description_width": "190px"},
        )

        EVENT_CHECKBOXES = {
            event_code: widgets.Checkbox(
                value=False,
                description=definition["label"],
                indent=False,
                layout=widgets.Layout(
                    width="100%",
                ),
            )
            for event_code, definition
            in EVENT_DEFINITIONS.items()
        }

        EVENT_ROWS = []

        for event_code, definition in (
            EVENT_DEFINITIONS.items()
        ):
            EVENT_ROWS.append(
                widgets.VBox(
                    [
                        EVENT_CHECKBOXES[event_code],
                        widgets.HTML(
                            value=(
                                "<div style='margin:0 0 8px 28px;"
                                "color:#5f6368;font-size:13px'>"
                                f"{definition['description']}"
                                "</div>"
                            )
                        ),
                    ]
                )
            )

        PAYMENT_OVERDUE_DAYS = widgets.Dropdown(
            options=(
                ("90 дней", 90),
                ("120 дней", 120),
            ),
            value=90,
            description="Просрочка от:",
            layout=widgets.Layout(width="500px"),
            style={"description_width": "190px"},
        )

        EXECUTION_MIN_COUNT = widgets.BoundedIntText(
            value=1,
            min=1,
            max=100000,
            step=1,
            description="Производств от:",
            layout=widgets.Layout(width="500px"),
            style={"description_width": "190px"},
        )

        EXECUTION_MIN_AMOUNT_RUB = widgets.BoundedFloatText(
            value=0.0,
            min=0.0,
            max=1_000_000_000_000_000.0,
            step=100_000.0,
            description="Общая сумма от, руб.:",
            layout=widgets.Layout(width="500px"),
            style={"description_width": "190px"},
        )

        TRAINING_HISTORY_WINDOW = widgets.Dropdown(
            options=(
                (
                    "Последние 3 завершённых года",
                    3,
                ),
                (
                    "Последние 5 завершённых лет",
                    5,
                ),
                (
                    "Вся накопленная история",
                    0,
                ),
            ),
            value=3,
            description="История обучения:",
            layout=widgets.Layout(width="760px"),
            style={"description_width": "190px"},
        )

        RETRAINING_BOX = widgets.VBox(
            [
                widgets.HTML(
                    value=(
                        "<p>При ежегодном обновлении программа будет "
                        "добавлять новый зафиксированный срез, дозаполнять "
                        "target только для завершённых горизонтов и формировать "
                        "обучающую выборку за выбранный период. Старые файлы "
                        "не перезаписываются.</p>"
                    )
                ),
                TRAINING_HISTORY_WINDOW,
                widgets.HTML(
                    value=(
                        "<p style='color:#5f6368'>Основной режим — скользящее "
                        "окно последних трёх лет. Частота обновления: один раз "
                        "в год. Ежегодные снимки сохраняются автоматически.</p>"
                    )
                ),
            ]
        )

        RETRAINING_ACCORDION = widgets.Accordion(
            children=[RETRAINING_BOX]
        )
        RETRAINING_ACCORDION.set_title(
            0,
            "Настройки ежегодного обновления модели",
        )
        RETRAINING_ACCORDION.selected_index = None

        PREVIEW = widgets.HTML()

        SAVE_BUTTON = widgets.Button(
            description="Сохранить правило цели",
            button_style="success",
            icon="save",
            layout=widgets.Layout(width="270px"),
        )

        SAVE_OUTPUT = widgets.Output()

        TARGET_MODE.observe(
            _apply_mode_template,
            names="value",
        )
        FORECAST_HORIZON.observe(
            _refresh_preview,
            names="value",
        )
        COMBINATION_OPERATOR.observe(
            _refresh_preview,
            names="value",
        )
        TRAINING_HISTORY_WINDOW.observe(
            _refresh_preview,
            names="value",
        )
        PAYMENT_OVERDUE_DAYS.observe(
            _refresh_preview,
            names="value",
        )
        EXECUTION_MIN_COUNT.observe(
            _refresh_preview,
            names="value",
        )
        EXECUTION_MIN_AMOUNT_RUB.observe(
            _refresh_preview,
            names="value",
        )

        for checkbox in EVENT_CHECKBOXES.values():
            checkbox.observe(
                _refresh_threshold_state,
                names="value",
            )
            checkbox.observe(
                _refresh_preview,
                names="value",
            )

        SAVE_BUTTON.on_click(
            _save_target_profile
        )

        _apply_mode_template()

        TARGET_INTERFACE = widgets.VBox(
            [
                widgets.HTML(
                    value=(
                        "<h3>Целевая переменная для модели дефолта</h3>"
                        "<p>Выберите правило, по которому после массового "
                        "сбора будет рассчитан столбец <b>target_default</b>. "
                        "Эта настройка не является обычным признаком и не "
                        "попадёт в список галочек следующей ячейки.</p>"
                        "<p><b>1</b> — дефолт наступил в выбранном горизонте; "
                        "<b>0</b> — полный горизонт проверен и дефолт не найден; "
                        "<b>пусто</b> — данных недостаточно или горизонт ещё "
                        "не завершён. Пустое значение никогда не заменяется "
                        "нулём.</p>"
                        "<p>Необходимые методы СПАРК программа подключит как "
                        "служебные источники target. Ваш набор методов из "
                        "пункта 4 при этом не изменяется.</p>"
                    )
                ),
                TARGET_PROFILE_NAME,
                TARGET_MODE,
                FORECAST_HORIZON,
                widgets.HTML(
                    value=(
                        "<b>Дата оценки:</b> 1 июня года, следующего "
                        "за отчётным. Это правило уже согласовано "
                        "с заказчиком."
                    )
                ),
                widgets.HTML(
                    value=(
                        "<h4>События, формирующие target</h4>"
                        "<p style='color:#5f6368'>Готовые правила фиксируют "
                        "состав событий. Чтобы менять отдельные галочки и "
                        "условие И/ИЛИ, выберите «Ручная настройка».</p>"
                    )
                ),
                *EVENT_ROWS,
                COMBINATION_OPERATOR,
                widgets.HTML(
                    value="<h4>Пороги для расширенного правила</h4>"
                ),
                PAYMENT_OVERDUE_DAYS,
                EXECUTION_MIN_COUNT,
                EXECUTION_MIN_AMOUNT_RUB,
                PREVIEW,
                RETRAINING_ACCORDION,
                SAVE_BUTTON,
                SAVE_OUTPUT,
            ],
            layout=widgets.Layout(
                width="100%",
                max_width="1050px",
            ),
        )

        display(TARGET_INTERFACE)


### 5.2. Параметры методов СПАРК

<details>
<summary><b>Техническая справка: что делает эта служебная ячейка</b></summary>

Эта ячейка автоматически подготавливает параметры запросов для методов СПАРК, выбранных в пункте 4. Ничего вводить, выбирать или редактировать в ней не требуется.

**Что нужно сделать**

1. Убедитесь, что в пункте 4 сохранён набор методов СПАРК.
2. Убедитесь, что в пункте 5.1 сохранено правило целевой переменной.
3. Запустите кодовую ячейку 5.2.
4. Проверьте, что вывод начинается со строки **«ПАРАМЕТРЫ МЕТОДОВ ПОДГОТОВЛЕНЫ»**.
5. После успешного выполнения перейдите к пункту 5.3 — выбору полей и признаков.

Эту ячейку нельзя пропускать в обычном порядке запуска ноутбука. Она связывает выбранные методы с отчётным годом целевой переменной и формирует технический план запросов, который будет использоваться при проверке одной компании и при массовой обработке.

**Какие параметры устанавливаются автоматически**

Служебные параметры применяются только к тем методам, которые уже выбраны в пункте 4. Новые пользовательские методы ячейка не добавляет.

| Метод                                                  | Что настраивается                                                                       |
| ------------------------------------------------------ | --------------------------------------------------------------------------------------- |
| Договоры лизинга                                       | Получение объектов со всеми доступными статусами                                        |
| Договоры залога                                        | Получение объектов со всеми доступными статусами                                        |
| Сертификаты                                            | Получение записей со всеми доступными статусами                                         |
| Исполнительные производства, где компания — взыскатель | Получение производств со всеми доступными статусами                                     |
| Недвижимость                                           | Включение исторических сведений                                                         |
| Патенты                                                | Включение исторических сведений                                                         |
| Проверки надзорных органов                             | Получение всех типов и статусов проверок без ограничения по региону и наличию нарушений |
| Вакансии                                               | Ограничение периода отчётным годом и защитный лимит до 50 вакансий                      |
| Государственные контракты                              | Отдельные запросы по 44-ФЗ и 223-ФЗ за отчётный год                                     |

Отчётный год берётся из правила целевой переменной, сохранённого в пункте 5.1. Например, если последний полностью наблюдаемый отчётный год равен 2024, вакансии и государственные контракты будут подготовлены для 2024 года.

Для государственных контрактов создаются два запроса на организацию:

* один запрос по 44-ФЗ;
* один запрос по 223-ФЗ.

Поэтому количество подготовленных запросов может быть больше количества методов, которым назначены параметры.

**Дополнительные параметры из пункта 4**

Если в расширенной настройке пункта 4 был указан JSON с параметрами разработчика, его значения применяются поверх стандартных служебных настроек.

Программа проверяет, что:

* JSON является объектом;
* параметры относятся только к выбранным методам;
* для каждого метода передан отдельный объект параметров;
* запросы государственных контрактов содержат обязательные поля `fz` и `year`.

При ошибке программа покажет понятное сообщение и предложит проверить настройки пунктов 4 и 5.1.

**Что сохраняется**

После подготовки ячейка:

1. формирует итоговые параметры выбранных методов;
2. создаёт реестр фактического выполнения методов;
3. обновляет сохранённый профиль методов;
4. сохраняет технические файлы профиля и экспортные версии;
5. записывает настройки в общий `workflow`;
6. помечает этап подготовки методов как завершённый.

Если параметры, отчётный год или правило целевой переменной изменились, ранее сохранённый выбор полей и результат теста одной компании становятся устаревшими. После этого необходимо повторно выполнить пункт 5.3 и последующие этапы.

Если конфигурация не изменилась и сохранённый профиль существует, программа повторно использует уже подготовленный профиль. Поэтому ячейку безопасно запускать повторно.

**Что эта ячейка не делает**

В этой ячейке:

* не выполняются запросы по организациям к REST API СПАРК;
* не загружаются данные компаний;
* не рассчитывается `target_default`;
* не выбираются поля и ML-признаки;
* не изменяется пользовательский набор методов;
* не подключаются банковские методы;
* не выполняется массовая обработка организаций.

**Успешный результат**

После выполнения должна появиться сводка со следующими данными:

* отчётный год из правила target;
* количество выбранных методов;
* количество методов со служебными параметрами;
* количество методов с дополнениями из JSON;
* итоговое количество настроенных методов;
* количество подготовленных запросов на одну организацию;
* перечень служебно настроенных методов.

Последняя строка успешного вывода должна сообщать:

**«Следующий шаг: выберите поля и признаки в ячейке 5.3.»**

После этого переходите к пункту 5.3.

</details>


In [ ]:
# @title 5.2. Подготовить параметры методов

try:
    from src.notebook_runtime import run_notebook_cell
except Exception as _cell_runtime_error:
    print(
        "Не удалось запустить этап «Параметры методов». "
        "Сначала выполните пункт 2. "
        f"Причина: {type(_cell_runtime_error).__name__}: {_cell_runtime_error}"
    )
else:
    with run_notebook_cell(
        stage="Параметры методов",
        next_step="Повторите пункты 4, 5.1 и 5.2.",
        project_root=globals().get("PROJECT_ROOT"),
    ):
        from collections.abc import Mapping
        from copy import deepcopy
        from datetime import datetime, timezone
        from pathlib import Path

        import pandas as pd

        import src.method_executor as method_executor
        import src.notebook_steps as notebook_steps
        import src.notebook_ui as notebook_ui
        from src.method_profiles import save_method_profile
        from src.profile_exports import export_method_profile_package
        from src.workflow import (
            invalidate_after, mark_stage, require_stages,
            save_workflow, update_workflow,
        )


        # Служебные значения меняются только в коде этой ячейки.
        # JSON из ячейки 4 применяется поверх них.
        PARAMETER_POLICY_VERSION = "spark_method_parameters_v2"
        ALL_STATUSES_CODE = 0
        INCLUDE_HISTORY_CODE = 1
        STATE_CONTRACT_LAWS = (44, 223)
        STATE_CONTRACT_YEAR_COUNT = 1
        MAX_VACANCIES = 50

        PARAMETER_SETTINGS_READY = False
        PARAMETER_SETTINGS_RESULT = None

        METHOD_LABELS = {
            "/rest/GetCompanyLeasings": "Договоры лизинга",
            "/rest/GetCompanyPledges": "Договоры залога",
            "/rest/GetCompanyRealEstate": "Недвижимость",
            "/rest/GetCompanyStateContracts": "Государственные контракты",
            "/rest/GetCompanyVacancies": "Вакансии",
            "/rest/GetCompanyInspections": "Проверки надзорных органов",
            "/rest/GetCompanyCertificates": "Сертификаты",
            "/rest/GetCompanyPatents": "Патенты",
            "/rest/GetClaimantCompanyExecutionProceedings": (
                "Исполнительные производства, где компания — взыскатель"
            ),
        }


        def _load_context():
            workflow = notebook_steps._latest_workflow(Path(PROJECT_ROOT))
            require_stages(workflow, "bootstrap", "authentication", "methods")
            target = workflow.get("target") or {}

            if (
                not isinstance(target, Mapping)
                or target.get("format") != "spark_target_profile_v1"
                or not target.get("configuration_hash")
            ):
                raise RuntimeError(
                    "Сначала сохраните правило целевой переменной в ячейке 5.1."
                )

            report_year = (target.get("training_history") or {}).get(
                "latest_matured_report_year_at_save"
            )
            if report_year is None:
                raise RuntimeError(
                    "В правиле target не найден завершённый отчётный год. "
                    "Повторно сохраните ячейку 5.1."
                )
            return workflow, target, int(report_year)


        def _recommended_parameters(selected, report_year):
            selected = set(selected)
            result = {}
            status_fields = {
                "/rest/GetCompanyLeasings": "leasingStatus",
                "/rest/GetCompanyPledges": "pledgeStatus",
                "/rest/GetCompanyCertificates": "certificateStatus",
                "/rest/GetClaimantCompanyExecutionProceedings": "proceedingStatus",
            }

            for endpoint, field_name in status_fields.items():
                if endpoint in selected:
                    result[endpoint] = {field_name: ALL_STATUSES_CODE}

            for endpoint in (
                "/rest/GetCompanyRealEstate",
                "/rest/GetCompanyPatents",
            ):
                if endpoint in selected:
                    result[endpoint] = {"historySearch": INCLUDE_HISTORY_CODE}

            if "/rest/GetCompanyInspections" in selected:
                result["/rest/GetCompanyInspections"] = {
                    "inspectionType": ALL_STATUSES_CODE,
                    "inspectionStatus": ALL_STATUSES_CODE,
                    "hasViolations": ALL_STATUSES_CODE,
                    "regionCode": None,
                }

            if "/rest/GetCompanyVacancies" in selected:
                result["/rest/GetCompanyVacancies"] = {
                    "vacancyStatus": ALL_STATUSES_CODE,
                    "publishedFrom": f"{report_year}-01-01T00:00:00",
                    "publishedTo": f"{report_year}-12-31T23:59:59",
                    "maxNumber": str(MAX_VACANCIES),
                }

            if "/rest/GetCompanyStateContracts" in selected:
                years = [
                    report_year - offset
                    for offset in range(STATE_CONTRACT_YEAR_COUNT)
                ]
                result["/rest/GetCompanyStateContracts"] = {
                    "__requests__": [
                        {"fz": law, "year": year}
                        for year in years
                        for law in STATE_CONTRACT_LAWS
                    ]
                }
            return result


        def _normalize_overrides(raw_overrides, selected):
            if not isinstance(raw_overrides, Mapping):
                raise ValueError(
                    "Дополнительные параметры из ячейки 4 должны быть JSON-объектом."
                )

            selected = set(selected)
            result = {}
            for endpoint, values in raw_overrides.items():
                endpoint = str(endpoint).strip()
                if endpoint not in selected:
                    raise ValueError(
                        f"В JSON ячейки 4 указан невыбранный метод: {endpoint}."
                    )
                if not isinstance(values, Mapping):
                    raise ValueError(
                        f"Параметры метода {endpoint} должны быть JSON-объектом."
                    )
                result[endpoint] = deepcopy(dict(values))
            return result


        def _deep_merge(base, override):
            """Словари объединяются, а списки, включая __requests__, заменяются."""
            result = deepcopy(dict(base))
            for key, value in override.items():
                if isinstance(value, Mapping) and isinstance(result.get(key), Mapping):
                    result[key] = _deep_merge(result[key], value)
                else:
                    result[key] = deepcopy(value)
            return result


        def _merge_parameters(recommended, overrides):
            result = deepcopy(recommended)
            for endpoint, values in overrides.items():
                merged = _deep_merge(result.get(endpoint) or {}, values)
                if merged:
                    result[endpoint] = merged
                else:
                    result.pop(endpoint, None)
            return result


        def _validate(parameters):
            contracts = parameters.get("/rest/GetCompanyStateContracts")
            if not isinstance(contracts, Mapping):
                return

            requests = contracts.get("__requests__")
            if not isinstance(requests, list) or not requests:
                raise ValueError(
                    "Для государственных контрактов список __requests__ пуст."
                )
            if any(
                not isinstance(item, Mapping)
                or "fz" not in item
                or "year" not in item
                for item in requests
            ):
                raise ValueError(
                    "Каждый запрос государственных контрактов должен содержать "
                    "поля fz и year."
                )


        def _request_count(parameters):
            total = 0
            for values in parameters.values():
                requests = values.get("__requests__") if isinstance(values, Mapping) else None
                total += len(requests) if isinstance(requests, list) else 1
            return total


        def _configuration_is_current(methods, settings, target, report_year, parameters):
            profile_path = methods.get("profile_path")
            return bool(
                methods.get("parameters_ready")
                and methods.get("parameters") == parameters
                and methods.get("parameter_report_year") == report_year
                and methods.get("parameter_target_hash") == target.get("configuration_hash")
                and settings.get("policy_version") == PARAMETER_POLICY_VERSION
                and profile_path
                and Path(profile_path).is_file()
            )


        def _save_configuration(
            root, workflow, target, report_year,
            methods, selected, parameters, recommended, overrides,
        ):
            catalog_state = notebook_steps._prepare_catalog(
                project_root=root,
                workflow=workflow,
                force_refresh=False,
            )
            workflow = catalog_state.get("workflow") or workflow
            catalog_result = catalog_state["catalog_result"]
            registry = method_executor.build_execution_registry(
                catalog_result=catalog_result,
                selected_endpoints=selected,
                method_parameters=parameters,
            )
            profile_name = str(methods.get("profile_name") or "Набор методов СПАРК")
            profile = save_method_profile(
                project_root=root,
                profile_name=profile_name,
                base_profile_name=str(methods.get("base_profile_name") or profile_name),
                selected_endpoints=selected,
                catalog_result=catalog_result,
                method_parameters=parameters,
                execution_registry_df=registry,
            )
            exports = export_method_profile_package(
                project_root=root,
                profile_save_result=profile,
                execution_registry_df=registry,
            )
            records = registry.where(pd.notna(registry), None).to_dict(orient="records")
            test_required = bool(
                registry["execution_status"].isin({"automatic", "configured"}).any()
            ) if not registry.empty else False
            settings = {
                "format": "spark_method_parameter_profile_v2",
                "policy_version": PARAMETER_POLICY_VERSION,
                "report_year": report_year,
                "target_configuration_hash": target["configuration_hash"],
                "recommended_endpoints": list(recommended),
                "overridden_endpoints": list(overrides),
                "saved_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            }
            updated_methods = {
                **methods,
                "parameters": parameters,
                "parameter_overrides": overrides,
                "parameters_ready": True,
                "parameter_report_year": report_year,
                "parameter_target_hash": target["configuration_hash"],
                "execution_registry": records,
                "profile_path": profile["profile_path"],
                "excel_path": exports["excel_path"],
                "zip_path": exports["zip_path"],
            }
            updated = update_workflow(
                workflow,
                methods=updated_methods,
                method_parameter_profile=settings,
                test={
                    **(workflow.get("test") or {}),
                    "required": test_required,
                    "completed": False,
                    "company_code": None,
                    "configuration_hash": None,
                    "summary": {},
                    "output_directory": None,
                },
            )
            updated = mark_stage(
                updated, "methods", ready=True,
                message="Методы и параметры запросов подготовлены.",
            )
            updated = invalidate_after(
                updated, "methods",
                reason=(
                    "Параметры запросов изменены. Повторно сохраните поля "
                    "и выполните тест одной компании."
                ),
            )
            saved = save_workflow(updated, workflow_root=root / "workflow")
            return saved["workflow"], profile["profile_path"]


        def _prepare():
            root = Path(PROJECT_ROOT)
            workflow, target, report_year = _load_context()
            methods = dict(workflow.get("methods") or {})
            selected = [str(endpoint) for endpoint in methods.get("selected_endpoints") or []]
            if not selected:
                raise RuntimeError("Набор методов пуст. Вернитесь к ячейке 4.")

            overrides = _normalize_overrides(
                methods.get("parameter_overrides") or {},
                selected,
            )
            recommended = _recommended_parameters(selected, report_year)
            parameters = _merge_parameters(recommended, overrides)
            _validate(parameters)
            settings = workflow.get("method_parameter_profile") or {}
            reused = _configuration_is_current(
                methods, settings, target, report_year, parameters,
            )
            profile_path = methods.get("profile_path")

            if not reused:
                workflow, profile_path = _save_configuration(
                    root, workflow, target, report_year,
                    methods, selected, parameters, recommended, overrides,
                )

            return {
                "success": True,
                "workflow": workflow,
                "parameters": parameters,
                "recommended": recommended,
                "overrides": overrides,
                "report_year": report_year,
                "method_profile_path": profile_path,
                "reused": reused,
            }


        def _show_result(result):
            workflow_methods = result["workflow"].get("methods") or {}
            print("ПАРАМЕТРЫ МЕТОДОВ ПОДГОТОВЛЕНЫ")
            print(f"Отчётный год из target: {result['report_year']}")
            print(f"Выбрано методов: {len(workflow_methods.get('selected_endpoints') or [])}")
            print(f"Служебные параметры применены к методам: {len(result['recommended'])}")
            print(f"Методов с дополнениями из JSON ячейки 4: {len(result['overrides'])}")
            print(f"Итоговых настроенных методов: {len(result['parameters'])}")
            print(
                "Запросов настроенных методов на одну организацию: "
                f"{_request_count(result['parameters'])}"
            )
            if result["recommended"]:
                print("Служебно настроенные методы:")
                for endpoint in result["recommended"]:
                    print(f"- {METHOD_LABELS[endpoint]}")
            if result["reused"]:
                print("Конфигурация не изменилась: сохранённый профиль уже актуален.")
            print("Банковские методы не добавлялись и запрашиваться не будут.")
            print("Запросы по организациям к СПАРК в этой ячейке не выполнялись.")
            print("Следующий шаг: выберите поля и признаки в ячейке 5.3.")


        try:
            PARAMETER_SETTINGS_RESULT = _prepare()
            PARAMETER_SETTINGS_READY = True
            _show_result(PARAMETER_SETTINGS_RESULT)
        except Exception as error:
            PARAMETER_SETTINGS_READY = False
            PARAMETER_SETTINGS_RESULT = {"success": False, "error": str(error)}
            notebook_ui.show_error(
                stage="Подготовка параметров методов",
                error=error,
                action=(
                    "Проверьте, что методы сохранены в ячейке 4, правило target "
                    "сохранено в ячейке 5.1, а JSON ячейки 4 содержит только "
                    "выбранные методы."
                ),
            )


### 5.3. Поля и признаки

<details>
<summary><b>Техническая справка: как работает выбор полей и признаков</b></summary>

В этом разделе формируется окончательный состав данных, которые будут получены из СПАРК и переданы в будущий ML-датасет.

Ячейка использует результаты предыдущих этапов:

* выбранные методы REST API СПАРК из пункта 4;
* правило целевой переменной из пункта 5.1;
* подготовленные параметры запросов из пункта 5.2;
* каталог методов и полей, построенный по OpenAPI-описанию СПАРК.

Перед открытием интерфейса программа проверяет, что пункты 5.1 и 5.2 выполнены, а сохранённый профиль методов соответствует текущему правилу target и выбранным методам. Если настройки устарели или неполны, программа остановит выполнение и предложит повторно запустить пункт 5.2.

**Что нужно сделать**

1. Запустите кодовую ячейку 5.3.
2. Выберите готовый режим или настройте состав данных вручную.
3. Проверьте количество выбранных показателей, расчётных признаков и дополнительных полей API.
4. Нажмите кнопку **«Сохранить выбранный набор»**.
5. Убедитесь, что появилась строка **«ПОЛЯ И ПРИЗНАКИ СОХРАНЕНЫ»**.
6. После успешного сохранения перейдите к проверке набора на одной компании.

Пропускать этот раздел нельзя. Без сохранённого профиля полей и признаков последующие этапы не смогут определить, какие данные необходимо извлекать и какие колонки должны войти в итоговый датасет.

**Доступные режимы**

В интерфейсе предусмотрены четыре режима.

| Режим                      | Назначение                                                                                        |
| -------------------------- | ------------------------------------------------------------------------------------------------- |
| Основной набор СПАРК       | Выбирает компактный набор основных показателей для первоначального датасета                       |
| СПАРК и расчётные признаки | Добавляет к основным показателям признаки, рассчитываемые внутри конвейера                        |
| Полная выгрузка            | Выбирает все доступные показатели, расчётные признаки и дополнительные поля выбранных методов API |
| Ручная настройка           | Позволяет самостоятельно включать и отключать отдельные признаки, поля, методы и категории        |

При ручном изменении состава данных интерфейс автоматически переводит выбор в режим ручной настройки.

**Показатели СПАРК**

Показатели СПАРК — это заранее зарегистрированные признаки, значения которых непосредственно получаются из ответов API или формируются на основе основных сведений СПАРК.

Для каждого показателя в реестре определены:

* техническое имя колонки;
* русское описание;
* категория;
* источник данных;
* тип значения;
* правила включения в итоговую таблицу.

Количество доступных показателей зависит от текущей версии реестра проекта.

**Расчётные признаки**

Расчётные признаки создаются внутри конвейера на основе полученных данных.

К ним могут относиться:

* количества объектов и событий;
* признаки наличия информации;
* агрегаты по спискам и связанным таблицам;
* возраст и длительность;
* показатели активности;
* признаки рисков;
* временные и логические преобразования.

Выбор расчётного признака означает, что после получения исходных данных конвейер должен рассчитать соответствующую колонку и включить её в ML-матрицу.

Количество расчётных признаков зависит от текущей версии встроенного реестра и расчётного модуля.

**Дополнительные поля API**

Дополнительные поля строятся по OpenAPI-каталогу методов, выбранных в пункте 4.

Для каждого поля сохраняются:

* метод REST API;
* JSON-путь внутри ответа;
* стабильный идентификатор `field_id`;
* имя выходной колонки;
* категория;
* тип данных;
* принадлежность к массиву или объекту;
* возможность прямого включения в ML-таблицу.

Количество дополнительных полей может изменяться в зависимости от:

* выбранных методов СПАРК;
* автоматически добавленных служебных методов;
* текущего OpenAPI-описания;
* доступных схем ответов;
* исключённых технических веток.

Поэтому количество полей в разных конфигурациях может отличаться. Например, при запуске режима **«Полная выгрузка»** будет выбрано 2254 дополнительных поля API.

**Выбор по категориям и методам**

Поля можно выбирать:

* по одному;
* сразу по категории;
* сразу по конкретному методу;
* через строку поиска;
* с помощью готового режима.

Интерфейс может показывать только ограниченное количество строк одновременно, чтобы не перегружать Google Colab. При этом кнопки выбора категории или метода работают со всем соответствующим набором, а не только с видимой частью списка.

**Основная таблица и связанные таблицы**

Не все ответы API можно безопасно поместить в одну строку основной таблицы.

Скалярные значения могут быть включены непосредственно в основную таблицу организации:

* числа;
* логические значения;
* строки;
* даты;
* одиночные показатели.

Повторяющиеся структуры и массивы сохраняются в связанных таблицах. Например, отдельными строками могут храниться:

* контракты;
* судебные дела;
* лицензии;
* проверки;
* вакансии;
* объекты имущества;
* учредители и связи;
* исполнительные производства.

Такой подход не позволяет потерять элементы массивов и предотвращает некорректное объединение нескольких объектов в одну ячейку.

**Что происходит при сохранении**

После нажатия кнопки **«Сохранить выбранный набор»** программа:

1. проверяет, что выбран хотя бы один признак;
2. проверяет существование выбранных полей в текущем реестре;
3. проверяет допустимость прямого включения полей в ML-таблицу;
4. формирует профиль выбранных полей и признаков;
5. сохраняет профиль на Google Диске;
6. записывает выбор в общий `workflow`;
7. создаёт Excel-файл с составом выбранных данных;
8. подготавливает файл для скачивания через браузер;
9. сбрасывает результаты предыдущего теста одной компании, если конфигурация изменилась;
10. помечает последующие результаты массовой обработки как устаревшие.

После завершения сохранения кнопка снова становится активной. Поэтому набор можно изменить и сохранить повторно без перезапуска всей ячейки.

**Что передаётся следующим этапам**

В общий рабочий процесс сохраняются как минимум:

```python
workflow["fields"]["selected_feature_names"]
workflow["fields"]["selected_field_ids"]
```

Последующие этапы используют этот выбор для:

* выполнения необходимых методов API;
* извлечения значений по JSON-путям;
* построения основной таблицы;
* формирования связанных таблиц;
* расчёта пользовательских признаков;
* создания итоговой ML-матрицы;
* проверки одной организации;
* массовой обработки списка компаний.

Тест одной компании и массовая обработка используют общий производственный механизм. Поэтому успешный тест подтверждает не только внешний вид интерфейса, но и фактическую передачу выбранной конфигурации в конвейер.

**Целевая переменная**

Целевая переменная не выбирается повторно в пункте 5.3.

Она автоматически берётся из правила, сохранённого в пункте 5.1. В текущей конфигурации используется:

```text
target_default
```

Целевая переменная добавляется к итоговому датасету отдельно от пользовательского набора признаков.

**Файл Excel**

После сохранения программа формирует Excel-файл с описанием выбранного набора.

Файл содержит сведения о:

* выбранных признаках;
* дополнительных полях API;
* методах-источниках;
* JSON-путях;
* категориях;
* типах данных;
* параметрах включения в ML;
* текущем режиме выбора.

Браузеру передаётся одно скачивание. Для повторной передачи файла необходимо повторно запустить ячейку и сохранить набор ещё раз.

**Что эта ячейка не делает**

В пункте 5.3:

* не выполняются запросы по конкретным организациям;
* не рассчитываются фактические значения признаков;
* не создаётся итоговый ML-датасет;
* не выполняется массовая обработка;
* не проверяется наполненность полей на реальных компаниях;
* не меняется правило целевой переменной;
* не добавляются методы, не выбранные в пункте 4.

Ячейка только формирует и сохраняет точную конфигурацию будущего сбора данных.

**Успешный результат**

После сохранения должна появиться сводка примерно следующего вида:

```text
ПОЛЯ И ПРИЗНАКИ СОХРАНЕНЫ
Режим: Полная выгрузка
Показатели СПАРК: 26
Расчётные признаки: 53
Дополнительные поля API: 2254
Целевая переменная: target_default — автоматически
Тест одной компании: обязателен
Следующий шаг: проверьте набор на одной компании.
```

Числа могут отличаться при изменении методов, реестра признаков или OpenAPI-каталога.

Успешным считается результат, при котором:

* профиль сохранён без ошибок;
* отображается выбранный режим;
* указано количество признаков и полей;
* указана целевая переменная;
* подготовлен Excel-файл;
* программа предлагает перейти к тесту одной компании.

После этого переходите к следующему разделу и проверяйте сохранённую конфигурацию на одной организации.

</details>


In [ ]:
# @title 5.3. Выбрать и сохранить поля и признаки

from __future__ import annotations

try:
    from src.notebook_runtime import run_notebook_cell
except Exception as _cell_runtime_error:
    print(
        "Не удалось запустить этап «Поля и признаки». "
        "Сначала выполните пункт 2. "
        f"Причина: {type(_cell_runtime_error).__name__}: {_cell_runtime_error}"
    )
else:
    with run_notebook_cell(
        stage="Поля и признаки",
        next_step="Повторите пункт 5.3.",
        project_root=globals().get("PROJECT_ROOT"),
    ):
        from collections.abc import Mapping
        from copy import deepcopy
        from datetime import datetime, timezone
        from pathlib import Path
        import hashlib
        import html
        import json
        import re

        import pandas as pd
        import ipywidgets as widgets
        from IPython.display import clear_output, display

        import src.notebook_steps as notebook_steps
        from src.feature_registry import build_feature_registry
        from src.field_registry import (
            build_field_registry,
            save_field_profile,
        )
        from src.notebook_ui import (
            protected_download_buttons,
            show_error,
        )
        from src.profile_exports import (
            export_field_profile_package,
        )
        from src.user_features_loader import (
            active_feature_source_state,
        )
        from src.method_policy import (
            validate_method_policy,
        )
        from src.workflow import (
            invalidate_after,
            mark_stage,
            save_workflow,
            update_workflow,
        )


        # Показатели, которые берутся из данных СПАРК без дополнительной
        # аналитической формулы. Техническое приведение типов и подсчёт
        # готовых объектов не меняют экономический смысл показателя.
        SPARK_SOURCE_FEATURES = frozenset(
            {
                "assets_rub",
                "equity_rub",
                "payables_rub",
                "inventories_rub",
                "other_current_assets_rub",
                "cash_rub",
                "revenue_rub",
                "operating_expenses_rub",
                "other_income_rub",
                "other_expenses_rub",
                "net_profit_rub",
                "company_type",
                "main_okved_code",
                "okopf_code",
                "charter_capital_rub",
                "leader_count",
                "phone_count",
                "same_address_count",
                "same_address_without_room_count",
                "same_address_not_affiliated_count",
                "same_address_fts_count",
                "same_manager_country_count",
                "same_manager_region_count",
                "same_manager_inn_count",
                "liquidated_same_address_count",
                "liquidated_same_address_without_room_count",
            }
        )


        # Формулы для ранее проверенных признаков основного конвейера.
        # Формулы custom__... уже хранятся в реестре признаков.
        CORE_FORMULAS = {
            "company_age_days": (
                "Дата оценки − дата регистрации, в днях."
            ),
            "company_age_years": (
                "Возраст компании в днях / 365,25."
            ),
            "equity_ratio": (
                "Собственный капитал / активы."
            ),
            "payables_to_assets": (
                "Кредиторская задолженность / активы."
            ),
            "cash_to_payables": (
                "Денежные средства / кредиторская задолженность."
            ),
            "disclosed_assets_to_payables": (
                "(Запасы + дебиторская задолженность + денежные средства) "
                "/ кредиторская задолженность."
            ),
            "inventory_share": (
                "Запасы / активы."
            ),
            "net_margin": (
                "Чистая прибыль или убыток / выручка."
            ),
            "return_on_assets": (
                "Чистая прибыль или убыток / средние активы за два года."
            ),
            "return_on_equity": (
                "Чистая прибыль или убыток / средний собственный капитал "
                "за два года."
            ),
            "asset_growth": (
                "Активы отчётного года / активы предыдущего года − 1."
            ),
            "equity_growth": (
                "Капитал отчётного года / капитал предыдущего года − 1."
            ),
            "payables_growth": (
                "Кредиторская задолженность отчётного года / значение "
                "предыдущего года − 1."
            ),
            "revenue_growth": (
                "Выручка отчётного года / выручка предыдущего года − 1."
            ),
            "profit_change_rub": (
                "Чистая прибыль отчётного года − чистая прибыль "
                "предыдущего года."
            ),
            "zero_revenue_flag": (
                "1, если выручка равна нулю; иначе 0."
            ),
            "loss_flag": (
                "1, если чистая прибыль отрицательная; иначе 0."
            ),
            "negative_equity_flag": (
                "1, если собственный капитал отрицательный; иначе 0."
            ),
            "has_phone_flag": (
                "1, если у компании найден хотя бы один телефон; иначе 0."
            ),
        }


        MODE_SPARK = "Основной набор СПАРК"
        MODE_SPARK_PLUS = "Основной СПАРК + расчётные признаки"
        MODE_FULL = "Полная выгрузка"
        MODE_MANUAL = "Ручная настройка"

        # Формат служебного профиля, который создаёт ячейка 5.2.
        # Проверка не изменяет параметры методов, а только не позволяет
        # открыть выбор полей на устаревшей или неполной конфигурации.
        METHOD_PARAMETER_PROFILE_FORMAT = "spark_method_parameter_profile_v2"

        # Формат и папка полных конфигураций пунктов 4–5.3.
        BUILD_CONFIGURATION_FORMAT = (
            "spark_build_configuration_profile_v1"
        )
        BUILD_CONFIGURATION_DIRECTORY = "configuration_profiles"

        # Рабочий финансовый конвейер получает перечень периодов через
        # GetCompanyAccountingReportDateList. Метод ...ReportPeriods является
        # альтернативным каталогом периодов и для расчёта признаков не нужен.
        REQUIRED_FEATURE_ENDPOINTS = frozenset(
            {
                "/rest/FindCompanyByCode",
                "/rest/GetCompanyShortReport",
                "/rest/GetCompanyAccountingReportDateList",
                "/rest/GetCompanyAccountingReport",
            }
        )

        MODE_DESCRIPTIONS = {
            MODE_SPARK: (
                "Исходные финансовые, регистрационные и счётные показатели "
                "из СПАРК. Дополнительные аналитические формулы не включаются."
            ),
            MODE_SPARK_PLUS: (
                "Основной набор СПАРК плюс все разрешённые и объяснимые "
                "признаки, которые рассчитывает ноутбук. Формулу каждого "
                "признака можно посмотреть ниже."
            ),
            MODE_FULL: (
                "Все разрешённые признаки и все доступные поля выбранных "
                "методов API. Массивы сохранятся в связанных таблицах. "
                "Файлы могут получиться большими."
            ),
            MODE_MANUAL: (
                "Ничего не выбирается автоматически. Можно включать и "
                "отключать целые категории, методы и отдельные поля."
            ),
        }


        def _safe(value) -> str:
            """Экранирует текст перед выводом в интерфейс."""

            return html.escape(str(value or ""))





        def _safe_filename(
            value: str,
            *,
            fallback: str = "configuration",
        ) -> str:
            """Возвращает безопасное имя JSON-файла без потери русского текста."""

            cleaned = re.sub(
                r'[<>:"/\\|?*\x00-\x1f]+',
                "_",
                str(value or "").strip(),
            )
            cleaned = re.sub(r"\s+", " ", cleaned).strip(" ._")
            return cleaned or fallback


        def _json_ready(value):
            """Преобразует снимок workflow к стабильному JSON-виду."""

            if isinstance(value, Mapping):
                return {
                    str(key): _json_ready(item)
                    for key, item in value.items()
                }

            if isinstance(value, (list, tuple, set, frozenset)):
                return [
                    _json_ready(item)
                    for item in value
                ]

            if isinstance(value, Path):
                return str(value)

            if isinstance(value, datetime):
                return value.isoformat(timespec="seconds")

            if value is None or isinstance(
                value,
                (str, int, float, bool),
            ):
                return value

            if hasattr(value, "item"):
                try:
                    return _json_ready(value.item())
                except Exception:
                    pass

            return str(value)


        def _canonical_sha256(value) -> str:
            """Считает стабильный SHA-256 содержимого конфигурации."""

            serialized = json.dumps(
                _json_ready(value),
                ensure_ascii=False,
                sort_keys=True,
                separators=(",", ":"),
            )
            return hashlib.sha256(
                serialized.encode("utf-8")
            ).hexdigest()


        def _write_json_atomic(
            path: Path,
            payload: Mapping,
        ) -> None:
            """Атомарно записывает JSON, не повреждая предыдущий файл."""

            path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )
            temporary_path = path.with_suffix(
                path.suffix + ".tmp"
            )
            temporary_path.write_text(
                json.dumps(
                    _json_ready(payload),
                    ensure_ascii=False,
                    indent=2,
                    sort_keys=True,
                ),
                encoding="utf-8",
            )
            temporary_path.replace(path)


        def _configuration_summary(
            configuration: Mapping,
        ) -> dict:
            """Готовит короткую русскую сводку сохранённого профиля."""

            methods = configuration.get("methods") or {}
            fields = configuration.get("fields") or {}
            target = configuration.get("target") or {}
            training_history = (
                target.get("training_history")
                if isinstance(target, Mapping)
                else {}
            ) or {}
            horizon_state = (
                target.get("horizon")
                if isinstance(target, Mapping)
                else {}
            ) or {}

            report_year = (
                training_history.get(
                    "latest_matured_report_year_at_save"
                )
                or target.get("report_year")
                or target.get("latest_report_year")
            )
            horizon_months = (
                target.get("horizon_months")
                or horizon_state.get("months")
                or target.get("forecast_horizon_months")
            )
            target_name = (
                target.get("profile_name")
                or target.get("rule_name")
                or target.get("target_name")
                or target.get("name")
                or "target_default"
            )

            return {
                "method_count": len(
                    methods.get("selected_endpoints") or []
                ),
                "method_profile_name": (
                    methods.get("profile_name")
                    or methods.get("base_profile_name")
                    or "Сохранённый набор методов"
                ),
                "runtime_policy_version": (
                    methods.get("runtime_policy_version")
                ),
                "feature_mode": (
                    fields.get("selection_mode")
                    or fields.get("feature_profile_name")
                    or "Ручная настройка"
                ),
                "feature_count": len(
                    fields.get("selected_feature_names") or []
                ),
                "field_count": len(
                    fields.get("selected_field_ids") or []
                ),
                "target_name": target_name,
                "target_report_year": report_year,
                "target_horizon_months": horizon_months,
                "custom_features_sha256": (
                    fields.get("custom_features_sha256")
                ),
            }


        def _build_configuration_payload(
            *,
            project_root: str | Path,
            workflow: Mapping,
            profile_name: str,
        ) -> dict:
            """Создаёт переносимый снимок API, target, параметров и признаков."""

            clean_name = str(profile_name or "").strip()

            if not clean_name:
                raise ValueError(
                    "Укажите название полной конфигурации."
                )

            methods = workflow.get("methods") or {}
            target = workflow.get("target") or {}
            parameter_profile = (
                workflow.get("method_parameter_profile") or {}
            )
            fields = workflow.get("fields") or {}

            if not isinstance(methods, Mapping) or not (
                methods.get("selected_endpoints") or []
            ):
                raise RuntimeError(
                    "Методы API ещё не сохранены. "
                    "Сначала завершите пункт 4."
                )

            if not bool(methods.get("parameters_ready")):
                raise RuntimeError(
                    "Параметры методов ещё не подготовлены. "
                    "Сначала завершите пункт 5.2."
                )

            if (
                not isinstance(target, Mapping)
                or target.get("format")
                != "spark_target_profile_v1"
                or not str(
                    target.get("configuration_hash") or ""
                ).strip()
            ):
                raise RuntimeError(
                    "Правило target ещё не сохранено. "
                    "Сначала завершите пункт 5.1."
                )

            if (
                not isinstance(parameter_profile, Mapping)
                or parameter_profile.get("format")
                != METHOD_PARAMETER_PROFILE_FORMAT
            ):
                raise RuntimeError(
                    "Служебный профиль параметров методов отсутствует. "
                    "Повторно сохраните пункт 5.2."
                )

            if not isinstance(fields, Mapping) or not (
                fields.get("selected_feature_names")
                or fields.get("selected_field_ids")
            ):
                raise RuntimeError(
                    "Поля и признаки ещё не сохранены. "
                    "Сначала нажмите «Сохранить выбранный набор»."
                )

            configuration = {
                "workflow_format": workflow.get("format"),
                "project_version": workflow.get("project_version"),
                "adapter_version": workflow.get("adapter_version"),
                "catalog": deepcopy(
                    dict(workflow.get("catalog") or {})
                ),
                "policies": deepcopy(
                    dict(workflow.get("policies") or {})
                ),
                "methods": deepcopy(dict(methods)),
                "target": deepcopy(dict(target)),
                "method_parameter_profile": deepcopy(
                    dict(parameter_profile)
                ),
                "fields": deepcopy(dict(fields)),
            }
            configuration = _json_ready(configuration)
            now = datetime.now(
                timezone.utc
            ).isoformat(timespec="seconds")

            return {
                "format": BUILD_CONFIGURATION_FORMAT,
                "profile_name": clean_name,
                "created_at": now,
                "updated_at": now,
                "revision": 1,
                "configuration_hash": _canonical_sha256(
                    configuration
                ),
                "summary": _configuration_summary(
                    configuration
                ),
                "configuration": configuration,
                "source": {
                    "project_root": str(Path(project_root)),
                    "workflow_fingerprints": deepcopy(
                        dict(workflow.get("fingerprints") or {})
                    ),
                },
            }


        def _save_build_configuration(
            *,
            project_root: str | Path,
            workflow: Mapping,
            profile_name: str,
        ) -> dict:
            """Сохраняет профиль и архивирует изменённую прошлую версию."""

            root = (
                Path(project_root)
                / BUILD_CONFIGURATION_DIRECTORY
            )
            clean_name = str(profile_name or "").strip()
            safe_name = _safe_filename(
                clean_name,
                fallback="configuration",
            )
            profile_path = root / f"{safe_name}.json"
            payload = _build_configuration_payload(
                project_root=project_root,
                workflow=workflow,
                profile_name=clean_name,
            )
            existing = None

            if profile_path.is_file():
                try:
                    loaded = json.loads(
                        profile_path.read_text(
                            encoding="utf-8"
                        )
                    )
                    if isinstance(loaded, Mapping):
                        existing = dict(loaded)
                except Exception:
                    existing = None

            # Разные названия, которые дали одинаковое безопасное имя,
            # не должны случайно перезаписывать друг друга.
            if (
                existing
                and str(existing.get("profile_name") or "").strip()
                and str(existing.get("profile_name")).strip()
                != clean_name
            ):
                suffix = hashlib.sha256(
                    clean_name.encode("utf-8")
                ).hexdigest()[:8]
                profile_path = root / (
                    f"{safe_name}_{suffix}.json"
                )
                existing = None

                if profile_path.is_file():
                    try:
                        loaded = json.loads(
                            profile_path.read_text(
                                encoding="utf-8"
                            )
                        )
                        if isinstance(loaded, Mapping):
                            existing = dict(loaded)
                    except Exception:
                        existing = None

            archived_path = None
            status = "created"

            if existing:
                old_hash = str(
                    existing.get("configuration_hash") or ""
                )
                new_hash = str(
                    payload.get("configuration_hash") or ""
                )
                created_at = str(
                    existing.get("created_at")
                    or payload["created_at"]
                )
                try:
                    old_revision = int(
                        existing.get("revision") or 1
                    )
                except (TypeError, ValueError):
                    old_revision = 1

                payload["created_at"] = created_at

                if old_hash == new_hash:
                    payload["revision"] = old_revision
                    status = "unchanged"
                else:
                    timestamp = datetime.now(
                        timezone.utc
                    ).strftime("%Y%m%d_%H%M%S")
                    history_root = (
                        root
                        / "history"
                        / profile_path.stem
                    )
                    archived_path = history_root / (
                        f"{timestamp}_v{old_revision}.json"
                    )
                    counter = 2

                    while archived_path.exists():
                        archived_path = history_root / (
                            f"{timestamp}_v{old_revision}_{counter}.json"
                        )
                        counter += 1

                    _write_json_atomic(
                        archived_path,
                        existing,
                    )
                    payload["revision"] = (
                        old_revision + 1
                    )
                    status = "updated"

            _write_json_atomic(
                profile_path,
                payload,
            )

            return {
                "status": status,
                "profile_path": str(profile_path),
                "archived_path": (
                    str(archived_path)
                    if archived_path is not None
                    else None
                ),
                "configuration_hash": payload[
                    "configuration_hash"
                ],
                "revision": payload["revision"],
                "summary": payload["summary"],
            }


        def _load_build_configuration_profile(
            profile_path: str | Path,
        ) -> dict:
            """Читает и полностью проверяет сохранённую конфигурацию."""

            path = Path(profile_path)

            if not path.is_file():
                raise FileNotFoundError(
                    f"Файл конфигурации не найден: {path}"
                )

            try:
                payload = json.loads(
                    path.read_text(encoding="utf-8")
                )
            except Exception as error:
                raise RuntimeError(
                    "JSON конфигурации повреждён или не читается."
                ) from error

            if not isinstance(payload, Mapping):
                raise RuntimeError(
                    "Файл конфигурации имеет неправильную структуру."
                )

            if payload.get("format") != BUILD_CONFIGURATION_FORMAT:
                raise RuntimeError(
                    "Файл создан в неподдерживаемом формате конфигурации."
                )

            configuration = payload.get("configuration")

            if not isinstance(configuration, Mapping):
                raise RuntimeError(
                    "В файле отсутствует снимок настроек пунктов 4–5.3."
                )

            expected_hash = str(
                payload.get("configuration_hash") or ""
            ).strip()
            actual_hash = _canonical_sha256(configuration)

            if not expected_hash or expected_hash != actual_hash:
                raise RuntimeError(
                    "Контрольная сумма конфигурации не совпала. "
                    "Файл мог быть изменён вручную или повреждён."
                )

            methods = configuration.get("methods") or {}
            target = configuration.get("target") or {}
            parameter_profile = (
                configuration.get("method_parameter_profile") or {}
            )
            fields = configuration.get("fields") or {}

            if not isinstance(methods, Mapping) or not (
                methods.get("selected_endpoints") or []
            ):
                raise RuntimeError(
                    "В конфигурации отсутствует сохранённый набор методов API."
                )

            if (
                not isinstance(target, Mapping)
                or target.get("format") != "spark_target_profile_v1"
                or not str(
                    target.get("configuration_hash") or ""
                ).strip()
            ):
                raise RuntimeError(
                    "В конфигурации отсутствует корректное правило target."
                )

            if (
                not isinstance(parameter_profile, Mapping)
                or parameter_profile.get("format")
                != METHOD_PARAMETER_PROFILE_FORMAT
            ):
                raise RuntimeError(
                    "В конфигурации отсутствуют параметры методов пункта 5.2."
                )

            if not isinstance(fields, Mapping) or not (
                fields.get("selected_feature_names")
                or fields.get("selected_field_ids")
            ):
                raise RuntimeError(
                    "В конфигурации отсутствует сохранённый набор признаков."
                )

            result = dict(payload)
            result["profile_path"] = str(path)
            return result


        def _discover_build_configuration_profiles(
            project_root: str | Path,
        ) -> tuple[list[dict], list[str]]:
            """Находит все текущие профили, не включая историю версий."""

            root = (
                Path(project_root)
                / BUILD_CONFIGURATION_DIRECTORY
            )
            profiles: list[dict] = []
            errors: list[str] = []

            if not root.is_dir():
                return profiles, errors

            for path in sorted(
                root.glob("*.json"),
                key=lambda item: item.stat().st_mtime,
                reverse=True,
            ):
                try:
                    payload = _load_build_configuration_profile(path)
                    payload["modified_at_timestamp"] = (
                        path.stat().st_mtime
                    )
                    profiles.append(payload)
                except Exception as error:
                    errors.append(
                        f"{path.name}: {error}"
                    )

            return profiles, errors


        def _configuration_profile_label(
            payload: Mapping,
        ) -> str:
            """Формирует понятную строку для выпадающего списка."""

            summary = payload.get("summary") or {}
            name = str(
                payload.get("profile_name")
                or Path(
                    str(payload.get("profile_path") or "configuration")
                ).stem
            )
            feature_mode = str(
                summary.get("feature_mode")
                or "Ручная настройка"
            )
            method_count = int(
                summary.get("method_count") or 0
            )
            feature_count = int(
                summary.get("feature_count") or 0
            )
            revision = int(payload.get("revision") or 1)
            return (
                f"{name} — {method_count} методов API — "
                f"{feature_mode}, {feature_count} признаков — v{revision}"
            )


        def _apply_build_configuration_profile(
            *,
            project_root: str | Path,
            workflow: Mapping,
            payload: Mapping,
            catalog_result: Mapping,
            known_feature_names: set[str],
        ) -> dict:
            """Применяет проверенный снимок, не затрагивая авторизацию и вход."""

            configuration = payload.get("configuration") or {}
            methods = deepcopy(
                dict(configuration.get("methods") or {})
            )
            target = deepcopy(
                dict(configuration.get("target") or {})
            )
            parameter_profile = deepcopy(
                dict(
                    configuration.get("method_parameter_profile")
                    or {}
                )
            )
            fields = deepcopy(
                dict(configuration.get("fields") or {})
            )

            policy_state = validate_method_policy(
                catalog_result
            )
            saved_policy_version = str(
                methods.get("runtime_policy_version") or ""
            ).strip()
            current_policy_version = str(
                policy_state.get("version") or ""
            ).strip()

            if (
                saved_policy_version
                and current_policy_version
                and saved_policy_version != current_policy_version
            ):
                raise RuntimeError(
                    "Конфигурация создана для другой версии политики "
                    "методов API. Откройте более новый профиль либо "
                    "настройте методы заново."
                )

            registry = catalog_result.get("endpoint_registry_df")
            catalog_endpoints = (
                set(registry["endpoint"].astype(str))
                if isinstance(registry, pd.DataFrame)
                and not registry.empty
                and "endpoint" in registry.columns
                else set()
            )
            saved_endpoints = {
                str(value)
                for value in (
                    methods.get("selected_endpoints") or []
                )
                if str(value).strip()
            }
            missing_endpoints = sorted(
                saved_endpoints - catalog_endpoints
            )

            if missing_endpoints:
                preview = ", ".join(missing_endpoints[:8])
                suffix = (
                    f" и ещё {len(missing_endpoints) - 8}"
                    if len(missing_endpoints) > 8
                    else ""
                )
                raise RuntimeError(
                    "В текущем каталоге СПАРК отсутствуют методы из "
                    f"профиля: {preview}{suffix}."
                )

            saved_features = {
                str(value)
                for value in (
                    fields.get("selected_feature_names") or []
                )
                if str(value).strip()
            }
            unknown_features = sorted(
                saved_features - set(known_feature_names)
            )

            if unknown_features:
                preview = ", ".join(unknown_features[:8])
                suffix = (
                    f" и ещё {len(unknown_features) - 8}"
                    if len(unknown_features) > 8
                    else ""
                )
                raise RuntimeError(
                    "В текущем реестре отсутствуют признаки из профиля: "
                    f"{preview}{suffix}."
                )

            current_feature_state = _json_ready(
                active_feature_source_state(
                    Path(project_root)
                )
            )
            current_feature_sha = str(
                (
                    current_feature_state
                    if isinstance(
                        current_feature_state,
                        Mapping,
                    )
                    else {}
                ).get("sha256")
                or ""
            ).strip()
            saved_feature_sha = str(
                fields.get("custom_features_sha256") or ""
            ).strip()

            if (
                saved_feature_sha
                and current_feature_sha
                and saved_feature_sha != current_feature_sha
            ):
                raise RuntimeError(
                    "Файл расчётных формул изменился после сохранения "
                    "этой конфигурации. Применение остановлено, чтобы "
                    "не смешать разные формулы в одном датасете."
                )

            updated = deepcopy(dict(workflow))
            updated["methods"] = methods
            updated["target"] = target
            updated["method_parameter_profile"] = (
                parameter_profile
            )
            updated["fields"] = fields

            saved_policies = configuration.get("policies")
            if isinstance(saved_policies, Mapping):
                updated["policies"] = deepcopy(
                    dict(saved_policies)
                )

            updated["active_build_configuration"] = {
                "format": BUILD_CONFIGURATION_FORMAT,
                "profile_name": str(
                    payload.get("profile_name") or ""
                ),
                "profile_path": str(
                    payload.get("profile_path") or ""
                ),
                "configuration_hash": str(
                    payload.get("configuration_hash") or ""
                ),
                "revision": int(payload.get("revision") or 1),
                "applied_at": datetime.now(
                    timezone.utc
                ).isoformat(timespec="seconds"),
            }
            updated["test"] = {
                "required": True,
                "completed": False,
                "company_code": None,
                "configuration_hash": None,
                "summary": {},
                "output_directory": None,
                "runtime_fields_path": None,
                "excel_path": None,
                "zip_path": None,
            }
            updated = mark_stage(
                updated,
                "methods",
                ready=True,
                message=(
                    "Методы и параметры восстановлены из полной "
                    "конфигурации."
                ),
            )
            updated = mark_stage(
                updated,
                "fields",
                ready=True,
                message=(
                    "Поля и признаки восстановлены из полной "
                    "конфигурации."
                ),
            )
            updated = invalidate_after(
                updated,
                "fields",
                reason=(
                    "Применена сохранённая полная конфигурация. "
                    "Повторите тест одной компании."
                ),
            )
            saved = save_workflow(
                updated,
                workflow_root=(
                    Path(project_root) / "workflow"
                ),
            )
            return saved["workflow"]


        def _ordered_values(
            dataframe: pd.DataFrame,
            column: str,
            selected: set[str],
        ) -> list[str]:
            """Возвращает выбранные значения в порядке реестра."""

            if dataframe.empty or column not in dataframe.columns:
                return []

            return [
                value
                for value in dataframe[column].astype(str).tolist()
                if value in selected
            ]


        def _require_current_method_parameters(
            *,
            workflow: Mapping,
            selected_endpoints: list[str],
        ) -> None:
            """Проверяет, что ячейка 5.2 выполнена для текущего target.

    Ячейка 5.3 не изменяет параметры запросов. Она только проверяет,
    что сохранённый профиль методов относится к текущему правилу
    целевой переменной, содержит актуальный отчётный год и реестр
    выполнения для всех выбранных методов.
    """

            target = workflow.get("target") or {}

            if (
                not isinstance(target, Mapping)
                or target.get("format") != "spark_target_profile_v1"
                or not str(target.get("configuration_hash") or "").strip()
            ):
                raise RuntimeError(
                    "Сначала сохраните правило целевой переменной "
                    "в ячейке 5.1."
                )

            target_hash = str(target["configuration_hash"])
            report_year = (
                target.get("training_history") or {}
            ).get("latest_matured_report_year_at_save")

            if report_year is None:
                raise RuntimeError(
                    "В правиле target не найден завершённый отчётный год. "
                    "Повторно сохраните ячейку 5.1."
                )

            methods = workflow.get("methods") or {}
            parameter_profile = (
                workflow.get("method_parameter_profile") or {}
            )

            if not bool(methods.get("parameters_ready")):
                raise RuntimeError(
                    "Параметры методов ещё не подготовлены. "
                    "Запустите ячейку 5.2."
                )

            if not isinstance(methods.get("parameters"), Mapping):
                raise RuntimeError(
                    "В workflow отсутствует корректный набор параметров "
                    "методов. Повторно запустите ячейку 5.2."
                )

            if str(methods.get("parameter_target_hash") or "") != target_hash:
                raise RuntimeError(
                    "Параметры методов относятся к другой версии правила "
                    "target. Повторно запустите ячейку 5.2."
                )

            try:
                parameter_report_year = int(
                    methods.get("parameter_report_year")
                )
                expected_report_year = int(report_year)
            except (TypeError, ValueError) as error:
                raise RuntimeError(
                    "В профиле методов отсутствует корректный отчётный год. "
                    "Повторно запустите ячейку 5.2."
                ) from error

            if parameter_report_year != expected_report_year:
                raise RuntimeError(
                    "Параметры методов подготовлены для другого отчётного "
                    "года. Повторно запустите ячейку 5.2."
                )

            if (
                not isinstance(parameter_profile, Mapping)
                or parameter_profile.get("format")
                != METHOD_PARAMETER_PROFILE_FORMAT
                or str(
                    parameter_profile.get("target_configuration_hash") or ""
                )
                != target_hash
            ):
                raise RuntimeError(
                    "Служебный профиль параметров методов отсутствует или "
                    "устарел. Повторно запустите ячейку 5.2."
                )

            try:
                profile_report_year = int(
                    parameter_profile.get("report_year")
                )
            except (TypeError, ValueError) as error:
                raise RuntimeError(
                    "В служебном профиле методов не указан корректный "
                    "отчётный год. Повторно запустите ячейку 5.2."
                ) from error

            if profile_report_year != expected_report_year:
                raise RuntimeError(
                    "Служебный профиль методов подготовлен для другого "
                    "отчётного года. Повторно запустите ячейку 5.2."
                )

            profile_path = str(methods.get("profile_path") or "").strip()

            if not profile_path or not Path(profile_path).is_file():
                raise RuntimeError(
                    "Сохранённый профиль методов не найден на Google Диске. "
                    "Повторно запустите ячейку 5.2."
                )

            execution_registry = methods.get("execution_registry")

            if not isinstance(execution_registry, list):
                raise RuntimeError(
                    "Реестр выполнения методов не подготовлен. "
                    "Повторно запустите ячейку 5.2."
                )

            registered_endpoints = {
                str(row.get("endpoint"))
                for row in execution_registry
                if isinstance(row, Mapping)
                and str(row.get("endpoint") or "").strip()
            }
            missing_endpoints = sorted(
                set(selected_endpoints) - registered_endpoints
            )

            if missing_endpoints:
                preview = ", ".join(missing_endpoints[:10])
                suffix = (
                    f" и ещё {len(missing_endpoints) - 10}"
                    if len(missing_endpoints) > 10
                    else ""
                )
                raise RuntimeError(
                    "В реестре выполнения отсутствуют выбранные методы: "
                    f"{preview}{suffix}. Повторно запустите ячейку 5.2."
                )


        def _show_simple_field_feature_selector(
            *,
            project_root: str | Path,
            workflow: Mapping,
            catalog_result: Mapping,
            runtime_fields_df: pd.DataFrame | None = None,
        ):
            """Показывает компактный конструктор датасета."""

            methods = workflow.get("methods") or {}
            selected_endpoints = [
                str(value)
                for value in (
                    methods.get("selected_endpoints") or []
                )
                if str(value).strip()
            ]

            if not selected_endpoints:
                raise RuntimeError(
                    "Сначала сохраните методы API в предыдущей ячейке."
                )

            _require_current_method_parameters(
                workflow=workflow,
                selected_endpoints=selected_endpoints,
            )

            response_fields_df = catalog_result.get(
                "response_fields_df"
            )

            if not isinstance(response_fields_df, pd.DataFrame):
                raise RuntimeError(
                    "Каталог полей API не подготовлен. Повторите "
                    "техническую подготовку и выбор методов."
                )

            field_registry_df = build_field_registry(
                response_fields_df=response_fields_df,
                selected_endpoints=selected_endpoints,
                runtime_fields_df=runtime_fields_df,
            ).copy()

            if not field_registry_df.empty:
                field_registry_df["_ui_group"] = (
                    field_registry_df.get(
                        "group_name",
                        pd.Series(
                            "Без категории",
                            index=field_registry_df.index,
                        ),
                    )
                    .fillna("Без категории")
                    .astype(str)
                )

            feature_registry_df = build_feature_registry()
            feature_registry_df = (
                feature_registry_df[
                    feature_registry_df["included_in_ml"].fillna(False)
                ]
                .copy()
                .reset_index(drop=True)
            )

            known_field_ids = set(
                field_registry_df.get(
                    "field_id",
                    pd.Series(dtype=str),
                ).astype(str)
            )
            known_feature_names = set(
                feature_registry_df["feature_name"].astype(str)
            )
            spark_feature_names = (
                known_feature_names & set(SPARK_SOURCE_FEATURES)
            )
            all_feature_names = set(known_feature_names)
            fields_state = workflow.get("fields") or {}
            saved_field_ids = {
                str(value)
                for value in (
                    fields_state.get("selected_field_ids") or []
                )
                if str(value) in known_field_ids
            }
            saved_feature_names = {
                str(value)
                for value in (
                    fields_state.get("selected_feature_names") or []
                )
                if str(value) in known_feature_names
            }

            selected_field_ids = set(saved_field_ids)
            selected_feature_names = set(saved_feature_names)

            if not selected_field_ids and not selected_feature_names:
                selected_feature_names.update(spark_feature_names)

            saved_mode = str(
                fields_state.get("selection_mode") or ""
            )

            if saved_mode in MODE_DESCRIPTIONS:
                initial_mode = saved_mode
            elif (
                selected_feature_names == spark_feature_names
                and not selected_field_ids
            ):
                initial_mode = MODE_SPARK
            elif (
                selected_feature_names == all_feature_names
                and not selected_field_ids
            ):
                initial_mode = MODE_SPARK_PLUS
            elif (
                selected_feature_names == all_feature_names
                and selected_field_ids == known_field_ids
                and bool(known_field_ids)
            ):
                initial_mode = MODE_FULL
            else:
                initial_mode = MODE_MANUAL

            state = {
                "updating": False,
                "dirty": False,
                "mode": initial_mode,
            }

            profile_name_input = widgets.Text(
                value=str(
                    fields_state.get("profile_name")
                    or "Мой набор данных"
                ),
                description="Название:",
                layout=widgets.Layout(width="95%"),
                style={"description_width": "120px"},
            )
            mode_dropdown = widgets.Dropdown(
                options=list(MODE_DESCRIPTIONS),
                value=initial_mode,
                description="Шаблон:",
                layout=widgets.Layout(width="95%"),
                style={"description_width": "120px"},
            )
            mode_help = widgets.HTML()
            apply_mode_button = widgets.Button(
                description="Применить шаблон",
                icon="check",
                button_style="info",
                layout=widgets.Layout(width="220px"),
            )
            select_all_button = widgets.Button(
                description="Выбрать всё",
                icon="check-square",
                button_style="success",
                layout=widgets.Layout(width="190px"),
            )
            clear_all_button = widgets.Button(
                description="Снять всё",
                icon="trash",
                button_style="warning",
                layout=widgets.Layout(width="190px"),
            )
            summary_html = widgets.HTML()
            save_button = widgets.Button(
                description="Сохранить выбранный набор",
                icon="save",
                button_style="success",
                layout=widgets.Layout(width="300px"),
            )
            save_output = widgets.Output(
                layout=widgets.Layout(
                    width="100%",
                    min_height="55px",
                )
            )
            configuration_name_input = widgets.Text(
                value=str(
                    fields_state.get("profile_name")
                    or "Моя конфигурация сборки"
                ),
                description="Название:",
                placeholder="Например: Основной дефолт 2024",
                layout=widgets.Layout(width="95%"),
                style={"description_width": "120px"},
            )
            configuration_save_button = widgets.Button(
                description="Сохранить полную конфигурацию",
                icon="archive",
                button_style="info",
                disabled=not bool(
                    fields_state.get("selected_feature_names")
                    or fields_state.get("selected_field_ids")
                ),
                layout=widgets.Layout(width="330px"),
                tooltip=(
                    "Сохранить методы API, параметры, target, "
                    "признаки и дополнительные поля одним профилем"
                ),
            )
            configuration_output = widgets.Output(
                layout=widgets.Layout(
                    width="100%",
                    min_height="45px",
                )
            )

            configuration_loader_state = {
                "profiles": {},
                "invalid_profiles": [],
            }
            configuration_profile_dropdown = widgets.Dropdown(
                options=[
                    (
                        "Выберите сохранённую конфигурацию",
                        "",
                    )
                ],
                value="",
                description="Конфигурация:",
                layout=widgets.Layout(width="95%"),
                style={"description_width": "130px"},
            )
            configuration_refresh_button = widgets.Button(
                description="Обновить список",
                icon="refresh",
                layout=widgets.Layout(width="210px"),
            )
            configuration_apply_button = widgets.Button(
                description="Применить выбранную конфигурацию",
                icon="check",
                button_style="success",
                disabled=True,
                layout=widgets.Layout(width="330px"),
            )
            configuration_preview_html = widgets.HTML()
            configuration_load_output = widgets.Output(
                layout=widgets.Layout(
                    width="100%",
                    min_height="45px",
                )
            )
            direct_ml_checkbox = widgets.Checkbox(
                value=any(
                    bool(
                        (option if isinstance(option, Mapping) else {}).get(
                            "include_in_ml"
                        )
                    )
                    for option in (
                        fields_state.get("field_options") or {}
                    ).values()
                ),
                description=(
                    "Добавлять допустимые числовые скалярные поля API "
                    "напрямую в ML-матрицу"
                ),
                indent=False,
                layout=widgets.Layout(width="100%"),
            )

            feature_checkboxes: dict[str, widgets.Checkbox] = {}
            feature_group_names: dict[str, set[str]] = {}

            def mark_manual() -> None:
                """Отмечает ручное изменение без автоматического сброса."""

                if state["updating"]:
                    return

                state["dirty"] = True
                state["mode"] = MODE_MANUAL
                previous = state["updating"]
                state["updating"] = True
                mode_dropdown.value = MODE_MANUAL
                state["updating"] = previous

            def update_mode_help(*_) -> None:
                mode = str(mode_dropdown.value)
                mode_help.value = (
                    "<div style='padding:10px 12px;background:#f5f7fa;"
                    "border-left:4px solid #1976d2;line-height:1.5'>"
                    f"{_safe(MODE_DESCRIPTIONS[mode])}"
                    "</div>"
                )

            def update_summary(*_) -> None:
                selected_rows = (
                    field_registry_df[
                        field_registry_df["field_id"].astype(str).isin(
                            selected_field_ids
                        )
                    ]
                    if not field_registry_df.empty
                    else pd.DataFrame()
                )
                collection_count = (
                    int(selected_rows["is_collection"].fillna(False).sum())
                    if (
                        not selected_rows.empty
                        and "is_collection" in selected_rows.columns
                    )
                    else 0
                )
                source_count = len(
                    selected_feature_names & spark_feature_names
                )
                calculated_count = len(
                    selected_feature_names - spark_feature_names
                )
                status = (
                    "Есть несохранённые изменения"
                    if state["dirty"]
                    else "Показан текущий сохранённый или начальный выбор"
                )
                summary_html.value = (
                    "<div style='padding:12px 14px;background:#e8f5e9;"
                    "border-left:5px solid #2e7d32;line-height:1.65'>"
                    "<b>Текущий состав датасета</b><br>"
                    f"Показатели СПАРК: <b>{source_count}</b><br>"
                    f"Расчётные признаки: <b>{calculated_count}</b><br>"
                    f"Дополнительные поля API: <b>{len(selected_field_ids)}</b> "
                    f"(в связанных таблицах: {collection_count})<br>"
                    "Целевая переменная: <b>target_default</b> — добавится "
                    "автоматически и не требует галочки.<br>"
                    f"<span style='color:#555'>{_safe(status)}</span>"
                    "</div>"
                )

            def on_feature_change(
                change,
                *,
                feature_name: str,
            ) -> None:
                if state["updating"] or change.get("name") != "value":
                    return

                if bool(change["new"]):
                    selected_feature_names.add(feature_name)
                else:
                    selected_feature_names.discard(feature_name)

                mark_manual()
                update_summary()

            feature_group_children = []
            feature_group_titles = []

            for group_name, group_df in feature_registry_df.groupby(
                "group",
                sort=False,
            ):
                group_name = str(group_name)
                group_names = set(
                    group_df["feature_name"].astype(str)
                )
                feature_group_names[group_name] = group_names
                items = []

                for row in group_df.to_dict(orient="records"):
                    feature_name = str(row["feature_name"])
                    russian_name = str(
                        row.get("russian_name") or feature_name
                    )
                    formula = str(
                        row.get("formula")
                        or CORE_FORMULAS.get(feature_name)
                        or (
                            "Исходное значение или готовый счётчик из СПАРК."
                            if feature_name in spark_feature_names
                            else "Проверенный расчётный признак."
                        )
                    )
                    checkbox = widgets.Checkbox(
                        value=feature_name in selected_feature_names,
                        description=f"{russian_name} [{feature_name}]",
                        indent=False,
                        layout=widgets.Layout(width="100%"),
                        tooltip=formula,
                    )
                    checkbox.observe(
                        lambda change, current=feature_name: on_feature_change(
                            change,
                            feature_name=current,
                        ),
                        names="value",
                    )
                    feature_checkboxes[feature_name] = checkbox
                    items.append(
                        widgets.VBox(
                            [
                                checkbox,
                                widgets.HTML(
                                    "<div style='margin:-5px 0 8px 26px;"
                                    "color:#666;font-size:12px'>"
                                    f"{_safe(formula)}"
                                    "</div>"
                                ),
                            ]
                        )
                    )

                select_group_button = widgets.Button(
                    description="Выбрать категорию",
                    icon="check",
                    layout=widgets.Layout(width="190px"),
                )
                clear_group_button = widgets.Button(
                    description="Снять категорию",
                    icon="times",
                    layout=widgets.Layout(width="180px"),
                )

                def set_feature_group(
                    value: bool,
                    *,
                    names=group_names,
                ) -> None:
                    previous = state["updating"]
                    state["updating"] = True

                    for feature_name in names:
                        feature_checkboxes[feature_name].value = value

                    if value:
                        selected_feature_names.update(names)
                    else:
                        selected_feature_names.difference_update(names)

                    state["updating"] = previous
                    mark_manual()
                    update_summary()

                select_group_button.on_click(
                    lambda _, names=group_names: set_feature_group(
                        True,
                        names=names,
                    )
                )
                clear_group_button.on_click(
                    lambda _, names=group_names: set_feature_group(
                        False,
                        names=names,
                    )
                )
                feature_group_children.append(
                    widgets.VBox(
                        [
                            widgets.HBox(
                                [
                                    select_group_button,
                                    clear_group_button,
                                ],
                                layout=widgets.Layout(
                                    gap="8px",
                                    flex_flow="row wrap",
                                ),
                            ),
                            widgets.VBox(items),
                        ]
                    )
                )
                feature_group_titles.append(
                    f"{group_name} ({len(group_names)})"
                )

            feature_accordion = widgets.Accordion(
                children=feature_group_children,
                selected_index=None,
            )

            for index, title in enumerate(feature_group_titles):
                feature_accordion.set_title(index, title)

            group_names = (
                field_registry_df["_ui_group"]
                .drop_duplicates()
                .tolist()
                if not field_registry_df.empty
                else []
            )
            field_ids_by_group = {
                group_name: set(
                    field_registry_df.loc[
                        field_registry_df["_ui_group"].eq(group_name),
                        "field_id",
                    ].astype(str)
                )
                for group_name in group_names
            }
            field_ids_by_endpoint = {
                str(endpoint): set(
                    group_df["field_id"].astype(str)
                )
                for endpoint, group_df in field_registry_df.groupby(
                    "endpoint",
                    sort=False,
                )
            } if not field_registry_df.empty else {}
            group_all_checkboxes = {}
            group_status_labels = {}
            visible_field_checkboxes = {}

            field_group_dropdown = widgets.Dropdown(
                options=[
                    (
                        f"{group_name} "
                        f"({len(field_ids_by_group[group_name])} полей)",
                        group_name,
                    )
                    for group_name in group_names
                ],
                description="Категория:",
                layout=widgets.Layout(width="95%"),
                style={"description_width": "120px"},
            )
            field_method_dropdown = widgets.Dropdown(
                options=[],
                description="Метод API:",
                layout=widgets.Layout(width="95%"),
                style={"description_width": "120px"},
            )
            field_search_input = widgets.Text(
                value="",
                description="Поиск:",
                placeholder="Русское описание, JSON-путь или имя поля",
                layout=widgets.Layout(width="95%"),
                style={"description_width": "120px"},
            )
            field_list_output = widgets.Output(
                layout=widgets.Layout(
                    border="1px solid #ddd",
                    padding="8px",
                    max_height="430px",
                    overflow_y="auto",
                    width="95%",
                )
            )
            select_category_button = widgets.Button(
                description="Выбрать всю категорию",
                icon="check",
                layout=widgets.Layout(width="220px"),
            )
            clear_category_button = widgets.Button(
                description="Снять всю категорию",
                icon="times",
                layout=widgets.Layout(width="210px"),
            )
            select_method_button = widgets.Button(
                description="Выбрать весь метод",
                icon="check",
                layout=widgets.Layout(width="200px"),
            )
            clear_method_button = widgets.Button(
                description="Снять весь метод",
                icon="times",
                layout=widgets.Layout(width="190px"),
            )

            def sync_group_controls() -> None:
                previous = state["updating"]
                state["updating"] = True

                for group_name in group_names:
                    group_ids = field_ids_by_group[group_name]
                    selected_count = len(
                        group_ids & selected_field_ids
                    )
                    checkbox = group_all_checkboxes.get(group_name)
                    label = group_status_labels.get(group_name)

                    if checkbox is not None:
                        checkbox.value = bool(
                            group_ids
                            and selected_count == len(group_ids)
                        )

                    if label is not None:
                        label.value = (
                            "<span style='color:#666'>"
                            f"Выбрано {selected_count} из {len(group_ids)}"
                            "</span>"
                        )

                state["updating"] = previous

            def sync_visible_fields() -> None:
                previous = state["updating"]
                state["updating"] = True

                for field_id, checkbox in visible_field_checkboxes.items():
                    checkbox.value = field_id in selected_field_ids

                state["updating"] = previous

            def after_field_change() -> None:
                mark_manual()
                sync_group_controls()
                sync_visible_fields()
                update_summary()

            def set_field_ids(ids: set[str], value: bool) -> None:
                if value:
                    selected_field_ids.update(ids)
                else:
                    selected_field_ids.difference_update(ids)

                after_field_change()

            category_rows = []

            for group_name in group_names:
                group_ids = field_ids_by_group[group_name]
                checkbox = widgets.Checkbox(
                    value=False,
                    description=(
                        f"Вся категория «{group_name}» "
                        f"({len(group_ids)} полей)"
                    ),
                    indent=False,
                    layout=widgets.Layout(width="560px"),
                )
                clear_button = widgets.Button(
                    description="Снять",
                    icon="times",
                    layout=widgets.Layout(width="105px"),
                )
                status_label = widgets.HTML()
                group_all_checkboxes[group_name] = checkbox
                group_status_labels[group_name] = status_label

                def on_all_group_change(
                    change,
                    *,
                    ids=group_ids,
                ) -> None:
                    if state["updating"] or change.get("name") != "value":
                        return

                    if bool(change["new"]):
                        set_field_ids(ids, True)

                checkbox.observe(
                    on_all_group_change,
                    names="value",
                )
                clear_button.on_click(
                    lambda _, ids=group_ids: set_field_ids(ids, False)
                )
                category_rows.append(
                    widgets.HBox(
                        [checkbox, clear_button, status_label],
                        layout=widgets.Layout(
                            gap="8px",
                            flex_flow="row wrap",
                            align_items="center",
                        ),
                    )
                )

            category_quick_box = widgets.VBox(category_rows)

            def refresh_methods(*_) -> None:
                group_name = field_group_dropdown.value

                if group_name is None:
                    field_method_dropdown.options = []
                    refresh_field_list()
                    return

                rows = field_registry_df[
                    field_registry_df["_ui_group"].eq(str(group_name))
                ]
                method_options = []

                for endpoint, endpoint_rows in rows.groupby(
                    "endpoint",
                    sort=False,
                ):
                    endpoint = str(endpoint)
                    summary = str(
                        endpoint_rows.iloc[0].get("summary") or ""
                    ).strip()
                    label = (
                        f"{summary} — {endpoint}"
                        if summary
                        else endpoint
                    )
                    method_options.append((label, endpoint))

                field_method_dropdown.options = method_options

                if method_options:
                    field_method_dropdown.value = method_options[0][1]

                refresh_field_list()

            def on_visible_field_change(
                change,
                *,
                field_id: str,
            ) -> None:
                if state["updating"] or change.get("name") != "value":
                    return

                if bool(change["new"]):
                    selected_field_ids.add(field_id)
                else:
                    selected_field_ids.discard(field_id)

                after_field_change()

            def refresh_field_list(*_) -> None:
                with field_list_output:
                    clear_output(wait=True)
                    visible_field_checkboxes.clear()
                    endpoint = field_method_dropdown.value

                    if not endpoint:
                        print("В выбранной категории нет доступных методов.")
                        return

                    rows = field_registry_df[
                        field_registry_df["endpoint"].astype(str).eq(
                            str(endpoint)
                        )
                    ].copy()
                    query = str(field_search_input.value or "").strip().lower()

                    if query:
                        mask = pd.Series(False, index=rows.index)

                        for column in (
                            "json_path",
                            "description",
                            "field_name",
                            "summary",
                        ):
                            if column in rows.columns:
                                mask = mask | (
                                    rows[column]
                                    .fillna("")
                                    .astype(str)
                                    .str.lower()
                                    .str.contains(query, regex=False)
                                )

                        rows = rows[mask]

                    if rows.empty:
                        print("По текущему фильтру поля не найдены.")
                        return

                    total = len(rows)

                    if total > 200:
                        print(
                            f"Найдено {total} полей. Показаны первые 200; "
                            "уточните поиск или выберите весь метод кнопкой."
                        )
                        rows = rows.head(200)

                    boxes = []

                    for row in rows.to_dict(orient="records"):
                        field_id = str(row["field_id"])
                        json_path = str(row.get("json_path") or "")
                        data_type = str(row.get("data_type") or "")
                        description = str(row.get("description") or "").strip()
                        runtime_mark = (
                            " — подтверждено тестом"
                            if bool(row.get("runtime_confirmed"))
                            else ""
                        )
                        checkbox = widgets.Checkbox(
                            value=field_id in selected_field_ids,
                            description=(
                                f"{json_path} [{data_type}]{runtime_mark}"
                            ),
                            indent=False,
                            layout=widgets.Layout(width="100%"),
                            tooltip=description,
                        )
                        checkbox.observe(
                            lambda change, current=field_id: on_visible_field_change(
                                change,
                                field_id=current,
                            ),
                            names="value",
                        )
                        visible_field_checkboxes[field_id] = checkbox
                        boxes.append(checkbox)

                    display(widgets.VBox(boxes))

            field_group_dropdown.observe(
                refresh_methods,
                names="value",
            )
            field_method_dropdown.observe(
                refresh_field_list,
                names="value",
            )
            field_search_input.observe(
                refresh_field_list,
                names="value",
            )
            select_category_button.on_click(
                lambda _: set_field_ids(
                    field_ids_by_group.get(
                        str(field_group_dropdown.value),
                        set(),
                    ),
                    True,
                )
            )
            clear_category_button.on_click(
                lambda _: set_field_ids(
                    field_ids_by_group.get(
                        str(field_group_dropdown.value),
                        set(),
                    ),
                    False,
                )
            )
            select_method_button.on_click(
                lambda _: set_field_ids(
                    field_ids_by_endpoint.get(
                        str(field_method_dropdown.value),
                        set(),
                    ),
                    True,
                )
            )
            clear_method_button.on_click(
                lambda _: set_field_ids(
                    field_ids_by_endpoint.get(
                        str(field_method_dropdown.value),
                        set(),
                    ),
                    False,
                )
            )

            def set_all_features(names: set[str]) -> None:
                previous = state["updating"]
                state["updating"] = True
                selected_feature_names.clear()
                selected_feature_names.update(names)

                for feature_name, checkbox in feature_checkboxes.items():
                    checkbox.value = feature_name in names

                state["updating"] = previous

            def select_all_features(_) -> None:
                """Выбирает все признаки, не меняя поля API."""

                set_all_features(set(all_feature_names))
                mark_manual()
                update_summary()

            def clear_all_features(_) -> None:
                """Снимает все признаки, не меняя поля API."""

                set_all_features(set())
                mark_manual()
                update_summary()

            def apply_mode(_) -> None:
                mode = str(mode_dropdown.value)
                state["mode"] = mode
                previous = state["updating"]
                state["updating"] = True

                if mode == MODE_SPARK:
                    selected_field_ids.clear()
                    set_all_features(set(spark_feature_names))
                    direct_ml_checkbox.value = False
                    profile_name_input.value = MODE_SPARK
                elif mode == MODE_SPARK_PLUS:
                    selected_field_ids.clear()
                    set_all_features(set(all_feature_names))
                    direct_ml_checkbox.value = False
                    profile_name_input.value = MODE_SPARK_PLUS
                elif mode == MODE_FULL:
                    selected_field_ids.clear()
                    selected_field_ids.update(known_field_ids)
                    set_all_features(set(all_feature_names))
                    direct_ml_checkbox.value = False
                    profile_name_input.value = MODE_FULL
                else:
                    state["updating"] = previous
                    state["dirty"] = True
                    sync_group_controls()
                    sync_visible_fields()
                    update_summary()
                    return

                state["updating"] = previous
                state["dirty"] = True
                sync_group_controls()
                sync_visible_fields()
                update_summary()

            def clear_all(_) -> None:
                previous = state["updating"]
                state["updating"] = True
                selected_field_ids.clear()
                set_all_features(set())
                direct_ml_checkbox.value = False
                mode_dropdown.value = MODE_MANUAL
                state["mode"] = MODE_MANUAL
                state["updating"] = previous
                state["dirty"] = True
                sync_group_controls()
                sync_visible_fields()
                update_summary()

            def select_everything(_) -> None:
                """Выбирает все признаки и все доступные поля API."""

                previous = state["updating"]
                state["updating"] = True
                selected_field_ids.clear()
                selected_field_ids.update(known_field_ids)
                set_all_features(set(all_feature_names))
                direct_ml_checkbox.value = False
                mode_dropdown.value = MODE_MANUAL
                state["mode"] = MODE_MANUAL
                state["updating"] = previous
                state["dirty"] = True
                sync_group_controls()
                sync_visible_fields()
                update_summary()

            mode_dropdown.observe(
                update_mode_help,
                names="value",
            )
            apply_mode_button.on_click(apply_mode)
            select_all_button.on_click(select_everything)
            clear_all_button.on_click(clear_all)
            direct_ml_checkbox.observe(
                lambda change: (
                    mark_manual(),
                    update_summary(),
                )
                if (
                    change.get("name") == "value"
                    and not state["updating"]
                )
                else None,
                names="value",
            )

            def save_selection(_) -> None:
                save_button.disabled = True

                with save_output:
                    clear_output(wait=True)

                    try:
                        ordered_fields = _ordered_values(
                            field_registry_df,
                            "field_id",
                            selected_field_ids,
                        )
                        ordered_features = _ordered_values(
                            feature_registry_df,
                            "feature_name",
                            selected_feature_names,
                        )

                        if not ordered_fields and not ordered_features:
                            raise ValueError(
                                "Выберите хотя бы один показатель, "
                                "признак или поле API."
                            )

                        if ordered_features:
                            missing_core_methods = sorted(
                                set(REQUIRED_FEATURE_ENDPOINTS)
                                - set(selected_endpoints)
                            )

                            if missing_core_methods:
                                raise ValueError(
                                    "Для выбранных признаков не хватает "
                                    "обязательных методов: "
                                    + ", ".join(missing_core_methods)
                                    + ". Вернитесь к выбору методов API."
                                )

                        selected_rows = field_registry_df[
                            field_registry_df["field_id"].astype(str).isin(
                                ordered_fields
                            )
                        ]
                        include_direct_ml = bool(
                            direct_ml_checkbox.value
                        )
                        field_options = {}

                        for row in selected_rows.to_dict(orient="records"):
                            field_id = str(row["field_id"])
                            field_options[field_id] = {
                                "include_in_dataset": True,
                                "include_in_ml": bool(
                                    include_direct_ml
                                    and row.get("direct_ml_allowed")
                                ),
                                "output_column": str(row["output_column"]),
                                "data_type": str(row.get("data_type") or ""),
                            }

                        endpoint_status = {
                            str(row.get("endpoint")): str(
                                row.get("execution_status")
                            )
                            for row in (
                                methods.get("execution_registry") or []
                            )
                            if isinstance(row, Mapping)
                        }
                        executable_statuses = {
                            "automatic",
                            "configured",
                            "legacy_adapter",
                            "company_lookup",
                            "requires_dependency",
                        }
                        blocked_methods = sorted(
                            {
                                str(row["endpoint"])
                                for row in selected_rows.to_dict(
                                    orient="records"
                                )
                                if endpoint_status.get(
                                    str(row["endpoint"])
                                ) not in executable_statuses
                            }
                        )

                        if blocked_methods:
                            raise ValueError(
                                "Выбраны поля методов, которые пока не могут "
                                "выполняться: "
                                + ", ".join(blocked_methods[:10])
                                + ". Снимите эти поля или настройте методы "
                                "в предыдущей ячейке."
                            )

                        mode_to_save = str(state["mode"] or MODE_MANUAL)
                        profile_name = str(
                            profile_name_input.value or ""
                        ).strip()
                        save_result = save_field_profile(
                            project_root=project_root,
                            profile_name=profile_name,
                            field_registry_df=field_registry_df,
                            selected_field_ids=ordered_fields,
                            selected_feature_names=ordered_features,
                            field_options=field_options,
                            feature_profile_name=mode_to_save,
                        )
                        export_result = export_field_profile_package(
                            project_root=project_root,
                            profile_save_result=save_result,
                        )
                        feature_source_state = _json_ready(
                            active_feature_source_state(
                                Path(project_root)
                            )
                        )
                        feature_source_sha = str(
                            (
                                feature_source_state
                                if isinstance(
                                    feature_source_state,
                                    Mapping,
                                )
                                else {}
                            ).get("sha256")
                            or ""
                        ).strip()
                        fields_update = {
                                "profile_name": profile_name,
                                "selection_mode": mode_to_save,
                                "selected_field_ids": ordered_fields,
                                "field_options": field_options,
                                "selected_feature_names": ordered_features,
                                "feature_profile_name": mode_to_save,
                                "spark_source_feature_names": [
                                    name
                                    for name in ordered_features
                                    if name in spark_feature_names
                                ],
                                "calculated_feature_names": [
                                    name
                                    for name in ordered_features
                                    if name not in spark_feature_names
                                ],
                                "profile_path": save_result["profile_path"],
                                "excel_path": export_result["excel_path"],
                                "zip_path": export_result["zip_path"],
                            }

                        if feature_source_sha:
                            fields_update[
                                "custom_features_sha256"
                            ] = feature_source_sha
                            fields_update[
                                "custom_features_source"
                            ] = feature_source_state

                        updated = update_workflow(
                            workflow,
                            fields=fields_update,
                            test={
                                "required": True,
                                "completed": False,
                                "company_code": None,
                                "configuration_hash": None,
                                "summary": {},
                                "output_directory": None,
                                "runtime_fields_path": None,
                                "excel_path": None,
                                "zip_path": None,
                            },
                        )
                        updated = mark_stage(
                            updated,
                            "fields",
                            ready=True,
                            message="Поля и признаки сохранены.",
                        )
                        updated = invalidate_after(
                            updated,
                            "fields",
                            reason=(
                                "Состав датасета изменён. Повторите тест "
                                "одной компании."
                            ),
                        )
                        workflow_save = save_workflow(
                            updated,
                            workflow_root=Path(project_root) / "workflow",
                        )
                        source_count = len(
                            set(ordered_features) & spark_feature_names
                        )
                        calculated_count = len(ordered_features) - source_count

                        print("ПОЛЯ И ПРИЗНАКИ СОХРАНЕНЫ")
                        print(f"Режим: {mode_to_save}")
                        print(f"Показатели СПАРК: {source_count}")
                        print(f"Расчётные признаки: {calculated_count}")
                        print(f"Дополнительные поля API: {len(ordered_fields)}")
                        print("Целевая переменная: target_default — автоматически")
                        print("Тест одной компании: обязателен")
                        print("Следующий шаг: проверьте набор на одной компании.")
                        protected_download_buttons(
                            {
                                "Скачать полный ZIP": export_result["zip_path"],
                                "Скачать Excel": export_result["excel_path"],
                                "Скачать JSON": export_result["json_path"],
                            },
                            recommended_label="Скачать полный ZIP",
                        )
                        workflow.clear()
                        workflow.update(workflow_save["workflow"])
                        state["dirty"] = False
                        configuration_save_button.disabled = False

                        if not str(
                            configuration_name_input.value or ""
                        ).strip():
                            configuration_name_input.value = (
                                profile_name
                                or mode_to_save
                                or "Моя конфигурация сборки"
                            )

                        update_summary()

                    except Exception as error:
                        show_error(
                            stage="Сохранение полей и признаков",
                            error=error,
                            action=(
                                "Исправьте указанный выбор и снова нажмите "
                                "«Сохранить выбранный набор»."
                            ),
                        )
                    finally:
                        # Кнопка блокируется только на время записи файлов.
                        # После успешного сохранения или ошибки пользователь
                        # может изменить состав и сохранить профиль повторно.
                        save_button.disabled = False





            def refresh_saved_configurations(_) -> None:
                """Обновляет список профилей без автоматического выбора."""

                configuration_profile_dropdown.disabled = True
                configuration_apply_button.disabled = True

                try:
                    profiles, errors = (
                        _discover_build_configuration_profiles(
                            project_root
                        )
                    )
                    profile_map = {
                        str(profile["profile_path"]): profile
                        for profile in profiles
                    }
                    configuration_loader_state["profiles"] = profile_map
                    configuration_loader_state["invalid_profiles"] = errors
                    options = [
                        (
                            "Выберите сохранённую конфигурацию",
                            "",
                        )
                    ]
                    options.extend(
                        (
                            _configuration_profile_label(profile),
                            str(profile["profile_path"]),
                        )
                        for profile in profiles
                    )
                    configuration_profile_dropdown.options = options
                    configuration_profile_dropdown.value = ""
                    configuration_preview_html.value = (
                        "<div style='padding:10px 12px;background:#f5f7fa;"
                        "border-left:4px solid #78909c;line-height:1.5'>"
                        f"Найдено сохранённых конфигураций: <b>{len(profiles)}</b>. "
                        "Ничего не выбрано и текущие настройки не изменены."
                        + (
                            "<br><span style='color:#b26a00'>"
                            f"Повреждённых или неподдерживаемых файлов: "
                            f"{len(errors)}.</span>"
                            if errors
                            else ""
                        )
                        + "</div>"
                    )
                except Exception as error:
                    configuration_profile_dropdown.options = [
                        (
                            "Не удалось прочитать список конфигураций",
                            "",
                        )
                    ]
                    configuration_profile_dropdown.value = ""
                    configuration_preview_html.value = (
                        "<div style='padding:10px 12px;background:#ffebee;"
                        "border-left:4px solid #c62828'>"
                        f"{_safe(error)}"
                        "</div>"
                    )
                finally:
                    configuration_profile_dropdown.disabled = False


            def preview_saved_configuration(change) -> None:
                """Показывает профиль, но не применяет его."""

                if change.get("name") != "value":
                    return

                selected_path = str(change.get("new") or "")
                payload = configuration_loader_state["profiles"].get(
                    selected_path
                )

                if not isinstance(payload, Mapping):
                    configuration_apply_button.disabled = True
                    return

                summary = payload.get("summary") or {}
                report_year = summary.get("target_report_year")
                horizon = summary.get("target_horizon_months")
                target_text = str(
                    summary.get("target_name") or "target_default"
                )

                if report_year is not None:
                    target_text += f", отчётный год {report_year}"

                if horizon is not None:
                    target_text += f", горизонт {horizon} мес."

                configuration_preview_html.value = (
                    "<div style='padding:12px 14px;background:#eef4fb;"
                    "border-left:5px solid #1976d2;line-height:1.65'>"
                    "<b>Конфигурация открыта только для просмотра</b><br>"
                    f"Название: <b>{_safe(payload.get('profile_name'))}</b><br>"
                    f"Методы API: <b>{int(summary.get('method_count') or 0)}</b><br>"
                    "Набор признаков: "
                    f"<b>{_safe(summary.get('feature_mode') or 'Ручная настройка')}</b><br>"
                    f"ML-признаков: <b>{int(summary.get('feature_count') or 0)}</b><br>"
                    "Дополнительных полей API: "
                    f"<b>{int(summary.get('field_count') or 0)}</b><br>"
                    f"Target: <b>{_safe(target_text)}</b><br>"
                    f"Версия профиля: <b>{int(payload.get('revision') or 1)}</b><br>"
                    "Текущие настройки пока не изменены. Для восстановления "
                    "нажмите кнопку ниже."
                    "</div>"
                )
                configuration_apply_button.disabled = False


            def apply_saved_configuration(_) -> None:
                """Применяет выбранный профиль и сохраняет его в workflow."""

                selected_path = str(
                    configuration_profile_dropdown.value or ""
                )
                configuration_apply_button.disabled = True

                with configuration_load_output:
                    clear_output(wait=True)

                    try:
                        if not selected_path:
                            raise ValueError(
                                "Сначала выберите конфигурацию в списке."
                            )

                        payload = _load_build_configuration_profile(
                            selected_path
                        )
                        restored_workflow = (
                            _apply_build_configuration_profile(
                                project_root=project_root,
                                workflow=workflow,
                                payload=payload,
                                catalog_result=catalog_result,
                                known_feature_names=set(
                                    known_feature_names
                                ),
                            )
                        )
                        workflow.clear()
                        workflow.update(restored_workflow)
                        summary = payload.get("summary") or {}

                        print("ПОЛНАЯ КОНФИГУРАЦИЯ ПРИМЕНЕНА")
                        print(
                            "Название: "
                            f"{payload.get('profile_name')}"
                        )
                        print(
                            "Методов API: "
                            f"{int(summary.get('method_count') or 0)}"
                        )
                        print(
                            "Набор признаков: "
                            f"{summary.get('feature_mode') or 'Ручная настройка'}"
                        )
                        print(
                            "ML-признаков: "
                            f"{int(summary.get('feature_count') or 0)}"
                        )
                        print(
                            "Дополнительных полей API: "
                            f"{int(summary.get('field_count') or 0)}"
                        )
                        print(
                            "Target: "
                            f"{summary.get('target_name') or 'target_default'}"
                        )
                        print()
                        print(
                            "Настройки пунктов 4–5.3 восстановлены и "
                            "сохранены на Google Диске."
                        )
                        print(
                            "Чтобы интерфейс ниже точно перестроился под "
                            "восстановленные методы и поля, повторно "
                            "запустите только ячейку 5.3."
                        )
                        print(
                            "После полного перезапуска среды выполните пункты "
                            "2, 2.1 и 3. Для быстрого восстановления всей "
                            "конфигурации откройте сохранённый профиль в пункте 4."
                        )

                        configuration_preview_html.value = (
                            "<div style='padding:12px 14px;background:#e8f5e9;"
                            "border-left:5px solid #2e7d32;line-height:1.6'>"
                            "<b>Конфигурация применена</b><br>"
                            f"{_safe(payload.get('profile_name'))}<br>"
                            "Повторно запустите только ячейку 5.3, чтобы "
                            "редактор ниже открылся уже с восстановленным "
                            "набором."
                            "</div>"
                        )
                        editor_section.layout.display = "none"

                    except Exception as error:
                        show_error(
                            stage="Открытие полной конфигурации",
                            error=error,
                            action=(
                                "Выберите другой профиль или обновите список. "
                                "Текущие настройки и сохранённые файлы не "
                                "изменялись."
                            ),
                        )
                        configuration_apply_button.disabled = False


            configuration_refresh_button.on_click(
                refresh_saved_configurations
            )
            configuration_profile_dropdown.observe(
                preview_saved_configuration,
                names="value",
            )
            configuration_apply_button.on_click(
                apply_saved_configuration
            )


            def save_full_configuration(_) -> None:
                """Сохраняет точный снимок пунктов 4–5.3 отдельным JSON."""

                configuration_save_button.disabled = True

                with configuration_output:
                    clear_output(wait=True)

                    try:
                        if state["dirty"]:
                            raise RuntimeError(
                                "В выборе признаков есть несохранённые "
                                "изменения. Сначала нажмите "
                                "«Сохранить выбранный набор»."
                            )

                        configuration_name = str(
                            configuration_name_input.value or ""
                        ).strip()
                        result = _save_build_configuration(
                            project_root=project_root,
                            workflow=workflow,
                            profile_name=configuration_name,
                        )
                        summary = result["summary"]
                        status_text = {
                            "created": "создана",
                            "updated": "обновлена",
                            "unchanged": (
                                "уже существовала без изменений"
                            ),
                        }.get(
                            result["status"],
                            "сохранена",
                        )

                        print("ПОЛНАЯ КОНФИГУРАЦИЯ СОХРАНЕНА")
                        print(
                            f"Название: {configuration_name}"
                        )
                        print(
                            f"Состояние: конфигурация {status_text}"
                        )
                        print(
                            "Методов API: "
                            f"{summary['method_count']}"
                        )
                        print(
                            "Набор признаков: "
                            f"{summary['feature_mode']}"
                        )
                        print(
                            "ML-признаков: "
                            f"{summary['feature_count']}"
                        )
                        print(
                            "Дополнительных полей API: "
                            f"{summary['field_count']}"
                        )
                        print(
                            "Target: "
                            f"{summary['target_name']}"
                        )

                        if (
                            summary.get("target_report_year")
                            is not None
                        ):
                            print(
                                "Отчётный год target: "
                                f"{summary['target_report_year']}"
                            )

                        if (
                            summary.get(
                                "target_horizon_months"
                            )
                            is not None
                        ):
                            print(
                                "Горизонт target: "
                                f"{summary['target_horizon_months']} мес."
                            )

                        print(
                            "Версия профиля: "
                            f"{result['revision']}"
                        )
                        print(
                            "SHA-256 конфигурации: "
                            f"{result['configuration_hash']}"
                        )
                        print(
                            "Путь: "
                            f"{result['profile_path']}"
                        )

                        if result.get("archived_path"):
                            print(
                                "Предыдущая версия сохранена: "
                                f"{result['archived_path']}"
                            )

                        print(
                            "Этот профиль можно открыть "
                            "в пункте 4 и применить целиком."
                        )
                        protected_download_buttons(
                            {
                                "Скачать JSON конфигурации": (
                                    result["profile_path"]
                                ),
                            },
                            recommended_label=(
                                "Скачать JSON конфигурации"
                            ),
                        )

                    except Exception as error:
                        show_error(
                            stage=(
                                "Сохранение полной конфигурации"
                            ),
                            error=error,
                            action=(
                                "Сначала сохраните методы, target, "
                                "параметры и выбранный набор признаков. "
                                "Затем повторите сохранение профиля."
                            ),
                        )
                    finally:
                        configuration_save_button.disabled = (
                            False
                        )


            configuration_save_button.on_click(
                save_full_configuration
            )

            save_button.on_click(save_selection)

            select_all_features_button = widgets.Button(
                description="Выбрать все признаки",
                icon="check-square",
                layout=widgets.Layout(width="220px"),
            )
            clear_all_features_button = widgets.Button(
                description="Снять все признаки",
                icon="times",
                layout=widgets.Layout(width="210px"),
            )
            select_all_features_button.on_click(select_all_features)
            clear_all_features_button.on_click(clear_all_features)

            select_all_api_button = widgets.Button(
                description="Выбрать все поля API",
                icon="check-square",
                layout=widgets.Layout(width="230px"),
            )
            clear_all_api_button = widgets.Button(
                description="Снять все поля API",
                icon="times",
                layout=widgets.Layout(width="220px"),
            )
            select_all_api_button.on_click(
                lambda _: set_field_ids(set(known_field_ids), True)
            )
            clear_all_api_button.on_click(
                lambda _: set_field_ids(set(known_field_ids), False)
            )

            feature_section = widgets.VBox(
                [
                    widgets.HTML(
                        "<p><b>Показатели СПАРК</b> — исходные значения и "
                        "готовые счётчики. <b>Расчётные признаки</b> — "
                        "прозрачные формулы ноутбука. Откройте категорию, "
                        "чтобы посмотреть формулы и изменить выбор.</p>"
                    ),
                    widgets.HBox(
                        [
                            select_all_features_button,
                            clear_all_features_button,
                        ],
                        layout=widgets.Layout(
                            gap="8px",
                            flex_flow="row wrap",
                        ),
                    ),
                    feature_accordion,
                ]
            )
            category_accordion = widgets.Accordion(
                children=[category_quick_box],
                selected_index=None,
            )
            category_accordion.set_title(
                0,
                "Быстрый выбор целых категорий API",
            )
            api_section = widgets.VBox(
                [
                    widgets.HTML(
                        "<p>Дополнительные поля — это полный ответ выбранных "
                        "методов СПАРК. Галочка категории выбирает её целиком; "
                        "кнопка «Снять» очищает категорию. Для точной настройки "
                        "используйте метод и поиск ниже.</p>"
                    ),
                    widgets.HBox(
                        [
                            select_all_api_button,
                            clear_all_api_button,
                        ],
                        layout=widgets.Layout(
                            gap="8px",
                            flex_flow="row wrap",
                        ),
                    ),
                    category_accordion,
                    field_group_dropdown,
                    widgets.HBox(
                        [select_category_button, clear_category_button],
                        layout=widgets.Layout(
                            gap="8px",
                            flex_flow="row wrap",
                        ),
                    ),
                    field_method_dropdown,
                    widgets.HBox(
                        [select_method_button, clear_method_button],
                        layout=widgets.Layout(
                            gap="8px",
                            flex_flow="row wrap",
                        ),
                    ),
                    field_search_input,
                    field_list_output,
                ]
            )
            developer_section = widgets.VBox(
                [
                    widgets.HTML(
                        "<p>По умолчанию дополнительные поля API сохраняются "
                        "в основной или связанных таблицах, но не подаются "
                        "напрямую в модель. Включайте настройку ниже только "
                        "для проверенных числовых скалярных полей. Источники "
                        "target всё равно будут исключены из ML.</p>"
                    ),
                    direct_ml_checkbox,
                ]
            )
            details_accordion = widgets.Accordion(
                children=[
                    feature_section,
                    api_section,
                    developer_section,
                ],
                selected_index=None,
            )
            details_accordion.set_title(
                0,
                f"Настроить признаки ({len(all_feature_names)})",
            )
            details_accordion.set_title(
                1,
                f"Настроить дополнительные поля API ({len(known_field_ids)})",
            )
            details_accordion.set_title(
                2,
                "Дополнительные настройки разработчика",
            )

            configuration_loader_box = widgets.VBox(
                [
                    widgets.HTML(
                        "<p>Откройте ранее сохранённый снимок методов API, "
                        "target, параметров и признаков. Выбор в списке "
                        "ничего не меняет до нажатия кнопки "
                        "«Применить выбранную конфигурацию».</p>"
                    ),
                    configuration_refresh_button,
                    configuration_profile_dropdown,
                    configuration_preview_html,
                    configuration_apply_button,
                    configuration_load_output,
                ],
                layout=widgets.Layout(
                    width="100%",
                    padding="8px",
                ),
            )
            configuration_loader_accordion = widgets.Accordion(
                children=[configuration_loader_box],
                selected_index=None,
            )
            configuration_loader_accordion.set_title(
                0,
                "Открыть сохранённую полную конфигурацию",
            )

            configuration_section = widgets.VBox(
                [
                    widgets.HTML(
                        "<div style='padding:12px 14px;background:#f5f7fa;"
                        "border-left:4px solid #1976d2;line-height:1.55'>"
                        "<b>Сохранить полную конфигурацию для повторного "
                        "использования</b><br>"
                        "Сначала сохраните выбранный набор выше. Затем "
                        "сохраните одним JSON методы API, их параметры, "
                        "target, ML-признаки и дополнительные поля. "
                        "Авторизация, входной файл и результаты запусков "
                        "в профиль не входят."
                        "</div>"
                    ),
                    configuration_name_input,
                    configuration_save_button,
                    configuration_output,
                ],
                layout=widgets.Layout(
                    width="100%",
                    border="1px solid #d8dee6",
                    padding="10px",
                    margin="12px 0 0 0",
                ),
            )

            editor_section = widgets.VBox(
                [
                    widgets.HTML(
                        "<h3>Соберите состав датасета</h3>"
                        "<p>Для быстрого запуска выберите один шаблон и "
                        "нажмите «Применить шаблон». Для точной настройки "
                        "раскройте нужный раздел. Целевая переменная "
                        "добавляется отдельно и здесь не выбирается.</p>"
                    ),
                    profile_name_input,
                    mode_dropdown,
                    mode_help,
                    widgets.HBox(
                        [
                            apply_mode_button,
                            select_all_button,
                            clear_all_button,
                        ],
                        layout=widgets.Layout(
                            gap="8px",
                            flex_flow="row wrap",
                        ),
                    ),
                    summary_html,
                    details_accordion,
                    save_button,
                    save_output,
                    configuration_section,
                ],
                layout=widgets.Layout(width="100%"),
            )

            interface = widgets.VBox(
                [
                    configuration_loader_accordion,
                    editor_section,
                ],
                layout=widgets.Layout(width="100%"),
            )

            refresh_saved_configurations(None)
            update_mode_help()
            sync_group_controls()
            refresh_methods()
            update_summary()
            display(interface)
            return interface


        # Подменяется только интерфейс. Каталог, сохранение workflow,
        # проверка target и дальнейший конвейер остаются общими.
        notebook_steps.show_field_feature_selector = (
            _show_simple_field_feature_selector
        )

        FIELD_STEP = notebook_steps.field_selection_step(
            project_root=PROJECT_ROOT
        )


## 6. Проверка настроек на одной компании

<details>
<summary><b>Техническая справка: как работает контрольный запуск</b></summary>

В этой главе текущая конфигурация конвейера проверяется на одной организации перед массовой обработкой списка компаний.

Программа использует тот же производственный механизм, который затем применяется при массовом запуске. Проверка не является отдельной демонстрацией и не формирует искусственный результат.

Во время теста программа:

* выполняет выбранные методы REST API СПАРК;
* применяет параметры запросов из пункта 5.2;
* использует выбранные признаки и дополнительные поля из пункта 5.3;
* рассчитывает ML-признаки;
* формирует целевую переменную `target_default`;
* создаёт основную таблицу и ML-матрицу;
* сохраняет журналы, Excel-файл и ZIP-архив;
* фиксирует текущую конфигурацию как проверенную.

**Что нужно сделать**

1. Убедитесь, что выполнены пункты 2, 2.1 и 3, а настройки датасета подготовлены одним из двух способов: применена полная конфигурация в пункте 4 либо вручную сохранены пункты 4, 5.1, 5.2 и 5.3.
2. Запустите кодовую ячейку главы 6.
3. Проверьте информационную панель с текущими настройками.
4. Оставьте предложенный ИНН либо введите ИНН или ОГРН другой нефинансовой организации.
5. При необходимости включите повторное получение данных из СПАРК.
6. Нажмите кнопку **«Проверить одну компанию»**.
7. Дождитесь итоговой сводки.
8. Переходите к следующему разделу только после сообщения **«Проверка успешно пройдена»**.

Эту главу нельзя пропускать. Массовый запуск разрешается только после успешной проверки текущей конфигурации.

**Требования к проверочной компании**

Можно указать:

* 10-значный ИНН российского юридического лица;
* 13-значный ОГРН российского юридического лица.

Используйте действующую нефинансовую организацию, по которой в СПАРК имеется бухгалтерская отчётность за выбранный отчётный период.

Не рекомендуется использовать:

* банки;
* страховые компании;
* кредитные организации;
* индивидуальных предпринимателей;
* ликвидированные организации без подходящей отчётности;
* организации, по которым отсутствуют финансовые данные за отчётный год.

**Предварительная проверка настроек**

Перед показом интерфейса программа проверяет готовность предыдущих этапов.

Проверяется:

* наличие корневой папки проекта;
* подключение Google Диска;
* наличие авторизованной сессии СПАРК;
* сохранение выбранных методов API;
* наличие подготовленного плана выполнения методов;
* подключение исполнителя методов детализации;
* наличие актуального правила целевой переменной;
* соответствие отчётного года правилу target;
* наличие профиля полей и признаков;
* количество выбранных ML-признаков;
* количество выбранных дополнительных полей API.

Если обязательный этап не выполнен, программа не начинает тест и показывает, какой пункт необходимо повторно запустить.

После перезапуска среды Google Colab повторно выполните пункты 2, 2.1 и 3. Сохранённые методы, target, параметры, признаки и поля остаются на Google Диске. Штатный обработчик target загружается в пункте 2, поэтому запускать пункт 5.1 только ради его подключения больше не требуется.

**Что показывает начальная сводка**

Перед тестом выводятся:

* количество выбранных методов API;
* количество строк подготовленного плана;
* количество подключённых правил детализации;
* количество выбранных ML-признаков;
* количество дополнительных полей API;
* отчётный год целевой переменной;
* горизонт наблюдения target.

Пример:

```text
Настройки готовы к проверке
Выбранных методов API: 41
Строк подготовленного плана: 41
Подключено правил детализации: 11
Выбрано ML-признаков: 79
Выбрано дополнительных полей API: 2254
Отчётный год target: 2024
Горизонт target: 12 мес.
```

Числа могут изменяться при выборе другого состава методов, признаков или полей.

**Использование сохранённых ответов**

Флажок **«Запросить данные заново, не использовать сохранённые ответы»** управляет использованием кэша.

Если флажок выключен:

* программа использует ранее сохранённые ответы, если они подходят текущему запросу;
* отсутствующие ответы запрашиваются у СПАРК;
* тест выполняется быстрее;
* уменьшается количество повторных API-запросов.

Если флажок включён:

* программа заново запрашивает данные;
* ранее сохранённые ответы для этой проверки не используются как основной источник;
* тест занимает больше времени;
* увеличивается количество HTTP-запросов.

При обычной проверке флажок рекомендуется оставить выключенным.

Включайте его, если:

* изменились данные организации;
* изменились параметры запросов;
* предыдущий ответ был неполным;
* необходимо проверить актуальное состояние API;
* требуется исключить влияние старого кэша.

**Какие настройки используются**

Тест загружает актуальные настройки из общего `workflow`.

Используются:

```python
workflow["methods"]
workflow["target"]
workflow["fields"]
```

Из них программа получает:

* выбранные endpoint;
* порядок выполнения методов;
* параметры запросов;
* правила детализации;
* отчётный год;
* горизонт target;
* источники целевой переменной;
* список ML-признаков;
* список дополнительных полей;
* настройки прямого включения полей API в ML.

Настройки не восстанавливаются из виджетов предыдущих ячеек. Они читаются из сохранённого состояния проекта на Google Диске.

**Выполнение методов СПАРК**

Для проверочной организации выполняется подготовленный план методов.

В итоговой сводке отдельно показываются:

* уникальные endpoint;
* операции в журнале;
* фактические HTTP-запросы;
* ответы, использованные из кэша;
* методы, вернувшие данные;
* служебные операции и адаптеры;
* методы без данных по организации;
* методы, недоступные по лицензии;
* операции, завершившиеся ошибкой.

Один endpoint может использоваться несколькими операциями, если для него подготовлены разные параметры.

Например, один метод государственных контрактов может выполняться отдельно:

* по 44-ФЗ;
* по 223-ФЗ.

Поэтому количество уникальных endpoint, операций и фактических HTTP-запросов может отличаться.

**Статусы методов**

Основные группы статусов:

| Группа                              | Значение                                                                              |
| ----------------------------------- | ------------------------------------------------------------------------------------- |
| Данные получены                     | Метод выполнен, и в ответе обнаружены данные                                          |
| Служебная операция, адаптер или кэш | Результат получен через служебный механизм, проверенный адаптер или сохранённый ответ |
| Данных нет                          | Метод выполнен корректно, но у выбранной организации нет соответствующих объектов     |
| Нет лицензии                        | Метод или данные недоступны по текущему тарифу СПАРК                                  |
| Требует исправления                 | Возникла ошибка авторизации, параметров, HTTP, ответа API или технического выполнения |

Отсутствие данных у конкретной организации не считается ошибкой.

Недоступность метода по лицензии также учитывается отдельно и не приводит к падению всего конвейера.

Массовый запуск не разрешается, если в тесте остаются операции, требующие исправления.

**Дополнительные поля API**

Для выбранных полей программа показывает:

* общее количество;
* количество полей, получивших значение;
* количество полей без значения;
* долю незаполненных полей;
* причины отсутствия значений.

Причины разделяются на группы:

* у организации нет соответствующих данных или связанных объектов;
* необязательное поле отсутствует в ответе;
* метод недоступен по лицензии;
* значение не получено из-за ошибки операции.

Пример:

```text
ВЫБРАННЫЕ ПОЛЯ API — всего: 2254
• Значения получены: 749
• Без значения у этой компании: 1505
```

Большое количество пустых полей при полной выгрузке является нормальным.

OpenAPI-схема описывает все потенциально доступные поля метода, однако отдельная организация может не иметь:

* судебных дел;
* контрактов;
* лицензий;
* залогов;
* недвижимости;
* вакансий;
* сертификатов;
* связанных объектов;
* некоторых необязательных атрибутов.

Пустое значение у одной компании не означает, что поле бесполезно для всего датасета.

**ML-признаки**

После получения исходных данных программа рассчитывает выбранные ML-признаки.

Проверяется:

* сформирована ли ML-матрица;
* создана ли строка для тестовой компании;
* присутствуют ли все выбранные признаки;
* отсутствуют ли потерянные колонки;
* сколько дополнительных API-полей включено непосредственно в ML.

Пример успешного результата:

```text
ML-ПРИЗНАКИ — выбрано: 79
• Рассчитано и найдено в ML-матрице: 79
• Не найдено в ML-матрице: 0
```

Успешной считается проверка, при которой все выбранные признаки присутствуют в итоговой ML-матрице.

Отдельные значения признаков могут оставаться `NaN`, если для выбранной компании не хватает исходных данных. Это не означает, что колонка не рассчитана или потеряна.

Например, значение может отсутствовать, если у компании:

* нет телефона;
* нет нескольких руководителей;
* отсутствуют связанные адреса;
* отсутствует уставный капитал;
* нет показателя, необходимого для расчёта отношения;
* нет финансового значения за требуемый период.

**Прямые поля API в ML**

Дополнительные поля API и зарегистрированные ML-признаки являются разными группами данных.

Дополнительное поле API может:

* сохраняться в основной таблице;
* сохраняться в связанной таблице;
* использоваться как источник расчётного признака;
* быть напрямую включено в ML-матрицу.

Строка:

```text
Прямые поля API в ML: 0 из 0
```

означает, что в текущем профиле ни одно дополнительное поле не было отдельно отмечено для прямого включения в ML.

Это не является ошибкой. Зарегистрированные ML-признаки при этом рассчитываются независимо.

**Целевая переменная**

Правило целевой переменной берётся из пункта 5.1.

В результате формируются как минимум:

```text
target_default
target_status
target_status_label
target_report_year
target_observation_date
target_horizon_end
target_source_coverage
```

Для `target_default` используются значения:

| Значение        | Смысл                                                |
| --------------- | ---------------------------------------------------- |
| `1`             | В установленном горизонте обнаружено событие дефолта |
| `0`             | Событие дефолта в горизонте не обнаружено            |
| пустое значение | Недостаточно данных для уверенного определения       |

Пример:

```text
target_default: 0
Статус: Дефолт в горизонте не обнаружен
Отчётный год: 2024
Покрытие источников: complete
```

Статус покрытия `complete` означает, что обязательные источники целевой переменной были проверены в соответствии с сохранённым правилом.

Наличие значения `0` считается корректным результатом, а не отсутствием target.

**Основная таблица и ML-матрица**

Проверка формирует две основные таблицы.

**Основной результат**

Содержит:

* идентификаторы организации;
* сведения СПАРК;
* выбранные поля API;
* технические статусы;
* целевую переменную;
* контрольные данные.

**ML-матрица**

Содержит:

* идентификаторы;
* выбранные признаки;
* расчётные признаки;
* целевую переменную;
* технические поля, необходимые для проверки происхождения строки.

Для одной тестовой компании нормальный результат:

```text
Строк основного результата: 1
Строк ML-матрицы: 1
```

Если основная строка существует, но ML-матрица пуста, проверка считается неуспешной.

Такое возможно, например, если у выбранной компании отсутствуют данные, необходимые базовому финансовому ядру.

В этом случае рекомендуется повторить тест на другой действующей нефинансовой организации с доступной бухгалтерской отчётностью.

**Нормализация `spark_id`**

Перед объединением частей датасета центральный модуль приводит `spark_id` к строковому типу.

Это предотвращает ошибку Pandas, когда:

* в одной таблице `spark_id` хранится как число;
* в другой таблице он хранится как строка или объект.

Нормализация находится в центральном модуле:

```text
src/pipeline_v2.py
```

Поэтому одинаково применяется:

* в проверке одной компании;
* в массовой обработке;
* после перезапуска Colab;
* независимо от порядка запуска пользовательских ячеек.

**Что сохраняется**

Каждая проверка сохраняется в отдельной папке на Google Диске.

Обычно создаются:

* Excel-файл проверки;
* ZIP-архив;
* журнал методов;
* HTTP-телеметрия;
* реестр фактически обнаруженных полей;
* итоговая JSON-сводка;
* связанные таблицы;
* сведения о текущей конфигурации.

Результат теста также записывается в:

```python
workflow["test"]
```

Сохраняются:

* код проверенной компании;
* признак успешного завершения;
* отпечаток конфигурации;
* пути к результатам;
* краткая сводка;
* время выполнения.

**Защита от устаревшей проверки**

После успешного теста программа сохраняет отпечаток текущей конфигурации.

Если после проверки изменить:

* методы API;
* параметры методов;
* правило target;
* отчётный год;
* горизонт target;
* признаки;
* дополнительные поля;
* настройки прямого включения полей в ML,

предыдущая проверка перестаёт считаться актуальной.

Перед массовым запуском программа сравнивает:

* сохранённый отпечаток успешного теста;
* отпечаток текущей конфигурации.

При несовпадении необходимо повторно выполнить главу 6.

**Файлы для скачивания**

После теста появляется интерфейс:

```text
Настроить состав Excel — Проверка_одной_компании.xlsx
```

Он позволяет выбрать вкладки, которые войдут в скачиваемый Excel-файл.

В зависимости от результатов могут быть доступны:

* основная таблица;
* ML-матрица;
* журнал методов;
* HTTP-телеметрия;
* контроль качества;
* связанные таблицы API;
* реестр полей;
* техническая сводка.

Полный набор также сохраняется на Google Диске независимо от скачивания через браузер.

**Расширенный критерий успешности**

Проверка считается успешно пройденной, если одновременно выполнены основные условия:

* основной результат содержит строку компании;
* ML-матрица сформирована;
* все выбранные признаки присутствуют;
* `target_default` сформирован;
* диагностические поля target присутствуют;
* нет операций со статусом ошибки;
* не все выбранные API-поля отсутствуют;
* результаты сохранены;
* конфигурационный отпечаток соответствует текущим настройкам.

Пример итогового сообщения:

```text
Проверка успешно пройдена
Текущая конфигурация сохранена как проверенная.
Можно переходить к следующему разделу.
```

**Что эта глава не делает**

Проверка одной компании:

* не изменяет исходный список организаций;
* не запускает массовую обработку;
* не добавляет новые методы;
* не изменяет правило target;
* не изменяет выбранные признаки;
* не заменяет профиль полей;
* не объединяет тестовую компанию с будущим массовым датасетом;
* не считает отсутствие необязательного поля ошибкой.

Тестовые результаты сохраняются отдельно от массовых запусков.

**Успешный результат**

Успешный вывод должен содержать:

* сообщение **«Проверка успешно пройдена»**;
* одну строку основного результата;
* одну строку ML-матрицы;
* ноль операций, требующих исправления;
* все выбранные ML-признаки;
* рассчитанный `target_default`;
* статус покрытия источников target;
* сообщение о сохранении текущей конфигурации как проверенной;
* элементы скачивания Excel или ZIP.

После успешной проверки переходите к следующей главе и настраивайте массовый запуск.

</details>


In [ ]:
# @title 6. Проверить настройки на одной компании

from __future__ import annotations

try:
    from src.notebook_runtime import run_notebook_cell
except Exception as _cell_runtime_error:
    print(
        "Не удалось запустить этап «Проверка одной компании». "
        "Сначала выполните пункт 2. "
        f"Причина: {type(_cell_runtime_error).__name__}: {_cell_runtime_error}"
    )
else:
    with run_notebook_cell(
        stage="Проверка одной компании",
        next_step="Исправьте указанный этап и повторите пункт 6.",
        project_root=globals().get("PROJECT_ROOT"),
    ):
        import importlib
        import inspect
        from collections.abc import Mapping
        from pathlib import Path

        import ipywidgets as widgets
        import pandas as pd
        import requests
        from IPython.display import clear_output, display

        try:
            import src.company_pipeline as company_pipeline
            import src.method_executor as method_executor
            import src.notebook_steps as notebook_steps
            import src.notebook_ui as notebook_ui
            import src.pipeline_v2 as pipeline_v2
            import src.workflow as workflow_module
            from src.file_loader import validate_legal_entity_code
            from src.notebook_steps import test_company_step
            from src.runtime_readiness import validate_runtime_readiness
            from src.method_policy import assess_method_log
            from src.user_features_loader import active_feature_source_state
        except ModuleNotFoundError as error:
            raise RuntimeError(
                "Технические модули автономного проекта не найдены. "
                "Сначала выполните ячейку 2, затем повторно запустите ячейку 6."
            ) from error


        ПРИМЕР_ИНН = "1215235283"
        ОЖИДАЕМАЯ_ВЕРСИЯ_ДЕТАЛИЗАЦИИ = "spark_detail_executor_v2"
        ОЖИДАЕМАЯ_ВЕРСИЯ_TARGET = "spark_target_calculator_v2"


        # Перечитываем только модуль пользовательского вывода. Центральный
        # pipeline_v2 не перезагружается, чтобы не удалить подключённый
        # в пункте 2.1 обработчик детализации и штатный target-runtime ядра.
        notebook_ui = importlib.reload(notebook_ui)

        try:
            _DOWNLOAD_SELECTOR_READY = (
                "Настроить состав Excel"
                in inspect.getsource(
                    notebook_ui.protected_download_buttons
                )
            )
        except (AttributeError, OSError, TypeError):
            _DOWNLOAD_SELECTOR_READY = False

        if not _DOWNLOAD_SELECTOR_READY:
            raise RuntimeError(
                "В техническом модуле не найдена актуальная функция выбора "
                "вкладок Excel. Повторно выполните обновлённую ячейку 2, "
                "а затем снова запустите ячейку 6."
            )

        from src.notebook_ui import show_error



        КОД_КОМПАНИИ = widgets.Text(
            value=ПРИМЕР_ИНН,
            description="ИНН или ОГРН:",
            placeholder="Введите 10-значный ИНН или 13-значный ОГРН",
            layout=widgets.Layout(width="95%"),
            style={"description_width": "140px"},
        )

        ОБНОВИТЬ_ДАННЫЕ = widgets.Checkbox(
            value=False,
            description=(
                "Запросить данные заново, не использовать сохранённые ответы"
            ),
            indent=False,
            layout=widgets.Layout(width="100%"),
            tooltip=(
                "Обычно галочку ставить не нужно. Используйте её, если хотите "
                "повторно запросить актуальные ответы СПАРК."
            ),
        )

        КНОПКА_ПРОВЕРКИ = widgets.Button(
            description="Проверить одну компанию",
            icon="play",
            button_style="success",
            layout=widgets.Layout(width="280px"),
            tooltip=(
                "Запустить текущие методы, признаки и правило target "
                "на одной организации"
            ),
        )

        ВЫВОД_ПРОВЕРКИ = widgets.Output(
            layout=widgets.Layout(
                width="100%",
                min_height="60px",
            )
        )

        TEST_RESULT = None
        TEST_COMPANY_CODE = ""


        СТАТУСЫ_С_ДАННЫМИ = frozenset(
            {
                "available_with_data",
            }
        )

        СТАТУСЫ_СЛУЖЕБНОГО_УСПЕХА = frozenset(
            {
                "success",
            }
        )

        СТАТУСЫ_БЕЗ_ДАННЫХ = frozenset(
            {
                "available_without_data",
                "available_without_related_objects",
                "not_found",
            }
        )

        СТАТУСЫ_БЕЗ_ЛИЦЕНЗИИ = frozenset(
            {
                "license_unavailable",
            }
        )

        СТАТУСЫ_ОШИБОК = frozenset(
            {
                "authentication_error",
                "temporary_error",
                "http_error",
                "technical_error",
                "invalid_parameters",
                "unexpected_response",
                "skipped_due_to_missing_runtime",
                "skipped_due_to_dependency",
            }
        )

        СТАТУСЫ_ПРОПУЩЕНЫ_ПО_ПЛАНУ = frozenset(
            {
                "skipped_by_plan",
                "skipped_not_applicable",
            }
        )


        def _перехватить_стандартный_вывод_теста(**kwargs):
            """Выполняет общий тест и откладывает таблицу и скачивания до сводки."""

            сохранённые_функции = {
                "show_panel": notebook_steps.show_panel,
                "show_error": notebook_steps.show_error,
                "dataframe_preview": notebook_steps.dataframe_preview,
                "protected_download_buttons": (
                    notebook_steps.protected_download_buttons
                ),
            }
            отложенный_вывод = {}
            исходный_обработчик_компании = (
                pipeline_v2.process_company_workflow
            )
            обновлять_данные = bool(
                kwargs.get("force_refresh")
            )

            def _панель(заголовок, сообщение, *, kind="info"):
                if заголовок in {
                    "Тест пройден",
                    "Тест требует внимания",
                }:
                    return None

                return сохранённые_функции["show_panel"](
                    заголовок,
                    сообщение,
                    kind=kind,
                )

            def _ошибка(*, stage, error, action):
                # Общий test_company_step сам показывает ошибку и затем возвращает
                # её вызывающей ячейке. Сохраняем данные, но не рисуем первую
                # техническую панель, чтобы пользователь увидел один итоговый вывод.
                отложенный_вывод["internal_error"] = {
                    "stage": stage,
                    "error": str(error),
                    "action": action,
                }
                return None

            def _таблица(*args, **table_kwargs):
                отложенный_вывод["table"] = (
                    args,
                    table_kwargs,
                )
                return None

            def _кнопки(*args, **button_kwargs):
                отложенный_вывод["downloads"] = (
                    args,
                    button_kwargs,
                )
                return None

            def _обработать_компанию_в_режиме_теста(**call_kwargs):
                """Изолирует кэш теста от режима будущего массового запуска."""

                workflow = workflow_module.normalize_workflow(
                    call_kwargs.get("workflow") or {}
                )
                policies = dict(
                    workflow.get("policies") or {}
                )
                policies["data_mode"] = (
                    "refresh_current_file"
                    if обновлять_данные
                    else "reuse"
                )
                workflow["policies"] = policies
                call_kwargs["workflow"] = workflow
                return исходный_обработчик_компании(
                    **call_kwargs
                )

            notebook_steps.show_panel = _панель
            notebook_steps.show_error = _ошибка
            notebook_steps.dataframe_preview = _таблица
            notebook_steps.protected_download_buttons = _кнопки
            pipeline_v2.process_company_workflow = (
                _обработать_компанию_в_режиме_теста
            )

            try:
                результат = test_company_step(**kwargs)
            finally:
                pipeline_v2.process_company_workflow = (
                    исходный_обработчик_компании
                )
                notebook_steps.show_panel = (
                    сохранённые_функции["show_panel"]
                )
                notebook_steps.show_error = (
                    сохранённые_функции["show_error"]
                )
                notebook_steps.dataframe_preview = (
                    сохранённые_функции["dataframe_preview"]
                )
                notebook_steps.protected_download_buttons = (
                    сохранённые_функции[
                        "protected_download_buttons"
                    ]
                )

            return (
                результат,
                отложенный_вывод,
                сохранённые_функции,
            )


        def _количество_статусов(статусы, разрешённые):
            """Считает методы с указанными техническими статусами."""

            return int(
                статусы.isin(разрешённые).sum()
            )


        def _разбивка_отсутствующих_полей(результат):
            """Объясняет пустые поля с учётом всех операций одного endpoint."""

            выбранные_поля = результат.get(
                "selected_fields_df"
            )
            журнал = результат.get("method_log_df")
            отсутствующие = {
                str(value)
                for value in (
                    результат.get(
                        "missing_selected_field_ids"
                    )
                    or []
                )
            }
            итог = {
                "без_данных": 0,
                "без_лицензии": 0,
                "из_за_ошибок": 0,
                "необязательные": len(отсутствующие),
            }

            if (
                not isinstance(выбранные_поля, pd.DataFrame)
                or выбранные_поля.empty
                or not {
                    "field_id",
                    "endpoint",
                }.issubset(выбранные_поля.columns)
                or not isinstance(журнал, pd.DataFrame)
                or журнал.empty
                or not {
                    "endpoint",
                    "result_status",
                }.issubset(журнал.columns)
            ):
                return итог

            статусы_endpoint = {
                str(endpoint): {
                    str(value)
                    for value in group[
                        "result_status"
                    ].dropna().astype(str)
                }
                for endpoint, group in журнал.groupby(
                    журнал["endpoint"].astype(str),
                    dropna=False,
                )
            }
            пропущенные_строки = выбранные_поля[
                выбранные_поля["field_id"]
                .astype(str)
                .isin(отсутствующие)
            ]
            итог = {
                "без_данных": 0,
                "без_лицензии": 0,
                "из_за_ошибок": 0,
                "необязательные": 0,
            }

            for endpoint in пропущенные_строки[
                "endpoint"
            ].astype(str):
                статусы = статусы_endpoint.get(
                    endpoint,
                    set(),
                )

                if статусы & СТАТУСЫ_ОШИБОК:
                    итог["из_за_ошибок"] += 1
                elif статусы & СТАТУСЫ_БЕЗ_ЛИЦЕНЗИИ:
                    итог["без_лицензии"] += 1
                elif (
                    статусы
                    and статусы.issubset(
                        СТАТУСЫ_БЕЗ_ДАННЫХ
                    )
                ):
                    итог["без_данных"] += 1
                else:
                    итог["необязательные"] += 1

            return итог


        def _показать_понятный_итог(результат, код_компании):
            """Показывает точную русскую сводку методов, полей, признаков и target."""

            журнал = результат.get("method_log_df")
            основная_таблица = результат.get(
                "main_dataset_df"
            )
            ml_матрица = результат.get(
                "ml_matrix_df"
            )
            выбранные_поля = результат.get(
                "selected_fields_df"
            )
            отсутствующие_поля = list(
                результат.get(
                    "missing_selected_field_ids"
                )
                or []
            )
            workflow = результат.get("workflow") or {}
            fields_state = workflow.get("fields") or {}
            выбранные_признаки = [
                str(value)
                for value in (
                    fields_state.get(
                        "selected_feature_names"
                    )
                    or []
                )
            ]

            if isinstance(журнал, pd.DataFrame):
                журнал = журнал.copy()
            else:
                журнал = pd.DataFrame()

            if (
                not журнал.empty
                and "result_status" in журнал.columns
            ):
                статусы = (
                    журнал["result_status"]
                    .fillna("unknown")
                    .astype(str)
                )
            else:
                статусы = pd.Series(dtype=str)

            операций_всего = len(журнал)
            уникальных_методов = (
                int(
                    журнал["endpoint"]
                    .dropna()
                    .astype(str)
                    .nunique()
                )
                if (
                    not журнал.empty
                    and "endpoint" in журнал.columns
                )
                else 0
            )
            http_запросов = (
                int(
                    pd.to_numeric(
                        журнал.get(
                            "http_request_count",
                            pd.Series(dtype=float),
                        ),
                        errors="coerce",
                    ).fillna(0).sum()
                )
                if not журнал.empty
                else 0
            )
            ответов_из_кэша = (
                int(
                    pd.to_numeric(
                        журнал.get(
                            "cache_hit_count",
                            pd.Series(dtype=float),
                        ),
                        errors="coerce",
                    ).fillna(0).sum()
                )
                if not журнал.empty
                else 0
            )
            операций_с_данными = _количество_статусов(
                статусы,
                СТАТУСЫ_С_ДАННЫМИ,
            )
            служебно_успешных = _количество_статусов(
                статусы,
                СТАТУСЫ_СЛУЖЕБНОГО_УСПЕХА,
            )
            операций_без_данных = _количество_статусов(
                статусы,
                СТАТУСЫ_БЕЗ_ДАННЫХ,
            )
            операций_без_лицензии = _количество_статусов(
                статусы,
                СТАТУСЫ_БЕЗ_ЛИЦЕНЗИИ,
            )
            оценка_журнала = assess_method_log(журнал)
            операций_с_ошибками = max(
                _количество_статусов(статусы, СТАТУСЫ_ОШИБОК),
                int(оценка_журнала.get("critical_count") or 0),
            )
            операций_пропущено_по_плану = _количество_статусов(
                статусы,
                СТАТУСЫ_ПРОПУЩЕНЫ_ПО_ПЛАНУ,
            )
            учтённых_операций = sum(
                (
                    операций_с_данными,
                    служебно_успешных,
                    операций_без_данных,
                    операций_без_лицензии,
                    операций_с_ошибками,
                    операций_пропущено_по_плану,
                )
            )
            операций_с_другим_статусом = max(
                0,
                операций_всего - учтённых_операций,
            )
            строк_результата = (
                len(основная_таблица)
                if isinstance(
                    основная_таблица,
                    pd.DataFrame,
                )
                else 0
            )
            строк_ml = (
                len(ml_матрица)
                if isinstance(
                    ml_матрица,
                    pd.DataFrame,
                )
                else 0
            )
            колонки_ml = (
                set(ml_матрица.columns.astype(str))
                if isinstance(ml_матрица, pd.DataFrame)
                else set()
            )
            рассчитанные_признаки = [
                name
                for name in выбранные_признаки
                if name in колонки_ml
            ]
            отсутствующие_признаки = [
                name
                for name in выбранные_признаки
                if name not in колонки_ml
            ]
            прямые_ml_поля = [
                str(options.get("output_column") or "")
                for options in (
                    fields_state.get("field_options")
                    or {}
                ).values()
                if (
                    isinstance(options, Mapping)
                    and bool(options.get("include_in_ml"))
                    and str(
                        options.get("output_column") or ""
                    ).strip()
                )
            ]
            прямые_ml_получены = sum(
                column in колонки_ml
                for column in прямые_ml_поля
            )
            полей_всего = (
                len(выбранные_поля)
                if isinstance(
                    выбранные_поля,
                    pd.DataFrame,
                )
                else 0
            )
            полей_без_значения = len(
                отсутствующие_поля
            )
            полей_получено = max(
                0,
                полей_всего - полей_без_значения,
            )
            доля_без_значения = (
                100 * полей_без_значения
                / полей_всего
                if полей_всего
                else 0.0
            )
            причины_пустых_полей = (
                _разбивка_отсутствующих_полей(
                    результат
                )
            )
            target_result = результат.get(
                "target_result"
            ) or {}
            target_значение = target_result.get(
                "target_default"
            )
            target_статус = str(
                target_result.get(
                    "target_status_label"
                )
                or target_result.get(
                    "target_status"
                )
                or "не определён"
            )
            target_год = target_result.get(
                "target_report_year"
            )
            target_покрытие = str(
                target_result.get(
                    "target_source_coverage"
                )
                or "не указано"
            )

            if операций_с_ошибками:
                заголовок = (
                    "Проверка завершена частично: "
                    "есть ошибки операций"
                )
                вид = "warning"
                решение = (
                    "Массовый запуск пока не начинайте: "
                    f"требуют исправления операций — "
                    f"{операций_с_ошибками}."
                )
            elif результат.get("success"):
                заголовок = "Проверка успешно пройдена"
                вид = "success"
                решение = (
                    "Текущая конфигурация сохранена как проверенная. "
                    "Можно переходить к следующему разделу."
                )
            else:
                заголовок = "Проверка не пройдена"
                вид = "error"
                решение = (
                    "Откройте журнал методов или тестовый ZIP "
                    "и устраните указанную причину."
                )

            строки = [
                f"Компания: {код_компании}",
                (
                    "Время выполнения: "
                    f"{notebook_ui.format_duration(результат.get('elapsed_seconds', 0))}"
                ),
                f"Строк основного результата: {строк_результата}",
                f"Строк ML-матрицы: {строк_ml}",
                "",
                f"МЕТОДЫ И ЗАПРОСЫ СПАРК",
                f"• Уникальных endpoint: {уникальных_методов}",
                f"• Операций в журнале: {операций_всего}",
                f"• Фактических HTTP-запросов: {http_запросов}",
                f"• Ответов использовано из кэша: {ответов_из_кэша}",
                f"• Данные получены: {операций_с_данными}",
                (
                    "• Служебный поиск, адаптер или кэш отработали: "
                    f"{служебно_успешных}"
                ),
                (
                    "• Выполнены корректно, но данных у компании нет: "
                    f"{операций_без_данных}"
                ),
                (
                    "• Недоступны по лицензии или тарифу: "
                    f"{операций_без_лицензии}"
                ),
                (
                    "• Пропущены по правилам плана: "
                    f"{операций_пропущено_по_плану}"
                ),
                f"• Требуют исправления: {операций_с_ошибками}",
            ]

            if операций_с_другим_статусом:
                строки.append(
                    "• Другой служебный статус: "
                    f"{операций_с_другим_статусом}"
                )

            строки.extend(
                [
                    "",
                    f"ВЫБРАННЫЕ ПОЛЯ API — всего: {полей_всего}",
                    f"• Значения получены: {полей_получено}",
                    (
                        "• Без значения у этой компании: "
                        f"{полей_без_значения} "
                        f"({доля_без_значения:.1f}%)"
                    ).replace(".", ","),
                    (
                        "  — у компании нет соответствующих данных "
                        "или связанных объектов: "
                        f"{причины_пустых_полей['без_данных']}"
                    ),
                    (
                        "  — необязательные поля отсутствуют "
                        "в полученных ответах: "
                        f"{причины_пустых_полей['необязательные']}"
                    ),
                    (
                        "  — методы недоступны по лицензии: "
                        f"{причины_пустых_полей['без_лицензии']}"
                    ),
                    (
                        "  — не получены из-за ошибок операций: "
                        f"{причины_пустых_полей['из_за_ошибок']}"
                    ),
                    "",
                    f"ML-ПРИЗНАКИ — выбрано: {len(выбранные_признаки)}",
                    f"• Рассчитано и найдено в ML-матрице: {len(рассчитанные_признаки)}",
                    f"• Не найдено в ML-матрице: {len(отсутствующие_признаки)}",
                    (
                        "• Прямые поля API в ML: "
                        f"{прямые_ml_получены} из {len(прямые_ml_поля)}"
                    ),
                    "",
                    "ЦЕЛЕВАЯ ПЕРЕМЕННАЯ",
                    f"• target_default: {target_значение}",
                    f"• Статус: {target_статус}",
                    f"• Отчётный год: {target_год}",
                    f"• Покрытие источников: {target_покрытие}",
                    "",
                    (
                        "Пустое API-поле не означает ошибку само по себе: "
                        "у другой компании оно может быть заполнено."
                    ),
                    решение,
                ]
            )
            notebook_ui.show_panel(
                заголовок,
                "\n".join(строки),
                kind=вид,
            )

            if отсутствующие_признаки:
                notebook_ui.show_panel(
                    "Какие признаки не попали в ML-матрицу",
                    "\n".join(
                        f"• {name}"
                        for name in отсутствующие_признаки
                    ),
                    kind="error",
                )

            if (
                операций_с_ошибками
                and not журнал.empty
            ):
                ошибки = журнал[
                    журнал["result_status"]
                    .astype(str)
                    .isin(СТАТУСЫ_ОШИБОК)
                ]
                описание_ошибок = []

                for row in ошибки.to_dict(
                    orient="records"
                ):
                    название = str(
                        row.get("summary")
                        or row.get("endpoint")
                        or "Неизвестный метод"
                    )
                    адрес = str(
                        row.get("endpoint") or ""
                    )
                    статус = str(
                        row.get("status_label")
                        or "Ошибка выполнения"
                    )
                    http_код = row.get(
                        "http_status_code"
                    )

                    if (
                        pd.notna(http_код)
                        and str(http_код).strip()
                    ):
                        try:
                            http_значение = str(
                                int(float(http_код))
                            )
                        except (TypeError, ValueError):
                            http_значение = str(http_код)

                        http_текст = (
                            f", HTTP {http_значение}"
                        )
                    else:
                        http_текст = ""
                    описание_ошибок.append(
                        f"• {название} [{адрес}] — "
                        f"{статус}{http_текст}"
                    )

                notebook_ui.show_panel(
                    (
                        "Какие операции нужно исправить — "
                        f"{операций_с_ошибками}"
                    ),
                    "\n".join(описание_ошибок),
                    kind="error",
                )


        def _показать_отложенные_результаты(
            отложенный_вывод,
            сохранённые_функции,
        ):
            """Возвращает предпросмотр и единые кнопки после понятной сводки."""

            if "table" in отложенный_вывод:
                args, kwargs = отложенный_вывод[
                    "table"
                ]
                сохранённые_функции[
                    "dataframe_preview"
                ](*args, **kwargs)

            if "downloads" in отложенный_вывод:
                args, kwargs = отложенный_вывод[
                    "downloads"
                ]
                сохранённые_функции[
                    "protected_download_buttons"
                ](*args, **kwargs)


        def _восстановить_служебные_метки_готовности():
            """Исправляет только отсутствующие метки, созданные прежней сборкой."""

            root_value = globals().get("PROJECT_ROOT")
            root = Path(root_value) if root_value else None
            if root is None or not root.is_dir():
                return

            # Обработчики target загружаются штатно вместе с ядром в пункте 2.
            # Проверка готовности дополнительно ожидает служебный атрибут,
            # который создаётся только во время первого запроса компании.
            # Создаём его заранее, но только если все реальные обработчики на месте.
            target_handlers_ready = all(
                (
                    hasattr(
                        pipeline_v2,
                        "_spark_target_base_process_company_workflow",
                    ),
                    getattr(
                        pipeline_v2.process_company_workflow,
                        "__name__",
                        "",
                    ) == "_process_company_with_target",
                    getattr(
                        pipeline_v2._execution_registry_dataframe,
                        "__name__",
                        "",
                    ) == "_target_execution_registry_dataframe",
                    getattr(
                        workflow_module.dataset_fingerprint_payload,
                        "__name__",
                        "",
                    ) == "_target_dataset_fingerprint_payload",
                )
            )
            if (
                target_handlers_ready
                and not hasattr(
                    company_pipeline,
                    "_spark_target_runtime",
                )
            ):
                company_pipeline._spark_target_runtime = None

            # В релизе 4.1.0 пункт 5.3 сохраняет выбранные признаки,
            # но не записывает SHA активного файла формул. Это создаёт
            # ложное сообщение об изменении файла. Дописываем только
            # отсутствующую метку. Уже существующий SHA не заменяем:
            # настоящее изменение формул по-прежнему будет обнаружено.
            workflow = workflow_module.load_workflow(
                workflow_root=root / "workflow",
                create_if_missing=False,
            )
            fields = workflow.get("fields") or {}
            stored_sha = str(
                fields.get("custom_features_sha256") or ""
            ).strip()
            if stored_sha:
                return

            source_state = active_feature_source_state(root)
            current_sha = str(
                source_state.get("sha256") or ""
            ).strip()
            if not current_sha:
                return

            updated = workflow_module.update_workflow(
                workflow,
                fields={
                    "custom_features_sha256": current_sha,
                    "custom_features_source": dict(source_state),
                },
            )
            workflow_module.save_workflow(
                updated,
                workflow_root=root / "workflow",
            )


        def _проверить_готовность_конвейера():
            """Использует единую проверку workflow и текущей памяти Colab."""

            _восстановить_служебные_метки_готовности()

            проверка = validate_runtime_readiness(
                project_root=globals().get("PROJECT_ROOT"),
                session=globals().get("SESSION"),
                require_test=False,
                require_input=False,
            )
            if not проверка.get("ready"):
                строки = []
                for issue in (проверка.get("issues") or {}).values():
                    строки.append(f"• {issue.get('message')}")
                    строки.append(f"  Что сделать: {issue.get('action')}")
                raise RuntimeError(
                    "Конвейер не готов к проверке:\n" + "\n".join(строки)
                )

            return {
                "workflow": проверка["workflow"],
                **dict(проверка.get("summary") or {}),
            }


        def _показать_готовность(готовность):
            """Показывает, какая сохранённая конфигурация будет проверена."""

            notebook_ui.show_panel(
                "Настройки готовы к проверке",
                "\n".join(
                    [
                        (
                            "Выбранных методов API: "
                            f"{готовность['selected_endpoint_count']}"
                        ),
                        (
                            "Строк подготовленного плана: "
                            f"{готовность['execution_row_count']}"
                        ),
                        (
                            "Подключено правил детализации: "
                            f"{готовность['detail_method_count']}"
                        ),
                        (
                            "Выбрано ML-признаков: "
                            f"{готовность['selected_feature_count']}"
                        ),
                        (
                            "Выбрано дополнительных полей API: "
                            f"{готовность['selected_field_count']}"
                        ),
                        (
                            "Отчётный год target: "
                            f"{готовность['target_report_year']}"
                        ),
                        (
                            "Горизонт target: "
                            f"{готовность['target_horizon_months']} мес."
                        ),
                        "Проверка будет сохранена отдельно на Google Диске.",
                    ]
                ),
                kind="info",
            )


        def _ошибки_контракта_результата(
            результат,
            workflow_before,
        ):
            """Проверяет обязательные выходы производственного теста."""

            ошибки = []
            основная = результат.get("main_dataset_df")
            ml_матрица = результат.get("ml_matrix_df")
            журнал = результат.get("method_log_df")
            выбранные_признаки = [
                str(value)
                for value in (
                    (
                        workflow_before.get("fields")
                        or {}
                    ).get("selected_feature_names")
                    or []
                )
            ]

            if (
                not isinstance(основная, pd.DataFrame)
                or основная.empty
            ):
                ошибки.append(
                    "Основная таблица не сформирована."
                )
            else:
                обязательные_target = {
                    "target_default",
                    "target_status",
                    "target_source_coverage",
                }
                отсутствующие_target = sorted(
                    обязательные_target
                    - set(основная.columns.astype(str))
                )

                if отсутствующие_target:
                    ошибки.append(
                        "В основной таблице отсутствуют поля target: "
                        + ", ".join(отсутствующие_target)
                        + "."
                    )

            if (
                not isinstance(ml_матрица, pd.DataFrame)
                or ml_матрица.empty
            ):
                ошибки.append(
                    "ML-матрица не сформирована."
                )
            else:
                ml_columns = set(
                    ml_матрица.columns.astype(str)
                )
                missing_features = [
                    name
                    for name in выбранные_признаки
                    if name not in ml_columns
                ]

                if missing_features:
                    preview = ", ".join(
                        missing_features[:10]
                    )
                    suffix = (
                        "..."
                        if len(missing_features) > 10
                        else ""
                    )
                    ошибки.append(
                        "В ML-матрице отсутствуют выбранные признаки: "
                        f"{preview}{suffix}"
                    )

                if "target_default" not in ml_columns:
                    ошибки.append(
                        "В ML-матрице отсутствует target_default."
                    )

            if isinstance(журнал, pd.DataFrame) and not журнал.empty:
                if "result_status" in журнал.columns:
                    error_count = int(
                        журнал["result_status"]
                        .astype(str)
                        .isin(СТАТУСЫ_ОШИБОК)
                        .sum()
                    )

                    if error_count:
                        ошибки.append(
                            "В журнале остались операции с ошибками: "
                            f"{error_count}."
                        )
            else:
                ошибки.append(
                    "Журнал выполнения методов не сформирован."
                )

            returned_workflow = результат.get(
                "workflow"
            ) or {}
            returned_test = returned_workflow.get(
                "test"
            ) or {}
            current_hash = (
                workflow_module.test_configuration_hash(
                    returned_workflow
                )
            )

            if (
                returned_test.get("configuration_hash")
                != current_hash
            ):
                ошибки.append(
                    "Сохранённый тест не соответствует текущему "
                    "отпечатку конфигурации."
                )

            for key, label in (
                ("excel_path", "тестовый Excel"),
                ("zip_path", "тестовый ZIP"),
                ("runtime_fields_path", "реестр фактических полей"),
            ):
                path_value = str(
                    результат.get(key) or ""
                ).strip()

                if not path_value or not Path(path_value).is_file():
                    ошибки.append(
                        f"Не сохранён {label}."
                    )

            return ошибки


        def _пометить_тест_непройденным(
            результат,
            ошибки,
        ):
            """Исправляет workflow, если расширенный контроль нашёл проблему."""

            workflow = результат.get("workflow")

            if not isinstance(workflow, Mapping):
                return

            test_state = dict(
                workflow.get("test") or {}
            )
            summary = dict(
                test_state.get("summary") or {}
            )
            summary["contract_errors"] = list(
                ошибки
            )
            test_state.update(
                {
                    "completed": False,
                    "summary": summary,
                }
            )
            updated = workflow_module.update_workflow(
                workflow,
                test=test_state,
            )
            updated = workflow_module.mark_stage(
                updated,
                "test",
                ready=False,
                message=(
                    "Расширенная проверка результата не пройдена."
                ),
            )
            saved = workflow_module.save_workflow(
                updated,
                workflow_root=(
                    Path(PROJECT_ROOT) / "workflow"
                ),
            )
            результат["workflow"] = saved["workflow"]
            результат["success"] = False
            результат["contract_errors"] = list(
                ошибки
            )


        def _понятная_причина_ошибки(error):
            """Заменяет внутренние имена Python на понятное описание проблемы."""

            message = str(error)
            lowered = message.lower()

            if "_spark_original_method_selection_panel" in lowered:
                return (
                    "После обновления технической ячейки 2 в памяти Colab "
                    "осталась устаревшая функция вывода из пункта 4. "
                    "Настройки методов, признаков и target при этом не потеряны."
                )

            if "has no attribute" in lowered and "_spark_" in lowered:
                return (
                    "В текущем сеансе Colab осталась устаревшая служебная "
                    "функция от предыдущей версии одной из ячеек. "
                    "Сохранённые настройки на Google Диске не повреждены."
                )

            return message


        def _рекомендация_после_ошибки(error):
            """Возвращает действие, соответствующее фактической причине ошибки."""

            message = str(error).lower()

            if "_spark_original_method_selection_panel" in message:
                return (
                    "Повторно запустите пункт 6. Ячейка уже восстановила "
                    "стандартный интерфейс. Если сообщение повторится, "
                    "выполните пункты 2 → 2.1 → 3, затем снова пункт 6. "
                    "Сохранённую конфигурацию повторно создавать не нужно."
                )
            if "не завершены обязательные этапы" in message:
                return (
                    "Выполните перечисленные в сообщении пункты сверху вниз, "
                    "нажимая кнопки сохранения внутри каждой ячейки, затем "
                    "снова запустите пункт 6."
                )

            if "детализац" in message:
                return "Повторно выполните пункт 2.1, затем снова запустите ячейку 6."
            if "сесс" in message or "авторизац" in message:
                return "Повторно выполните пункт 3 — авторизацию в СПАРК."
            if (
                "обработчик target" in message
                or "target runtime" in message
                or "target-runtime" in message
            ):
                return (
                    "Повторно выполните пункты 2 → 2.1 → 3, "
                    "затем снова запустите пункт 6."
                )

            if "target" in message or "целев" in message:
                return (
                    "Правило target отсутствует или устарело. "
                    "Примените полную конфигурацию в пункте 4 либо вручную "
                    "сохраните пункты 5.1 → 5.2 → 5.3, затем снова запустите пункт 6."
                )
            if "параметр" in message or "план выполнения" in message:
                return "Повторно выполните пункт 5.2."
            if "полей" in message or "признак" in message:
                return "Повторно сохраните набор в пункте 5.3."
            if "инн" in message or "огрн" in message or "код" in message:
                return (
                    "Введите корректный 10-значный ИНН или "
                    "13-значный ОГРН нефинансовой организации."
                )

            return (
                "Проверьте сообщение выше и повторите соответствующий "
                "предыдущий пункт ноутбука."
            )


        def _запустить_проверку(_):
            """Проверяет код и запускает общий конвейер для одной компании."""

            global TEST_RESULT
            global TEST_COMPANY_CODE

            КНОПКА_ПРОВЕРКИ.disabled = True
            КНОПКА_ПРОВЕРКИ.description = "Выполняется проверка..."

            with ВЫВОД_ПРОВЕРКИ:
                clear_output(wait=True)

                try:
                    готовность = (
                        _проверить_готовность_конвейера()
                    )
                    _показать_готовность(
                        готовность
                    )
                    проверка_кода = validate_legal_entity_code(
                        КОД_КОМПАНИИ.value
                    )

                    if not проверка_кода.get("is_valid"):
                        raise ValueError(
                            проверка_кода.get("reason")
                            or (
                                "Введите корректный ИНН или ОГРН "
                                "юридического лица."
                            )
                        )

                    TEST_COMPANY_CODE = str(
                        проверка_кода["normalized_code"]
                    )
                    (
                        TEST_RESULT,
                        отложенный_вывод,
                        сохранённые_функции,
                    ) = _перехватить_стандартный_вывод_теста(
                        project_root=PROJECT_ROOT,
                        session=globals().get(
                            "SESSION"
                        ),
                        run_test=True,
                        company_code=(
                            TEST_COMPANY_CODE
                        ),
                        force_refresh=bool(
                            ОБНОВИТЬ_ДАННЫЕ.value
                        ),
                    )

                    if TEST_RESULT.get("error"):
                        raise RuntimeError(
                            _понятная_причина_ошибки(
                                TEST_RESULT["error"]
                            )
                        )

                    ошибки_контракта = (
                        _ошибки_контракта_результата(
                            TEST_RESULT,
                            готовность["workflow"],
                        )
                    )

                    if ошибки_контракта:
                        _пометить_тест_непройденным(
                            TEST_RESULT,
                            ошибки_контракта,
                        )
                        notebook_ui.show_panel(
                            "Расширенная проверка не пройдена",
                            "\n".join(
                                f"• {message}"
                                for message in ошибки_контракта
                            ),
                            kind="error",
                        )

                    _показать_понятный_итог(
                        TEST_RESULT,
                        TEST_COMPANY_CODE,
                    )
                    _показать_отложенные_результаты(
                        отложенный_вывод,
                        сохранённые_функции,
                    )

                except Exception as error:
                    TEST_RESULT = {
                        "success": False,
                        "error": str(error),
                    }
                    show_error(
                        stage="Проверка одной компании",
                        error=error,
                        action=(
                            _рекомендация_после_ошибки(
                                error
                            )
                        ),
                    )

                finally:
                    КНОПКА_ПРОВЕРКИ.disabled = False
                    КНОПКА_ПРОВЕРКИ.description = (
                        "Проверить одну компанию"
                    )


        КНОПКА_ПРОВЕРКИ.on_click(_запустить_проверку)

        ИНТЕРФЕЙС_ПРОВЕРКИ = widgets.VBox(
            [
                widgets.HTML(
                    "<h3>Проверка настроек перед массовым запуском</h3>"
                    "<p>Программа выполнит для одной <b>нефинансовой "
                    "организации</b> тот же производственный конвейер, "
                    "который затем будет использоваться для всего списка: "
                    "проверит методы, поля, ML-признаки и "
                    "<code>target_default</code>.</p>"
                    "<p>В поле уже подставлен корректный пример ИНН. Можно "
                    "сразу нажать кнопку либо заменить его на ИНН/ОГРН "
                    "другой нефинансовой организации.</p>"
                    "<p>Тест не изменяет исходный список компаний. Excel, ZIP, "
                    "журналы и состояние проверки сохраняются отдельно на "
                    "вашем Google Диске.</p>"
                ),
                КОД_КОМПАНИИ,
                ОБНОВИТЬ_ДАННЫЕ,
                widgets.HTML(
                    "<div style='padding:10px 12px;background:#e3f2fd;"
                    "border-left:4px solid #1976d2;line-height:1.5'>"
                    "<b>Когда включать обновление данных?</b><br>"
                    "Оставьте галочку выключенной при обычной проверке: "
                    "программа сможет использовать совместимые сохранённые "
                    "ответы. Включите её, если нужно заново выполнить запросы "
                    "СПАРК для этой компании. Настройка массового запуска из "
                    "следующих разделов на поведение этой галочки не влияет."
                    "</div>"
                ),
                КНОПКА_ПРОВЕРКИ,
                ВЫВОД_ПРОВЕРКИ,
            ],
            layout=widgets.Layout(width="100%"),
        )

        display(ИНТЕРФЕЙС_ПРОВЕРКИ)

        with ВЫВОД_ПРОВЕРКИ:
            try:
                _ГОТОВНОСТЬ_ПРИ_ОТКРЫТИИ = (
                    _проверить_готовность_конвейера()
                )
                _показать_готовность(
                    _ГОТОВНОСТЬ_ПРИ_ОТКРЫТИИ
                )
            except Exception as error:
                КНОПКА_ПРОВЕРКИ.disabled = True
                show_error(
                    stage="Подготовка проверки одной компании",
                    error=error,
                    action=(
                        _рекомендация_после_ошибки(
                            error
                        )
                    ),
                )


## 7. Входной файл и параметры запуска

Загрузите таблицу с ИНН или ОГРН компаний, выберите нужный лист и при необходимости укажите название столбца вручную.

Если столбец называется `ИНН`, `ОГРН`, `company_code` или `код компании`, программа попробует определить его автоматически.

После этого выберите режим обновления данных, настройку ликвидированных компаний и размер контрольного сохранения, затем нажмите **«Подготовить входной файл»**.

<details>
<summary><b>Техническая справка</b></summary>

Поддерживаются файлы `.csv`, `.xls`, `.xlsx` и `.xlsb`.

В выбранном столбце должны находиться:

* ИНН юридического лица — 10 цифр;
* либо ОГРН юридического лица — 13 цифр.

Если название столбца нестандартное, укажите его вручную точно так, как оно записано в таблице. Программа проверит значения, исключит некорректные строки и удалит дубликаты.

Файл можно загрузить с компьютера или выбрать на личном Google Диске. В обоих случаях рабочая копия сохраняется в папке `input_files/` проекта.

Режим **«Использовать сохранённые ответы»** позволяет продолжить совместимый предыдущий запуск и не повторять уже успешно выполненные запросы.

Режим **«Обновить компании текущего файла»** заново обрабатывает все корректные компании выбранного файла.

**Размер контрольного сохранения**

Этот параметр определяет, через сколько обработанных компаний программа будет сохранять промежуточный результат на Google Диск.

Например, значение `25` означает, что после каждых 25 компаний программа сохраняет:

* уже полученные ответы СПАРК;
* результаты обработанных компаний;
* журналы выполнения;
* текущее состояние запуска;
* информацию о том, с какого места продолжать работу.

Контрольное сохранение защищает от потери прогресса, если:

* среда Google Colab отключилась;
* закончился сеанс;
* пропало интернет-соединение;
* выполнение было остановлено вручную;
* произошла временная ошибка.

После повторного запуска ноутбука программа найдёт совместимое сохранение на Google Диске, загрузит уже полученные результаты и продолжит работу с оставшихся компаний. Повторно загружать исходный файл и заново запрашивать уже обработанные компании обычно не потребуется.

Рекомендуемое значение — `25`.

Меньшее значение сохраняет прогресс чаще, но немного увеличивает количество операций записи на Google Диск. Большее значение сохраняет реже, но при неожиданной остановке может потребоваться повторно обработать больше последних компаний.

В этой ячейке запросы к СПАРК ещё не выполняются. Она только проверяет входной файл и сохраняет настройки для массового запуска.

</details>


In [ ]:
# @title 7. Подготовить входной файл и параметры запуска

from __future__ import annotations

try:
    from src.notebook_runtime import run_notebook_cell
except Exception as _cell_runtime_error:
    print(
        "Не удалось запустить этап «Подготовка входного файла». "
        "Сначала выполните пункт 2. "
        f"Причина: {type(_cell_runtime_error).__name__}: {_cell_runtime_error}"
    )
else:
    with run_notebook_cell(
        stage="Подготовка входного файла",
        next_step="Проверьте файл и повторите пункт 7.",
        project_root=globals().get("PROJECT_ROOT"),
    ):
        from datetime import datetime
        from pathlib import Path
        from uuid import uuid4

        import ipywidgets as widgets
        from IPython.display import clear_output, display

        from src.artifact_utils import safe_filename
        from src.notebook_steps import prepare_input_step
        from src.notebook_ui import dataframe_preview, show_panel
        from src.workflow import load_workflow


        # Результат сохраняется для просмотра в текущем сеансе.
        # Массовый запуск использует не эту переменную, а сохранённый workflow.
        INPUT_RESULT = None

        # Файл с компьютера сначала сохраняется в input_files,
        # а затем передаётся неизменённой центральной функции подготовки.
        _ЗАГРУЖЕННЫЙ_ПУТЬ = ""
        _ЗАГРУЖЕННОЕ_ИМЯ = ""
        _ИДЁТ_ПОДГОТОВКА = False


        def _загрузить_текущие_настройки() -> tuple[dict, str | None]:
            """Возвращает сохранённый workflow и понятное описание ошибки."""

            try:
                workflow = load_workflow(
                    workflow_root=(
                        Path(PROJECT_ROOT) / "workflow"
                    )
                )
                return workflow, None

            except Exception as error:
                return {}, str(error)


        _СОХРАНЁННЫЙ_WORKFLOW, _ОШИБКА_ЗАГРУЗКИ = (
            _загрузить_текущие_настройки()
        )
        _СОХРАНЁННЫЙ_ВХОД = (
            _СОХРАНЁННЫЙ_WORKFLOW.get("input") or {}
        )
        _СОХРАНЁННЫЕ_ПРАВИЛА = (
            _СОХРАНЁННЫЙ_WORKFLOW.get("policies") or {}
        )
        _СОХРАНЁННЫЙ_ПУТЬ = str(
            _СОХРАНЁННЫЙ_ВХОД.get("file_path") or ""
        )
        _СОХРАНЁННЫЙ_ФАЙЛ_ДОСТУПЕН = bool(
            _СОХРАНЁННЫЙ_ПУТЬ
            and Path(_СОХРАНЁННЫЙ_ПУТЬ).is_file()
        )


        ИСТОЧНИК_ФАЙЛА = widgets.Dropdown(
            options=[
                "Загрузить с компьютера",
                "Взять файл с Google Диска",
            ],
            value=(
                "Взять файл с Google Диска"
                if _СОХРАНЁННЫЙ_ФАЙЛ_ДОСТУПЕН
                else "Загрузить с компьютера"
            ),
            description="Источник:",
            layout=widgets.Layout(width="430px"),
            style={"description_width": "110px"},
        )

        ЗАГРУЗКА_С_КОМПЬЮТЕРА = widgets.FileUpload(
            accept=".csv,.xls,.xlsx,.xlsb",
            multiple=False,
            description="Загрузить файл",
            icon="upload",
            button_style="primary",
            layout=widgets.Layout(width="230px"),
        )

        ПУТЬ_К_ФАЙЛУ = widgets.Text(
            value=(
                _СОХРАНЁННЫЙ_ПУТЬ
                if _СОХРАНЁННЫЙ_ФАЙЛ_ДОСТУПЕН
                else ""
            ),
            description="Путь на Диске:",
            placeholder=(
                "/content/drive/MyDrive/папка/компании.xlsx"
            ),
            layout=widgets.Layout(width="100%"),
            style={"description_width": "170px"},
        )

        НАЗВАНИЕ_СТОЛБЦА = widgets.Text(
            value=str(
                _СОХРАНЁННЫЙ_ВХОД.get("column_name") or ""
            ),
            description="Столбец с кодом:",
            placeholder=(
                "Оставьте пустым для автоматического определения"
            ),
            layout=widgets.Layout(width="100%"),
            style={"description_width": "170px"},
        )

        _СОХРАНЁННЫЙ_ЛИСТ = _СОХРАНЁННЫЙ_ВХОД.get(
            "sheet_name",
            0,
        )

        ЛИСТ_EXCEL = widgets.Text(
            value=str(
                0
                if _СОХРАНЁННЫЙ_ЛИСТ is None
                else _СОХРАНЁННЫЙ_ЛИСТ
            ),
            description="Лист Excel:",
            placeholder="Номер с 0 или точное название листа",
            layout=widgets.Layout(width="100%"),
            style={"description_width": "170px"},
        )

        ВКЛЮЧАТЬ_ЛИКВИДИРОВАННЫЕ = widgets.Checkbox(
            value=bool(
                _СОХРАНЁННЫЕ_ПРАВИЛА.get(
                    "include_liquidated_companies",
                    False,
                )
            ),
            description=(
                "Включать ликвидированные компании, если доступен "
                "подходящий период до даты ликвидации"
            ),
            indent=False,
            layout=widgets.Layout(width="100%"),
        )

        РЕЖИМ_ДАННЫХ = widgets.Dropdown(
            options=[
                (
                    "Использовать сохранённые ответы и продолжить checkpoint",
                    "reuse",
                ),
                (
                    "Заново запросить все компании текущего файла",
                    "refresh_current_file",
                ),
            ],
            value=(
                str(
                    _СОХРАНЁННЫЕ_ПРАВИЛА.get(
                        "data_mode",
                        "reuse",
                    )
                )
                if str(
                    _СОХРАНЁННЫЕ_ПРАВИЛА.get(
                        "data_mode",
                        "reuse",
                    )
                )
                in {"reuse", "refresh_current_file"}
                else "reuse"
            ),
            description="Режим данных:",
            layout=widgets.Layout(width="100%"),
            style={"description_width": "170px"},
        )

        РАЗМЕР_CHECKPOINT = widgets.IntText(
            value=max(
                1,
                int(
                    _СОХРАНЁННЫЕ_ПРАВИЛА.get(
                        "checkpoint_every",
                        25,
                    )
                    or 25
                ),
            ),
            description="Компаний в пакете:",
            layout=widgets.Layout(width="360px"),
            style={"description_width": "170px"},
        )

        ПОДСКАЗКА_ИСТОЧНИКА = widgets.HTML()
        СТАТУС_ВЫБОРА_ФАЙЛА = widgets.HTML()

        КНОПКА_ПОДГОТОВКИ = widgets.Button(
            description="Подготовить входной файл",
            icon="folder-open",
            button_style="success",
            layout=widgets.Layout(width="290px"),
            tooltip=(
                "Проверить файл, сохранить его рабочую копию и "
                "записать настройки массового запуска"
            ),
        )

        ВЫВОД_ПОДГОТОВКИ = widgets.Output(
            layout=widgets.Layout(
                width="100%",
                min_height="60px",
            )
        )


        def _html_статус_файла(
            *,
            вид: str,
            заголовок: str,
            текст: str,
        ) -> str:
            """Возвращает компактный цветной статус выбора файла."""

            оформление = {
                "neutral": ("#f8f9fa", "#9aa0a6", "#3c4043"),
                "success": ("#e6f4ea", "#34a853", "#137333"),
                "warning": ("#fff8e1", "#f9ab00", "#8a5300"),
                "error": ("#fce8e6", "#d93025", "#b3261e"),
            }
            фон, граница, цвет = оформление.get(
                вид,
                оформление["neutral"],
            )
            return (
                f"<div style='padding:9px 12px;background:{фон};"
                f"border-left:4px solid {граница};line-height:1.45'>"
                f"<b style='color:{цвет}'>{заголовок}</b><br>"
                f"<span style='color:#3c4043'>{текст}</span></div>"
            )


        def _извлечь_файл_из_виджета(value):
            """Поддерживает форматы FileUpload из ipywidgets 7 и 8."""

            if not value:
                return None

            if isinstance(value, dict):
                name, info = next(iter(value.items()))
                content = (
                    info.get("content")
                    if isinstance(info, dict)
                    else None
                )
                return str(name), bytes(content or b"")

            if isinstance(value, (tuple, list)):
                info = value[0]
                name = (
                    info.get("name")
                    or (info.get("metadata") or {}).get("name")
                    or "uploaded_file"
                )
                content = info.get("content") or b""
                if isinstance(content, memoryview):
                    content = content.tobytes()
                return str(name), bytes(content)

            return None


        def _обновить_доступность_подготовки() -> None:
            """Разрешает подготовку только после реального выбора файла."""

            if _ИДЁТ_ПОДГОТОВКА:
                КНОПКА_ПОДГОТОВКИ.disabled = True
                return

            if ИСТОЧНИК_ФАЙЛА.value == "Загрузить с компьютера":
                файл_готов = bool(
                    _ЗАГРУЖЕННЫЙ_ПУТЬ
                    and Path(_ЗАГРУЖЕННЫЙ_ПУТЬ).is_file()
                )
            else:
                путь = ПУТЬ_К_ФАЙЛУ.value.strip()
                файл_готов = bool(
                    путь
                    and Path(путь).is_file()
                )

            КНОПКА_ПОДГОТОВКИ.disabled = not файл_готов
            КНОПКА_ПОДГОТОВКИ.button_style = (
                "success" if файл_готов else ""
            )
            КНОПКА_ПОДГОТОВКИ.tooltip = (
                "Проверить выбранный файл и сохранить настройки"
                if файл_готов
                else "Сначала выберите существующий файл"
            )


        def _обновить_статус_источника() -> None:
            """Показывает, какой файл сейчас будет подготовлен."""

            if ИСТОЧНИК_ФАЙЛА.value == "Загрузить с компьютера":
                if (
                    _ЗАГРУЖЕННЫЙ_ПУТЬ
                    and Path(_ЗАГРУЖЕННЫЙ_ПУТЬ).is_file()
                ):
                    СТАТУС_ВЫБОРА_ФАЙЛА.value = _html_статус_файла(
                        вид="success",
                        заголовок="Файл загружен и готов к подготовке",
                        текст=(
                            f"{_ЗАГРУЖЕННОЕ_ИМЯ}<br>"
                            f"Рабочая копия: {_ЗАГРУЖЕННЫЙ_ПУТЬ}"
                        ),
                    )
                else:
                    СТАТУС_ВЫБОРА_ФАЙЛА.value = _html_статус_файла(
                        вид="neutral",
                        заголовок="Файл ещё не выбран",
                        текст=(
                            "Нажмите «Загрузить файл». После выбора станет "
                            "доступна кнопка «Подготовить входной файл»."
                        ),
                    )
                return

            путь = ПУТЬ_К_ФАЙЛУ.value.strip()
            if not путь:
                СТАТУС_ВЫБОРА_ФАЙЛА.value = _html_статус_файла(
                    вид="neutral",
                    заголовок="Путь к файлу не указан",
                    текст="Введите полный путь к файлу на Google Диске.",
                )
            elif Path(путь).is_file():
                СТАТУС_ВЫБОРА_ФАЙЛА.value = _html_статус_файла(
                    вид="success",
                    заголовок="Файл найден и готов к подготовке",
                    текст=путь,
                )
            else:
                СТАТУС_ВЫБОРА_ФАЙЛА.value = _html_статус_файла(
                    вид="warning",
                    заголовок="Файл по указанному пути не найден",
                    текст=(
                        "Проверьте путь. Кнопка подготовки станет доступна, "
                        "когда файл будет найден."
                    ),
                )


        def _обновить_вид_источника(*_) -> None:
            """Показывает только элементы выбранного способа загрузки."""

            загрузка_с_компьютера = (
                ИСТОЧНИК_ФАЙЛА.value
                == "Загрузить с компьютера"
            )
            ЗАГРУЗКА_С_КОМПЬЮТЕРА.layout.display = (
                "flex" if загрузка_с_компьютера else "none"
            )
            ПУТЬ_К_ФАЙЛУ.disabled = загрузка_с_компьютера
            ПУТЬ_К_ФАЙЛУ.layout.display = (
                "none" if загрузка_с_компьютера else "flex"
            )

            if загрузка_с_компьютера:
                ПОДСКАЗКА_ИСТОЧНИКА.value = (
                    "<div style='font-size:13px;color:#5f6368;"
                    "margin:2px 0 6px 0'>Выберите один файл CSV, XLS, "
                    "XLSX или XLSB. Он сразу сохранится в папке "
                    "<code>input_files</code> на вашем Google Диске.</div>"
                )
            else:
                ПОДСКАЗКА_ИСТОЧНИКА.value = (
                    "<div style='font-size:13px;color:#5f6368;"
                    "margin:2px 0 6px 0'>Укажите полный путь к уже "
                    "существующему файлу на подключённом Google Диске.</div>"
                )

            _обновить_статус_источника()
            _обновить_доступность_подготовки()


        def _сохранить_загруженный_файл(change) -> None:
            """Сохраняет выбранный в браузере файл в input_files."""

            global _ЗАГРУЖЕННЫЙ_ПУТЬ
            global _ЗАГРУЖЕННОЕ_ИМЯ

            выбранный = _извлечь_файл_из_виджета(change.get("new"))
            if выбранный is None:
                return

            имя, содержимое = выбранный
            расширение = Path(имя).suffix.lower()

            if расширение not in {".csv", ".xls", ".xlsx", ".xlsb"}:
                _ЗАГРУЖЕННЫЙ_ПУТЬ = ""
                _ЗАГРУЖЕННОЕ_ИМЯ = ""
                СТАТУС_ВЫБОРА_ФАЙЛА.value = _html_статус_файла(
                    вид="error",
                    заголовок="Формат файла не поддерживается",
                    текст="Выберите CSV, XLS, XLSX или XLSB.",
                )
                _обновить_доступность_подготовки()
                return

            if not содержимое:
                _ЗАГРУЖЕННЫЙ_ПУТЬ = ""
                _ЗАГРУЖЕННОЕ_ИМЯ = ""
                СТАТУС_ВЫБОРА_ФАЙЛА.value = _html_статус_файла(
                    вид="error",
                    заголовок="Файл пустой",
                    текст="Выберите другой файл.",
                )
                _обновить_доступность_подготовки()
                return

            папка = Path(PROJECT_ROOT) / "input_files"
            папка.mkdir(parents=True, exist_ok=True)
            имя_копии = (
                f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_"
                f"{uuid4().hex[:8]}_{safe_filename(имя)}"
            )
            путь = папка / имя_копии
            путь.write_bytes(содержимое)

            _ЗАГРУЖЕННЫЙ_ПУТЬ = str(путь)
            _ЗАГРУЖЕННОЕ_ИМЯ = имя
            ЗАГРУЗКА_С_КОМПЬЮТЕРА.description = "Заменить файл"

            with ВЫВОД_ПОДГОТОВКИ:
                clear_output(wait=True)

            _обновить_статус_источника()
            _обновить_доступность_подготовки()


        def _путь_на_диске_изменён(change) -> None:
            """Обновляет статус при ручном вводе пути."""

            if ИСТОЧНИК_ФАЙЛА.value == "Взять файл с Google Диска":
                _обновить_статус_источника()
                _обновить_доступность_подготовки()


        ИСТОЧНИК_ФАЙЛА.observe(
            _обновить_вид_источника,
            names="value",
        )
        ЗАГРУЗКА_С_КОМПЬЮТЕРА.observe(
            _сохранить_загруженный_файл,
            names="value",
        )
        ПУТЬ_К_ФАЙЛУ.observe(
            _путь_на_диске_изменён,
            names="value",
        )
        _обновить_вид_источника()

        def _проверить_настройки_интерфейса() -> None:
            """Проверяет пользовательские значения до запуска загрузчика."""

            if ИСТОЧНИК_ФАЙЛА.value == "Загрузить с компьютера":
                if not (
                    _ЗАГРУЖЕННЫЙ_ПУТЬ
                    and Path(_ЗАГРУЖЕННЫЙ_ПУТЬ).is_file()
                ):
                    raise ValueError(
                        "Сначала нажмите «Загрузить файл» и выберите файл."
                    )
            else:
                путь = ПУТЬ_К_ФАЙЛУ.value.strip()
                if not путь:
                    raise ValueError(
                        "Укажите полный путь к файлу на Google Диске."
                    )
                if not Path(путь).is_file():
                    raise FileNotFoundError(
                        f"Файл на Google Диске не найден: {путь}"
                    )

            if int(РАЗМЕР_CHECKPOINT.value) < 1:
                raise ValueError(
                    "Размер checkpoint должен быть не меньше одной компании."
                )


        def _статус_проверки_главы_6(workflow: dict) -> tuple[bool, str]:
            """Определяет, остаётся ли успешный тест актуальным."""

            test = workflow.get("test") or {}
            dataset_hash = (
                workflow.get("fingerprints") or {}
            ).get("dataset")
            if not bool(test.get("required")):
                return (
                    True,
                    "Для текущей конфигурации отдельный тест главы 6 не обязателен.",
                )

            test_current = bool(
                test.get("completed")
                and test.get("configuration_hash")
                and test.get("configuration_hash") == dataset_hash
            )

            if test_current:
                return (
                    True,
                    "Проверка главы 6 соответствует текущей конфигурации.",
                )

            return (
                False,
                (
                    "После сохранения настроек проверка главы 6 не соответствует "
                    "текущей конфигурации. Повторно проверьте одну компанию "
                    "перед массовым запуском."
                ),
            )


        def _показать_итог(результат: dict) -> None:
            """Показывает понятную сводку сохранённых входных настроек."""

            workflow = результат.get("workflow") or {}
            input_settings = workflow.get("input") or {}
            policies = workflow.get("policies") or {}
            load_result = результат.get("load_result") or {}
            strict_result = результат.get("strict_result") or {}
            file_info = load_result.get("file_info") or {}
            basic_result = (
                strict_result.get("basic_preparation_result") or {}
            )
            basic_summary = basic_result.get("summary") or {}
            valid_codes = strict_result.get("valid_codes") or []
            rejected_df = strict_result.get("rejected_codes_df")
            rejected_count = (
                len(rejected_df)
                if hasattr(rejected_df, "__len__")
                else int(input_settings.get("invalid_count") or 0)
            )
            duplicates = int(
                basic_summary.get("duplicates", 0) or 0
            )
            invalid_values = max(
                0,
                rejected_count - duplicates,
            )
            test_current, test_message = (
                _статус_проверки_главы_6(workflow)
            )
            data_mode_label = (
                "Использовать сохранённые ответы и продолжать совместимый checkpoint"
                if policies.get("data_mode") == "reuse"
                else "Заново запросить все компании текущего файла"
            )
            liquidated_label = (
                "включать при наличии допустимого периода до даты ликвидации"
                if policies.get("include_liquidated_companies")
                else "исключать"
            )

            show_panel(
                "Входной файл и настройки сохранены",
                "\n".join(
                    [
                        f"Файл: {input_settings.get('file_path')}",
                        (
                            "Формат: "
                            f"{str(file_info.get('file_extension') or '').upper()}"
                        ),
                        (
                            "Лист: "
                            f"{input_settings.get('sheet_name', 0)}"
                        ),
                        (
                            "Столбец с ИНН/ОГРН: "
                            f"{input_settings.get('column_name')}"
                        ),
                        (
                            "Исходных строк: "
                            f"{file_info.get('rows', 0)}"
                        ),
                        (
                            "Корректных уникальных компаний: "
                            f"{len(valid_codes)}"
                        ),
                        f"Удалено дубликатов: {duplicates}",
                        (
                            "Некорректных значений: "
                            f"{invalid_values}"
                        ),
                        "",
                        f"Режим данных: {data_mode_label}",
                        (
                            "Checkpoint: каждые "
                            f"{policies.get('checkpoint_every', 25)} компаний"
                        ),
                        (
                            "Ликвидированные компании: "
                            f"{liquidated_label}"
                        ),
                        (
                            "Финансовый сектор с кодами ОКВЭД 64–66: "
                            "исключать автоматически"
                        ),
                        "",
                        test_message,
                        (
                            "Следующий шаг: глава 8 — массовый запуск."
                            if test_current
                            else (
                                "Следующий шаг: повторите главу 6, затем "
                                "переходите к массовому запуску."
                            )
                        ),
                    ]
                ),
                kind=("success" if test_current else "warning"),
            )

            if rejected_count and rejected_df is not None:
                show_panel(
                    "Часть значений исключена",
                    (
                        f"Исключено строк: {rejected_count}. "
                        "Ниже показаны первые значения и причины. "
                        "Они не попадут в массовую обработку."
                    ),
                    kind="warning",
                )
                dataframe_preview(
                    rejected_df,
                    rows=10,
                )


        def _подготовить_файл(_button) -> None:
            """Запускает центральную подготовку файла и сохраняет workflow."""

            global INPUT_RESULT
            global INPUT_SOURCE
            global DRIVE_FILE_PATH
            global COLUMN_NAME
            global SHEET_NAME
            global INCLUDE_LIQUIDATED
            global DATA_MODE_LABEL
            global DATA_MODE
            global CHECKPOINT_EVERY

            global _ИДЁТ_ПОДГОТОВКА

            _ИДЁТ_ПОДГОТОВКА = True
            КНОПКА_ПОДГОТОВКИ.disabled = True
            КНОПКА_ПОДГОТОВКИ.description = "Подготовка файла..."

            with ВЫВОД_ПОДГОТОВКИ:
                clear_output(wait=True)

                try:
                    _проверить_настройки_интерфейса()

                    # Сохраняем прежние совместимые имена переменных.
                    INPUT_SOURCE = str(ИСТОЧНИК_ФАЙЛА.value)
                    if INPUT_SOURCE == "Загрузить с компьютера":
                        DRIVE_FILE_PATH = _ЗАГРУЖЕННЫЙ_ПУТЬ
                        _ИСТОЧНИК_ДЛЯ_ПОДГОТОВКИ = (
                            "Взять файл с Google Диска"
                        )
                    else:
                        DRIVE_FILE_PATH = str(
                            ПУТЬ_К_ФАЙЛУ.value
                        ).strip()
                        _ИСТОЧНИК_ДЛЯ_ПОДГОТОВКИ = INPUT_SOURCE
                    COLUMN_NAME = str(
                        НАЗВАНИЕ_СТОЛБЦА.value
                    ).strip()
                    SHEET_NAME = str(
                        ЛИСТ_EXCEL.value
                    ).strip() or "0"
                    INCLUDE_LIQUIDATED = bool(
                        ВКЛЮЧАТЬ_ЛИКВИДИРОВАННЫЕ.value
                    )
                    DATA_MODE = str(РЕЖИМ_ДАННЫХ.value)
                    DATA_MODE_LABEL = (
                        "Использовать сохранённые ответы"
                        if DATA_MODE == "reuse"
                        else "Обновить компании текущего файла"
                    )
                    CHECKPOINT_EVERY = int(
                        РАЗМЕР_CHECKPOINT.value
                    )

                    INPUT_RESULT = prepare_input_step(
                        project_root=PROJECT_ROOT,
                        source_mode=_ИСТОЧНИК_ДЛЯ_ПОДГОТОВКИ,
                        drive_file_path=DRIVE_FILE_PATH,
                        column_name=COLUMN_NAME,
                        sheet_name=SHEET_NAME,
                        include_liquidated_companies=(
                            INCLUDE_LIQUIDATED
                        ),
                        data_mode=DATA_MODE,
                        checkpoint_every=CHECKPOINT_EVERY,
                    )

                    if INPUT_RESULT.get("success"):
                        _показать_итог(INPUT_RESULT)

                except Exception as error:
                    INPUT_RESULT = {
                        "success": False,
                        "error": str(error),
                    }
                    show_panel(
                        "Не удалось подготовить входной файл",
                        (
                            f"Причина: {error}\n"
                            "Проверьте источник, путь, лист, название "
                            "столбца и размер checkpoint."
                        ),
                        kind="error",
                    )

                finally:
                    _ИДЁТ_ПОДГОТОВКА = False
                    КНОПКА_ПОДГОТОВКИ.description = (
                        "Подготовить входной файл"
                    )
                    _обновить_доступность_подготовки()


        КНОПКА_ПОДГОТОВКИ.on_click(
            _подготовить_файл
        )


        _ТЕКУЩИЙ_СТАТУС = ""

        if _ОШИБКА_ЗАГРУЗКИ:
            _ТЕКУЩИЙ_СТАТУС = (
                "<div style='padding:9px 12px;background:#fce8e6;"
                "border-left:4px solid #d93025;line-height:1.45'>"
                "<b>Сохранённые настройки пока не прочитаны.</b><br>"
                f"{_ОШИБКА_ЗАГРУЗКИ}</div>"
            )
        elif _СОХРАНЁННЫЙ_ФАЙЛ_ДОСТУПЕН:
            _ТЕКУЩИЙ_СТАТУС = (
                "<div style='padding:9px 12px;background:#e6f4ea;"
                "border-left:4px solid #34a853;line-height:1.45'>"
                "<b>Найден ранее подготовленный файл.</b><br>"
                f"{_СОХРАНЁННЫЙ_ПУТЬ}<br>"
                "Можно повторно использовать его или выбрать новый файл."
                "</div>"
            )


        ИНТЕРФЕЙС_ВХОДНОГО_ФАЙЛА = widgets.VBox(
            [
                widgets.HTML(
                    "<h3>Подготовка списка компаний для массового запуска</h3>"
                    "<p>Выберите один файл со столбцом ИНН или ОГРН. "
                    "Программа сохранит рабочую копию на личном Google "
                    "Диске, проверит контрольные разряды, удалит дубликаты "
                    "и запишет настройки в общий workflow.</p>"
                    "<p><b>Поддерживаются:</b> CSV, XLS, XLSX и XLSB. "
                    "Если название столбца стандартное, поле «Столбец с "
                    "кодом» можно оставить пустым.</p>"
                ),
                widgets.HTML(_ТЕКУЩИЙ_СТАТУС),
                widgets.HBox(
                    [
                        ИСТОЧНИК_ФАЙЛА,
                        ЗАГРУЗКА_С_КОМПЬЮТЕРА,
                    ],
                    layout=widgets.Layout(
                        width="100%",
                        align_items="center",
                        flex_flow="row wrap",
                    ),
                ),
                ПУТЬ_К_ФАЙЛУ,
                ПОДСКАЗКА_ИСТОЧНИКА,
                СТАТУС_ВЫБОРА_ФАЙЛА,
                widgets.HTML(
                    "<h4 style='margin-bottom:6px'>Структура файла</h4>"
                ),
                НАЗВАНИЕ_СТОЛБЦА,
                ЛИСТ_EXCEL,
                widgets.HTML(
                    "<div style='font-size:13px;color:#5f6368;"
                    "margin:-2px 0 8px 170px'>Для CSV значение листа "
                    "игнорируется. Для Excel укажите номер листа, начиная "
                    "с 0, либо его точное название.</div>"
                ),
                widgets.HTML(
                    "<h4 style='margin-bottom:6px'>Правила массовой обработки</h4>"
                ),
                ВКЛЮЧАТЬ_ЛИКВИДИРОВАННЫЕ,
                РЕЖИМ_ДАННЫХ,
                РАЗМЕР_CHECKPOINT,
                widgets.HTML(
                    "<div style='padding:9px 12px;background:#fff8e1;"
                    "border-left:4px solid #f9ab00;line-height:1.45'>"
                    "<b>Финансовые организации исключаются автоматически.</b> "
                    "Checkpoint сохраняется после указанного количества "
                    "компаний; стандартное значение 25 подходит для "
                    "большинства запусков.</div>"
                ),
                КНОПКА_ПОДГОТОВКИ,
                ВЫВОД_ПОДГОТОВКИ,
            ],
            layout=widgets.Layout(width="100%"),
        )


        display(ИНТЕРФЕЙС_ВХОДНОГО_ФАЙЛА)


### 7.1. Накопительный датасет

Включайте накопление только тогда, когда подходящие компании нужно собирать из нескольких запусков. Повторное название открывает существующее накопление и не удаляет данные.

<details>
<summary><b>Техническая документация</b></summary>

Данные хранятся в `dataset_goals/<название>/`. Продолжение разрешается только при совпадении методов, target, признаков, пользовательских формул и правил принятия.

</details>


In [ ]:
# @title 7.1. Выбрать или создать накопительный датасет

try:
    from src.notebook_runtime import run_notebook_cell
except Exception as _cell_runtime_error:
    print(
        "Не удалось запустить этап «Накопительный датасет». "
        "Сначала выполните пункт 2. "
        f"Причина: {type(_cell_runtime_error).__name__}: {_cell_runtime_error}"
    )
else:
    with run_notebook_cell(
        stage="Накопительный датасет",
        next_step="Проверьте сообщение в пункте 7.1 и повторите только нужное действие.",
        project_root=globals().get("PROJECT_ROOT"),
    ):
        from copy import deepcopy
        from datetime import datetime, timezone
        from html import escape
        from pathlib import Path
        import json

        import ipywidgets as widgets
        from IPython.display import HTML, display
        from IPython.utils.capture import capture_output

        from src.artifact_utils import safe_filename, write_json_atomic
        import src.dataset_goals as dataset_goals_module
        from src.dataset_goals import (
            GOAL_ROOT_DIRECTORY,
            export_dataset_goal_snapshot,
            goal_acceptance_hash,
            goal_configuration_hash,
        )
        from src.notebook_steps import dataset_goal_step
        from src.notebook_ui import protected_download_buttons
        from src.workflow import load_workflow, refresh_fingerprints


        # -------------------------------------------------------------------------
        # Пути и внутреннее состояние интерфейса
        # -------------------------------------------------------------------------

        _КОРЕНЬ_WORKFLOW = Path(PROJECT_ROOT) / "workflow"
        _КОРЕНЬ_НАКОПЛЕНИЙ = (
            Path(PROJECT_ROOT) / GOAL_ROOT_DIRECTORY
        )
        _КОРЕНЬ_НАКОПЛЕНИЙ.mkdir(
            parents=True,
            exist_ok=True,
        )

        _STATE = {
            "entries": {},
            "updating_list": False,
            "action": None,
        }


        # -------------------------------------------------------------------------
        # Общие функции
        # -------------------------------------------------------------------------

        def _прочитать_json(
            path: Path,
            default=None,
        ):
            if not path.is_file():
                return default

            try:
                value = json.loads(
                    path.read_text(encoding="utf-8")
                )
            except Exception:
                return default

            return value


        def _пути_таблиц_накопления(
            *,
            folder: Path,
            goal: dict,
        ) -> dict[str, Path]:
            """Возвращает реальные пути таблиц рядом с выбранным goal.json."""

            saved_paths = dict(
                goal.get("paths") or {}
            )

            def resolve_path(
                key: str,
                filename: str,
            ) -> Path:
                local_path = folder / filename
                saved_value = str(
                    saved_paths.get(key) or ""
                ).strip()
                saved_path = (
                    Path(saved_value)
                    if saved_value
                    else None
                )

                # Сначала проверяем файлы рядом с фактически найденным goal.json.
                # Это защищает от старых абсолютных путей после копирования папки.
                for candidate in (
                    local_path,
                    saved_path,
                ):
                    if candidate is None:
                        continue

                    if (
                        candidate.is_file()
                        or candidate.with_suffix(".csv").is_file()
                    ):
                        return candidate

                return local_path

            return {
                "registry": resolve_path(
                    "registry",
                    "companies_registry.parquet",
                ),
                "accepted_ml": resolve_path(
                    "accepted_ml",
                    "cumulative_ml_matrix.parquet",
                ),
                "accepted_main": resolve_path(
                    "accepted_main",
                    "cumulative_main_dataset.parquet",
                ),
            }


        def _фактическое_состояние_накопления(
            *,
            folder: Path,
            goal: dict,
        ) -> dict:
            """
            Проверяет реальные таблицы накопления тем же чтением,
            которое использует центральный модуль dataset_goals.

            Нулевые счётчики progress.json сами по себе не считаются
            доказательством пустоты: решение принимается по реестру,
            ML-матрице и основной накопительной таблице.
            """

            paths = _пути_таблиц_накопления(
                folder=folder,
                goal=goal,
            )
            reader = getattr(
                dataset_goals_module,
                "_read_parquet",
                None,
            )
            registry_columns = list(
                getattr(
                    dataset_goals_module,
                    "REGISTRY_COLUMNS",
                    [],
                )
            )

            if not callable(reader):
                return {
                    "readable": False,
                    "empty": False,
                    "accepted_company_count": 0,
                    "seen_company_count": 0,
                    "message": (
                        "В установленном ядре не найдена штатная функция "
                        "чтения накопительных таблиц. Повторно выполните пункт 2."
                    ),
                }

            try:
                registry_df = reader(
                    paths["registry"],
                    registry_columns,
                )
                accepted_ml_df = reader(
                    paths["accepted_ml"]
                )
                accepted_main_df = reader(
                    paths["accepted_main"]
                )
            except Exception as error:
                return {
                    "readable": False,
                    "empty": False,
                    "accepted_company_count": 0,
                    "seen_company_count": 0,
                    "message": (
                        "Не удалось безопасно прочитать фактические таблицы "
                        f"накопления: {type(error).__name__}: {error}"
                    ),
                }

            if (
                not registry_df.empty
                and "requested_code" in registry_df.columns
            ):
                seen_count = int(
                    registry_df["requested_code"]
                    .dropna()
                    .astype(str)
                    .nunique()
                )
            else:
                seen_count = 0

            if (
                not accepted_ml_df.empty
                and "requested_code" in accepted_ml_df.columns
            ):
                accepted_count = int(
                    accepted_ml_df["requested_code"]
                    .dropna()
                    .astype(str)
                    .nunique()
                )
            else:
                accepted_count = int(
                    len(accepted_ml_df)
                )

            is_empty = bool(
                registry_df.empty
                and accepted_ml_df.empty
                and accepted_main_df.empty
            )
            return {
                "readable": True,
                "empty": is_empty,
                "accepted_company_count": accepted_count,
                "seen_company_count": seen_count,
                "registry_row_count": int(len(registry_df)),
                "accepted_ml_row_count": int(len(accepted_ml_df)),
                "accepted_main_row_count": int(len(accepted_main_df)),
                "message": "",
            }


        def _обновить_счётчики_из_хранилища(
            *,
            progress: dict,
            storage_state: dict,
        ) -> dict:
            """Использует фактические таблицы вместо возможного старого progress.json."""

            updated = dict(progress or {})

            if bool(storage_state.get("readable")):
                updated["accepted_company_count"] = int(
                    storage_state.get(
                        "accepted_company_count",
                        0,
                    )
                    or 0
                )
                updated["seen_company_count"] = int(
                    storage_state.get(
                        "seen_company_count",
                        0,
                    )
                    or 0
                )

            return updated


        def _показать_панель(
            *,
            заголовок: str,
            сообщение: str,
            тип: str = "info",
        ) -> None:
            стили = {
                "success": ("#e6f4ea", "#34a853", "#137333"),
                "warning": ("#fef7e0", "#f9ab00", "#8a4b00"),
                "error": ("#fce8e6", "#d93025", "#b3261e"),
                "info": ("#e8f0fe", "#1a73e8", "#174ea6"),
            }
            фон, граница, цвет = стили.get(
                тип,
                стили["info"],
            )

            display(
                HTML(
                    "<div style='max-width:860px; padding:12px 14px; "
                    f"background:{фон}; border-left:5px solid {граница}; "
                    f"border-radius:7px; color:{цвет}; line-height:1.5;'>"
                    f"<b>{escape(str(заголовок))}</b><br>"
                    f"{сообщение}"
                    "</div>"
                )
            )


        def _текущий_workflow() -> dict:
            return load_workflow(
                workflow_root=_КОРЕНЬ_WORKFLOW,
            )


        _НАЗВАНИЯ_ЭТАПОВ_НАКОПЛЕНИЯ = {
            "bootstrap": (
                "пункт 2 «Установить и проверить автономный проект»"
            ),
            "methods": (
                "пункт 4 «Выбрать и сохранить методы API»"
            ),
            "fields": (
                "пункт 5.3 «Выбрать и сохранить поля и признаки»"
            ),
            "input": (
                "пункт 7 «Подготовить входной файл»"
            ),
        }


        def _проверить_готовность(
            workflow: dict,
        ) -> list[str]:
            stages = dict(workflow.get("stages") or {})
            required = [
                "bootstrap",
                "methods",
                "fields",
                "input",
            ]
            return [
                stage
                for stage in required
                if not bool(
                    (stages.get(stage) or {}).get("ready")
                )
            ]


        def _сообщение_о_неготовности(
            missing: list[str],
        ) -> str:
            if not missing:
                return ""

            пункты = [
                _НАЗВАНИЯ_ЭТАПОВ_НАКОПЛЕНИЯ.get(
                    stage,
                    stage,
                )
                for stage in missing
            ]

            if len(пункты) == 1:
                return (
                    "Для применения накопления выполните "
                    f"<b>{escape(пункты[0])}</b>."
                )

            return (
                "Для применения накопления выполните:<br>• "
                + "<br>• ".join(
                    escape(value)
                    for value in пункты
                )
            )


        def _список_различий(
            сохранённые: list[str],
            текущие: list[str],
        ) -> str:
            old_set = set(сохранённые)
            current_set = set(текущие)
            added = sorted(current_set - old_set)
            removed = sorted(old_set - current_set)
            parts = []

            if added:
                shown = ", ".join(
                    escape(value)
                    for value in added[:8]
                )
                tail = (
                    ""
                    if len(added) <= 8
                    else f" и ещё {len(added) - 8}"
                )
                parts.append(
                    f"сейчас добавлены: {shown}{tail}"
                )

            if removed:
                shown = ", ".join(
                    escape(value)
                    for value in removed[:8]
                )
                tail = (
                    ""
                    if len(removed) <= 8
                    else f" и ещё {len(removed) - 8}"
                )
                parts.append(
                    f"сейчас отсутствуют: {shown}{tail}"
                )

            return (
                "; ".join(parts)
                or "названия признаков совпадают"
            )


        def _совместимость(
            *,
            goal: dict,
            workflow: dict,
            folder: Path | None = None,
            storage_state: dict | None = None,
        ) -> dict:
            missing = _проверить_готовность(workflow)

            if "fields" in missing:
                return {
                    "code": "unknown",
                    "can_apply": False,
                    "label": "нельзя проверить",
                    "kind": "warning",
                    "message": (
                        "Сначала сохраните поля и ML-признаки "
                        "в пункте 5.3."
                    ),
                }

            resolved_folder = Path(
                folder
                or (
                    (goal.get("paths") or {}).get("root")
                    or (
                        _КОРЕНЬ_НАКОПЛЕНИЙ
                        / safe_filename(
                            str(goal.get("goal_name") or "dataset_goal"),
                            fallback="dataset_goal",
                        )
                    )
                )
            )
            actual_storage = dict(
                storage_state
                or _фактическое_состояние_накопления(
                    folder=resolved_folder,
                    goal=goal,
                )
            )

            if not bool(actual_storage.get("readable")):
                return {
                    "code": "storage_unreadable",
                    "can_apply": False,
                    "label": "не удалось проверить файлы",
                    "kind": "warning",
                    "message": (
                        escape(
                            str(
                                actual_storage.get("message")
                                or "Фактические таблицы накопления не читаются."
                            )
                        )
                        + "<br>Для безопасности накопление не применяется, "
                        "пока его реальные таблицы не будут прочитаны."
                    ),
                }

            current_features = sorted(
                str(value)
                for value in (
                    (workflow.get("fields") or {}).get(
                        "selected_feature_names"
                    )
                    or []
                )
            )
            stored_features = sorted(
                str(value)
                for value in (
                    goal.get("selected_feature_names")
                    or []
                )
            )
            minimum_fill = goal.get(
                "minimum_feature_fill_percent"
            )
            current_configuration_hash = (
                goal_configuration_hash(workflow)
            )
            current_acceptance_hash = goal_acceptance_hash(
                workflow=workflow,
                minimum_feature_fill_percent=minimum_fill,
            )
            stored_configuration_hash = str(
                goal.get("configuration_hash") or ""
            )
            stored_acceptance_hash = str(
                goal.get("acceptance_hash") or ""
            )

            # Пустое накопление не содержит ни одной записи в реестре,
            # ML-матрице и основной таблице. Его можно безопасно привязать
            # к текущей конфигурации: центральный dataset_goal_step повторно
            # проверит пустоту и обновит только служебные настройки папки.
            if bool(actual_storage.get("empty")):
                return {
                    "code": "empty_rebind",
                    "can_apply": not bool(missing),
                    "label": "пустое — можно привязать",
                    "kind": "info",
                    "message": (
                        "Фактические таблицы накопления полностью пусты: "
                        "нет записей в реестре компаний, ML-матрице и "
                        "основном датасете.<br>Поэтому папку можно безопасно "
                        "привязать к текущим методам, признакам, target и правилам. "
                        "Накопленных строк нет, смешивание разных конфигураций "
                        "невозможно. При применении центральное ядро ещё раз "
                        "проверит пустоту и обновит служебные отпечатки."
                        + (
                            ""
                            if not missing
                            else (
                                "<br>"
                                + _сообщение_о_неготовности(
                                    missing
                                )
                            )
                        )
                    ),
                    "new_configuration_hash": (
                        current_configuration_hash
                    ),
                    "new_acceptance_hash": (
                        current_acceptance_hash
                    ),
                    "current_feature_names": list(
                        current_features
                    ),
                }

            if stored_features != current_features:
                return {
                    "code": "different_features",
                    "can_apply": False,
                    "label": "другие признаки",
                    "kind": "warning",
                    "stored_feature_count": len(stored_features),
                    "current_feature_count": len(current_features),
                    "message": (
                        "Это накопление можно открыть и скачать, "
                        "но сейчас нельзя продолжать его массовый сбор.<br>"
                        f"В накоплении: <b>{len(stored_features)}</b> "
                        "ML-признаков; сейчас в пункте 5.3 выбрано: "
                        f"<b>{len(current_features)}</b>.<br>"
                        f"Различия: {_список_различий(stored_features, current_features)}.<br>"
                        "Чтобы продолжить именно это накопление, "
                        "верните в пункте 5.3 прежний набор признаков. "
                        "Для текущих признаков создайте новое накопление."
                    ),
                }

            if (
                stored_configuration_hash
                == current_configuration_hash
                and stored_acceptance_hash
                == current_acceptance_hash
            ):
                return {
                    "code": "compatible",
                    "can_apply": not bool(missing),
                    "label": "совместимо",
                    "kind": "success",
                    "message": (
                        "Методы, ML-признаки, target и правила "
                        "совпадают с текущей конфигурацией."
                        + (
                            ""
                            if not missing
                            else (
                                "<br>"
                                + _сообщение_о_неготовности(
                                    missing
                                )
                            )
                        )
                    ),
                }

            legacy_workflow = deepcopy(workflow)
            legacy_fields = dict(
                legacy_workflow.get("fields") or {}
            )
            legacy_fields.pop(
                "custom_features_sha256",
                None,
            )
            legacy_fields.pop(
                "custom_features_source",
                None,
            )
            legacy_workflow["fields"] = legacy_fields
            legacy_workflow = refresh_fingerprints(
                legacy_workflow
            )

            legacy_configuration_hash = (
                goal_configuration_hash(
                    legacy_workflow
                )
            )
            legacy_acceptance_hash = goal_acceptance_hash(
                workflow=legacy_workflow,
                minimum_feature_fill_percent=minimum_fill,
            )

            if (
                stored_configuration_hash
                == legacy_configuration_hash
                and stored_acceptance_hash
                == legacy_acceptance_hash
            ):
                return {
                    "code": "legacy_compatible",
                    "can_apply": not bool(missing),
                    "label": "совместимо после обновления",
                    "kind": "info",
                    "message": (
                        "Состав данных совпадает. В старой папке "
                        "отсутствует только служебная SHA-метка "
                        "расчётных формул. При применении будет "
                        "обновлён только goal.json; строки датасета "
                        "не изменятся."
                        + (
                            ""
                            if not missing
                            else (
                                "<br>"
                                + _сообщение_о_неготовности(
                                    missing
                                )
                            )
                        )
                    ),
                    "new_configuration_hash": (
                        current_configuration_hash
                    ),
                    "new_acceptance_hash": (
                        current_acceptance_hash
                    ),
                }

            if (
                stored_configuration_hash
                == current_configuration_hash
            ):
                return {
                    "code": "different_acceptance",
                    "can_apply": False,
                    "label": "другой критерий отбора",
                    "kind": "warning",
                    "message": (
                        "Методы, признаки и target совпадают, "
                        "но отличается критерий принятия строк — "
                        "чаще всего минимальная заполненность "
                        "признаков. Для продолжения верните прежний "
                        "порог или создайте новое накопление."
                    ),
                }

            return {
                "code": "different_configuration",
                "can_apply": False,
                "label": "другая конфигурация",
                "kind": "warning",
                "message": (
                    "Названия ML-признаков совпадают, но отличаются "
                    "другие настройки: методы API, параметры методов, "
                    "target, правила исключения компаний либо версия "
                    "производственной политики. Это накопление доступно "
                    "для просмотра и скачивания, но не для продолжения "
                    "с текущей конфигурацией."
                ),
            }


        def _время_сортировки(
            goal: dict,
            goal_path: Path,
        ) -> float:
            for key in (
                "updated_at",
                "created_at",
            ):
                raw = str(goal.get(key) or "").strip()

                if raw:
                    try:
                        return datetime.fromisoformat(
                            raw.replace("Z", "+00:00")
                        ).timestamp()
                    except Exception:
                        pass

            try:
                return goal_path.stat().st_mtime
            except Exception:
                return 0.0


        def _пути_накопления(
            *,
            folder: Path,
            goal: dict,
        ) -> dict:
            saved_paths = dict(
                goal.get("paths") or {}
            )

            return {
                "root": str(
                    saved_paths.get("root")
                    or folder
                ),
                "excel": str(
                    saved_paths.get("excel")
                    or (
                        folder
                        / "Накопительный_датасет.xlsx"
                    )
                ),
                "zip": str(
                    saved_paths.get("zip")
                    or (
                        folder
                        / "Накопительный_датасет.zip"
                    )
                ),
            }


        def _найти_накопления(
            workflow: dict,
        ) -> tuple[list[dict], list[str]]:
            """Находит все накопления в общей папке Google Диска."""

            entries = []
            broken = []
            goal_paths = []
            seen_goal_paths = set()

            if not _КОРЕНЬ_НАКОПЛЕНИЙ.is_dir():
                return entries, broken

            # Основной формат: dataset_goals/<название>/goal.json.
            # Дополнительный уровень оставлен для совместимости со старыми
            # папками, если накопление когда-либо переносилось вручную.
            candidates = list(
                _КОРЕНЬ_НАКОПЛЕНИЙ.glob("*/goal.json")
            ) + list(
                _КОРЕНЬ_НАКОПЛЕНИЙ.glob("*/*/goal.json")
            )

            for goal_path in candidates:
                try:
                    key = str(goal_path.resolve())
                except Exception:
                    key = str(goal_path)

                if key in seen_goal_paths:
                    continue

                seen_goal_paths.add(key)
                goal_paths.append(goal_path)

            for goal_path in sorted(
                goal_paths,
                key=lambda value: str(value.parent).lower(),
            ):
                folder = goal_path.parent
                goal = _прочитать_json(
                    goal_path,
                    default=None,
                )

                if not isinstance(goal, dict):
                    broken.append(
                        f"{folder.name}: goal.json не читается"
                    )
                    continue

                progress = _прочитать_json(
                    folder / "progress.json",
                    default={},
                )

                if not isinstance(progress, dict):
                    progress = {}

                name = str(
                    goal.get("goal_name")
                    or folder.name
                ).strip()

                if not name:
                    name = folder.name

                storage_state = (
                    _фактическое_состояние_накопления(
                        folder=folder,
                        goal=goal,
                    )
                )
                progress = _обновить_счётчики_из_хранилища(
                    progress=progress,
                    storage_state=storage_state,
                )
                compatibility = _совместимость(
                    goal=goal,
                    workflow=workflow,
                    folder=folder,
                    storage_state=storage_state,
                )
                paths = _пути_накопления(
                    folder=folder,
                    goal=goal,
                )

                entries.append(
                    {
                        "name": name,
                        "folder": folder,
                        "goal_path": goal_path,
                        "goal": goal,
                        "progress": progress,
                        "paths": paths,
                        "storage_state": storage_state,
                        "compatibility": compatibility,
                        "sort_time": _время_сортировки(
                            goal,
                            goal_path,
                        ),
                    }
                )

            entries.sort(
                key=lambda item: (
                    item.get("sort_time", 0.0),
                    item.get("name", "").lower(),
                ),
                reverse=True,
            )
            return entries, broken

        def _активное_накопление(
            workflow: dict,
        ) -> tuple[bool, str]:
            settings = dict(
                workflow.get("dataset_goal") or {}
            )
            enabled = bool(
                settings.get("enabled", False)
            )
            name = str(
                settings.get("goal_name") or ""
            ).strip()
            return enabled, name


        def _название_набора_признаков(
            *,
            goal: dict,
            workflow: dict,
            compatibility: dict | None = None,
        ) -> str:
            """Возвращает понятное название шаблона или ручного набора."""

            fields = dict(workflow.get("fields") or {})
            stored_features = {
                str(value)
                for value in (
                    goal.get("selected_feature_names") or []
                )
            }
            feature_count = len(stored_features)
            stored_mode = str(
                goal.get("selection_mode")
                or goal.get("feature_profile_name")
                or ""
            ).strip()
            current_mode = str(
                fields.get("selection_mode")
                or fields.get("feature_profile_name")
                or ""
            ).strip()
            current_features = {
                str(value)
                for value in (
                    fields.get("selected_feature_names") or []
                )
            }
            spark_source_features = {
                str(value)
                for value in (
                    fields.get("spark_source_feature_names") or []
                )
            }
            known_modes = {
                "Основной набор СПАРК",
                "Основной СПАРК + расчётные признаки",
                "СПАРК и расчётные признаки",
                "Полная выгрузка",
                "Ручная настройка",
            }

            mode = stored_mode if stored_mode in known_modes else ""

            if not mode and stored_features == current_features:
                mode = current_mode if current_mode in known_modes else ""

            if not mode and spark_source_features:
                if stored_features == spark_source_features:
                    mode = "Основной набор СПАРК"

            if mode in {
                "Основной СПАРК + расчётные признаки",
                "СПАРК и расчётные признаки",
            }:
                return "Основной СПАРК + расчётные признаки"

            if mode == "Ручная настройка":
                return f"Ручная настройка — {feature_count} признаков"

            if mode:
                return mode

            if feature_count:
                return (
                    "Ручной или старый набор — "
                    f"{feature_count} признаков"
                )

            return "Набор признаков не сохранён"


        def _краткая_сводка(
            *,
            entry: dict,
            workflow: dict,
            include_path: bool = False,
        ) -> str:
            progress = dict(entry.get("progress") or {})
            goal = dict(entry.get("goal") or {})
            paths = dict(entry.get("paths") or {})
            compatibility = dict(
                entry.get("compatibility") or {}
            )
            accepted = int(
                progress.get("accepted_company_count") or 0
            )
            target = int(
                progress.get("target_company_count")
                or goal.get("target_company_count")
                or 0
            )
            remaining = max(0, target - accepted)
            seen = int(
                progress.get("seen_company_count") or 0
            )
            stop_after_goal = bool(
                goal.get("stop_when_target_reached", True)
            )
            minimum_fill = goal.get(
                "minimum_feature_fill_percent"
            )
            feature_set = _название_набора_признаков(
                goal=goal,
                workflow=workflow,
                compatibility=compatibility,
            )
            stop_text = (
                "Да — сбор остановится после достижения цели"
                if stop_after_goal
                else "Нет — сбор продолжится после достижения цели"
            )
            fill_text = (
                "не используется"
                if minimum_fill is None
                else f"не менее {float(minimum_fill):g}%"
            )
            parts = [
                f"Название: <b>{escape(entry.get('name', ''))}</b>",
                f"Набор признаков: <b>{escape(feature_set)}</b>",
                f"Принято компаний: <b>{accepted}</b>",
                f"Цель: <b>{target}</b>",
                f"Осталось: <b>{remaining}</b>",
                f"Проверено кандидатов: <b>{seen}</b>",
                f"Остановить после достижения цели: <b>{escape(stop_text)}</b>",
                f"Минимальная заполненность: <b>{escape(fill_text)}</b>",
            ]

            if include_path:
                parts.append(
                    "Папка: "
                    f"<code>{escape(str(paths.get('root') or ''))}</code>"
                )

            return "<br>".join(parts)


        def _метка_элемента(
            *,
            entry: dict,
            workflow: dict,
        ) -> str:
            progress = dict(entry.get("progress") or {})
            accepted = int(
                progress.get("accepted_company_count") or 0
            )
            compatibility = dict(
                entry.get("compatibility") or {}
            )
            feature_set = _название_набора_признаков(
                goal=dict(entry.get("goal") or {}),
                workflow=workflow,
                compatibility=compatibility,
            )
            return (
                f"{entry.get('name')} — {accepted} компаний — "
                f"{feature_set} — "
                f"{compatibility.get('label', 'не проверено')}"
            )


        # -------------------------------------------------------------------------
        # Виджеты: включение интерфейса и выбор существующего накопления
        # -------------------------------------------------------------------------

        режим_накопления = widgets.ToggleButtons(
            options=[
                ("Нет", False),
                ("Да", True),
            ],
            value=False,
            description="Накопление:",
            style={"description_width": "120px"},
            layout=widgets.Layout(width="500px"),
        )

        описание_режима = widgets.HTML()
        вывод_списка = widgets.Output()
        вывод_просмотра = widgets.Output()
        вывод_действия = widgets.Output()

        кнопка_открыть_существующий = widgets.Button(
            description="Открыть существующий датасет",
            icon="folder-open",
            layout=widgets.Layout(width="370px"),
        )
        кнопка_показать_создание = widgets.Button(
            description="Создать новый датасет",
            icon="plus",
            layout=widgets.Layout(width="330px"),
        )
        панель_выбора_действия = widgets.HBox(
            [
                кнопка_открыть_существующий,
                кнопка_показать_создание,
            ],
            layout=widgets.Layout(
                width="860px",
                gap="12px",
                flex_flow="row wrap",
                margin="8px 0 4px 0",
            ),
        )

        выбор_накопления = widgets.Dropdown(
            options=[("— Выберите накопительный датасет —", None)],
            value=None,
            description="Накопление:",
            layout=widgets.Layout(width="860px"),
            style={"description_width": "120px"},
        )

        кнопка_обновить_список = widgets.Button(
            description="Обновить список",
            icon="refresh",
            layout=widgets.Layout(width="180px"),
        )

        кнопка_обновить_файлы = widgets.Button(
            description="Обновить Excel и ZIP",
            icon="file-archive-o",
            layout=widgets.Layout(width="210px"),
        )

        кнопка_применить = widgets.Button(
            description="Применить выбранное",
            button_style="success",
            icon="check",
            disabled=True,
            layout=widgets.Layout(width="220px"),
        )

        кнопка_выключить = widgets.Button(
            description="Применить режим без накопления",
            button_style="warning",
            icon="ban",
            layout=widgets.Layout(width="290px"),
        )


        # -------------------------------------------------------------------------
        # Виджеты: создание нового накопления
        # -------------------------------------------------------------------------

        новое_название = widgets.Text(
            value="",
            placeholder="Например: Дефолт_2024_10000",
            description="Название:",
            layout=widgets.Layout(width="650px"),
            style={"description_width": "120px"},
        )

        новая_цель = widgets.BoundedIntText(
            value=10_000,
            min=1,
            max=100_000_000,
            step=100,
            description="Цель:",
            layout=widgets.Layout(width="360px"),
            style={"description_width": "120px"},
        )

        новая_автоостановка = widgets.Checkbox(
            value=True,
            description=(
                "Остановить массовый сбор после достижения цели"
            ),
            indent=False,
            layout=widgets.Layout(width="650px"),
        )

        использовать_новый_порог = widgets.Checkbox(
            value=False,
            description=(
                "Использовать минимальную заполненность признаков"
            ),
            indent=False,
            layout=widgets.Layout(width="650px"),
        )

        новый_порог = widgets.BoundedFloatText(
            value=80.0,
            min=0.0,
            max=100.0,
            step=1.0,
            description="Заполненность, %:",
            layout=widgets.Layout(width="360px"),
            style={"description_width": "150px"},
        )

        кнопка_создать = widgets.Button(
            description="Создать и применить новое",
            button_style="success",
            icon="plus",
            layout=widgets.Layout(width="260px"),
        )

        блок_создания = widgets.VBox(
            [
                widgets.HTML(
                    value=(
                        "<h4 style='margin:4px 0 6px 0;'>"
                        "Создать новое накопление"
                        "</h4>"
                        "<div style='margin-bottom:8px; color:#5f6368;'>"
                        "Новое название не должно совпадать "
                        "с уже существующей папкой. Созданное "
                        "накопление сразу станет активным."
                        "</div>"
                    )
                ),
                новое_название,
                новая_цель,
                новая_автоостановка,
                использовать_новый_порог,
                новый_порог,
                кнопка_создать,
            ],
            layout=widgets.Layout(
                border="1px solid #dadce0",
                padding="12px",
                margin="12px 0 0 0",
                width="860px",
                display="none",
            ),
        )

        блок_существующих = widgets.VBox(
            [
                вывод_списка,
                выбор_накопления,
                widgets.HBox(
                    [
                        кнопка_обновить_список,
                        кнопка_обновить_файлы,
                        кнопка_применить,
                    ]
                ),
                вывод_просмотра,
            ],
            layout=widgets.Layout(
                width="860px",
                display="none",
            ),
        )


        def _очистить_выбор_накопления() -> None:
            _STATE["updating_list"] = True

            try:
                выбор_накопления.value = None
            finally:
                _STATE["updating_list"] = False

            кнопка_применить.disabled = True
            кнопка_применить.description = "Применить выбранный датасет"
            кнопка_обновить_файлы.disabled = True

            with вывод_просмотра:
                вывод_просмотра.clear_output(wait=True)


        def _скрыть_разделы_действия() -> None:
            _STATE["action"] = None
            блок_существующих.layout.display = "none"
            блок_создания.layout.display = "none"
            кнопка_открыть_существующий.button_style = ""
            кнопка_показать_создание.button_style = ""
            _очистить_выбор_накопления()


        def _открыть_существующие(_button=None) -> None:
            _STATE["action"] = "existing"
            блок_существующих.layout.display = "flex"
            блок_создания.layout.display = "none"
            кнопка_открыть_существующий.button_style = "info"
            кнопка_показать_создание.button_style = ""

            with вывод_действия:
                вывод_действия.clear_output(wait=True)

            _обновить_список()


        def _открыть_создание(_button=None) -> None:
            _STATE["action"] = "new"
            блок_существующих.layout.display = "none"
            блок_создания.layout.display = "flex"
            кнопка_открыть_существующий.button_style = ""
            кнопка_показать_создание.button_style = "info"

            with вывод_действия:
                вывод_действия.clear_output(wait=True)

            _очистить_выбор_накопления()


        def _обновить_видимость_порога(*_args) -> None:
            новый_порог.layout.display = (
                "flex"
                if bool(использовать_новый_порог.value)
                else "none"
            )


        использовать_новый_порог.observe(
            _обновить_видимость_порога,
            names="value",
        )
        _обновить_видимость_порога()


        def _обновить_описание_режима() -> None:
            workflow = _текущий_workflow()
            enabled, active_name = _активное_накопление(
                workflow
            )

            if enabled and active_name:
                current_text = (
                    "Сейчас в пункте 8 сохранено накопление: "
                    f"<b>{escape(active_name)}</b>. "
                    "Переключатель ниже сам ничего не меняет."
                )
            else:
                current_text = (
                    "Сейчас пункт 8 настроен на обычный запуск "
                    "без накопления."
                )

            if bool(режим_накопления.value):
                action_text = (
                    "Меню открыто. Сначала выберите действие: "
                    "<b>открыть существующий</b> или "
                    "<b>создать новый</b> датасет. "
                    "Ни один датасет автоматически не выбирается."
                )
            else:
                action_text = (
                    "Меню накоплений скрыто. Чтобы выбрать или создать "
                    "накопление, нажмите <b>«Да»</b>. Чтобы явно "
                    "отключить уже выбранное накопление, нажмите "
                    "<b>«Применить режим без накопления»</b>."
                )

            описание_режима.value = (
                "<div style='max-width:860px; padding:10px 12px; "
                "background:#f8f9fa; border-left:4px solid #5f6368; "
                "border-radius:6px; line-height:1.5;'>"
                f"{current_text}<br>{action_text}"
                "</div>"
            )


        def _обновить_видимость_меню() -> None:
            show_menu = bool(режим_накопления.value)
            меню_накоплений.layout.display = (
                "flex" if show_menu else "none"
            )
            блок_без_накопления.layout.display = (
                "none" if show_menu else "flex"
            )

            if not show_menu:
                _скрыть_разделы_действия()

            _обновить_описание_режима()


        # -------------------------------------------------------------------------
        # Просмотр выбранного накопления
        # -------------------------------------------------------------------------

        def _выбранная_запись():
            selected = выбор_накопления.value

            if not selected:
                return None

            return _STATE["entries"].get(
                str(selected)
            )


        def _показать_выбранное(
            *_args,
        ) -> None:
            if _STATE["updating_list"]:
                return

            entry = _выбранная_запись()

            with вывод_просмотра:
                вывод_просмотра.clear_output(wait=True)

                if entry is None:
                    кнопка_применить.disabled = True
                    кнопка_обновить_файлы.disabled = True
                    кнопка_применить.description = (
                        "Применить выбранный датасет"
                    )
                    return

                workflow = _текущий_workflow()
                enabled, active_name = _активное_накопление(
                    workflow
                )
                storage_state = (
                    _фактическое_состояние_накопления(
                        folder=Path(entry["folder"]),
                        goal=entry["goal"],
                    )
                )
                entry["storage_state"] = storage_state
                entry["progress"] = (
                    _обновить_счётчики_из_хранилища(
                        progress=dict(entry.get("progress") or {}),
                        storage_state=storage_state,
                    )
                )
                compatibility = _совместимость(
                    goal=entry["goal"],
                    workflow=workflow,
                    folder=Path(entry["folder"]),
                    storage_state=storage_state,
                )
                entry["compatibility"] = compatibility
                is_active = bool(
                    enabled and entry["name"] == active_name
                )
                can_apply = bool(
                    compatibility.get("can_apply")
                )
                active_and_compatible = bool(
                    is_active
                    and compatibility.get("code") in {
                        "compatible",
                        "legacy_compatible",
                    }
                    and can_apply
                )

                кнопка_обновить_файлы.disabled = False
                кнопка_применить.disabled = (
                    active_and_compatible or not can_apply
                )

                if active_and_compatible:
                    кнопка_применить.description = "Уже применяется"
                elif can_apply:
                    кнопка_применить.description = (
                        "Применить выбранный датасет"
                    )
                else:
                    кнопка_применить.description = "Нельзя применить"

                if active_and_compatible:
                    action_text = (
                        "Этот датасет уже применяется в пункте 8. "
                        "Переходите к массовому сбору."
                    )
                elif (
                    can_apply
                    and compatibility.get("code") == "empty_rebind"
                ):
                    action_text = (
                        "Накопление фактически пусто. Нажмите "
                        "<b>«Применить выбранный датасет»</b>, чтобы "
                        "безопасно привязать его к текущей конфигурации."
                    )
                elif can_apply:
                    action_text = (
                        "Настройки совместимы. Нажмите "
                        "<b>«Применить выбранный датасет»</b>."
                    )
                else:
                    action_text = str(
                        compatibility.get("message")
                        or "Датасет несовместим с текущими настройками."
                    )

                _показать_панель(
                    заголовок=(
                        "Накопительный датасет готов"
                        if active_and_compatible
                        else "Проверка выбранного датасета"
                    ),
                    сообщение=(
                        _краткая_сводка(
                            entry=entry,
                            workflow=workflow,
                            include_path=False,
                        )
                        + "<br><br>"
                        + action_text
                    ),
                    тип=(
                        "success"
                        if active_and_compatible
                        else (
                            "info" if can_apply else "warning"
                        )
                    ),
                )

                paths = dict(entry.get("paths") or {})
                existing_downloads = {
                    label: path
                    for label, path in {
                        "Скачать накопительный Excel": paths.get("excel"),
                        "Скачать полный накопительный ZIP": paths.get("zip"),
                    }.items()
                    if path and Path(path).is_file()
                }

                if existing_downloads:
                    protected_download_buttons(
                        existing_downloads,
                        recommended_label=(
                            "Скачать накопительный Excel"
                        ),
                    )


        # -------------------------------------------------------------------------
        # Обновление списка
        # -------------------------------------------------------------------------

        def _обновить_список(
            *,
            preferred_name: str | None = None,
        ) -> None:
            workflow = _текущий_workflow()
            entries, broken = _найти_накопления(workflow)
            _STATE["entries"] = {
                entry["name"]: entry
                for entry in entries
            }

            selected_name = (
                preferred_name
                if preferred_name in _STATE["entries"]
                else None
            )
            options = [
                ("— Выберите накопительный датасет —", None)
            ] + [
                (
                    _метка_элемента(
                        entry=entry,
                        workflow=workflow,
                    ),
                    entry["name"],
                )
                for entry in entries
            ]

            _STATE["updating_list"] = True

            try:
                выбор_накопления.options = options
                выбор_накопления.value = selected_name
            finally:
                _STATE["updating_list"] = False

            with вывод_списка:
                вывод_списка.clear_output(wait=True)
                message = (
                    f"Найдено накоплений: <b>{len(entries)}</b>.<br>"
                    "В списке сразу показаны набор признаков и "
                    "совместимость с текущей конфигурацией.<br>"
                    "Ничего не выбрано автоматически."
                )

                if broken:
                    message += (
                        "<br><br>Не удалось прочитать папки: "
                        + ", ".join(
                            escape(value)
                            for value in broken
                        )
                    )

                _показать_панель(
                    заголовок="Выберите накопительный датасет",
                    сообщение=message,
                    тип=("info" if entries else "warning"),
                )

            if selected_name:
                _показать_выбранное()
            else:
                _очистить_выбор_накопления()


        # -------------------------------------------------------------------------
        # Применение существующего накопления
        # -------------------------------------------------------------------------

        def _обновить_служебный_отпечаток(
            entry: dict,
            compatibility: dict,
        ) -> None:
            if (
                compatibility.get("code")
                != "legacy_compatible"
            ):
                return

            goal = dict(entry["goal"])
            now = datetime.now(
                timezone.utc
            ).isoformat(timespec="seconds")
            goal["configuration_hash"] = (
                compatibility[
                    "new_configuration_hash"
                ]
            )
            goal["acceptance_hash"] = (
                compatibility[
                    "new_acceptance_hash"
                ]
            )
            goal["updated_at"] = now
            goal["compatibility_migration"] = {
                "type": (
                    "add_custom_features_sha256_v4_1_0"
                ),
                "migrated_at": now,
                "changed_data_files": False,
            }
            write_json_atomic(
                entry["goal_path"],
                goal,
            )
            entry["goal"] = goal


        def _применить_выбранное(_button) -> None:
            global DATASET_GOAL_RESULT

            entry = _выбранная_запись()

            with вывод_действия:
                вывод_действия.clear_output(
                    wait=True
                )

                if entry is None:
                    _показать_панель(
                        заголовок=(
                            "Накопление не применено"
                        ),
                        сообщение=(
                            "Сначала выберите накопление "
                            "в списке."
                        ),
                        тип="error",
                    )
                    return

                workflow = _текущий_workflow()
                missing = _проверить_готовность(
                    workflow
                )

                if missing:
                    _показать_панель(
                        заголовок=(
                            "Накопление не применено"
                        ),
                        сообщение=(
                            _сообщение_о_неготовности(
                                missing
                            )
                        ),
                        тип="error",
                    )
                    return

                storage_state = (
                    _фактическое_состояние_накопления(
                        folder=Path(entry["folder"]),
                        goal=entry["goal"],
                    )
                )
                entry["storage_state"] = storage_state
                entry["progress"] = (
                    _обновить_счётчики_из_хранилища(
                        progress=dict(entry.get("progress") or {}),
                        storage_state=storage_state,
                    )
                )
                compatibility = _совместимость(
                    goal=entry["goal"],
                    workflow=workflow,
                    folder=Path(entry["folder"]),
                    storage_state=storage_state,
                )
                empty_rebind = bool(
                    compatibility.get("code")
                    == "empty_rebind"
                )

                if not bool(
                    compatibility.get("can_apply")
                ):
                    _показать_панель(
                        заголовок=(
                            "Накопление не применено"
                        ),
                        сообщение=str(
                            compatibility.get(
                                "message"
                            )
                        ),
                        тип="error",
                    )
                    return

                кнопка_применить.disabled = True

                try:
                    _обновить_служебный_отпечаток(
                        entry,
                        compatibility,
                    )
                    goal = dict(entry["goal"])
                    with capture_output():
                        result = dataset_goal_step(
                            project_root=PROJECT_ROOT,
                            enabled=True,
                            goal_name=entry["name"],
                            target_company_count=int(
                                goal.get(
                                    "target_company_count"
                                )
                                or 10_000
                            ),
                            stop_when_target_reached=bool(
                                goal.get(
                                    "stop_when_target_reached",
                                    True,
                                )
                            ),
                            minimum_feature_fill_percent=(
                                goal.get(
                                    "minimum_feature_fill_percent"
                                )
                            ),
                            reset_existing=False,
                        )
                    DATASET_GOAL_RESULT = result

                    if not bool(
                        result.get("success")
                    ):
                        _показать_выбранное()
                        return

                    refreshed = export_dataset_goal_snapshot(
                        project_root=PROJECT_ROOT,
                        goal_name=entry["name"],
                    )
                    entry["goal"] = dict(
                        refreshed.get("goal") or entry.get("goal") or {}
                    )
                    entry["progress"] = dict(
                        refreshed.get("progress") or {}
                    )
                    entry["paths"] = dict(
                        refreshed.get("paths") or entry.get("paths") or {}
                    )
                    refreshed_storage_state = (
                        _фактическое_состояние_накопления(
                            folder=Path(entry["folder"]),
                            goal=entry["goal"],
                        )
                    )
                    entry["storage_state"] = refreshed_storage_state
                    entry["progress"] = (
                        _обновить_счётчики_из_хранилища(
                            progress=entry["progress"],
                            storage_state=refreshed_storage_state,
                        )
                    )
                    entry["compatibility"] = _совместимость(
                        goal=entry["goal"],
                        workflow=_текущий_workflow(),
                        folder=Path(entry["folder"]),
                        storage_state=refreshed_storage_state,
                    )
                    _обновить_список()
                    _обновить_описание_режима()
                    _показать_панель(
                        заголовок="Накопительный датасет готов",
                        сообщение=(
                            _краткая_сводка(
                                entry=entry,
                                workflow=_текущий_workflow(),
                                include_path=True,
                            )
                            + (
                                "<br><br>Пустое накопление безопасно "
                                "привязано к текущей конфигурации."
                                if empty_rebind
                                else ""
                            )
                            + "<br><br><b>Переходите к пункту 8.</b>"
                        ),
                        тип="success",
                    )

                except Exception as error:
                    _показать_панель(
                        заголовок=(
                            "Накопление не применено"
                        ),
                        сообщение=(
                            f"{escape(str(error))}<br>"
                            "Файлы накопительного датасета "
                            "не изменялись."
                        ),
                        тип="error",
                    )
                    _показать_выбранное()


        # -------------------------------------------------------------------------
        # Создание нового накопления
        # -------------------------------------------------------------------------

        def _создать_новое(_button) -> None:
            global DATASET_GOAL_RESULT

            with вывод_действия:
                вывод_действия.clear_output(
                    wait=True
                )
                workflow = _текущий_workflow()
                missing = _проверить_готовность(
                    workflow
                )

                if missing:
                    _показать_панель(
                        заголовок=(
                            "Новое накопление не создано"
                        ),
                        сообщение=(
                            _сообщение_о_неготовности(
                                missing
                            )
                        ),
                        тип="error",
                    )
                    return

                name = str(
                    новое_название.value
                ).strip()

                if not name:
                    _показать_панель(
                        заголовок=(
                            "Новое накопление не создано"
                        ),
                        сообщение=(
                            "Укажите название нового "
                            "накопительного датасета."
                        ),
                        тип="error",
                    )
                    return

                folder = (
                    _КОРЕНЬ_НАКОПЛЕНИЙ
                    / safe_filename(
                        name,
                        fallback="dataset_goal",
                    )
                )

                if (
                    (folder / "goal.json").is_file()
                    or name in _STATE["entries"]
                ):
                    existing_entry_name = next(
                        (
                            entry_name
                            for entry_name, entry_value
                            in _STATE["entries"].items()
                            if Path(entry_value["folder"]) == folder
                        ),
                        name,
                    )
                    _открыть_существующие()
                    _обновить_список(
                        preferred_name=existing_entry_name,
                    )

                    _показать_панель(
                        заголовок=(
                            "Такое накопление уже существует"
                        ),
                        сообщение=(
                            f"<b>{escape(existing_entry_name)}</b> "
                            "показано в списке существующих "
                            "накоплений.<br>"
                            "Ничего не перезаписано. Проверьте "
                            "его совместимость и нажмите "
                            "<b>«Применить выбранное»</b>."
                        ),
                        тип="warning",
                    )
                    return

                minimum_fill = (
                    float(новый_порог.value)
                    if bool(
                        использовать_новый_порог.value
                    )
                    else None
                )
                кнопка_создать.disabled = True

                try:
                    with capture_output():
                        result = dataset_goal_step(
                            project_root=PROJECT_ROOT,
                            enabled=True,
                            goal_name=name,
                            target_company_count=int(
                                новая_цель.value
                            ),
                            stop_when_target_reached=bool(
                                новая_автоостановка.value
                            ),
                            minimum_feature_fill_percent=(
                                minimum_fill
                            ),
                            reset_existing=False,
                        )
                    DATASET_GOAL_RESULT = result

                    if not bool(
                        result.get("success")
                    ):
                        return

                    created_entry = {
                        "name": name,
                        "goal": dict(result.get("goal") or {}),
                        "progress": dict(result.get("progress") or {}),
                        "paths": dict(result.get("paths") or {}),
                    }
                    created_entry["compatibility"] = _совместимость(
                        goal=created_entry["goal"],
                        workflow=_текущий_workflow(),
                    )
                    новое_название.value = ""
                    _обновить_описание_режима()
                    _показать_панель(
                        заголовок="Накопительный датасет готов",
                        сообщение=(
                            _краткая_сводка(
                                entry=created_entry,
                                workflow=_текущий_workflow(),
                                include_path=True,
                            )
                            + "<br><br><b>Переходите к пункту 8.</b>"
                        ),
                        тип="success",
                    )

                finally:
                    кнопка_создать.disabled = False


        # -------------------------------------------------------------------------
        # Обновление выгрузок и выключение накопления
        # -------------------------------------------------------------------------

        def _обновить_выгрузки(_button) -> None:
            entry = _выбранная_запись()

            with вывод_действия:
                вывод_действия.clear_output(
                    wait=True
                )

                if entry is None:
                    _показать_панель(
                        заголовок=(
                            "Файлы не обновлены"
                        ),
                        сообщение=(
                            "Сначала выберите накопление."
                        ),
                        тип="error",
                    )
                    return

                кнопка_обновить_файлы.disabled = True

                try:
                    snapshot = (
                        export_dataset_goal_snapshot(
                            project_root=PROJECT_ROOT,
                            goal_name=entry["name"],
                        )
                    )
                    progress = dict(
                        snapshot.get("progress") or {}
                    )
                    paths = dict(
                        snapshot.get("paths") or {}
                    )
                    entry["progress"] = progress
                    entry["paths"] = paths
                    refreshed_goal = dict(
                        snapshot.get("goal") or entry.get("goal") or {}
                    )
                    entry["goal"] = refreshed_goal
                    storage_state = (
                        _фактическое_состояние_накопления(
                            folder=Path(entry["folder"]),
                            goal=refreshed_goal,
                        )
                    )
                    entry["storage_state"] = storage_state
                    entry["progress"] = (
                        _обновить_счётчики_из_хранилища(
                            progress=entry["progress"],
                            storage_state=storage_state,
                        )
                    )

                    _показать_панель(
                        заголовок=(
                            "Excel и ZIP обновлены"
                        ),
                        сообщение=(
                            f"Накопление: "
                            f"<b>{escape(entry['name'])}</b>.<br>"
                            "Строки датасета не изменялись; "
                            "пересобраны только файлы "
                            "для скачивания."
                        ),
                        тип="info",
                    )
                    _показать_выбранное()

                except Exception as error:
                    _показать_панель(
                        заголовок=(
                            "Файлы не обновлены"
                        ),
                        сообщение=(
                            f"{escape(str(error))}<br>"
                            "Накопленный датасет не удалялся."
                        ),
                        тип="error",
                    )

                finally:
                    кнопка_обновить_файлы.disabled = False


        def _выключить_накопление(_button) -> None:
            global DATASET_GOAL_RESULT

            with вывод_действия:
                вывод_действия.clear_output(
                    wait=True
                )
                workflow = _текущий_workflow()
                missing = _проверить_готовность(
                    workflow
                )

                if missing:
                    _показать_панель(
                        заголовок=(
                            "Режим не изменён"
                        ),
                        сообщение=(
                            _сообщение_о_неготовности(
                                missing
                            )
                        ),
                        тип="error",
                    )
                    return

                with capture_output():
                    result = dataset_goal_step(
                        project_root=PROJECT_ROOT,
                        enabled=False,
                    )
                DATASET_GOAL_RESULT = result

                if not bool(
                    result.get("success")
                ):
                    return

                _показать_панель(
                    заголовок=(
                        "Накопительный режим выключен"
                    ),
                    сообщение=(
                        "Пункт 8 обработает только текущий "
                        "входной файл. Все ранее созданные "
                        "накопительные папки сохранены."
                    ),
                    тип="info",
                )
                _обновить_список()
                _обновить_описание_режима()


        # -------------------------------------------------------------------------
        # События: все callback-и защищены от красного traceback
        # -------------------------------------------------------------------------

        def _безопасный_callback(
            action_name: str,
            callback,
        ):
            def wrapped(*args, **kwargs):
                try:
                    return callback(*args, **kwargs)
                except Exception as error:
                    with вывод_действия:
                        вывод_действия.clear_output(
                            wait=True
                        )
                        _показать_панель(
                            заголовок=(
                                f"Действие «{escape(action_name)}» "
                                "не выполнено"
                            ),
                            сообщение=(
                                f"{escape(str(error))}<br>"
                                "Ранее созданные файлы на Google Диске "
                                "не изменялись. Повторите только это действие."
                            ),
                            тип="error",
                        )
                    return None

            return wrapped


        def _при_смене_режима(change) -> None:
            _обновить_видимость_меню()

            if bool(режим_накопления.value):
                _скрыть_разделы_действия()


        кнопка_открыть_существующий.on_click(
            _безопасный_callback(
                "Открыть существующие накопления",
                _открыть_существующие,
            )
        )
        кнопка_показать_создание.on_click(
            _безопасный_callback(
                "Открыть создание накопления",
                _открыть_создание,
            )
        )
        выбор_накопления.observe(
            _безопасный_callback(
                "Показать выбранное накопление",
                _показать_выбранное,
            ),
            names="value",
        )
        режим_накопления.observe(
            _безопасный_callback(
                "Открыть меню накоплений",
                _при_смене_режима,
            ),
            names="value",
        )
        кнопка_обновить_список.on_click(
            _безопасный_callback(
                "Обновить список накоплений",
                lambda _button: _обновить_список(),
            )
        )
        кнопка_применить.on_click(
            _безопасный_callback(
                "Применить выбранное накопление",
                _применить_выбранное,
            )
        )
        кнопка_создать.on_click(
            _безопасный_callback(
                "Создать новое накопление",
                _создать_новое,
            )
        )
        кнопка_обновить_файлы.on_click(
            _безопасный_callback(
                "Обновить Excel и ZIP",
                _обновить_выгрузки,
            )
        )
        кнопка_выключить.on_click(
            _безопасный_callback(
                "Применить режим без накопления",
                _выключить_накопление,
            )
        )

        # -------------------------------------------------------------------------
        # Интерфейс
        # -------------------------------------------------------------------------

        вводная = widgets.HTML(
            value=(
                "<div style='max-width:860px; padding:10px 12px; "
                "background:#f8f9fa; border-left:4px solid #5f6368; "
                "border-radius:6px; line-height:1.5;'>"
                "<b>Как пользоваться:</b><br>"
                "1. Нажмите <b>«Да»</b>, чтобы открыть меню накоплений.<br>"
                "2. Нажмите <b>«Открыть существующий датасет»</b> "
                "или <b>«Создать новый датасет»</b>.<br>"
                "3. В списке ничего не выбирается автоматически.<br>"
                "4. После выбора совместимого датасета нажмите "
                "<b>«Применить выбранный датасет»</b>.<br>"
                "5. После применения останется одна зелёная сводка "
                "с настройками готового датасета."
                "</div>"
            )
        )

        меню_накоплений = widgets.VBox(
            [
                вводная,
                панель_выбора_действия,
                блок_существующих,
                блок_создания,
            ],
            layout=widgets.Layout(
                width="880px",
                display="none",
            ),
        )

        блок_без_накопления = widgets.VBox(
            [
                кнопка_выключить,
            ],
            layout=widgets.Layout(
                width="880px",
                display="flex",
                margin="8px 0 0 0",
            ),
        )

        display(
            widgets.VBox(
                [
                    widgets.HTML(
                        value=(
                            "<h3 style='margin:0 0 8px 0;'>"
                            "Накопительный датасет"
                            "</h3>"
                        )
                    ),
                    режим_накопления,
                    описание_режима,
                    блок_без_накопления,
                    меню_накоплений,
                    вывод_действия,
                ],
                layout=widgets.Layout(
                    width="880px",
                ),
            )
        )

        # При каждом запуске ячейки интерфейс начинается с безопасного
        # положения «Нет». Существующая настройка workflow при этом не
        # изменяется, пока пользователь не нажмёт явную кнопку действия.
        _скрыть_разделы_действия()
        _обновить_видимость_меню()


## 8. Массовый сбор

Проверьте итоговую сводку, включите подтверждение и запустите сбор.

<details>
<summary><b>Техническая документация раздела 8</b></summary>


**Назначение раздела**

Раздел запускает производственный сбор данных СПАРК для всех компаний из подготовленного входного файла. Для каждой организации выполняются сохранённые методы API, правила детализации, расчёт выбранных ML-признаков и формирование `target_default`.

Перед запуском программа проверяет готовность предыдущих этапов:

* техническую подготовку проекта;
* активную авторизацию в СПАРК;
* сохранённый набор методов;
* настройку параметров методов;
* выбранные признаки и дополнительные поля;
* обязательную проверку на одной компании;
* подготовленный входной файл;
* настройки накопительного датасета, если этот режим включён.

Если какой-либо этап не завершён, массовый сбор не начинается. В сообщении указывается конкретный пункт ноутбука и действие, которое необходимо выполнить.

**Подтверждение запуска**

Параметр `ПОДТВЕРЖДАЮ_МАССОВЫЙ_ЗАПУСК` защищает от случайного запуска большого количества запросов.

При выключенном параметре ячейка только сообщает, что массовый сбор не подтверждён. Для запуска необходимо включить галочку и повторно выполнить ячейку.

**Обработка компаний**

Компании обрабатываются последовательно. Для каждой организации сохраняются:

* основной результат;
* ML-матрица;
* журнал выполненных методов;
* количество HTTP-запросов;
* использование сохранённых ответов;
* итоговый статус обработки;
* результат расчёта целевой переменной;
* причины исключения или неполного результата.

Повторное отсутствие данных у конкретного метода не обязательно считается ошибкой. У компании может не быть соответствующих сведений, связанных объектов или доступа к методу по текущей лицензии.

**Checkpoint и продолжение работы**

Через заданное в пункте 7 количество компаний создаётся контрольное сохранение.

Checkpoint содержит:

* уже обработанные компании;
* основные таблицы;
* ML-матрицу;
* журналы методов и компаний;
* состояние текущего запуска;
* позицию продолжения обработки.

Если среда Colab, интернет-соединение или сессия прервутся, совместимый checkpoint позволяет продолжить обработку без повторного выполнения всей работы.

**Накопительный датасет**

Если в пункте 7.1 включён накопительный режим, результаты дополнительно сохраняются в выбранной папке накопительного датасета.

Программа учитывает:

* ранее принятые компании;
* окончательно исключённые компании;
* временно ошибочные компании;
* новые компании из текущего файла;
* установленную цель по количеству подходящих строк.

Уже зарегистрированные компании не запрашиваются повторно, кроме результатов, которым назначена повторная обработка.

Если включена автоматическая остановка, массовый сбор завершается после достижения заданного количества подходящих компаний. Оставшиеся организации из входного файла не запрашиваются.

**Итоговые статусы компаний**

После завершения компании распределяются по понятным категориям:

* обработано без критических технических ошибок;
* строка сформирована, но часть методов не дала результат;
* отсутствует отчётность за выбранный год;
* отсутствует подходящая бухгалтерская отчётность;
* исключён финансовый сектор;
* компания ликвидирована или находится в процессе ликвидации;
* компания не действует;
* компания не найдена;
* требуется повторная обработка;
* возникла техническая, сетевая или авторизационная ошибка.

Статус «обработано без критических технических ошибок» означает, что техническая обработка компании завершилась без критической ошибки. Он не означает, что заполнены значения всех выбранных ML-признаков: полнота значений показывается отдельно.

Нулевые категории в итоговой сводке не показываются.

**Итоговая сводка**

После завершения выводятся:

* количество компаний в текущем файле;
* количество обработанных компаний;
* количество результатов, загруженных из checkpoint;
* число компаний, обработанных без критических технических ошибок;
* число неполных и исключённых результатов;
* количество строк основного датасета;
* количество строк ML-матрицы;
* полнота значений выбранных признаков: строки, где заполнены все признаки, и строки хотя бы с одним пустым значением;
* распределение `target_default`;
* количество операций методов СПАРК;
* фактическое количество HTTP-запросов;
* использование кэша;
* количество повторных попыток;
* количество операций методов с ошибочным статусом;
* число checkpoint-пакетов;
* время выполнения;
* папка с результатами.

**Предпросмотр результата**

В ноутбуке показываются только первые 5 строк и не более 10 основных колонок.

Предпросмотр предназначен для быстрой проверки того, что строки сформированы и основные идентификаторы присутствуют. Полный состав данных необходимо анализировать в итоговом Excel или ZIP-пакете.

**Сохраняемые результаты**

После массового запуска на Google Диске сохраняются:

* основной датасет;
* ML-матрица;
* журналы компаний;
* журналы методов;
* реестр выполнения;
* checkpoint;
* Excel-файл;
* ZIP-пакет;
* техническая сводка запуска.

Стандартные результаты текущего запуска и накопительный датасет хранятся раздельно.

</details>


In [ ]:
# @title 8. Выполнить массовый сбор

from __future__ import annotations

try:
    from src.notebook_runtime import run_notebook_cell
except Exception as _cell_runtime_error:
    print(
        "Не удалось запустить этап «Массовый сбор». "
        "Сначала выполните пункт 2. "
        f"Причина: {type(_cell_runtime_error).__name__}: {_cell_runtime_error}"
    )
else:
    with run_notebook_cell(
        stage="Массовый сбор",
        next_step="Исправьте указанный этап и повторите пункт 8.",
        project_root=globals().get("PROJECT_ROOT"),
    ):
        import builtins
        import html
        import re
        import time
        from collections import Counter
        from collections.abc import Mapping
        from pathlib import Path

        import ipywidgets as widgets
        import pandas as pd
        import requests
        from IPython.display import display

        import src.file_loader as file_loader
        import src.notebook_steps as notebook_steps
        import src.notebook_ui as notebook_ui
        import src.pipeline_v2 as pipeline_v2
        import src.workflow as workflow_module
        from src.runtime_readiness import validate_runtime_readiness


        ПОДТВЕРЖДАЮ_МАССОВЫЙ_ЗАПУСК = False  # @param {type:"boolean"}


        # Технические статусы переводятся только для итоговой сводки.
        # Центральный конвейер и сохранённые журналы не изменяются.
        _СТАТУСЫ_КОМПАНИЙ = {
            "success": "Обработано без критических технических ошибок",
            "partial_or_failed": (
                "Строка сформирована, но часть методов не дала результат"
            ),
            "target_report_year_missing": (
                "Не добавлено: отсутствует отчётность за выбранный год"
            ),
            "no_accounting_data": (
                "Не добавлено: отсутствует подходящая бухгалтерская отчётность"
            ),
            "excluded_financial_sector": (
                "Исключено: финансовый сектор по ОКВЭД 64–66"
            ),
            "excluded_liquidated": (
                "Исключено: компания ликвидирована"
            ),
            "excluded_liquidation_in_progress": (
                "Исключено: компания находится в процессе ликвидации"
            ),
            "excluded_inactive_other": (
                "Исключено: компания не действует"
            ),
            "no_pre_liquidation_accounting": (
                "Не добавлено: нет подходящей отчётности до ликвидации"
            ),
            "liquidation_date_missing": (
                "Не добавлено: отсутствует точная дата ликвидации"
            ),
            "not_found": (
                "Не добавлено: компания не найдена в СПАРК"
            ),
            "invalid_code": (
                "Не добавлено: некорректный ИНН или ОГРН"
            ),
            "unexpected_company_error": (
                "Техническая ошибка компании: требуется повторный запуск"
            ),
            "authentication_error": (
                "Ошибка авторизации: требуется повторный запуск"
            ),
            "network_error": (
                "Сетевая ошибка: требуется повторный запуск"
            ),
            "rate_limit": (
                "Превышен лимит запросов: требуется повторный запуск"
            ),
            "temporary_server_error": (
                "Временная ошибка СПАРК: требуется повторный запуск"
            ),
            "response_format_error": (
                "Не распознан ответ СПАРК: требуется проверка"
            ),
            "data_validation_error": (
                "Ошибка проверки данных: требуется проверка"
            ),
            "unexpected_error": (
                "Непредвиденная ошибка: требуется повторный запуск"
            ),
        }

        _СТАТУСЫ_ТРЕБУЮЩИЕ_ПОВТОРА = {
            "unexpected_company_error",
            "authentication_error",
            "network_error",
            "rate_limit",
            "temporary_server_error",
            "response_format_error",
            "data_validation_error",
            "unexpected_error",
        }

        _ПРИОРИТЕТНЫЕ_КОЛОНКИ = [
            "requested_code",
            "inn",
            "ogrn",
            "company_name",
            "report_year",
            "target_default",
            "target_status_label",
            "assets_rub",
            "revenue_rub",
            "net_profit_rub",
        ]


        RUN_RESULT = None


        _ОБЯЗАТЕЛЬНЫЕ_ЭТАПЫ = (
            "bootstrap",
            "authentication",
            "methods",
            "fields",
            "input",
        )

        _НАЗВАНИЯ_ЭТАПОВ = {
            "bootstrap": "Пункт 2 — техническая подготовка проекта",
            "authentication": "Пункт 3 — авторизация в СПАРК",
            "methods": "Пункты 4, 5.1 и 5.2 — методы, target и параметры методов",
            "fields": "Пункт 5.3 — выбор и сохранение ML-признаков",
            "test": "Пункт 6 — проверка настроек на одной компании",
            "input": "Пункт 7 — подготовка входного файла и настроек",
            "detail_runtime": "Пункт 2.1 — методы детализации",
            "method_policy": "Пункт 2 — политика методов",
            "target": "Пункт 5.1 — правило target",
            "target_runtime": "Пункт 2 — штатный обработчик target",
            "method_parameters": "Пункт 5.2 — параметры методов",
            "dataset_goal": "Пункт 7.1 — накопительный датасет",
        }

        _ДЕЙСТВИЯ_ПО_ЭТАПАМ = {
            "bootstrap": (
                "Запустите пункт 2. После перезапуска среды повторите "
                "пункты 2.1 и 3, затем снова запустите пункт 8. "
                "Сохранённые настройки и правило target повторно создавать не нужно."
            ),
            "authentication": (
                "Запустите пункт 3 и завершите авторизацию в СПАРК."
            ),
            "methods": (
                "Вернитесь к пункту 4, сохраните методы, затем повторно "
                "выполните пункты 5.1 и 5.2."
            ),
            "fields": (
                "Вернитесь к пункту 5.3, выберите признаки и нажмите кнопку "
                "сохранения набора."
            ),
            "test": (
                "Вернитесь к пункту 6 и нажмите «Проверить одну компанию». "
                "Дождитесь сообщения «Проверка успешно пройдена»."
            ),
            "detail_runtime": "Повторно выполните пункт 2.1.",
            "method_policy": "Повторно выполните пункт 2 и сохраните методы в пункте 4.",
            "target": "Сохраните правило target в пункте 5.1.",
            "target_runtime": "Повторно выполните пункт 2, затем пункты 2.1 и 3.",
            "method_parameters": "Повторно выполните пункт 5.2.",
            "dataset_goal": "Проверьте настройки пункта 7.1.",
            "input": (
                "Вернитесь к пункту 7, выберите или загрузите таблицу и "
                "обязательно нажмите «Подготовить входной файл». "
                "Простого запуска ячейки кнопкой Play недостаточно. "
                "Дождитесь сообщения «Входной файл готов»."
            ),
        }


        def _добавить_проблему(
            проблемы: dict,
            этап: str,
            причина: str = "",
        ) -> None:
            """Добавляет одну понятную проблему без дублирования этапов."""

            if этап in проблемы:
                return

            проблемы[этап] = {
                "title": _НАЗВАНИЯ_ЭТАПОВ.get(
                    этап,
                    f"Предыдущий этап [{этап}]",
                ),
                "action": _ДЕЙСТВИЯ_ПО_ЭТАПАМ.get(
                    этап,
                    "Повторно выполните соответствующий предыдущий пункт.",
                ),
                "reason": str(причина or "").strip(),
            }


        def _проверить_готовность_массового_запуска() -> dict:
            """Проверяет workflow и временные обработчики одной функцией ядра."""

            проверка = validate_runtime_readiness(
                project_root=globals().get("PROJECT_ROOT"),
                session=globals().get("SESSION"),
                require_test=True,
                require_input=True,
                require_dataset_goal=True,
            )
            проблемы = {}
            for code, issue in (проверка.get("issues") or {}).items():
                _добавить_проблему(
                    проблемы,
                    code,
                    issue.get("message") or "",
                )
                проблемы[code]["action"] = (
                    issue.get("action") or проблемы[code]["action"]
                )
            return {
                "ready": not проблемы,
                "issues": проблемы,
                "workflow": проверка.get("workflow") or {},
                "summary": проверка.get("summary") or {},
            }


        def _показать_незавершённые_этапы(
            проверка: Mapping,
        ) -> None:
            """Показывает номера пунктов и точные действия вместо внутренних имён."""

            проблемы = проверка.get("issues") or {}
            строки = [
                "Массовый сбор не запускался.",
                "",
                "Не завершены обязательные действия:",
            ]

            for index, issue in enumerate(
                проблемы.values(),
                start=1,
            ):
                строки.extend(
                    [
                        "",
                        f"{index}. {issue['title']}",
                        f"   Что сделать: {issue['action']}",
                    ]
                )

                reason = str(
                    issue.get("reason") or ""
                ).strip()

                if reason:
                    строки.append(
                        f"   Сохранённое состояние: {reason}"
                    )

            строки.extend(
                [
                    "",
                    (
                        "После выполнения указанных пунктов снова запустите "
                        "ячейку 8. Запросы к СПАРК не отправлялись, "
                        "checkpoint и ранее сохранённые файлы не изменялись."
                    ),
                ]
            )

            notebook_ui.show_panel(
                "Массовый сбор пока не готов к запуску",
                "\n".join(строки),
                kind="warning",
            )


        def _безопасное_число(value, default=0):
            """Преобразует значение сводки в число без исключения."""

            try:
                if pd.isna(value):
                    return default
                return int(value)
            except (TypeError, ValueError):
                return default


        def _таблица_состояния(result: Mapping, name: str) -> pd.DataFrame:
            """Возвращает таблицу из состояния массового запуска."""

            state = result.get("state") or {}
            dataframe = state.get(name)

            if isinstance(dataframe, pd.DataFrame):
                return dataframe.copy()

            return pd.DataFrame()


        def _коды_текущего_файла(result: Mapping) -> list[str]:
            """Возвращает корректные уникальные коды текущего файла."""

            strict_result = result.get("strict_input_result") or {}
            valid_codes = strict_result.get("valid_codes") or []

            if valid_codes:
                return list(
                    dict.fromkeys(
                        str(value)
                        for value in valid_codes
                        if str(value).strip()
                    )
                )

            company_log_df = _таблица_состояния(
                result,
                "company_log_df",
            )

            if (
                not company_log_df.empty
                and "requested_code" in company_log_df.columns
            ):
                return list(
                    dict.fromkeys(
                        company_log_df["requested_code"]
                        .dropna()
                        .astype(str)
                        .tolist()
                    )
                )

            return []


        def _статусы_компаний(result: Mapping) -> tuple[Counter, int, int]:
            """Считает статусы и компании, оставшиеся для повторной обработки."""

            company_log_df = _таблица_состояния(
                result,
                "company_log_df",
            )

            if company_log_df.empty:
                return Counter(), 0, 0

            if "requested_code" in company_log_df.columns:
                company_log_df = company_log_df.drop_duplicates(
                    subset=["requested_code"],
                    keep="last",
                )

            status_series = (
                company_log_df.get(
                    "status",
                    pd.Series(index=company_log_df.index, dtype="object"),
                )
                .fillna("unknown")
                .astype(str)
            )

            if "success" in company_log_df.columns:
                success_mask = (
                    company_log_df["success"]
                    .fillna(False)
                    .astype(bool)
                )
                fully_successful = int(success_mask.sum())
                status_counts = Counter(
                    status_series.loc[~success_mask].tolist()
                )
            else:
                status_counts = Counter(status_series.tolist())
                fully_successful = status_counts.pop("success", 0)

            if "final" in company_log_df.columns:
                retry_pending = int(
                    (~company_log_df["final"].fillna(False).astype(bool)).sum()
                )
            else:
                retry_pending = sum(
                    int(status_counts.get(status, 0))
                    for status in _СТАТУСЫ_ТРЕБУЮЩИЕ_ПОВТОРА
                )

            return status_counts, fully_successful, retry_pending


        def _статистика_ml(result: Mapping) -> dict:
            """Считает полноту выбранных признаков и значения target."""

            ml_df = _таблица_состояния(
                result,
                "ml_matrix_df",
            )
            main_df = _таблица_состояния(
                result,
                "main_dataset_df",
            )
            workflow = result.get("workflow") or {}
            selected_features = [
                str(value)
                for value in (
                    (workflow.get("fields") or {}).get(
                        "selected_feature_names"
                    )
                    or []
                )
            ]
            existing_features = [
                name
                for name in selected_features
                if name in ml_df.columns
            ]
            missing_feature_columns = [
                name
                for name in selected_features
                if name not in ml_df.columns
            ]

            if not ml_df.empty and existing_features:
                missing_mask = ml_df[existing_features].isna().any(axis=1)
                rows_with_missing_features = int(missing_mask.sum())
                rows_without_missing_features = int((~missing_mask).sum())
            else:
                rows_with_missing_features = 0
                rows_without_missing_features = 0

            target_source = (
                main_df
                if "target_default" in main_df.columns
                else ml_df
            )

            if (
                isinstance(target_source, pd.DataFrame)
                and not target_source.empty
                and "target_default" in target_source.columns
            ):
                target_numeric = pd.to_numeric(
                    target_source["target_default"],
                    errors="coerce",
                )
                target_zero = int(target_numeric.eq(0).sum())
                target_one = int(target_numeric.eq(1).sum())
                target_unknown = int(target_numeric.isna().sum())
            else:
                target_zero = 0
                target_one = 0
                target_unknown = 0

            return {
                "ml_rows": len(ml_df),
                "selected_features": len(selected_features),
                "existing_features": len(existing_features),
                "missing_feature_columns": missing_feature_columns,
                "rows_with_missing_features": rows_with_missing_features,
                "rows_without_missing_features": rows_without_missing_features,
                "target_zero": target_zero,
                "target_one": target_one,
                "target_unknown": target_unknown,
            }


        def _кэш_и_методы(result: Mapping) -> dict:
            """Считает кэш, операции методов и технические ошибки."""

            company_log_df = _таблица_состояния(
                result,
                "company_log_df",
            )
            method_log_df = _таблица_состояния(
                result,
                "method_log_df",
            )
            summary = result.get("summary") or {}

            method_cache_hits = 0
            legacy_cache_hits = 0

            if not company_log_df.empty:
                if "method_cache_hit_count" in company_log_df.columns:
                    method_cache_hits = int(
                        pd.to_numeric(
                            company_log_df["method_cache_hit_count"],
                            errors="coerce",
                        )
                        .fillna(0)
                        .sum()
                    )

                if "legacy_cache_hit" in company_log_df.columns:
                    legacy_cache_hits = int(
                        company_log_df["legacy_cache_hit"]
                        .fillna(False)
                        .astype(bool)
                        .sum()
                    )

            method_errors = _безопасное_число(
                summary.get("method_errors"),
                default=0,
            )

            if (
                method_errors == 0
                and not method_log_df.empty
                and "result_status" in method_log_df.columns
            ):
                method_errors = int(
                    method_log_df["result_status"]
                    .astype(str)
                    .isin(
                        {
                            "authentication_error",
                            "temporary_error",
                            "http_error",
                            "technical_error",
                            "invalid_parameters",
                            "unexpected_response",
                        }
                    )
                    .sum()
                )

            return {
                "method_operations": len(method_log_df),
                "method_errors": method_errors,
                "method_cache_hits": method_cache_hits,
                "legacy_cache_hits": legacy_cache_hits,
                "http_requests": _безопасное_число(
                    summary.get("http_request_count"),
                    default=0,
                ),
                "http_responses": _безопасное_число(
                    summary.get("http_response_count"),
                    default=0,
                ),
                "http_retries": _безопасное_число(
                    summary.get("retry_count"),
                    default=0,
                ),
                "response_bytes": _безопасное_число(
                    summary.get("response_bytes"),
                    default=0,
                ),
            }


        def _строки_статусов(status_counts: Counter) -> tuple[list[str], int]:
            """Формирует понятные строки только для ненулевых статусов."""

            lines = []
            retry_required = 0

            preferred_order = [
                "partial_or_failed",
                "target_report_year_missing",
                "no_accounting_data",
                "excluded_financial_sector",
                "excluded_liquidated",
                "excluded_liquidation_in_progress",
                "excluded_inactive_other",
                "no_pre_liquidation_accounting",
                "liquidation_date_missing",
                "not_found",
                "invalid_code",
                "unexpected_company_error",
                "authentication_error",
                "network_error",
                "rate_limit",
                "temporary_server_error",
                "response_format_error",
                "data_validation_error",
                "unexpected_error",
            ]

            handled = set()

            for status in preferred_order:
                count = int(status_counts.get(status, 0))

                if count <= 0:
                    continue

                handled.add(status)
                lines.append(
                    f"• {_СТАТУСЫ_КОМПАНИЙ.get(status, status)}: {count}"
                )

                if status in _СТАТУСЫ_ТРЕБУЮЩИЕ_ПОВТОРА:
                    retry_required += count

            for status, count in sorted(status_counts.items()):
                if status in handled or int(count) <= 0:
                    continue

                label = _СТАТУСЫ_КОМПАНИЙ.get(
                    status,
                    f"Другой результат [{status}]",
                )
                lines.append(f"• {label}: {int(count)}")

            return lines, retry_required


        def _формат_числа(value) -> str:
            """Форматирует целое число с пробелами между разрядами."""

            try:
                return f"{int(value):,}".replace(",", " ")
            except (TypeError, ValueError):
                return "0"


        def _показать_цветной_блок(
            title: str,
            *,
            rows: list[tuple[str, object]] | None = None,
            bullets: list[str] | None = None,
            message: str = "",
            tone: str = "info",
        ) -> None:
            """Показывает один компактный цветной блок."""

            palettes = {
                "success": ("#e6f4ea", "#34a853", "#137333"),
                "info": ("#e8f0fe", "#4285f4", "#174ea6"),
                "warning": ("#fef7e0", "#f9ab00", "#8d4e00"),
                "error": ("#fce8e6", "#d93025", "#a50e0e"),
                "purple": ("#f3e8fd", "#a142f4", "#681da8"),
                "neutral": ("#f8f9fa", "#9aa0a6", "#3c4043"),
            }
            background, border, title_color = palettes.get(
                tone,
                palettes["info"],
            )

            row_html = "".join(
                (
                    "<div style='color:#3c4043'>"
                    f"{html.escape(str(label))}</div>"
                    "<div style='font-weight:600;text-align:right;"
                    "overflow-wrap:anywhere;color:#202124'>"
                    f"{html.escape(str(value))}</div>"
                )
                for label, value in (rows or [])
            )
            rows_html = (
                "<div style='display:grid;"
                "grid-template-columns:minmax(230px,1fr) minmax(90px,auto);"
                "gap:7px 18px;margin-top:10px;align-items:start'>"
                f"{row_html}</div>"
                if row_html
                else ""
            )

            bullet_html = "".join(
                f"<li style='margin:4px 0'>{html.escape(str(item))}</li>"
                for item in (bullets or [])
                if str(item).strip()
            )
            bullets_html = (
                f"<ul style='margin:10px 0 0 20px;padding:0'>{bullet_html}</ul>"
                if bullet_html
                else ""
            )

            message_html = (
                "<div style='margin-top:9px;line-height:1.45;color:#3c4043'>"
                f"{html.escape(str(message)).replace(chr(10), '<br>')}</div>"
                if str(message).strip()
                else ""
            )

            display(
                widgets.HTML(
                    value=(
                        f"<div style='background:{background};"
                        f"border-left:6px solid {border};border-radius:8px;"
                        "padding:14px 16px;margin:10px 0;"
                        "font-family:Arial,sans-serif;box-sizing:border-box;width:100%'>"
                        f"<div style='font-size:16px;font-weight:700;color:{title_color}'>"
                        f"{html.escape(str(title))}</div>"
                        f"{message_html}{rows_html}{bullets_html}</div>"
                    )
                )
            )


        def _показать_итог_массового_сбора(
            result: Mapping,
            *,
            full_elapsed_seconds: float | None = None,
        ) -> None:
            """Показывает итог отдельными цветными смысловыми блоками."""

            summary = result.get("summary") or {}
            company_codes = _коды_текущего_файла(result)
            status_counts, fully_successful, retry_pending = _статусы_компаний(result)
            ml_stats = _статистика_ml(result)
            technical = _кэш_и_методы(result)

            total_companies = len(company_codes)
            processed_current = int(result.get("processed_company_count") or 0)
            reused_checkpoint = int(result.get("reused_company_count") or 0)
            planned_current = int(result.get("planned_company_count") or 0)
            unprocessed_current = int(result.get("unprocessed_company_count") or 0)
            accounted_companies = fully_successful + sum(status_counts.values())
            main_rows = _безопасное_число(
                summary.get("main_dataset_rows"),
                default=len(_таблица_состояния(result, "main_dataset_df")),
            )
            checkpoint_count = len(result.get("checkpoint_paths") or [])

            core_elapsed = float(summary.get("elapsed_seconds") or 0)
            full_elapsed = max(
                core_elapsed,
                float(
                    full_elapsed_seconds
                    if full_elapsed_seconds is not None
                    else core_elapsed
                ),
            )
            preparation_export_elapsed = max(0.0, full_elapsed - core_elapsed)

            partial_count = int(status_counts.get("partial_or_failed", 0))
            no_report_count = int(status_counts.get("target_report_year_missing", 0))
            no_accounting_count = int(status_counts.get("no_accounting_data", 0))
            has_attention = bool(
                retry_pending
                or partial_count
                or no_report_count
                or no_accounting_count
                or technical["method_errors"]
                or ml_stats["missing_feature_columns"]
            )

            _показать_цветной_блок(
                "Массовый сбор завершён",
                tone="success",
                message=(
                    "Обработка завершена, checkpoint обновлён, итоговые файлы "
                    "и накопительный датасет сохранены."
                ),
                rows=[
                    ("Компаний во входном файле", _формат_числа(total_companies)),
                    ("Новых компаний обработано", _формат_числа(processed_current)),
                    ("Взято из совместимого checkpoint", _формат_числа(reused_checkpoint)),
                    ("Всего учтено в результате", _формат_числа(accounted_companies)),
                    ("Строк сформировано в основном датасете", _формат_числа(main_rows)),
                ],
            )

            company_rows = [
                ("Запланировано новых компаний", _формат_числа(planned_current)),
                ("Фактически обработано новых", _формат_числа(processed_current)),
                ("Повторно использовано", _формат_числа(reused_checkpoint)),
                (
                    "Обработано без критических технических ошибок",
                    _формат_числа(fully_successful),
                ),
            ]
            preferred_status_order = [
                "partial_or_failed",
                "target_report_year_missing",
                "no_accounting_data",
                "excluded_financial_sector",
                "excluded_liquidated",
                "excluded_liquidation_in_progress",
                "excluded_inactive_other",
                "no_pre_liquidation_accounting",
                "liquidation_date_missing",
                "not_found",
                "invalid_code",
                "unexpected_company_error",
                "authentication_error",
                "network_error",
                "rate_limit",
                "temporary_server_error",
                "response_format_error",
                "data_validation_error",
                "unexpected_error",
            ]
            handled = set()

            for status in preferred_status_order:
                count = int(status_counts.get(status, 0))
                if count <= 0:
                    continue
                handled.add(status)
                company_rows.append(
                    (
                        _СТАТУСЫ_КОМПАНИЙ.get(status, status),
                        _формат_числа(count),
                    )
                )

            for status, count in sorted(status_counts.items()):
                if status in handled or int(count) <= 0:
                    continue
                company_rows.append(
                    (
                        _СТАТУСЫ_КОМПАНИЙ.get(status, f"Другой результат [{status}]"),
                        _формат_числа(count),
                    )
                )

            company_rows.extend(
                [
                    ("Осталось необработанными", _формат_числа(unprocessed_current)),
                    ("Требуют повторной обработки", _формат_числа(retry_pending)),
                ]
            )

            if retry_pending:
                company_tone = "error"
                company_message = (
                    "Часть компаний не получила окончательный результат. "
                    "При следующем запуске с checkpoint они будут обработаны повторно."
                )
            elif has_attention:
                company_tone = "warning"
                company_message = (
                    "Сбор завершён без обязательного повтора. Технический статус компании "
                    "показывает, завершилась ли её обработка без критической ошибки; "
                    "он не означает, что заполнены все выбранные ML-признаки. "
                    "Часть компаний сохранена с неполными данными или не вошла "
                    "в датасет по правилам отбора."
                )
            else:
                company_tone = "success"
                company_message = (
                    "Все компании получили окончательный технический результат. "
                    "Полнота значений ML-признаков показана отдельным блоком ниже."
                )

            _показать_цветной_блок(
                "Технический результат обработки компаний",
                tone=company_tone,
                message=company_message,
                rows=company_rows,
            )

            if result.get("dataset_goal_result"):
                _показать_цветной_блок(
                    "Накопительный датасет",
                    tone="purple",
                    message=(
                        "Это отдельный показатель: число принятых компаний "
                        "не равно числу строк, где заполнены все ML-признаки."
                    ),
                    rows=[
                        (
                            "Компаний принято до запуска",
                            _формат_числа(result.get("accepted_before_run") or 0),
                        ),
                        (
                            "Компаний принято в текущем запуске",
                            _формат_числа(result.get("accepted_in_current_run") or 0),
                        ),
                        (
                            "Всего компаний принято в накопительный датасет",
                            _формат_числа(result.get("accepted_total") or 0),
                        ),
                        (
                            "Осталось до цели",
                            _формат_числа(result.get("remaining_to_goal") or 0),
                        ),
                        (
                            "Проверено кандидатов всего",
                            _формат_числа(result.get("examined_candidate_count") or 0),
                        ),
                        (
                            "Остановка после достижения цели",
                            (
                                "Да, цель достигнута"
                                if result.get("stopped_after_goal")
                                else "Нет"
                            ),
                        ),
                    ],
                )

            all_feature_columns_present = not ml_stats["missing_feature_columns"]
            _показать_цветной_блок(
                "ML-датасет и целевая переменная",
                tone="success" if all_feature_columns_present else "error",
                message=(
                    "Все выбранные признаки присутствуют как колонки ML-матрицы. "
                    "Это не означает, что в каждой строке заполнено значение "
                    "по каждому признаку."
                    if all_feature_columns_present
                    else (
                        "Часть выбранных колонок отсутствует в ML-матрице. "
                        "Список показан ниже."
                    )
                ),
                rows=[
                    ("Строк ML-матрицы", _формат_числа(ml_stats["ml_rows"])),
                    (
                        "Найдено выбранных признаков",
                        (
                            f"{_формат_числа(ml_stats['existing_features'])} "
                            f"из {_формат_числа(ml_stats['selected_features'])}"
                        ),
                    ),
                    (
                        "Строк, где заполнены все выбранные признаки",
                        (
                            f"{_формат_числа(ml_stats['rows_without_missing_features'])} "
                            f"из {_формат_числа(ml_stats['ml_rows'])}"
                        ),
                    ),
                    (
                        "Строк хотя бы с одним пустым значением среди выбранных признаков",
                        (
                            f"{_формат_числа(ml_stats['rows_with_missing_features'])} "
                            f"из {_формат_числа(ml_stats['ml_rows'])}"
                        ),
                    ),
                    ("target_default = 0", _формат_числа(ml_stats["target_zero"])),
                    ("target_default = 1", _формат_числа(ml_stats["target_one"])),
                    (
                        "target_default не определён",
                        _формат_числа(ml_stats["target_unknown"]),
                    ),
                ],
            )

            _показать_цветной_блок(
                "Запросы СПАРК, кэш и checkpoint",
                tone=(
                    "warning"
                    if technical["method_errors"] or technical["http_retries"]
                    else "neutral"
                ),
                rows=[
                    (
                        "Операций методов в журнале",
                        _формат_числа(technical["method_operations"]),
                    ),
                    (
                        "Фактических HTTP-запросов",
                        _формат_числа(technical["http_requests"]),
                    ),
                    (
                        "Получено HTTP-ответов",
                        _формат_числа(technical["http_responses"]),
                    ),
                    (
                        "Повторных HTTP-попыток",
                        _формат_числа(technical["http_retries"]),
                    ),
                    (
                        "Ответов методов взято из кэша",
                        _формат_числа(technical["method_cache_hits"]),
                    ),
                    (
                        "Компаний с финансовым ядром из кэша",
                        _формат_числа(technical["legacy_cache_hits"]),
                    ),
                    (
                        "Операций методов с ошибочным статусом",
                        _формат_числа(technical["method_errors"]),
                    ),
                    ("Checkpoint-пакетов", _формат_числа(checkpoint_count)),
                    (
                        "Объём полученных HTTP-ответов",
                        notebook_ui.format_file_size(technical["response_bytes"]),
                    ),
                ],
            )

            _показать_цветной_блок(
                "Время и сохранённые результаты",
                tone="info",
                message=(
                    "Полное время включает обработку компаний, объединение checkpoint, "
                    "обновление накопительного датасета, экспорт и запись на Google Диск."
                ),
                rows=[
                    (
                        "Основной конвейер",
                        notebook_ui.format_duration(core_elapsed),
                    ),
                    (
                        "Подготовка, экспорт и запись файлов",
                        notebook_ui.format_duration(preparation_export_elapsed),
                    ),
                    (
                        "Полное время массового запуска",
                        notebook_ui.format_duration(full_elapsed),
                    ),
                    (
                        "Папка результата",
                        result.get("output_directory") or "не указана",
                    ),
                ],
            )

            if ml_stats["missing_feature_columns"]:
                preview = list(ml_stats["missing_feature_columns"][:20])
                if len(ml_stats["missing_feature_columns"]) > 20:
                    preview.append("…")
                _показать_цветной_блок(
                    "Отсутствующие колонки признаков",
                    tone="error",
                    message=(
                        "Эти выбранные признаки полностью отсутствуют "
                        "в сформированной ML-матрице."
                    ),
                    bullets=preview,
                )

            if retry_pending:
                action_tone = "warning"
                action_title = "Что произойдёт при следующем запуске"
                action_message = (
                    "Результаты уже сохранены. Компании с временными ошибками будут "
                    "обработаны повторно при продолжении совместимого checkpoint."
                )
            elif has_attention:
                action_tone = "info"
                action_title = "Что делать дальше"
                action_message = (
                    "Обязательного повторного запуска нет. Для анализа неполных данных "
                    "откройте журналы компаний и методов в итоговом Excel или ZIP-пакете."
                )
            else:
                action_tone = "success"
                action_title = "Готово"
                action_message = (
                    "Результаты сохранены. Для дальнейшего анализа используйте "
                    "итоговый Excel или ZIP-пакет."
                )

            _показать_цветной_блок(
                action_title,
                tone=action_tone,
                message=action_message,
            )


        def _компактный_предпросмотр(result: Mapping) -> None:
            """Показывает 5 строк и не более 10 понятных колонок."""

            dataframe = _таблица_состояния(
                result,
                "main_dataset_df",
            )

            if dataframe.empty:
                dataframe = _таблица_состояния(
                    result,
                    "ml_matrix_df",
                )

            if dataframe.empty:
                _показать_цветной_блок(
                    "Нет данных для предпросмотра",
                    tone="info",
                    message=(
                        "Полные результаты сохранены "
                        "в Excel и ZIP-пакете."
                    ),
                )
                return

            selected_columns = [
                column
                for column in _ПРИОРИТЕТНЫЕ_КОЛОНКИ
                if column in dataframe.columns
            ]

            if len(selected_columns) < 10:
                selected_columns.extend(
                    column
                    for column in dataframe.columns
                    if column not in selected_columns
                )

            selected_columns = selected_columns[:10]

            _показать_цветной_блок(
                "Краткий предпросмотр результата",
                tone="neutral",
                message=(
                    "Показаны первые 5 строк и не более 10 основных колонок. "
                    "Полный состав данных находится в Excel и ZIP-пакете."
                ),
            )
            display(
                dataframe.loc[
                    :,
                    selected_columns,
                ].head(5)
            )


        def _выполнить_массовый_сбор() -> dict:
            """
            Запускает ядро, показывает один обновляемый индикатор прогресса
            и заменяет только стандартный итоговый вывод.
            """

            full_run_started_at = time.perf_counter()

            saved_functions = {
                "show_panel": notebook_steps.show_panel,
                "dataframe_preview": notebook_steps.dataframe_preview,
                "protected_download_buttons": (
                    notebook_steps.protected_download_buttons
                ),
            }
            delayed_output = {}

            # В pipeline_v2 прогресс печатается двумя строками:
            # [номер/всего] ИНН
            #   готово / завершено со статусом ...
            # Временно перехватываем только эти две строки и обновляем
            # один виджет. Журналы, checkpoint и логика ядра не меняются.
            _print_not_defined = object()
            saved_pipeline_print = getattr(
                pipeline_v2,
                "print",
                _print_not_defined,
            )
            saved_file_loader_print = getattr(
                file_loader,
                "print",
                _print_not_defined,
            )

            progress_state = {
                "position": 0,
                "total": 0,
                "company_code": "",
                "status": "Подготовка массового запуска…",
            }
            preparation_state = {
                "capturing": False,
                "lines": [],
                "shown": False,
            }

            preparation_info = widgets.HTML(
                value=(
                    "<div style='background:#e8f0fe;border-left:5px solid #4285f4;"
                    "border-radius:7px;padding:10px 12px;margin-bottom:8px'>"
                    "<b style='color:#174ea6'>Подготовка входного файла</b><br>"
                    "<span style='color:#3c4043'>Чтение файла и проверка ИНН/ОГРН…</span>"
                    "</div>"
                )
            )
            progress_title = widgets.HTML(
                "<b>Массовый сбор выполняется</b>"
            )
            progress_bar = widgets.IntProgress(
                value=0,
                min=0,
                max=1,
                description="",
                bar_style="info",
                orientation="horizontal",
                layout=widgets.Layout(width="100%"),
            )
            progress_text = widgets.HTML(
                "<span>Подготовка списка и проверка checkpoint…</span>"
            )
            progress_box = widgets.VBox(
                [
                    preparation_info,
                    progress_title,
                    progress_bar,
                    progress_text,
                ],
                layout=widgets.Layout(
                    width="100%",
                    border="1px solid #dadce0",
                    padding="10px",
                ),
            )
            display(progress_box)

            def _показать_подготовку_файла() -> None:
                """Сворачивает технический текст проверки файла в один блок."""

                if preparation_state["shown"]:
                    return

                preparation_state["shown"] = True
                raw_lines = [
                    str(value).rstrip()
                    for value in preparation_state["lines"]
                    if str(value).strip()
                    and not re.fullmatch(r"=+", str(value).strip())
                ]

                values = {}
                for line in raw_lines:
                    if ":" not in line:
                        continue
                    key, value = line.split(":", 1)
                    values[key.strip()] = value.strip()

                summary_parts = []
                for key in (
                    "Файл",
                    "Формат",
                    "Строк",
                    "Выбранный столбец",
                    "Уникальных корректных кодов",
                    "Кодов с корректным контрольным разрядом",
                    "Дополнительно исключено",
                ):
                    if key in values:
                        summary_parts.append(
                            f"<b>{html.escape(key)}:</b> "
                            f"{html.escape(values[key])}"
                        )

                technical_text = html.escape(
                    "\n".join(raw_lines)
                )
                details_html = (
                    "<details style='margin-top:8px'>"
                    "<summary style='cursor:pointer;color:#174ea6'>"
                    "Показать технические подробности проверки файла"
                    "</summary>"
                    "<pre style='white-space:pre-wrap;margin:8px 0 0;"
                    "font-size:12px;line-height:1.35;color:#3c4043'>"
                    f"{technical_text}</pre></details>"
                    if raw_lines
                    else ""
                )
                preparation_info.value = (
                    "<div style='background:#e8f0fe;border-left:5px solid #4285f4;"
                    "border-radius:7px;padding:10px 12px;margin-bottom:8px'>"
                    "<b style='color:#174ea6'>Входной файл проверен</b><br>"
                    "<span style='color:#3c4043;line-height:1.5'>"
                    + "<br>".join(summary_parts)
                    + "</span>"
                    + details_html
                    + "</div>"
                )

            def _обновить_прогресс(
                *,
                completed: int | None = None,
                total: int | None = None,
                company_code: str | None = None,
                status: str | None = None,
            ) -> None:
                """Обновляет один компактный блок без добавления новых строк."""

                if total is not None:
                    progress_state["total"] = max(
                        0,
                        int(total),
                    )

                if completed is not None:
                    progress_state["position"] = max(
                        0,
                        int(completed),
                    )

                if company_code is not None:
                    progress_state["company_code"] = str(
                        company_code
                    ).strip()

                if status is not None:
                    progress_state["status"] = str(
                        status
                    ).strip()

                current_total = max(
                    1,
                    int(progress_state["total"] or 1),
                )
                current_position = min(
                    current_total,
                    max(
                        0,
                        int(progress_state["position"] or 0),
                    ),
                )
                progress_bar.max = current_total
                progress_bar.value = current_position

                percentage = (
                    100.0
                    * current_position
                    / current_total
                    if progress_state["total"]
                    else 0.0
                )
                safe_code = html.escape(
                    progress_state["company_code"]
                )
                safe_status = html.escape(
                    progress_state["status"]
                )
                code_part = (
                    f" · Компания: <code>{safe_code}</code>"
                    if safe_code
                    else ""
                )

                progress_text.value = (
                    "<span>"
                    f"Обработано: <b>{current_position:,}</b> "
                    f"из <b>{int(progress_state['total'] or 0):,}</b> "
                    f"({percentage:.1f}%)"
                    f"{code_part}<br>"
                    f"Статус: {safe_status}"
                    "</span>"
                ).replace(",", " ")

            def _progress_print(
                *args,
                sep=" ",
                end="\n",
                file=None,
                flush=False,
                **kwargs,
            ):
                """
                Сворачивает техническую подготовку и обновляет одну строку прогресса.
                Неожиданный вывод продолжает печататься как обычно.
                """

                message = sep.join(str(value) for value in args)
                normalized = message.strip()
                progress_match = re.fullmatch(
                    r"\[(\d+)/(\d+)\]\s+(.+)",
                    normalized,
                )

                if (
                    normalized.startswith("ЗАГРУЗКА СПИСКА КОМПАНИЙ")
                    and not preparation_state["capturing"]
                ):
                    preparation_state["capturing"] = True

                if preparation_state["capturing"] and not progress_match:
                    preparation_state["lines"].append(message)
                    return None

                if progress_match:
                    if preparation_state["capturing"]:
                        preparation_state["capturing"] = False
                        _показать_подготовку_файла()

                    position = int(progress_match.group(1))
                    total = int(progress_match.group(2))
                    company_code = progress_match.group(3).strip()

                    _обновить_прогресс(
                        completed=max(0, position - 1),
                        total=total,
                        company_code=company_code,
                        status=f"Обрабатывается компания {position} из {total}",
                    )
                    return None

                if (
                    normalized == "готово"
                    or normalized.startswith("завершено со статусом ")
                ):
                    current_position = int(progress_state["position"] or 0)
                    total = int(progress_state["total"] or 0)
                    completed = min(total, current_position + 1)

                    if total > 0 and completed >= total:
                        progress_title.value = (
                            "<b>Компании обработаны — сохраняются результаты</b>"
                        )
                        status = (
                            "Все компании обработаны. Обновляется накопительный датасет, "
                            "объединяются checkpoint и формируются итоговые файлы…"
                        )
                    else:
                        status = (
                            "Готово"
                            if normalized == "готово"
                            else normalized
                        )

                    _обновить_прогресс(
                        completed=completed,
                        status=status,
                    )
                    return None

                return builtins.print(
                    *args,
                    sep=sep,
                    end=end,
                    file=file,
                    flush=flush,
                    **kwargs,
                )

            def _panel(title, message, *, kind="info"):
                if title == "Массовый сбор завершён":
                    delayed_output["standard_summary"] = {
                        "title": title,
                        "message": message,
                        "kind": kind,
                    }
                    return None

                return saved_functions["show_panel"](
                    title,
                    message,
                    kind=kind,
                )

            def _preview(*args, **kwargs):
                delayed_output["preview"] = (args, kwargs)
                return None

            def _downloads(*args, **kwargs):
                delayed_output["downloads"] = (args, kwargs)
                return None

            notebook_steps.show_panel = _panel
            notebook_steps.dataframe_preview = _preview
            notebook_steps.protected_download_buttons = _downloads
            pipeline_v2.print = _progress_print
            file_loader.print = _progress_print

            try:
                result = notebook_steps.full_run_step(
                    project_root=PROJECT_ROOT,
                    session=globals().get("SESSION"),
                    confirm_run=True,
                )
            finally:
                notebook_steps.show_panel = saved_functions["show_panel"]
                notebook_steps.dataframe_preview = saved_functions[
                    "dataframe_preview"
                ]
                notebook_steps.protected_download_buttons = saved_functions[
                    "protected_download_buttons"
                ]

                if saved_pipeline_print is _print_not_defined:
                    try:
                        delattr(
                            pipeline_v2,
                            "print",
                        )
                    except AttributeError:
                        pass
                else:
                    pipeline_v2.print = saved_pipeline_print

                if saved_file_loader_print is _print_not_defined:
                    try:
                        delattr(
                            file_loader,
                            "print",
                        )
                    except AttributeError:
                        pass
                else:
                    file_loader.print = saved_file_loader_print

            full_elapsed_seconds = (
                time.perf_counter()
                - full_run_started_at
            )

            if not preparation_state["shown"]:
                preparation_state["capturing"] = False
                _показать_подготовку_файла()

            if result.get("success"):
                processed_count = int(
                    result.get(
                        "processed_company_count"
                    )
                    or progress_state["position"]
                    or 0
                )
                reused_count = int(
                    result.get(
                        "reused_company_count"
                    )
                    or 0
                )
                planned_count = int(
                    result.get(
                        "planned_company_count"
                    )
                    or progress_state["total"]
                    or 0
                )
                total_in_file = len(
                    _коды_текущего_файла(result)
                )

                if processed_count == 0 and reused_count > 0:
                    # В этом сценарии новых запросов нет: весь текущий
                    # файл уже присутствует в checkpoint. Показываем
                    # фактически завершённые компании, а не вводящие
                    # в заблуждение 0 из 0.
                    final_total = max(
                        total_in_file,
                        reused_count,
                    )
                    final_completed = min(
                        final_total,
                        reused_count,
                    )
                    final_status = (
                        "Новых запросов не потребовалось: "
                        f"{reused_count} компаний взяты из checkpoint"
                    )
                else:
                    final_total = max(
                        planned_count,
                        processed_count,
                    )
                    final_completed = processed_count

                    if result.get("stopped_after_goal"):
                        final_status = (
                            "Сбор остановлен после достижения "
                            "цели накопительного датасета"
                        )
                    elif reused_count:
                        final_status = (
                            "Массовый сбор завершён. "
                            f"Из checkpoint использовано: {reused_count}"
                        )
                    else:
                        final_status = "Массовый сбор завершён"

                _обновить_прогресс(
                    completed=final_completed,
                    total=final_total,
                    company_code="",
                    status=final_status,
                )
                progress_bar.bar_style = "success"
                progress_title.value = (
                    "<b>Массовый сбор завершён</b>"
                )

                _показать_итог_массового_сбора(
                    result,
                    full_elapsed_seconds=full_elapsed_seconds,
                )
                _компактный_предпросмотр(result)

                if "downloads" in delayed_output:
                    args, kwargs = delayed_output["downloads"]
                    saved_functions["protected_download_buttons"](
                        *args,
                        **kwargs,
                    )
            else:
                progress_bar.bar_style = "danger"
                progress_title.value = (
                    "<b>Массовый сбор остановлен</b>"
                )
                _обновить_прогресс(
                    status=(
                        result.get("error")
                        or result.get("message")
                        or "Сбор не завершён"
                    ),
                )

            return result

        if not ПОДТВЕРЖДАЮ_МАССОВЫЙ_ЗАПУСК:
            notebook_ui.show_panel(
                "Массовый сбор не запущен",
                (
                    "Включите параметр «ПОДТВЕРЖДАЮ_МАССОВЫЙ_ЗАПУСК», "
                    "затем повторно запустите эту ячейку кнопкой Play."
                ),
                kind="info",
            )
            RUN_RESULT = {
                "success": False,
                "not_started": True,
                "message": "Массовый запуск не подтверждён.",
            }
        else:
            _ПРОВЕРКА_ГОТОВНОСТИ = (
                _проверить_готовность_массового_запуска()
            )

            if not _ПРОВЕРКА_ГОТОВНОСТИ["ready"]:
                _показать_незавершённые_этапы(
                    _ПРОВЕРКА_ГОТОВНОСТИ
                )
                RUN_RESULT = {
                    "success": False,
                    "not_started": True,
                    "missing_stages": list(
                        _ПРОВЕРКА_ГОТОВНОСТИ[
                            "issues"
                        ].keys()
                    ),
                    "message": (
                        "Массовый запуск не начат: "
                        "не завершены обязательные пункты ноутбука."
                    ),
                }
            else:
                RUN_RESULT = _выполнить_массовый_сбор()


## 9. Результаты и выгрузка

Эта ячейка не обращается к API. Она читает последний паспорт
запуска с вашего Диска, показывает сводку и создаёт защищённые
кнопки скачивания:

- **Excel** — основной понятный файл для просмотра;
- **ZIP** — полный пакет CSV/Parquet и журналов;
- **паспорт** — конфигурация и статистика запуска;
- **manifest** — размеры и SHA‑256 файлов.

Все файлы уже остаются в `runs_v2/`, даже если ничего не
скачивать через браузер.


In [ ]:
# @title 9. Открыть и скачать последний результат

from __future__ import annotations

try:
    from src.notebook_runtime import run_notebook_cell
except Exception as _cell_runtime_error:
    print(
        "Не удалось запустить этап «Выгрузка результата». "
        "Сначала выполните пункт 2. "
        f"Причина: {type(_cell_runtime_error).__name__}: {_cell_runtime_error}"
    )
else:
    with run_notebook_cell(
        stage="Выгрузка результата",
        next_step="Проверьте наличие запуска и повторите пункт 9.",
        project_root=globals().get("PROJECT_ROOT"),
    ):
        import importlib
        import inspect
        from pathlib import Path

        try:
            import src.notebook_steps as notebook_steps
            import src.notebook_ui as notebook_ui
        except ModuleNotFoundError as error:
            raise RuntimeError(
                "Технические модули проекта не найдены. "
                "Сначала выполните ячейку 2, затем снова запустите пункт 9."
            ) from error


        # Загружаем актуальную версию пользовательского интерфейса с Google Диска.
        # API СПАРК в этой ячейке не вызывается.
        notebook_ui = importlib.reload(notebook_ui)

        try:
            _ГИБКАЯ_ВЫГРУЗКА_ДОСТУПНА = (
                "Настроить состав Excel"
                in inspect.getsource(
                    notebook_ui.protected_download_buttons
                )
            )
        except (AttributeError, OSError, TypeError):
            _ГИБКАЯ_ВЫГРУЗКА_ДОСТУПНА = False

        if not _ГИБКАЯ_ВЫГРУЗКА_ДОСТУПНА:
            raise RuntimeError(
                "В проекте не найдена актуальная функция настройки Excel. "
                "Повторно выполните обновлённую ячейку 2, "
                "затем снова запустите пункт 9."
            )


        RESULTS = None


        def _папка_результата(files: dict) -> Path | None:
            """Определяет папку последнего запуска по найденным файлам."""

            for raw_path in files.values():
                path = Path(raw_path)

                if path.is_file():
                    return path.parent

            return None


        def _название_запуска(workflow: dict, folder: Path | None) -> str:
            """Возвращает идентификатор последнего сохранённого запуска."""

            run_state = workflow.get("run") or {}

            for key in (
                "run_id",
                "id",
            ):
                value = str(
                    run_state.get(key) or ""
                ).strip()

                if value:
                    return value

            if folder is not None:
                return folder.name

            return "не определён"


        def _понятное_действие_после_ошибки(error: Exception) -> tuple[str, str]:
            """Переводит техническую причину в понятный следующий шаг."""

            message = str(error).lower()

            if (
                "google" in message
                or "диск" in message
                or "project_root" in message
            ):
                return (
                    "Google Диск или проект не подключён",
                    (
                        "Сначала выполните пункты 1 и 2, "
                        "затем снова запустите пункт 9."
                    ),
                )

            if (
                "results" in message
                or "этап" in message
                or "последнего запуска" in message
                or "не найдены" in message
            ):
                return (
                    "Сохранённый массовый результат не найден",
                    (
                        "Сначала завершите пункт 8 хотя бы один раз. "
                        "После этого пункт 9 сможет открыть готовые файлы "
                        "без повторного обращения к СПАРК."
                    ),
                )

            return (
                "Последний результат не удалось открыть",
                (
                    "Проверьте, что Google Диск подключён и файлы последнего "
                    "запуска не были перемещены или удалены. Затем повторите пункт 9."
                ),
            )


        def _открыть_последний_результат() -> dict:
            """Открывает последний результат, не повторяя вывод пункта 8."""

            saved_functions = {
                "show_panel": notebook_steps.show_panel,
                "dataframe_preview": notebook_steps.dataframe_preview,
                "protected_download_buttons": (
                    notebook_steps.protected_download_buttons
                ),
                "show_error": notebook_steps.show_error,
            }
            captured_error = {}

            def _не_показывать_стандартную_панель(*args, **kwargs):
                return None

            def _не_показывать_таблицу(*args, **kwargs):
                return None

            def _не_показывать_стандартные_кнопки(*args, **kwargs):
                return None

            def _перехватить_ошибку(*, stage, error, action):
                captured_error["error"] = error
                captured_error["stage"] = stage
                captured_error["action"] = action
                return None

            notebook_steps.show_panel = _не_показывать_стандартную_панель
            notebook_steps.dataframe_preview = _не_показывать_таблицу
            notebook_steps.protected_download_buttons = (
                _не_показывать_стандартные_кнопки
            )
            notebook_steps.show_error = _перехватить_ошибку

            try:
                result = notebook_steps.results_step(
                    project_root=PROJECT_ROOT,
                )
            finally:
                notebook_steps.show_panel = saved_functions["show_panel"]
                notebook_steps.dataframe_preview = (
                    saved_functions["dataframe_preview"]
                )
                notebook_steps.protected_download_buttons = (
                    saved_functions["protected_download_buttons"]
                )
                notebook_steps.show_error = saved_functions["show_error"]

            if not result.get("success"):
                error = captured_error.get("error") or RuntimeError(
                    result.get("error")
                    or "Последний сохранённый результат не найден."
                )
                title, action = _понятное_действие_после_ошибки(error)

                notebook_ui.show_panel(
                    title,
                    "\n".join(
                        [
                            f"Причина: {error}",
                            "",
                            f"Что сделать: {action}",
                            "",
                            "Новые запросы к СПАРК не выполнялись.",
                        ]
                    ),
                    kind="error",
                )
                return result

            workflow = result.get("workflow") or {}
            original_files = result.get("files") or {}
            files = {}

            for label, raw_path in original_files.items():
                new_label = (
                    "Скачать технический manifest"
                    if label == "Скачать manifest"
                    else label
                )
                files[new_label] = raw_path

            folder = _папка_результата(files)
            run_name = _название_запуска(
                workflow,
                folder,
            )
            available_labels = list(files)

            notebook_ui.show_panel(
                "Последний сохранённый результат открыт",
                "\n".join(
                    [
                        f"Запуск: {run_name}",
                        (
                            f"Папка: {folder}"
                            if folder is not None
                            else "Папка результата не определена"
                        ),
                        (
                            "Доступные файлы: "
                            + ", ".join(
                                label.removeprefix("Скачать ")
                                for label in available_labels
                            )
                        ),
                        "",
                        (
                            "Ниже можно выбрать состав Excel: рекомендуемый набор, "
                            "только ML-матрицу, полный файл или нужные листы вручную."
                        ),
                        (
                            "Исходные файлы на Google Диске не изменяются. "
                            "Новые запросы к СПАРК не выполняются."
                        ),
                    ]
                ),
                kind="success",
            )

            notebook_ui.protected_download_buttons(
                files,
                recommended_label="Скачать Excel",
            )

            return result


        RESULTS = _открыть_последний_результат()


## 10. Техническая справка

---

<details>
<summary><b>Где искать файлы и как восстановить работу</b></summary>

**Основные служебные файлы**

* Текущие настройки проекта: `workflow/workflow_config.json`
* Сохранённые ответы методов СПАРК: `method_responses/`
* Контрольные сохранения массового запуска: `checkpoints_v2/`
* Результаты отдельных запусков: `runs_v2/`
* Накопительные датасеты: `dataset_goals/`
* Итоговые Excel-файлы: сохраняются внутри папки соответствующего запуска или накопительного датасета.

Служебные файлы рекомендуется не перемещать и не редактировать вручную.

**Продолжение после остановки или перезапуска Colab**

После перезапуска среды при продолжении ранее сохранённого запуска последовательно выполните:

1. Пункт 1 — подключение Google Диска.
2. Пункт 2 — установку и проверку автономного проекта.
3. Пункт 2.1 — подготовку методов детализации.
4. Пункт 3 — авторизацию в СПАРК.
5. Пункт 8 — продолжение массового запуска.

Сохранённые методы, target, параметры, признаки, входной файл, накопительный датасет и checkpoint повторно создавать не требуется. Штатный обработчик target загружается в пункте 2.

Если пункт 8 сообщит, что входной файл не подготовлен, вернитесь в пункт 7 и нажмите **«Подготовить входной файл»**.

**Куда возвращаться при ошибках**

* Ошибка подключения детализации — пункт 2.1.
* Ошибка авторизации — пункт 3.
* Не выбраны методы API — пункт 4.
* Не загружен штатный обработчик target — повторно выполните пункты 2, 2.1 и 3.
* Не сохранено или устарело правило target — примените полную конфигурацию в пункте 4 либо сохраните пункты 5.1, 5.2 и 5.3 вручную.
* Ошибка параметров методов или плана выполнения — пункт 5.2.
* Не выбраны признаки или дополнительные поля — пункт 5.3.
* Не пройдена проверка одной компании — пункт 6.
* Не подготовлен входной файл — пункт 7.
* Ошибка накопительного датасета — пункт 7.1.
* Массовый запуск не подтверждён — включите галочку в пункте 8.
* Нужно повторно скачать последний результат — пункт 9.

**Изменение настроек**

Изменение методов, параметров, признаков, дополнительных полей или target намеренно сбрасывает готовность зависимых этапов. После изменения программа может потребовать повторно выполнить пункты 5.2, 5.3 или 6.

Ранее сохранённые ответы, checkpoint, Excel, ZIP и результаты прошлых запусков при этом не удаляются.

Накопительный датасет нельзя продолжать с другим составом методов, признаков или target под тем же названием. Для другой конфигурации необходимо создать отдельное накопление.

</details>
